In [ ]:
# ============================================================
# STEP 1 — READ-ONLY ORGANIZER DATA + GPU AUDIT
# No files are modified, extracted, cleaned, or overwritten.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import io
import json
import re
import statistics
import sys
import zipfile

import torch


INPUT_ROOT = Path("/kaggle/input")

ALLOWED_LABELS = {
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
}

REQUIRED_FIELDS = {
    "id",
    "claim",
    "evidence",
    "label",
}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def normalize_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_evidence(evidence, order_invariant=False):
    if not isinstance(evidence, list):
        return normalize_text(evidence)

    passages = [normalize_text(x) for x in evidence]

    if order_invariant:
        passages = sorted(passages)

    return "\x1e".join(passages)


def content_fingerprint(row, order_invariant=False):
    claim = normalize_text(row.get("claim"))
    evidence = normalize_evidence(
        row.get("evidence"),
        order_invariant=order_invariant,
    )
    return claim + "\x1f" + evidence


def evidence_fingerprint(row, order_invariant=False):
    return normalize_evidence(
        row.get("evidence"),
        order_invariant=order_invariant,
    )


def read_jsonl(raw_bytes, source_name):
    text = raw_bytes.decode("utf-8-sig")
    rows = []

    for line_number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue

        try:
            rows.append(json.loads(line))
        except Exception as exc:
            raise RuntimeError(
                f"Invalid JSON in {source_name}, "
                f"line {line_number}: {exc}"
            )

    return rows


def percentile(values, p):
    if not values:
        return None

    values = sorted(values)

    if len(values) == 1:
        return values[0]

    position = (len(values) - 1) * p
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    fraction = position - lower

    return (
        values[lower] * (1 - fraction)
        + values[upper] * fraction
    )


def print_example(row):
    claim = normalize_text(row.get("claim"))
    evidence = row.get("evidence")

    if isinstance(evidence, list):
        evidence_preview = " | ".join(
            normalize_text(x) for x in evidence
        )
    else:
        evidence_preview = normalize_text(evidence)

    print("  id      :", repr(row.get("id")))
    print("  label   :", repr(row.get("label")))
    print("  claim   :", claim[:180])
    print("  evidence:", evidence_preview[:280])


# ------------------------------------------------------------
# A. GPU / runtime
# ------------------------------------------------------------

print("=" * 78)
print("A. RUNTIME / GPU")
print("=" * 78)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024 ** 3)

    print(
        f"GPU {i}: {props.name} | "
        f"VRAM={vram_gb:.2f} GB | "
        f"compute capability={props.major}.{props.minor}"
    )


# ------------------------------------------------------------
# B. Find organizer package
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B. ORGANIZER FILE DISCOVERY")
print("=" * 78)

zip_candidates = sorted(INPUT_ROOT.rglob("*.zip"))

valid_zip_packages = []

for zip_path in zip_candidates:
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = [
                name.replace("\\", "/")
                for name in zf.namelist()
                if not name.startswith("__MACOSX/")
            ]

            train_hits = [
                n for n in names
                if n.endswith("/data/train.jsonl")
                or n == "data/train.jsonl"
            ]

            val_hits = [
                n for n in names
                if n.endswith("/data/validation.jsonl")
                or n == "data/validation.jsonl"
            ]

            if len(train_hits) == 1 and len(val_hits) == 1:
                valid_zip_packages.append(
                    (zip_path, train_hits[0], val_hits[0])
                )
    except zipfile.BadZipFile:
        pass


package_mode = None
package_path = None
train_bytes = None
val_bytes = None
train_source = None
val_source = None
manifest_text = None


if len(valid_zip_packages) == 1:

    package_mode = "ZIP"
    package_path, train_member, val_member = valid_zip_packages[0]

    with zipfile.ZipFile(package_path, "r") as zf:
        train_bytes = zf.read(train_member)
        val_bytes = zf.read(val_member)

        manifest_hits = [
            n for n in zf.namelist()
            if n.replace("\\", "/").endswith("/checksums.sha256")
            or n.replace("\\", "/") == "checksums.sha256"
        ]

        if len(manifest_hits) == 1:
            manifest_text = zf.read(
                manifest_hits[0]
            ).decode("utf-8", errors="replace")

    train_source = f"{package_path.name} :: {train_member}"
    val_source = f"{package_path.name} :: {val_member}"

elif len(valid_zip_packages) > 1:

    print("Multiple organizer-like ZIPs were found:")
    for item in valid_zip_packages:
        print(" -", item[0])

    raise RuntimeError(
        "More than one ZIP contains train.jsonl and validation.jsonl. "
        "Please remove duplicate dataset uploads before continuing."
    )

else:

    train_files = sorted(INPUT_ROOT.rglob("train.jsonl"))
    val_files = sorted(INPUT_ROOT.rglob("validation.jsonl"))

    if len(train_files) != 1 or len(val_files) != 1:
        print("train.jsonl candidates:")
        for path in train_files:
            print(" -", path)

        print("validation.jsonl candidates:")
        for path in val_files:
            print(" -", path)

        raise RuntimeError(
            "Could not uniquely identify organizer train/validation files."
        )

    package_mode = "DIRECT FILES"

    train_path = train_files[0]
    val_path = val_files[0]

    train_bytes = train_path.read_bytes()
    val_bytes = val_path.read_bytes()

    train_source = str(train_path)
    val_source = str(val_path)


print("Package mode:", package_mode)

if package_path is not None:
    print("Package:", package_path)
    print("Package SHA-256:", hashlib.sha256(
        package_path.read_bytes()
    ).hexdigest())

print("Train source:", train_source)
print("Validation source:", val_source)

print("Train SHA-256:", sha256_bytes(train_bytes))
print("Validation SHA-256:", sha256_bytes(val_bytes))


# ------------------------------------------------------------
# C. Compare dataset hashes with organizer checksum manifest
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("C. ORGANIZER CHECKSUM CHECK")
print("=" * 78)

if manifest_text:

    manifest = {}

    for line in manifest_text.splitlines():
        line = line.strip()

        if not line:
            continue

        parts = line.split(maxsplit=1)

        if len(parts) == 2:
            digest, relpath = parts
            manifest[relpath.strip()] = digest.strip()

    checks = [
        (
            "data/train.jsonl",
            sha256_bytes(train_bytes),
        ),
        (
            "data/validation.jsonl",
            sha256_bytes(val_bytes),
        ),
    ]

    for relpath, actual in checks:
        expected = manifest.get(relpath)

        print(relpath)
        print("  expected:", expected)
        print("  actual  :", actual)
        print("  match   :", expected == actual)

else:
    print("No checksum manifest located inside the package.")


# ------------------------------------------------------------
# D. Load JSONL in memory only
# ------------------------------------------------------------

train = read_jsonl(train_bytes, "train.jsonl")
validation = read_jsonl(val_bytes, "validation.jsonl")


# ------------------------------------------------------------
# E. Split audit
# ------------------------------------------------------------

def audit_split(rows, split_name):

    print("\n" + "=" * 78)
    print(f"{split_name.upper()} AUDIT")
    print("=" * 78)

    print("Rows:", len(rows))

    all_keys = Counter()

    for row in rows:
        all_keys.update(row.keys())

    print("Fields and occurrence counts:")
    for key, count in sorted(all_keys.items()):
        print(f"  {key!r}: {count}")

    missing_required = {
        field: sum(
            field not in row
            for row in rows
        )
        for field in REQUIRED_FIELDS
    }

    print("\nMissing required fields:")
    for field, count in sorted(missing_required.items()):
        print(f"  {field}: {count}")

    print("\nField value types:")

    for field in sorted(REQUIRED_FIELDS):
        type_counts = Counter(
            type(row.get(field)).__name__
            for row in rows
        )
        print(f"  {field}: {dict(type_counts)}")

    labels = Counter(
        row.get("label")
        for row in rows
    )

    print("\nRaw label distribution:")
    for label, count in sorted(
        labels.items(),
        key=lambda x: repr(x[0]),
    ):
        print(f"  {label!r}: {count}")

    invalid_label_rows = [
        row
        for row in rows
        if row.get("label") not in ALLOWED_LABELS
    ]

    print(
        "\nRows whose raw label is NOT one of "
        "the three official strings:",
        len(invalid_label_rows),
    )

    ids = [
        row.get("id")
        for row in rows
    ]

    duplicate_id_excess = (
        len(ids) - len(set(ids))
    )

    print("Duplicate ID excess:", duplicate_id_excess)

    whole_rows = [
        json.dumps(
            row,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        for row in rows
    ]

    exact_whole_row_excess = (
        len(whole_rows)
        - len(set(whole_rows))
    )

    print(
        "Exact whole-row duplicate excess:",
        exact_whole_row_excess,
    )

    blank_claims = 0
    non_list_evidence = 0
    empty_evidence_lists = 0
    blank_or_invalid_passages = 0

    passage_counts = []
    character_lengths = []

    for row in rows:

        claim = row.get("claim")
        evidence = row.get("evidence")

        if (
            not isinstance(claim, str)
            or not claim.strip()
        ):
            blank_claims += 1

        if not isinstance(evidence, list):
            non_list_evidence += 1
            passages = []
        else:
            passages = evidence
            passage_counts.append(len(passages))

            if len(passages) == 0:
                empty_evidence_lists += 1

            for passage in passages:
                if (
                    not isinstance(passage, str)
                    or not passage.strip()
                ):
                    blank_or_invalid_passages += 1

        total_chars = len(normalize_text(claim))

        if isinstance(evidence, list):
            total_chars += sum(
                len(normalize_text(x))
                for x in evidence
            )

        character_lengths.append(total_chars)

    print("\nClaim/evidence quality:")
    print("  Blank/invalid claims:", blank_claims)
    print("  Evidence values not lists:", non_list_evidence)
    print("  Empty evidence lists:", empty_evidence_lists)
    print(
        "  Blank/invalid evidence passages:",
        blank_or_invalid_passages,
    )

    if passage_counts:
        print(
            "  Evidence passage-count distribution:",
            dict(sorted(Counter(passage_counts).items())),
        )

    if character_lengths:
        print("\nClaim + evidence character lengths:")
        print("  min :", min(character_lengths))
        print(
            "  p50 :",
            round(percentile(character_lengths, 0.50), 1),
        )
        print(
            "  p90 :",
            round(percentile(character_lengths, 0.90), 1),
        )
        print(
            "  p95 :",
            round(percentile(character_lengths, 0.95), 1),
        )
        print(
            "  p99 :",
            round(percentile(character_lengths, 0.99), 1),
        )
        print("  max :", max(character_lengths))

    ordered_groups = defaultdict(list)
    unordered_groups = defaultdict(list)

    for row in rows:
        ordered_groups[
            content_fingerprint(
                row,
                order_invariant=False,
            )
        ].append(row)

        unordered_groups[
            content_fingerprint(
                row,
                order_invariant=True,
            )
        ].append(row)

    ordered_duplicates = [
        group
        for group in ordered_groups.values()
        if len(group) > 1
    ]

    unordered_duplicates = [
        group
        for group in unordered_groups.values()
        if len(group) > 1
    ]

    ordered_excess = sum(
        len(group) - 1
        for group in ordered_duplicates
    )

    unordered_excess = sum(
        len(group) - 1
        for group in unordered_duplicates
    )

    conflicting_duplicate_groups = [
        group
        for group in unordered_duplicates
        if len({
            row.get("label")
            for row in group
        }) > 1
    ]

    print("\nNormalized claim+evidence duplicate diagnostics:")
    print(
        "  Ordered duplicate groups:",
        len(ordered_duplicates),
    )
    print(
        "  Ordered duplicate excess rows:",
        ordered_excess,
    )
    print(
        "  Order-invariant duplicate groups:",
        len(unordered_duplicates),
    )
    print(
        "  Order-invariant duplicate excess rows:",
        unordered_excess,
    )
    print(
        "  Duplicate groups with conflicting raw labels:",
        len(conflicting_duplicate_groups),
    )

    if conflicting_duplicate_groups:

        print("\nFirst conflicting duplicate groups:")

        for group in conflicting_duplicate_groups[:5]:

            print("\n  Group:")

            for row in group:
                print(
                    "   ",
                    repr(row.get("id")),
                    "|",
                    repr(row.get("label")),
                )

            print(
                "    Claim:",
                normalize_text(
                    group[0].get("claim")
                )[:180],
            )

    print("\nRepresentative examples:")

    labels_to_show = [
        "SUPPORTS",
        "REFUTES",
        "NOT_ENOUGH_INFO",
    ]

    for target_label in labels_to_show:

        example = next(
            (
                row
                for row in rows
                if row.get("label") == target_label
            ),
            None,
        )

        if example is not None:
            print(f"\n{target_label}:")
            print_example(example)

    return {
        "ordered": set(ordered_groups.keys()),
        "unordered": set(unordered_groups.keys()),
        "claims": {
            normalize_text(row.get("claim"))
            for row in rows
        },
        "evidence_ordered": {
            evidence_fingerprint(
                row,
                order_invariant=False,
            )
            for row in rows
        },
        "ids": set(ids),
    }


train_info = audit_split(
    train,
    "train",
)

validation_info = audit_split(
    validation,
    "validation",
)


# ------------------------------------------------------------
# F. Cross-split overlap diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("CROSS-SPLIT DIAGNOSTICS")
print("=" * 78)

print(
    "Shared IDs:",
    len(
        train_info["ids"]
        & validation_info["ids"]
    ),
)

print(
    "Exact normalized claim+evidence overlaps:",
    len(
        train_info["ordered"]
        & validation_info["ordered"]
    ),
)

print(
    "Order-invariant claim+evidence overlaps:",
    len(
        train_info["unordered"]
        & validation_info["unordered"]
    ),
)

print(
    "Shared normalized claims:",
    len(
        train_info["claims"]
        & validation_info["claims"]
    ),
)

print(
    "Shared normalized evidence blocks:",
    len(
        train_info["evidence_ordered"]
        & validation_info["evidence_ordered"]
    ),
)


# ------------------------------------------------------------
# G. Final audit-only assertions
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("AUDIT COMPLETED")
print("=" * 78)

print("Train rows loaded      :", len(train))
print("Validation rows loaded :", len(validation))
print("No organizer files were modified.")
print("No rows were cleaned, deleted, relabeled, or saved.")

In [ ]:
# ============================================================
# STEP 2 — CONSERVATIVE TRAIN CLEANING V1
# Creates derived files only in /kaggle/working.
# Official train/validation files remain untouched.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import re


INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

EXPECTED_TRAIN_SHA256 = (
    "26ea9a6998815d0f99f45aff2206f435"
    "781cc71bd92638101d7ea08c2e175d3c"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

OFFICIAL_LABELS = {
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def find_file_by_hash(filename, expected_hash):
    matches = []

    for path in INPUT_ROOT.rglob(filename):
        if path.is_file():
            digest = sha256_bytes(path.read_bytes())
            if digest == expected_hash:
                matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename} with organizer hash "
            f"{expected_hash}, found {len(matches)}: {matches}"
        )

    return matches[0]


def load_jsonl(path):
    rows = []

    with path.open("r", encoding="utf-8-sig") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON at {path}, line {line_number}: {exc}"
                )

    return rows


def canonicalize_label(raw_label):
    if not isinstance(raw_label, str):
        return None

    normalized = (
        raw_label
        .strip()
        .upper()
        .replace(" ", "_")
    )

    mapping = {
        "SUPPORTS": "SUPPORTS",
        "SUPPORTED": "SUPPORTS",
        "REFUTES": "REFUTES",
        "REFUTED": "REFUTES",
        "NOT_ENOUGH_INFO": "NOT_ENOUGH_INFO",
        "NEI": "NOT_ENOUGH_INFO",
    }

    return mapping.get(normalized)


def normalize_text(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def sanitize_evidence(evidence):
    if not isinstance(evidence, list):
        raise TypeError(
            f"Evidence must be a list, got {type(evidence).__name__}"
        )

    cleaned = []

    for passage in evidence:
        if not isinstance(passage, str):
            raise TypeError(
                f"Evidence passage must be str, "
                f"got {type(passage).__name__}"
            )

        stripped = passage.strip()

        if stripped:
            cleaned.append(stripped)

    return cleaned


def content_fingerprint(row):
    claim = normalize_text(row["claim"])

    evidence = sorted(
        normalize_text(p)
        for p in row["evidence"]
    )

    return (
        claim
        + "\x1f"
        + "\x1e".join(evidence)
    )


# ------------------------------------------------------------
# A. Resolve immutable organizer files by known hashes
# ------------------------------------------------------------

train_path = find_file_by_hash(
    "train.jsonl",
    EXPECTED_TRAIN_SHA256,
)

validation_path = find_file_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

train_bytes = train_path.read_bytes()
validation_bytes = validation_path.read_bytes()

assert sha256_bytes(train_bytes) == EXPECTED_TRAIN_SHA256
assert sha256_bytes(validation_bytes) == EXPECTED_VAL_SHA256

train_raw = load_jsonl(train_path)
validation_raw = load_jsonl(validation_path)

assert len(train_raw) == 1000
assert len(validation_raw) == 300


# ------------------------------------------------------------
# B. Audit raw label normalization
# ------------------------------------------------------------

raw_label_counts = Counter(
    row["label"]
    for row in train_raw
)

label_normalization_counts = Counter()

unresolved_label_rows = []

for row in train_raw:
    raw_label = row["label"]
    canonical = canonicalize_label(raw_label)

    if canonical is None:
        unresolved_label_rows.append(row)
    elif raw_label != canonical:
        label_normalization_counts[
            (raw_label, canonical)
        ] += 1


# Safety: unresolved labels must only be blank strings.
unexpected_unresolved = [
    row
    for row in unresolved_label_rows
    if not (
        isinstance(row["label"], str)
        and row["label"].strip() == ""
    )
]

assert not unexpected_unresolved, (
    "Unexpected unknown labels were found. "
    "Do not continue automatically."
)

assert len(unresolved_label_rows) == 10


# ------------------------------------------------------------
# C. Canonicalize labels + remove whitespace-only passages
# ------------------------------------------------------------

working_rows = []

evidence_cleaned_ids = []
blank_passages_removed = 0

for original_index, row in enumerate(train_raw):

    canonical_label = canonicalize_label(
        row["label"]
    )

    if canonical_label is None:
        continue

    original_evidence = row["evidence"]
    cleaned_evidence = sanitize_evidence(
        original_evidence
    )

    removed_here = (
        len(original_evidence)
        - len(cleaned_evidence)
    )

    if removed_here:
        evidence_cleaned_ids.append(
            row["id"]
        )
        blank_passages_removed += removed_here

    cleaned_row = {
        "id": row["id"],
        "claim": row["claim"],
        "evidence": cleaned_evidence,
        "label": canonical_label,
        "_original_index": original_index,
    }

    working_rows.append(cleaned_row)


assert blank_passages_removed == 16
assert len(evidence_cleaned_ids) == 16


# ------------------------------------------------------------
# D. Deduplicate AFTER evidence sanitation
# ------------------------------------------------------------

groups = defaultdict(list)

for row in working_rows:
    groups[
        content_fingerprint(row)
    ].append(row)


clean_rows = []

redundant_removed_ids = []
same_label_duplicate_groups = []

conflict_removed_ids = []
conflicting_groups = []


for group in groups.values():

    canonical_labels = {
        row["label"]
        for row in group
    }

    # Exact same semantic input but contradictory targets:
    # choose neither target.
    if len(canonical_labels) > 1:

        group_sorted = sorted(
            group,
            key=lambda x: x["_original_index"],
        )

        conflicting_groups.append({
            "ids": [
                row["id"]
                for row in group_sorted
            ],
            "labels": [
                row["label"]
                for row in group_sorted
            ],
        })

        conflict_removed_ids.extend(
            row["id"]
            for row in group_sorted
        )

        continue

    # Same content + same canonical target:
    # deterministically keep earliest organizer row.
    group_sorted = sorted(
        group,
        key=lambda x: x["_original_index"],
    )

    kept = group_sorted[0]

    if len(group_sorted) > 1:

        removed = group_sorted[1:]

        same_label_duplicate_groups.append({
            "kept_id": kept["id"],
            "removed_ids": [
                row["id"]
                for row in removed
            ],
            "label": kept["label"],
        })

        redundant_removed_ids.extend(
            row["id"]
            for row in removed
        )

    clean_rows.append({
        "id": kept["id"],
        "claim": kept["claim"],
        "evidence": kept["evidence"],
        "label": kept["label"],
    })


# Restore deterministic organizer order.
original_order = {
    row["id"]: i
    for i, row in enumerate(train_raw)
}

clean_rows.sort(
    key=lambda row: original_order[row["id"]]
)


# ------------------------------------------------------------
# E. Integrity checks
# ------------------------------------------------------------

clean_distribution = Counter(
    row["label"]
    for row in clean_rows
)

assert len(redundant_removed_ids) == 52
assert len(conflict_removed_ids) == 2
assert len(conflicting_groups) == 1

assert len(clean_rows) == 936

assert clean_distribution == Counter({
    "SUPPORTS": 348,
    "REFUTES": 313,
    "NOT_ENOUGH_INFO": 275,
})

assert all(
    row["label"] in OFFICIAL_LABELS
    for row in clean_rows
)

assert len({
    row["id"]
    for row in clean_rows
}) == len(clean_rows)

assert all(
    isinstance(row["claim"], str)
    and row["claim"].strip()
    for row in clean_rows
)

assert all(
    isinstance(row["evidence"], list)
    and len(row["evidence"]) > 0
    for row in clean_rows
)

assert all(
    isinstance(passage, str)
    and passage.strip()
    for row in clean_rows
    for passage in row["evidence"]
)

clean_fingerprints = [
    content_fingerprint(row)
    for row in clean_rows
]

assert (
    len(clean_fingerprints)
    == len(set(clean_fingerprints))
), "Duplicate cleaned inputs remain."


# ------------------------------------------------------------
# F. Verify no full train↔validation leakage after cleaning
# ------------------------------------------------------------

validation_fingerprints = {
    content_fingerprint({
        "claim": row["claim"],
        "evidence": sanitize_evidence(
            row["evidence"]
        ),
    })
    for row in validation_raw
}

post_clean_cross_split_overlap = (
    set(clean_fingerprints)
    & validation_fingerprints
)

assert len(post_clean_cross_split_overlap) == 0


# ------------------------------------------------------------
# G. Save derived Dataset V1 + reproducibility audit
# ------------------------------------------------------------

clean_path = (
    OUTPUT_ROOT
    / "train_clean_v1.jsonl"
)

audit_path = (
    OUTPUT_ROOT
    / "cleaning_audit_v1.json"
)


with clean_path.open(
    "w",
    encoding="utf-8",
) as f:

    for row in clean_rows:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )


audit = {
    "version": "train_clean_v1",
    "source": {
        "train_path": str(train_path),
        "validation_path": str(validation_path),
        "train_sha256": EXPECTED_TRAIN_SHA256,
        "validation_sha256": EXPECTED_VAL_SHA256,
        "raw_train_rows": len(train_raw),
        "raw_validation_rows": len(validation_raw),
    },
    "policy": {
        "validation_modified": False,
        "label_policy": (
            "Normalize deterministic aliases only; "
            "remove unresolved blank targets."
        ),
        "evidence_policy": (
            "Remove whitespace-only evidence passages; "
            "preserve non-empty evidence."
        ),
        "duplicate_policy": (
            "Keep earliest row for identical content "
            "with same canonical label; remove all rows "
            "from identical-content groups with "
            "conflicting canonical labels."
        ),
    },
    "label_normalizations": [
        {
            "raw": raw,
            "canonical": canonical,
            "count": count,
        }
        for (raw, canonical), count
        in sorted(
            label_normalization_counts.items(),
            key=lambda x: (
                repr(x[0][0]),
                x[0][1],
            ),
        )
    ],
    "removed": {
        "unresolved_label_ids": [
            row["id"]
            for row in unresolved_label_rows
        ],
        "redundant_duplicate_ids": (
            redundant_removed_ids
        ),
        "conflicting_duplicate_ids": (
            conflict_removed_ids
        ),
    },
    "evidence_cleanup": {
        "rows_affected": len(
            evidence_cleaned_ids
        ),
        "passages_removed": (
            blank_passages_removed
        ),
        "affected_ids": (
            evidence_cleaned_ids
        ),
    },
    "duplicate_groups": {
        "same_label": (
            same_label_duplicate_groups
        ),
        "conflicting": (
            conflicting_groups
        ),
    },
    "result": {
        "clean_train_rows": len(clean_rows),
        "class_distribution": dict(
            clean_distribution
        ),
        "train_validation_full_overlap": (
            len(post_clean_cross_split_overlap)
        ),
    },
}


with audit_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


clean_sha256 = sha256_bytes(
    clean_path.read_bytes()
)


# ------------------------------------------------------------
# H. Concise report
# ------------------------------------------------------------

print("=" * 76)
print("TRAIN CLEANING V1 — RESULT")
print("=" * 76)

print("\nSource integrity")
print("  Raw train rows       :", len(train_raw))
print("  Validation rows      :", len(validation_raw))
print("  Validation modified  :", False)

print("\nLabel repair")
print(
    "  Deterministic aliases normalized:",
    sum(label_normalization_counts.values()),
)
print(
    "  Blank-label rows removed         :",
    len(unresolved_label_rows),
)

print("\nEvidence repair")
print(
    "  Rows with blank passages:",
    len(evidence_cleaned_ids),
)
print(
    "  Blank passages removed  :",
    blank_passages_removed,
)

print("\nDeduplication")
print(
    "  Same-label redundant rows removed:",
    len(redundant_removed_ids),
)
print(
    "  Conflicting-content rows removed  :",
    len(conflict_removed_ids),
)
print(
    "  Genuine conflicting groups        :",
    len(conflicting_groups),
)

for group in conflicting_groups:
    print(
        "   Conflict:",
        list(
            zip(
                group["ids"],
                group["labels"],
            )
        ),
    )

print("\nFinal Dataset V1")
print("  Rows:", len(clean_rows))
print(
    "  Distribution:",
    dict(clean_distribution),
)
print(
    "  Remaining duplicate inputs:",
    (
        len(clean_fingerprints)
        - len(set(clean_fingerprints))
    ),
)
print(
    "  Train↔validation full overlaps:",
    len(post_clean_cross_split_overlap),
)

print("\nSaved")
print(" ", clean_path)
print(
    "  Clean SHA-256:",
    clean_sha256,
)
print(" ", audit_path)

print("\nAll integrity assertions passed.")
print("Official validation data was not modified.")

In [ ]:
# ============================================================
# STEP 3 — DETERMINISTIC STRATIFIED INTERNAL DEV SPLIT
# 85% fit / 15% internal development, stratified by class.
# Official organizer validation remains untouched.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import random


WORK_ROOT = Path("/kaggle/working")

CLEAN_PATH = WORK_ROOT / "train_clean_v1.jsonl"

EXPECTED_CLEAN_SHA256 = (
    "af28bb4ac1186ffec8c544e377091427"
    "ab912c9ba0c9afd42185d11d9e318def"
)

FIT_PATH = WORK_ROOT / "train_fit_v1.jsonl"
DEV_PATH = WORK_ROOT / "internal_dev_v1.jsonl"
SPLIT_AUDIT_PATH = WORK_ROOT / "split_audit_v1.json"

SEED = 42
DEV_FRACTION = 0.15

OFFICIAL_LABELS = (
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def load_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON at line "
                    f"{line_number}: {exc}"
                )

    return rows


def save_jsonl(rows, path):

    with path.open(
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def fingerprint(row):

    normalized_claim = " ".join(
        row["claim"].split()
    )

    normalized_evidence = sorted(
        " ".join(passage.split())
        for passage in row["evidence"]
    )

    payload = (
        normalized_claim
        + "\x1f"
        + "\x1e".join(normalized_evidence)
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# A. Verify Dataset V1
# ------------------------------------------------------------

assert CLEAN_PATH.exists(), (
    f"Missing {CLEAN_PATH}. "
    "Run the cleaning cell first."
)

actual_clean_hash = sha256_file(
    CLEAN_PATH
)

assert actual_clean_hash == EXPECTED_CLEAN_SHA256, (
    "train_clean_v1.jsonl does not match "
    "the frozen Dataset V1 hash."
)

rows = load_jsonl(
    CLEAN_PATH
)

assert len(rows) == 936

raw_distribution = Counter(
    row["label"]
    for row in rows
)

assert raw_distribution == Counter({
    "SUPPORTS": 348,
    "REFUTES": 313,
    "NOT_ENOUGH_INFO": 275,
})


# ------------------------------------------------------------
# B. Group by label
# ------------------------------------------------------------

by_label = defaultdict(list)

for original_index, row in enumerate(rows):

    assert row["label"] in OFFICIAL_LABELS

    row_copy = dict(row)
    row_copy["_original_index"] = original_index

    by_label[
        row["label"]
    ].append(row_copy)


# ------------------------------------------------------------
# C. Deterministic stratified selection
# ------------------------------------------------------------

dev_ids = set()

per_class_plan = {}

for label_index, label in enumerate(
    OFFICIAL_LABELS
):

    class_rows = by_label[label]

    # Separate reproducible RNG stream per class.
    rng = random.Random(
        SEED + label_index
    )

    shuffled = list(class_rows)

    rng.shuffle(shuffled)

    dev_count = round(
        len(class_rows)
        * DEV_FRACTION
    )

    selected = shuffled[
        :dev_count
    ]

    dev_ids.update(
        row["id"]
        for row in selected
    )

    per_class_plan[label] = {
        "total": len(class_rows),
        "dev": dev_count,
        "fit": (
            len(class_rows)
            - dev_count
        ),
    }


# ------------------------------------------------------------
# D. Restore organizer order in each split
# ------------------------------------------------------------

fit_rows = []
dev_rows = []

for row in rows:

    if row["id"] in dev_ids:
        dev_rows.append(row)
    else:
        fit_rows.append(row)


# ------------------------------------------------------------
# E. Integrity checks
# ------------------------------------------------------------

fit_distribution = Counter(
    row["label"]
    for row in fit_rows
)

dev_distribution = Counter(
    row["label"]
    for row in dev_rows
)

assert len(fit_rows) == 796
assert len(dev_rows) == 140

assert fit_distribution == Counter({
    "SUPPORTS": 296,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})

assert dev_distribution == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


fit_ids = {
    row["id"]
    for row in fit_rows
}

dev_ids_check = {
    row["id"]
    for row in dev_rows
}

assert fit_ids.isdisjoint(
    dev_ids_check
)

assert (
    len(fit_ids | dev_ids_check)
    == len(rows)
)

assert (
    fit_ids | dev_ids_check
    == {
        row["id"]
        for row in rows
    }
)


fit_fingerprints = {
    fingerprint(row)
    for row in fit_rows
}

dev_fingerprints = {
    fingerprint(row)
    for row in dev_rows
}

assert fit_fingerprints.isdisjoint(
    dev_fingerprints
)

assert len(fit_fingerprints) == len(
    fit_rows
)

assert len(dev_fingerprints) == len(
    dev_rows
)


# ------------------------------------------------------------
# F. Save derived splits
# ------------------------------------------------------------

save_jsonl(
    fit_rows,
    FIT_PATH,
)

save_jsonl(
    dev_rows,
    DEV_PATH,
)

fit_hash = sha256_file(
    FIT_PATH
)

dev_hash = sha256_file(
    DEV_PATH
)


# ------------------------------------------------------------
# G. Save split manifest
# ------------------------------------------------------------

split_audit = {
    "version": "split_v1",
    "source": {
        "dataset": "train_clean_v1",
        "rows": len(rows),
        "sha256": actual_clean_hash,
    },
    "split_policy": {
        "seed": SEED,
        "dev_fraction": DEV_FRACTION,
        "stratified": True,
        "official_validation_used": False,
        "official_validation_modified": False,
    },
    "per_class": per_class_plan,
    "result": {
        "fit_rows": len(fit_rows),
        "dev_rows": len(dev_rows),
        "fit_distribution": dict(
            fit_distribution
        ),
        "dev_distribution": dict(
            dev_distribution
        ),
        "fit_dev_shared_ids": len(
            fit_ids & dev_ids_check
        ),
        "fit_dev_shared_fingerprints": len(
            fit_fingerprints
            & dev_fingerprints
        ),
        "fit_sha256": fit_hash,
        "dev_sha256": dev_hash,
    },
}


with SPLIT_AUDIT_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        split_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# H. Concise report
# ------------------------------------------------------------

print("=" * 76)
print("INTERNAL SPLIT V1 — RESULT")
print("=" * 76)

print("\nFrozen source")
print(
    "  Dataset:",
    CLEAN_PATH.name,
)
print(
    "  SHA-256:",
    actual_clean_hash,
)
print(
    "  Rows:",
    len(rows),
)

print("\nSplit policy")
print(
    "  Seed:",
    SEED,
)
print(
    "  Internal dev fraction:",
    DEV_FRACTION,
)
print(
    "  Stratified:",
    True,
)
print(
    "  Official validation used:",
    False,
)
print(
    "  Official validation modified:",
    False,
)

print("\nFit split")
print(
    "  Rows:",
    len(fit_rows),
)
print(
    "  Distribution:",
    dict(fit_distribution),
)
print(
    "  SHA-256:",
    fit_hash,
)

print("\nInternal dev split")
print(
    "  Rows:",
    len(dev_rows),
)
print(
    "  Distribution:",
    dict(dev_distribution),
)
print(
    "  SHA-256:",
    dev_hash,
)

print("\nLeakage checks")
print(
    "  Shared IDs:",
    len(
        fit_ids
        & dev_ids_check
    ),
)
print(
    "  Shared content fingerprints:",
    len(
        fit_fingerprints
        & dev_fingerprints
    ),
)

print("\nSaved")
print(" ", FIT_PATH)
print(" ", DEV_PATH)
print(" ", SPLIT_AUDIT_PATH)

print("\nAll split integrity assertions passed.")

In [ ]:
# ============================================================
# STEP 4 — GEMMA TOKENIZER + TOKEN-LENGTH PREFLIGHT
# No model weights are loaded and no training occurs.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import json
import math
import os
import statistics
import subprocess
import sys


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

FIT_PATH = WORK_ROOT / "train_fit_v1.jsonl"
DEV_PATH = WORK_ROOT / "internal_dev_v1.jsonl"

EXPECTED_FIT_SHA256 = (
    "57439029cc20cf538386b75f1d5de730"
    "f57168d0899052a4bdded9f8887368c9"
)

EXPECTED_DEV_SHA256 = (
    "af58d06b6f896d7a1d7210f929499c0"
    "f8c110015fd44e9a2d844df2d177efb1f"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

OFFICIAL_LABELS = (
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def load_jsonl(path):
    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON in {path.name}, "
                    f"line {line_number}: {exc}"
                )

    return rows


def find_by_hash(filename, expected_hash):
    matches = []

    for path in INPUT_ROOT.rglob(filename):

        if (
            path.is_file()
            and sha256_file(path) == expected_hash
        ):
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one organizer {filename} "
            f"with the frozen hash. Found: {matches}"
        )

    return matches[0]


def percentile(values, p):
    values = sorted(values)

    if not values:
        return None

    if len(values) == 1:
        return values[0]

    position = (len(values) - 1) * p

    low = math.floor(position)
    high = math.ceil(position)

    if low == high:
        return values[low]

    weight = position - low

    return (
        values[low] * (1 - weight)
        + values[high] * weight
    )


# ------------------------------------------------------------
# A. Verify frozen datasets
# ------------------------------------------------------------

assert FIT_PATH.exists()
assert DEV_PATH.exists()

assert sha256_file(FIT_PATH) == EXPECTED_FIT_SHA256
assert sha256_file(DEV_PATH) == EXPECTED_DEV_SHA256

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

fit_rows = load_jsonl(FIT_PATH)
dev_rows = load_jsonl(DEV_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(fit_rows) == 796
assert len(dev_rows) == 140
assert len(val_rows) == 300

assert all(
    row["label"] in OFFICIAL_LABELS
    for row in fit_rows + dev_rows + val_rows
)


# ------------------------------------------------------------
# B. Ensure tokenizer dependencies
# ------------------------------------------------------------

required_transformers = "5.10.1"

try:
    installed_transformers = (
        importlib.metadata.version("transformers")
    )
except importlib.metadata.PackageNotFoundError:
    installed_transformers = None

if installed_transformers != required_transformers:

    print(
        "Installing organizer-compatible tokenizer stack..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"transformers=={required_transformers}",
        "huggingface_hub",
        "sentencepiece",
    ])


from transformers import AutoTokenizer
import transformers


# ------------------------------------------------------------
# C. Retrieve HF token securely
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:

    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN was not found. Add a Kaggle secret named "
        "'HF_TOKEN' containing a Hugging Face read token from "
        "an account with access to google/gemma-2-2b-it, "
        "then rerun this cell."
    )


# ------------------------------------------------------------
# D. Load TOKENIZER ONLY
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

assert tokenizer.chat_template is not None, (
    "Gemma tokenizer has no chat template."
)


# ------------------------------------------------------------
# E. Prompt V1
# ------------------------------------------------------------

PROMPT_VERSION = "v1_minimal_organizer_semantics"

SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def format_evidence(items):

    return "\n".join(
        f"[{i}] {text}"
        for i, text in enumerate(items, 1)
    )


def build_user_content(example):

    return (
        f"{SYSTEM_INSTRUCTION}"
        f"\n\nClaim:\n{example['claim']}"
        f"\n\nEvidence:\n"
        f"{format_evidence(example['evidence'])}"
    )


def make_prompt(example):

    messages = [{
        "role": "user",
        "content": build_user_content(example),
    }]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def make_full_training_text(example):

    messages = [
        {
            "role": "user",
            "content": build_user_content(example),
        },
        {
            "role": "assistant",
            "content": example["label"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def token_count(text):

    return len(
        tokenizer(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


# ------------------------------------------------------------
# F. Verify prompt/completion structure
# ------------------------------------------------------------

sample = fit_rows[0]

sample_prompt = make_prompt(sample)
sample_full = make_full_training_text(sample)

assert sample_full.startswith(sample_prompt), (
    "Full Gemma training text does not begin with "
    "the generation prompt. Stop before training."
)

sample_completion = sample_full[
    len(sample_prompt):
]

assert sample["label"] in sample_completion


# ------------------------------------------------------------
# G. Token-length audit
# ------------------------------------------------------------

def audit_lengths(rows, split_name):

    records = []

    for row in rows:

        prompt = make_prompt(row)
        full = make_full_training_text(row)

        prompt_tokens = token_count(prompt)
        full_tokens = token_count(full)
        completion_tokens = (
            full_tokens - prompt_tokens
        )

        records.append({
            "id": row["id"],
            "label": row["label"],
            "passages": len(row["evidence"]),
            "prompt_tokens": prompt_tokens,
            "full_tokens": full_tokens,
            "completion_tokens": completion_tokens,
            "characters": (
                len(row["claim"])
                + sum(
                    len(x)
                    for x in row["evidence"]
                )
            ),
        })

    full_lengths = [
        x["full_tokens"]
        for x in records
    ]

    prompt_lengths = [
        x["prompt_tokens"]
        for x in records
    ]

    completion_lengths = [
        x["completion_tokens"]
        for x in records
    ]

    summary = {
        "rows": len(records),
        "prompt_min": min(prompt_lengths),
        "prompt_p50": percentile(
            prompt_lengths, 0.50
        ),
        "prompt_p95": percentile(
            prompt_lengths, 0.95
        ),
        "prompt_p99": percentile(
            prompt_lengths, 0.99
        ),
        "prompt_max": max(prompt_lengths),
        "full_min": min(full_lengths),
        "full_p50": percentile(
            full_lengths, 0.50
        ),
        "full_p90": percentile(
            full_lengths, 0.90
        ),
        "full_p95": percentile(
            full_lengths, 0.95
        ),
        "full_p99": percentile(
            full_lengths, 0.99
        ),
        "full_max": max(full_lengths),
        "completion_lengths": dict(
            sorted(
                Counter(
                    completion_lengths
                ).items()
            )
        ),
    }

    longest = sorted(
        records,
        key=lambda x: x["full_tokens"],
        reverse=True,
    )[:5]

    print(
        "\n" + "-" * 76
    )
    print(
        f"{split_name.upper()} TOKEN LENGTHS"
    )
    print(
        "-" * 76
    )

    print("Rows       :", summary["rows"])

    print("\nPrompt tokens")
    print(
        "  min :",
        summary["prompt_min"],
    )
    print(
        "  p50 :",
        round(summary["prompt_p50"], 1),
    )
    print(
        "  p95 :",
        round(summary["prompt_p95"], 1),
    )
    print(
        "  p99 :",
        round(summary["prompt_p99"], 1),
    )
    print(
        "  max :",
        summary["prompt_max"],
    )

    print("\nFull training tokens")
    print(
        "  min :",
        summary["full_min"],
    )
    print(
        "  p50 :",
        round(summary["full_p50"], 1),
    )
    print(
        "  p90 :",
        round(summary["full_p90"], 1),
    )
    print(
        "  p95 :",
        round(summary["full_p95"], 1),
    )
    print(
        "  p99 :",
        round(summary["full_p99"], 1),
    )
    print(
        "  max :",
        summary["full_max"],
    )

    print(
        "\nCompletion-token lengths:",
        summary["completion_lengths"],
    )

    print("\nFive longest examples:")

    for item in longest:
        print(
            " ",
            item["id"],
            "|",
            item["label"],
            "| full=",
            item["full_tokens"],
            "| prompt=",
            item["prompt_tokens"],
            "| passages=",
            item["passages"],
            "| chars=",
            item["characters"],
        )

    return summary, records


fit_summary, fit_records = audit_lengths(
    fit_rows,
    "fit",
)

dev_summary, dev_records = audit_lengths(
    dev_rows,
    "internal dev",
)

val_summary, val_records = audit_lengths(
    val_rows,
    "official validation",
)


# ------------------------------------------------------------
# H. Combined distribution
# ------------------------------------------------------------

combined_records = (
    fit_records
    + dev_records
    + val_records
)

combined_lengths = [
    x["full_tokens"]
    for x in combined_records
]

overall_max = max(combined_lengths)

overall_longest = max(
    combined_records,
    key=lambda x: x["full_tokens"],
)


# ------------------------------------------------------------
# I. Label tokenization
# ------------------------------------------------------------

label_tokenization = {}

for label in OFFICIAL_LABELS:

    encoded = tokenizer(
        label,
        add_special_tokens=False,
    )["input_ids"]

    label_tokenization[label] = {
        "count": len(encoded),
        "ids": encoded,
        "pieces": tokenizer.convert_ids_to_tokens(
            encoded
        ),
    }


# ------------------------------------------------------------
# J. Report
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("GEMMA TOKENIZER PREFLIGHT — RESULT")
print("=" * 76)

print("\nEnvironment")
print(
    "  Transformers:",
    transformers.__version__,
)
print(
    "  Model/tokenizer:",
    MODEL_ID,
)
print(
    "  Tokenizer class:",
    tokenizer.__class__.__name__,
)
print(
    "  Vocab size:",
    len(tokenizer),
)
print(
    "  Declared model_max_length:",
    tokenizer.model_max_length,
)

print("\nSpecial tokens")
print(
    "  BOS:",
    repr(tokenizer.bos_token),
    tokenizer.bos_token_id,
)
print(
    "  EOS:",
    repr(tokenizer.eos_token),
    tokenizer.eos_token_id,
)
print(
    "  PAD:",
    repr(tokenizer.pad_token),
    tokenizer.pad_token_id,
)
print(
    "  Padding side:",
    tokenizer.padding_side,
)
print(
    "  Chat template present:",
    tokenizer.chat_template is not None,
)

print("\nPrompt")
print(
    "  Version:",
    PROMPT_VERSION,
)

print("\nAllowed-label tokenization")

for label, info in label_tokenization.items():
    print(
        f"  {label}: "
        f"{info['count']} tokens | "
        f"ids={info['ids']} | "
        f"pieces={info['pieces']}"
    )

print("\nCombined fit + internal dev + validation")
print(
    "  Rows:",
    len(combined_records),
)
print(
    "  Full-token min:",
    min(combined_lengths),
)
print(
    "  Full-token p50:",
    round(
        percentile(combined_lengths, 0.50),
        1,
    ),
)
print(
    "  Full-token p90:",
    round(
        percentile(combined_lengths, 0.90),
        1,
    ),
)
print(
    "  Full-token p95:",
    round(
        percentile(combined_lengths, 0.95),
        1,
    ),
)
print(
    "  Full-token p99:",
    round(
        percentile(combined_lengths, 0.99),
        1,
    ),
)
print(
    "  Full-token max:",
    overall_max,
)

print(
    "  Longest example:",
    overall_longest["id"],
    "|",
    overall_longest["label"],
    "|",
    overall_longest["full_tokens"],
    "tokens",
)

print("\nSample training structure")
print(
    "  Sample ID:",
    sample["id"],
)
print(
    "  Prompt tokens:",
    token_count(sample_prompt),
)
print(
    "  Full tokens:",
    token_count(sample_full),
)
print(
    "  Completion repr:",
    repr(sample_completion),
)

print("\nSample rendered prompt:")
print("-" * 76)
print(sample_prompt)
print("-" * 76)

print(
    "\nNo model weights were loaded. "
    "No training occurred."
)

In [ ]:
# ============================================================
# STEP 5 — FINALIZE DATASET V2 + INTERNAL SPLIT
# Conservative normalization, repeated-passage removal,
# deterministic deduplication, 15% stratified internal dev.
# Official validation remains completely untouched.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import random
import re
import unicodedata
import os


INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

EXPECTED_TRAIN_SHA256 = (
    "26ea9a6998815d0f99f45aff2206f435"
    "781cc71bd92638101d7ea08c2e175d3c"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

CLEAN_PATH = WORK_ROOT / "train_clean_v2.jsonl"
FIT_PATH = WORK_ROOT / "train_fit_v2.jsonl"
DEV_PATH = WORK_ROOT / "internal_dev_v2.jsonl"
AUDIT_PATH = WORK_ROOT / "dataset_v2_audit.json"

SEED = 42
DEV_FRACTION = 0.15
MODEL_ID = "google/gemma-2-2b-it"
MAX_LENGTH_CANDIDATE = 256

LABEL_ORDER = (
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
)

OFFICIAL_LABELS = set(LABEL_ORDER)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def find_by_hash(filename, expected_hash):
    matches = []

    for path in INPUT_ROOT.rglob(filename):
        if path.is_file() and sha256_file(path) == expected_hash:
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename} with frozen organizer hash; "
            f"found {len(matches)}: {matches}"
        )

    return matches[0]


def read_jsonl(path):
    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line {line_number}: {exc}"
                )

    return rows


def write_jsonl(rows, path):
    with Path(path).open(
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def normalize_text(value):
    value = unicodedata.normalize(
        "NFKC",
        value,
    )

    return re.sub(
        r"\s+",
        " ",
        value,
    ).strip()


def canonicalize_label(value):
    if not isinstance(value, str):
        return None

    key = (
        normalize_text(value)
        .upper()
        .replace(" ", "_")
    )

    aliases = {
        "SUPPORTS": "SUPPORTS",
        "SUPPORTED": "SUPPORTS",
        "REFUTES": "REFUTES",
        "REFUTED": "REFUTES",
        "NOT_ENOUGH_INFO": "NOT_ENOUGH_INFO",
        "NEI": "NOT_ENOUGH_INFO",
    }

    return aliases.get(key)


def clean_evidence(items):
    cleaned = []
    seen = set()

    blank_removed = 0
    repeated_removed = 0
    normalized_changed = False

    for item in items:

        if not isinstance(item, str):
            raise TypeError(
                f"Evidence item is {type(item).__name__}, not str"
            )

        normalized = normalize_text(item)

        if normalized != item:
            normalized_changed = True

        if not normalized:
            blank_removed += 1
            continue

        if normalized in seen:
            repeated_removed += 1
            continue

        seen.add(normalized)
        cleaned.append(normalized)

    return (
        cleaned,
        blank_removed,
        repeated_removed,
        normalized_changed,
    )


def content_fingerprint(row):
    payload = {
        "claim": normalize_text(row["claim"]),
        "evidence": sorted(
            normalize_text(item)
            for item in row["evidence"]
        ),
    }

    encoded = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def percentile(values, p):
    values = sorted(values)

    if len(values) == 1:
        return values[0]

    position = (
        (len(values) - 1)
        * p
    )

    low = math.floor(position)
    high = math.ceil(position)

    if low == high:
        return values[low]

    fraction = position - low

    return (
        values[low] * (1 - fraction)
        + values[high] * fraction
    )


# ------------------------------------------------------------
# A. Resolve immutable organizer data
# ------------------------------------------------------------

TRAIN_PATH = find_by_hash(
    "train.jsonl",
    EXPECTED_TRAIN_SHA256,
)

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

raw_train = read_jsonl(TRAIN_PATH)
validation = read_jsonl(VAL_PATH)

assert len(raw_train) == 1000
assert len(validation) == 300

validation_hash_before = sha256_file(
    VAL_PATH
)


# ------------------------------------------------------------
# B. Conservative normalization
# ------------------------------------------------------------

working = []

label_alias_repairs = 0
blank_label_ids = []

normalized_claim_rows = 0
normalized_evidence_rows = 0

blank_passages_removed = 0
blank_passage_rows = 0

repeated_passages_removed = 0
repeated_passage_rows = 0


for original_index, source in enumerate(raw_train):

    canonical_label = canonicalize_label(
        source["label"]
    )

    if canonical_label is None:

        assert (
            isinstance(source["label"], str)
            and not source["label"].strip()
        )

        blank_label_ids.append(
            source["id"]
        )

        continue

    if source["label"] != canonical_label:
        label_alias_repairs += 1

    clean_claim = normalize_text(
        source["claim"]
    )

    if clean_claim != source["claim"]:
        normalized_claim_rows += 1

    (
        evidence,
        blank_removed,
        repeated_removed,
        evidence_changed,
    ) = clean_evidence(
        source["evidence"]
    )

    if evidence_changed:
        normalized_evidence_rows += 1

    if blank_removed:
        blank_passage_rows += 1
        blank_passages_removed += blank_removed

    if repeated_removed:
        repeated_passage_rows += 1
        repeated_passages_removed += repeated_removed

    assert clean_claim
    assert evidence

    working.append({
        "id": source["id"],
        "claim": clean_claim,
        "evidence": evidence,
        "label": canonical_label,
        "_original_index": original_index,
    })


assert label_alias_repairs == 45
assert len(blank_label_ids) == 10

assert normalized_claim_rows == 55
assert normalized_evidence_rows == 71

assert blank_passages_removed == 16
assert blank_passage_rows == 16

assert repeated_passages_removed == 30
assert repeated_passage_rows == 30


# ------------------------------------------------------------
# C. Detect canonical duplicate groups
# ------------------------------------------------------------

groups = defaultdict(list)

for row in working:
    groups[
        content_fingerprint(row)
    ].append(row)


clean_train = []

redundant_removed_ids = []
conflict_removed_ids = []
conflicting_groups = []


for group in groups.values():

    group = sorted(
        group,
        key=lambda x: x["_original_index"],
    )

    labels = {
        row["label"]
        for row in group
    }

    # Identical input with contradictory supervision:
    # remove every member rather than guessing.
    if len(labels) > 1:

        conflicting_groups.append([
            {
                "id": row["id"],
                "label": row["label"],
            }
            for row in group
        ])

        conflict_removed_ids.extend(
            row["id"]
            for row in group
        )

        continue

    # Same canonical input and same label:
    # preserve earliest organizer occurrence.
    kept = group[0]

    clean_train.append({
        "id": kept["id"],
        "claim": kept["claim"],
        "evidence": kept["evidence"],
        "label": kept["label"],
    })

    redundant_removed_ids.extend(
        row["id"]
        for row in group[1:]
    )


clean_train.sort(
    key=lambda row: next(
        i
        for i, source in enumerate(raw_train)
        if source["id"] == row["id"]
    )
)


# ------------------------------------------------------------
# D. Dataset V2 integrity
# ------------------------------------------------------------

clean_distribution = Counter(
    row["label"]
    for row in clean_train
)

assert len(conflicting_groups) == 1
assert len(conflict_removed_ids) == 2
assert len(redundant_removed_ids) == 53

assert len(clean_train) == 935

assert clean_distribution == Counter({
    "SUPPORTS": 347,
    "REFUTES": 313,
    "NOT_ENOUGH_INFO": 275,
})

assert all(
    row["label"] in OFFICIAL_LABELS
    for row in clean_train
)

assert len({
    row["id"]
    for row in clean_train
}) == 935

assert len({
    content_fingerprint(row)
    for row in clean_train
}) == 935

assert all(
    row["claim"] == normalize_text(row["claim"])
    for row in clean_train
)

assert all(
    passage
    and passage == normalize_text(passage)
    for row in clean_train
    for passage in row["evidence"]
)

assert all(
    len(row["evidence"])
    == len(set(row["evidence"]))
    for row in clean_train
)


# ------------------------------------------------------------
# E. Train ↔ organizer-validation leakage check
# ------------------------------------------------------------

validation_fingerprints = {
    content_fingerprint(row)
    for row in validation
}

clean_fingerprints = {
    content_fingerprint(row)
    for row in clean_train
}

cross_split_overlap = (
    clean_fingerprints
    & validation_fingerprints
)

assert not cross_split_overlap


# ------------------------------------------------------------
# F. Deterministic 15% stratified internal dev
# ------------------------------------------------------------

by_label = defaultdict(list)

for row in clean_train:
    by_label[row["label"]].append(row)


dev_ids = set()
class_plan = {}


for class_index, label in enumerate(
    LABEL_ORDER
):

    candidates = list(
        by_label[label]
    )

    rng = random.Random(
        SEED + class_index
    )

    rng.shuffle(
        candidates
    )

    dev_count = round(
        len(candidates)
        * DEV_FRACTION
    )

    selected = candidates[
        :dev_count
    ]

    dev_ids.update(
        row["id"]
        for row in selected
    )

    class_plan[label] = {
        "total": len(candidates),
        "fit": len(candidates) - dev_count,
        "dev": dev_count,
    }


fit_rows = [
    row
    for row in clean_train
    if row["id"] not in dev_ids
]

dev_rows = [
    row
    for row in clean_train
    if row["id"] in dev_ids
]


fit_distribution = Counter(
    row["label"]
    for row in fit_rows
)

dev_distribution = Counter(
    row["label"]
    for row in dev_rows
)


assert len(fit_rows) == 795
assert len(dev_rows) == 140

assert fit_distribution == Counter({
    "SUPPORTS": 295,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})

assert dev_distribution == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


assert {
    row["id"]
    for row in fit_rows
}.isdisjoint({
    row["id"]
    for row in dev_rows
})

assert {
    content_fingerprint(row)
    for row in fit_rows
}.isdisjoint({
    content_fingerprint(row)
    for row in dev_rows
})


# ------------------------------------------------------------
# G. Save Dataset V2
# ------------------------------------------------------------

write_jsonl(
    clean_train,
    CLEAN_PATH,
)

write_jsonl(
    fit_rows,
    FIT_PATH,
)

write_jsonl(
    dev_rows,
    DEV_PATH,
)


clean_hash = sha256_file(
    CLEAN_PATH
)

fit_hash = sha256_file(
    FIT_PATH
)

dev_hash = sha256_file(
    DEV_PATH
)


# ------------------------------------------------------------
# H. Final tokenizer check against FINAL data
# ------------------------------------------------------------

from transformers import AutoTokenizer


hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:

    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token if hf_token else True,
)


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def full_training_tokens(row):

    messages = [
        {
            "role": "user",
            "content": user_content(row),
        },
        {
            "role": "assistant",
            "content": row["label"],
        },
    ]

    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    return len(ids)


final_length_records = []

for split_name, rows in (
    ("fit", fit_rows),
    ("internal_dev", dev_rows),
    ("official_validation", validation),
):

    for row in rows:

        length = full_training_tokens(
            row
        )

        final_length_records.append({
            "split": split_name,
            "id": row["id"],
            "label": row["label"],
            "tokens": length,
        })


lengths = [
    item["tokens"]
    for item in final_length_records
]

longest = max(
    final_length_records,
    key=lambda x: x["tokens"],
)


assert max(lengths) <= MAX_LENGTH_CANDIDATE


# ------------------------------------------------------------
# I. Confirm validation remained immutable
# ------------------------------------------------------------

validation_hash_after = sha256_file(
    VAL_PATH
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# J. Save audit manifest
# ------------------------------------------------------------

audit = {
    "dataset_version": "clean_v2",
    "source_hashes": {
        "train": EXPECTED_TRAIN_SHA256,
        "validation": EXPECTED_VAL_SHA256,
    },
    "cleaning": {
        "raw_train_rows": len(raw_train),
        "label_alias_repairs": label_alias_repairs,
        "blank_label_rows_removed": len(blank_label_ids),
        "normalized_claim_rows": normalized_claim_rows,
        "normalized_evidence_rows": normalized_evidence_rows,
        "blank_evidence_passages_removed": blank_passages_removed,
        "repeated_evidence_passages_removed": repeated_passages_removed,
        "conflicting_rows_removed": len(conflict_removed_ids),
        "redundant_rows_removed": len(redundant_removed_ids),
        "final_rows": len(clean_train),
        "class_distribution": dict(clean_distribution),
    },
    "split": {
        "seed": SEED,
        "dev_fraction": DEV_FRACTION,
        "fit_rows": len(fit_rows),
        "dev_rows": len(dev_rows),
        "fit_distribution": dict(fit_distribution),
        "dev_distribution": dict(dev_distribution),
    },
    "tokenization": {
        "model": MODEL_ID,
        "prompt": "v1_minimal_organizer_semantics",
        "max_length_selected": MAX_LENGTH_CANDIDATE,
        "observed_min": min(lengths),
        "observed_p50": percentile(lengths, 0.50),
        "observed_p95": percentile(lengths, 0.95),
        "observed_p99": percentile(lengths, 0.99),
        "observed_max": max(lengths),
        "longest_example": longest,
    },
    "output_hashes": {
        CLEAN_PATH.name: clean_hash,
        FIT_PATH.name: fit_hash,
        DEV_PATH.name: dev_hash,
    },
    "official_validation_modified": False,
}


AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# K. Concise report
# ------------------------------------------------------------

print("=" * 76)
print("DATASET V2 FREEZE — RESULT")
print("=" * 76)

print("\nCleaning")
print(
    "  Raw rows:",
    len(raw_train),
)
print(
    "  Label aliases repaired:",
    label_alias_repairs,
)
print(
    "  Blank-label rows removed:",
    len(blank_label_ids),
)
print(
    "  Claims normalized:",
    normalized_claim_rows,
)
print(
    "  Evidence rows normalized:",
    normalized_evidence_rows,
)
print(
    "  Blank evidence passages removed:",
    blank_passages_removed,
)
print(
    "  Repeated evidence passages removed:",
    repeated_passages_removed,
)
print(
    "  Conflicting rows removed:",
    len(conflict_removed_ids),
)
print(
    "  Redundant rows removed:",
    len(redundant_removed_ids),
)

print("\nFinal clean dataset")
print(
    "  Rows:",
    len(clean_train),
)
print(
    "  Distribution:",
    dict(clean_distribution),
)
print(
    "  Train↔validation full overlaps:",
    len(cross_split_overlap),
)
print(
    "  SHA-256:",
    clean_hash,
)

print("\nInternal split")
print(
    "  Fit:",
    len(fit_rows),
    dict(fit_distribution),
)
print(
    "  Internal dev:",
    len(dev_rows),
    dict(dev_distribution),
)
print(
    "  Fit SHA-256:",
    fit_hash,
)
print(
    "  Dev SHA-256:",
    dev_hash,
)

print("\nFinal token-length check")
print(
    "  Model/tokenizer:",
    MODEL_ID,
)
print(
    "  Rows measured:",
    len(final_length_records),
)
print(
    "  min:",
    min(lengths),
)
print(
    "  p50:",
    round(percentile(lengths, 0.50), 1),
)
print(
    "  p95:",
    round(percentile(lengths, 0.95), 1),
)
print(
    "  p99:",
    round(percentile(lengths, 0.99), 1),
)
print(
    "  max:",
    max(lengths),
)
print(
    "  longest:",
    longest,
)
print(
    "  selected max_length:",
    MAX_LENGTH_CANDIDATE,
)
print(
    "  truncation required:",
    max(lengths) > MAX_LENGTH_CANDIDATE,
)

print("\nOfficial validation")
print(
    "  Rows:",
    len(validation),
)
print(
    "  Modified:",
    False,
)
print(
    "  SHA-256:",
    validation_hash_after,
)

print("\nSaved")
print(" ", CLEAN_PATH)
print(" ", FIT_PATH)
print(" ", DEV_PATH)
print(" ", AUDIT_PATH)

print("\nAll Dataset V2 freeze assertions passed.")

In [ ]:
# ============================================================
# STEP 5B — REPAIR FINAL V2 TOKEN-LENGTH AUDIT
# Dataset files are NOT modified.
# Only token lengths are re-measured correctly.
# ============================================================

from pathlib import Path
import hashlib
import json
import math
import os

from transformers import AutoTokenizer


WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

MODEL_ID = "google/gemma-2-2b-it"
MAX_LENGTH_CANDIDATE = 256

FIT_PATH = WORK_ROOT / "train_fit_v2.jsonl"
DEV_PATH = WORK_ROOT / "internal_dev_v2.jsonl"
CLEAN_PATH = WORK_ROOT / "train_clean_v2.jsonl"
AUDIT_PATH = WORK_ROOT / "dataset_v2_audit.json"

EXPECTED_CLEAN_SHA256 = (
    "011a8cd33f499e83e6bfc08e61b27de8"
    "d784fc0c0fc75dc444accbbd6fa92337"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):
    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line {line_number}: {exc}"
                )

    return rows


def find_by_hash(filename, expected_hash):
    matches = []

    for path in INPUT_ROOT.rglob(filename):

        if (
            path.is_file()
            and sha256_file(path) == expected_hash
        ):
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely locate {filename} "
            f"with the expected organizer hash. "
            f"Matches: {matches}"
        )

    return matches[0]


def percentile(values, p):
    values = sorted(values)

    if not values:
        return None

    if len(values) == 1:
        return values[0]

    position = (
        (len(values) - 1)
        * p
    )

    low = math.floor(position)
    high = math.ceil(position)

    if low == high:
        return values[low]

    fraction = position - low

    return (
        values[low] * (1 - fraction)
        + values[high] * fraction
    )


# ------------------------------------------------------------
# A. Verify frozen V2 files
# ------------------------------------------------------------

assert sha256_file(CLEAN_PATH) == EXPECTED_CLEAN_SHA256
assert sha256_file(FIT_PATH) == EXPECTED_FIT_SHA256
assert sha256_file(DEV_PATH) == EXPECTED_DEV_SHA256

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

fit_rows = read_jsonl(FIT_PATH)
dev_rows = read_jsonl(DEV_PATH)
val_rows = read_jsonl(VAL_PATH)

assert len(fit_rows) == 795
assert len(dev_rows) == 140
assert len(val_rows) == 300


# ------------------------------------------------------------
# B. Load tokenizer
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )
    except Exception:
        hf_token = None


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token if hf_token else True,
)

assert tokenizer.chat_template is not None


# ------------------------------------------------------------
# C. Frozen Prompt V1
# ------------------------------------------------------------

SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    messages = [{
        "role": "user",
        "content": build_user_content(row),
    }]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def render_full(row):

    messages = [
        {
            "role": "user",
            "content": build_user_content(row),
        },
        {
            "role": "assistant",
            "content": row["label"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def token_ids_from_rendered_text(text):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
    )

    ids = encoded["input_ids"]

    assert isinstance(ids, list)
    assert all(
        isinstance(token_id, int)
        for token_id in ids
    )

    return ids


# ------------------------------------------------------------
# D. Structural verification
# ------------------------------------------------------------

sample = fit_rows[0]

sample_prompt = render_prompt(sample)
sample_full = render_full(sample)

assert sample_full.startswith(sample_prompt), (
    "Full training sequence does not start with "
    "the generation prompt."
)

sample_completion = sample_full[
    len(sample_prompt):
]

sample_prompt_ids = token_ids_from_rendered_text(
    sample_prompt
)

sample_full_ids = token_ids_from_rendered_text(
    sample_full
)

assert len(sample_full_ids) > 100
assert len(sample_full_ids) > len(sample_prompt_ids)

assert sample["label"] in sample_completion


# ------------------------------------------------------------
# E. Measure all final V2 sequences
# ------------------------------------------------------------

records = []

for split_name, rows in (
    ("fit", fit_rows),
    ("internal_dev", dev_rows),
    ("official_validation", val_rows),
):

    for row in rows:

        prompt_text = render_prompt(row)
        full_text = render_full(row)

        assert full_text.startswith(
            prompt_text
        )

        prompt_ids = token_ids_from_rendered_text(
            prompt_text
        )

        full_ids = token_ids_from_rendered_text(
            full_text
        )

        records.append({
            "split": split_name,
            "id": row["id"],
            "label": row["label"],
            "passages": len(row["evidence"]),
            "prompt_tokens": len(prompt_ids),
            "full_tokens": len(full_ids),
            "completion_tokens": (
                len(full_ids)
                - len(prompt_ids)
            ),
        })


full_lengths = [
    row["full_tokens"]
    for row in records
]

prompt_lengths = [
    row["prompt_tokens"]
    for row in records
]

completion_lengths = [
    row["completion_tokens"]
    for row in records
]

longest = max(
    records,
    key=lambda row: row["full_tokens"],
)

top_10 = sorted(
    records,
    key=lambda row: row["full_tokens"],
    reverse=True,
)[:10]


# ------------------------------------------------------------
# F. Critical assertions
# ------------------------------------------------------------

assert len(records) == 1235

assert min(full_lengths) > 100, (
    "Token lengths are still suspiciously small."
)

assert max(full_lengths) < 256, (
    "At least one sequence exceeds max_length=256."
)

assert max(full_lengths) > 150, (
    "Unexpected token-length distribution."
)

assert set(completion_lengths).issubset({
    4,
    8,
}), (
    f"Unexpected completion lengths: "
    f"{sorted(set(completion_lengths))}"
)


# ------------------------------------------------------------
# G. Patch ONLY tokenization section of audit manifest
# ------------------------------------------------------------

audit = json.loads(
    AUDIT_PATH.read_text(
        encoding="utf-8"
    )
)

audit["tokenization"] = {
    "model": MODEL_ID,
    "prompt": "v1_minimal_organizer_semantics",
    "measurement_method": (
        "apply_chat_template(tokenize=False), then "
        "tokenizer(rendered_text, add_special_tokens=False)"
    ),
    "max_length_selected": MAX_LENGTH_CANDIDATE,
    "observed_prompt_min": min(prompt_lengths),
    "observed_prompt_p50": percentile(
        prompt_lengths,
        0.50,
    ),
    "observed_prompt_p95": percentile(
        prompt_lengths,
        0.95,
    ),
    "observed_prompt_p99": percentile(
        prompt_lengths,
        0.99,
    ),
    "observed_prompt_max": max(prompt_lengths),
    "observed_full_min": min(full_lengths),
    "observed_full_p50": percentile(
        full_lengths,
        0.50,
    ),
    "observed_full_p90": percentile(
        full_lengths,
        0.90,
    ),
    "observed_full_p95": percentile(
        full_lengths,
        0.95,
    ),
    "observed_full_p99": percentile(
        full_lengths,
        0.99,
    ),
    "observed_full_max": max(full_lengths),
    "longest_example": longest,
    "truncation_required": (
        max(full_lengths)
        > MAX_LENGTH_CANDIDATE
    ),
}

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# H. Concise report
# ------------------------------------------------------------

print("=" * 76)
print("V2 TOKEN AUDIT REPAIR — RESULT")
print("=" * 76)

print("\nDataset integrity")
print(
    "  Clean SHA-256:",
    sha256_file(CLEAN_PATH),
)
print(
    "  Fit SHA-256:",
    sha256_file(FIT_PATH),
)
print(
    "  Dev SHA-256:",
    sha256_file(DEV_PATH),
)
print(
    "  Dataset files modified:",
    False,
)

print("\nTokenizer")
print(
    "  Model:",
    MODEL_ID,
)
print(
    "  Class:",
    tokenizer.__class__.__name__,
)
print(
    "  Chat template:",
    tokenizer.chat_template is not None,
)

print("\nPrompt tokens")
print(
    "  min:",
    min(prompt_lengths),
)
print(
    "  p50:",
    round(
        percentile(
            prompt_lengths,
            0.50,
        ),
        1,
    ),
)
print(
    "  p95:",
    round(
        percentile(
            prompt_lengths,
            0.95,
        ),
        1,
    ),
)
print(
    "  p99:",
    round(
        percentile(
            prompt_lengths,
            0.99,
        ),
        1,
    ),
)
print(
    "  max:",
    max(prompt_lengths),
)

print("\nFull training tokens")
print(
    "  min:",
    min(full_lengths),
)
print(
    "  p50:",
    round(
        percentile(
            full_lengths,
            0.50,
        ),
        1,
    ),
)
print(
    "  p90:",
    round(
        percentile(
            full_lengths,
            0.90,
        ),
        1,
    ),
)
print(
    "  p95:",
    round(
        percentile(
            full_lengths,
            0.95,
        ),
        1,
    ),
)
print(
    "  p99:",
    round(
        percentile(
            full_lengths,
            0.99,
        ),
        1,
    ),
)
print(
    "  max:",
    max(full_lengths),
)

print(
    "\nCompletion lengths:",
    sorted(set(completion_lengths)),
)

print("\nLongest example")
print(" ", longest)

print("\nTop 10 longest")
for row in top_10:
    print(
        " ",
        row["split"],
        "|",
        row["id"],
        "|",
        row["label"],
        "| full=",
        row["full_tokens"],
        "| prompt=",
        row["prompt_tokens"],
        "| passages=",
        row["passages"],
    )

print("\nSample structure")
print(
    "  ID:",
    sample["id"],
)
print(
    "  Prompt tokens:",
    len(sample_prompt_ids),
)
print(
    "  Full tokens:",
    len(sample_full_ids),
)
print(
    "  Completion:",
    repr(sample_completion),
)

print("\nContext decision")
print(
    "  Selected max_length:",
    MAX_LENGTH_CANDIDATE,
)
print(
    "  Observed maximum:",
    max(full_lengths),
)
print(
    "  Safety margin:",
    MAX_LENGTH_CANDIDATE
    - max(full_lengths),
    "tokens",
)
print(
    "  Truncation required:",
    max(full_lengths)
    > MAX_LENGTH_CANDIDATE,
)

print("\nAudit manifest repaired:")
print(" ", AUDIT_PATH)

print("\nAll corrected token-length assertions passed.")

In [ ]:
# ============================================================
# STEP 6 — GEMMA 2 2B QLORA ARCHITECTURE + MEMORY PREFLIGHT
#
# - Uses GPU 0 only
# - Loads Gemma 2 2B IT in 4-bit NF4
# - Inspects exact LoRA targets
# - Applies PEFT/QLoRA preparation
# - Verifies completion-only loss masking
# - Runs real forward + backward memory probes
# - NO optimizer.step() is performed
# - NO model parameter is updated
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys

import torch


# ------------------------------------------------------------
# Configuration — Baseline A
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

FIT_PATH = Path("/kaggle/working/train_fit_v2.jsonl")

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

MAX_LENGTH = 256

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Probe large → small.
BATCH_CANDIDATES = [16, 12, 8, 4, 2, 1]

# Leave a healthy amount of T4 VRAM unused rather than
# selecting a batch that only barely survives.
MIN_HEADROOM_GB = 1.5


# ------------------------------------------------------------
# A. Environment / package preflight
# ------------------------------------------------------------

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1

gpu = torch.cuda.get_device_properties(0)

assert gpu.major == 7 and gpu.minor == 5, (
    f"Expected T4-class compute capability 7.5, got "
    f"{gpu.major}.{gpu.minor}"
)

print("=" * 78)
print("A. GPU")
print("=" * 78)

print("GPU 0:", gpu.name)
print(
    "VRAM:",
    round(gpu.total_memory / 1024**3, 2),
    "GB",
)
print(
    "GPU 1 visible:",
    torch.cuda.device_count() > 1,
)

if torch.cuda.device_count() > 1:
    print(
        "GPU 1:",
        torch.cuda.get_device_name(1),
        "(will remain unused)",
    )


# ------------------------------------------------------------
# B. Ensure QLoRA dependencies
# ------------------------------------------------------------

packages_needed = []

for module_name, pip_name in [
    ("bitsandbytes", "bitsandbytes"),
    ("peft", "peft"),
    ("accelerate", "accelerate"),
]:

    if importlib.util.find_spec(module_name) is None:
        packages_needed.append(pip_name)


if packages_needed:

    print(
        "\nInstalling:",
        ", ".join(packages_needed),
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages_needed,
    ])


from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

import bitsandbytes
import peft
import transformers


print("\nPackage versions")
print("  torch       :", torch.__version__)
print("  transformers:", transformers.__version__)
print("  peft        :", peft.__version__)
print("  bitsandbytes:", bitsandbytes.__version__)


# ------------------------------------------------------------
# C. Verify frozen fit dataset
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def load_jsonl(path):

    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:
            if line.strip():
                rows.append(
                    json.loads(line)
                )

    return rows


assert FIT_PATH.exists()
assert sha256_file(FIT_PATH) == EXPECTED_FIT_SHA256

fit_rows = load_jsonl(FIT_PATH)

assert len(fit_rows) == 795


# ------------------------------------------------------------
# D. Secure Hugging Face authentication
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:

    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is unavailable in this runtime."
    )


# ------------------------------------------------------------
# E. Tokenizer + frozen Prompt V1
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

tokenizer.padding_side = "right"

assert tokenizer.pad_token_id is not None
assert tokenizer.chat_template is not None


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(row),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


def render_full(row):

    return tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": build_user_content(row),
            },
            {
                "role": "assistant",
                "content": row["label"],
            },
        ],
        tokenize=False,
        add_generation_prompt=False,
    )


# ------------------------------------------------------------
# F. Build completion-only training tensor
# ------------------------------------------------------------

def encode_training_row(row):

    prompt_text = render_prompt(row)
    full_text = render_full(row)

    assert full_text.startswith(prompt_text)

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]

    # Critical masking assumption:
    # separately tokenized prompt must be an exact token prefix.
    assert (
        full_ids[:len(prompt_ids)]
        == prompt_ids
    ), (
        "Prompt tokenization is not an exact prefix "
        "of the full training sequence."
    )

    assert len(full_ids) <= MAX_LENGTH

    supervised_count = (
        len(full_ids)
        - len(prompt_ids)
    )

    assert supervised_count in {4, 8}

    pad_count = (
        MAX_LENGTH
        - len(full_ids)
    )

    input_ids = (
        full_ids
        + [tokenizer.pad_token_id] * pad_count
    )

    attention_mask = (
        [1] * len(full_ids)
        + [0] * pad_count
    )

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
        + [-100] * pad_count
    )

    assert len(input_ids) == MAX_LENGTH
    assert len(attention_mask) == MAX_LENGTH
    assert len(labels) == MAX_LENGTH

    assert all(
        x == -100
        for x in labels[:len(prompt_ids)]
    )

    assert all(
        x != -100
        for x in labels[
            len(prompt_ids):len(full_ids)
        ]
    )

    assert all(
        x == -100
        for x in labels[len(full_ids):]
    )

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            attention_mask,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
        "prompt_tokens": len(prompt_ids),
        "full_tokens": len(full_ids),
        "supervised_tokens": supervised_count,
    }


# Find the longest FINAL fit example dynamically.
encoded_lengths = []

for row in fit_rows:

    full_ids = tokenizer(
        render_full(row),
        add_special_tokens=False,
    )["input_ids"]

    encoded_lengths.append(
        (len(full_ids), row)
    )


longest_length, longest_row = max(
    encoded_lengths,
    key=lambda x: x[0],
)

assert longest_length <= MAX_LENGTH

encoded = encode_training_row(
    longest_row
)


# ------------------------------------------------------------
# G. Explicit completion-only masking inspection
# ------------------------------------------------------------

supervised_ids = encoded["labels"][
    encoded["labels"] != -100
].tolist()

supervised_pieces = (
    tokenizer.convert_ids_to_tokens(
        supervised_ids
    )
)

supervised_decoded = tokenizer.decode(
    supervised_ids,
    skip_special_tokens=False,
)


print("\n" + "=" * 78)
print("B. COMPLETION-ONLY MASKING")
print("=" * 78)

print(
    "Longest fit example:",
    longest_row["id"],
)
print(
    "Label:",
    longest_row["label"],
)
print(
    "Full tokens:",
    encoded["full_tokens"],
)
print(
    "Prompt tokens:",
    encoded["prompt_tokens"],
)
print(
    "Supervised tokens:",
    encoded["supervised_tokens"],
)
print(
    "Masked prompt tokens:",
    encoded["prompt_tokens"],
)
print(
    "Padding tokens:",
    MAX_LENGTH - encoded["full_tokens"],
)
print(
    "Supervised token IDs:",
    supervised_ids,
)
print(
    "Supervised pieces:",
    supervised_pieces,
)
print(
    "Supervised decoded:",
    repr(supervised_decoded),
)

assert longest_row["label"] in supervised_decoded


# ------------------------------------------------------------
# H. Clear GPUs before model load
# ------------------------------------------------------------

gc.collect()

torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()

torch.cuda.reset_peak_memory_stats(0)


# ------------------------------------------------------------
# I. 4-bit NF4 configuration
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


# ------------------------------------------------------------
# J. Load BASE MODEL on GPU 0 only
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("C. 4-BIT MODEL LOAD")
print("=" * 78)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

assert model.is_loaded_in_4bit

print(
    "Model class:",
    model.__class__.__name__,
)

print(
    "Hidden layers:",
    model.config.num_hidden_layers,
)

print(
    "HF device map:",
    getattr(
        model,
        "hf_device_map",
        None,
    ),
)

gpu1_allocated = (
    torch.cuda.memory_allocated(1)
    if torch.cuda.device_count() > 1
    else 0
)

print(
    "GPU 0 allocated after base load:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1 allocated:",
    round(
        gpu1_allocated
        / 1024**3,
        3,
    ),
    "GB",
)

assert gpu1_allocated < 50 * 1024**2, (
    "GPU 1 unexpectedly contains significant model memory."
)


# ------------------------------------------------------------
# K. Inspect exact LoRA target modules
# ------------------------------------------------------------

matched = {
    suffix: []
    for suffix in TARGET_MODULES
}

for name, module in model.named_modules():

    suffix = name.split(".")[-1]

    if suffix in matched:
        matched[suffix].append(name)


num_layers = model.config.num_hidden_layers

print("\n" + "=" * 78)
print("D. LORA TARGET INSPECTION")
print("=" * 78)

print(
    "Architecture layers:",
    num_layers,
)

for suffix in TARGET_MODULES:

    names = matched[suffix]

    print(
        f"{suffix:10}:",
        len(names),
        "| sample:",
        names[:2],
    )

    # Gemma 2 should expose one of each target per decoder layer.
    assert len(names) == num_layers, (
        f"Unexpected {suffix} target count: "
        f"{len(names)} vs {num_layers} layers"
    )

    assert all(
        name.startswith("model.layers.")
        for name in names
    )

    assert not any(
        any(
            bad in name.lower()
            for bad in (
                "vision",
                "multimodal",
                "image",
            )
        )
        for name in names
    )


total_target_modules = sum(
    len(names)
    for names in matched.values()
)

assert total_target_modules == (
    num_layers * len(TARGET_MODULES)
)

print(
    "\nTotal intended language modules:",
    total_target_modules,
)


# ------------------------------------------------------------
# L. Prepare for k-bit training
# ------------------------------------------------------------

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# Explicitly preserve the training configuration.
model.config.use_cache = False

if hasattr(
    model,
    "enable_input_require_grads",
):
    model.enable_input_require_grads()


# ------------------------------------------------------------
# M. Attach LoRA
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(
    model,
    lora_config,
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

all_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_pct = (
    100
    * trainable_params
    / all_params
)


print("\n" + "=" * 78)
print("E. QLORA CONFIGURATION")
print("=" * 78)

print(
    "Quantization:",
    "4-bit NF4 + double quant",
)

print(
    "Compute dtype:",
    "float16",
)

print(
    "Gradient checkpointing:",
    model.is_gradient_checkpointing,
)

print(
    "use_cache:",
    model.config.use_cache,
)

print(
    "LoRA r:",
    LORA_R,
)

print(
    "LoRA alpha:",
    LORA_ALPHA,
)

print(
    "LoRA dropout:",
    LORA_DROPOUT,
)

print(
    "Targets:",
    TARGET_MODULES,
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)

print(
    "Total parameters:",
    f"{all_params:,}",
)

print(
    "Trainable percentage:",
    f"{trainable_pct:.4f}%",
)


assert trainable_params > 0
assert trainable_pct < 5.0


# ------------------------------------------------------------
# N. Numerical forward preflight
# ------------------------------------------------------------

model.train()

single_input_ids = (
    encoded["input_ids"]
    .unsqueeze(0)
    .to("cuda:0")
)

single_attention = (
    encoded["attention_mask"]
    .unsqueeze(0)
    .to("cuda:0")
)

single_labels = (
    encoded["labels"]
    .unsqueeze(0)
    .to("cuda:0")
)


with torch.no_grad():

    numerical_output = model(
        input_ids=single_input_ids,
        attention_mask=single_attention,
    )

    final_position = (
        encoded["full_tokens"] - 1
    )

    final_logits = (
        numerical_output.logits[
            0,
            final_position,
            :
        ]
    )

    logits_finite = bool(
        torch.isfinite(
            final_logits
        ).all().item()
    )


assert logits_finite, (
    "NaN/Inf detected in base-model logits."
)

print("\n" + "=" * 78)
print("F. NUMERICAL PREFLIGHT")
print("=" * 78)

print(
    "Final-token logits finite:",
    logits_finite,
)

print(
    "Final-token logit dtype:",
    final_logits.dtype,
)

del numerical_output
del final_logits

torch.cuda.empty_cache()


# ------------------------------------------------------------
# O. Real backward-pass batch probe
#
# This allocates training activations + gradients but NEVER
# calls optimizer.step(), so no model weights are changed.
# ------------------------------------------------------------

base_input = encoded["input_ids"]
base_attention = encoded["attention_mask"]
base_labels = encoded["labels"]


probe_results = []

selected_batch = None


print("\n" + "=" * 78)
print("G. WORST-CASE TRAINING MEMORY PROBE")
print("=" * 78)

print(
    "Sequence length:",
    MAX_LENGTH,
)

print(
    "Example:",
    longest_row["id"],
    "| actual tokens:",
    encoded["full_tokens"],
)

print(
    "Batch candidates:",
    BATCH_CANDIDATES,
)


for batch_size in BATCH_CANDIDATES:

    model.zero_grad(
        set_to_none=True,
    )

    gc.collect()
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(0)

    input_ids = (
        base_input
        .unsqueeze(0)
        .repeat(batch_size, 1)
        .to("cuda:0")
    )

    attention_mask = (
        base_attention
        .unsqueeze(0)
        .repeat(batch_size, 1)
        .to("cuda:0")
    )

    labels = (
        base_labels
        .unsqueeze(0)
        .repeat(batch_size, 1)
        .to("cuda:0")
    )

    try:

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

        loss = outputs.loss

        assert torch.isfinite(loss), (
            f"Non-finite loss at batch {batch_size}"
        )

        loss_value = float(
            loss.detach().cpu()
        )

        loss.backward()

        peak_allocated = (
            torch.cuda.max_memory_allocated(0)
            / 1024**3
        )

        peak_reserved = (
            torch.cuda.max_memory_reserved(0)
            / 1024**3
        )

        total_vram = (
            gpu.total_memory
            / 1024**3
        )

        headroom = (
            total_vram
            - peak_reserved
        )

        status = (
            "SAFE"
            if headroom >= MIN_HEADROOM_GB
            else "TIGHT"
        )

        probe_results.append({
            "batch": batch_size,
            "status": status,
            "loss": loss_value,
            "peak_allocated_gb": peak_allocated,
            "peak_reserved_gb": peak_reserved,
            "headroom_gb": headroom,
        })

        print(
            f"batch={batch_size:2d} | "
            f"{status:5} | "
            f"loss={loss_value:.4f} | "
            f"peak_alloc={peak_allocated:.2f} GB | "
            f"peak_reserved={peak_reserved:.2f} GB | "
            f"headroom={headroom:.2f} GB"
        )

        if (
            selected_batch is None
            and status == "SAFE"
        ):
            selected_batch = batch_size

        del outputs
        del loss

    except torch.OutOfMemoryError:

        probe_results.append({
            "batch": batch_size,
            "status": "OOM",
        })

        print(
            f"batch={batch_size:2d} | OOM"
        )

    finally:

        model.zero_grad(
            set_to_none=True,
        )

        for variable_name in (
            "input_ids",
            "attention_mask",
            "labels",
        ):
            if variable_name in locals():
                del locals()[variable_name]

        gc.collect()
        torch.cuda.empty_cache()


if selected_batch is None:
    raise RuntimeError(
        "No candidate batch size met the required "
        "VRAM headroom."
    )


# ------------------------------------------------------------
# P. Confirm no optimizer update occurred
# ------------------------------------------------------------

# Gradients have been cleared and no optimizer was created.
model.zero_grad(
    set_to_none=True,
)


# ------------------------------------------------------------
# Q. Recommended effective batch
# ------------------------------------------------------------

TARGET_EFFECTIVE_BATCH = 16

if selected_batch >= TARGET_EFFECTIVE_BATCH:

    recommended_physical_batch = (
        TARGET_EFFECTIVE_BATCH
    )

    recommended_grad_accum = 1

else:

    recommended_physical_batch = (
        selected_batch
    )

    recommended_grad_accum = max(
        1,
        TARGET_EFFECTIVE_BATCH
        // recommended_physical_batch,
    )


effective_batch = (
    recommended_physical_batch
    * recommended_grad_accum
)


# ------------------------------------------------------------
# R. Final report
# ------------------------------------------------------------

allocated_now = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)

reserved_now = (
    torch.cuda.memory_reserved(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("QLORA PREFLIGHT — RESULT")
print("=" * 78)

print("\nModel")
print(
    "  Checkpoint:",
    MODEL_ID,
)
print(
    "  Architecture:",
    model.base_model.model.__class__.__name__,
)
print(
    "  Decoder layers:",
    num_layers,
)

print("\nData / prompt")
print(
    "  Dataset:",
    "train_fit_v2",
)
print(
    "  Rows:",
    len(fit_rows),
)
print(
    "  max_length:",
    MAX_LENGTH,
)
print(
    "  Prompt:",
    "v1_minimal_organizer_semantics",
)

print("\nQLoRA")
print(
    "  4-bit:",
    True,
)
print(
    "  Quant type:",
    "NF4",
)
print(
    "  Double quant:",
    True,
)
print(
    "  Compute dtype:",
    "float16",
)
print(
    "  LoRA target modules:",
    total_target_modules,
)
print(
    "  r / alpha / dropout:",
    LORA_R,
    "/",
    LORA_ALPHA,
    "/",
    LORA_DROPOUT,
)
print(
    "  Trainable parameters:",
    f"{trainable_params:,}",
)
print(
    "  Trainable %:",
    f"{trainable_pct:.4f}%",
)

print("\nMasking")
print(
    "  Prompt tokens supervised:",
    0,
)
print(
    "  Longest-example completion tokens supervised:",
    encoded["supervised_tokens"],
)
print(
    "  Supervised decoded:",
    repr(supervised_decoded),
)

print("\nNumerics")
print(
    "  Base logits finite:",
    logits_finite,
)

print("\nMemory-probe recommendation")
print(
    "  Largest candidate with >=",
    MIN_HEADROOM_GB,
    "GB headroom:",
    selected_batch,
)
print(
    "  Recommended physical batch:",
    recommended_physical_batch,
)
print(
    "  Recommended gradient accumulation:",
    recommended_grad_accum,
)
print(
    "  Effective batch:",
    effective_batch,
)

print("\nGPU state after probe")
print(
    "  GPU 0 allocated:",
    round(allocated_now, 3),
    "GB",
)
print(
    "  GPU 0 reserved:",
    round(reserved_now, 3),
    "GB",
)
print(
    "  GPU 1 allocated:",
    round(
        (
            torch.cuda.memory_allocated(1)
            if torch.cuda.device_count() > 1
            else 0
        )
        / 1024**3,
        3,
    ),
    "GB",
)

print("\nNo optimizer.step() was executed.")
print("No model parameter update occurred.")
print("Model + tokenizer remain loaded for the next step.")
print("\nAll QLoRA preflight assertions passed.")

In [ ]:
# ============================================================
# STEP 7A — DIAGNOSE NON-FINITE GRADIENT NORM
#
# Replays the first 16-example accumulation window WITHOUT
# GradScaler and WITHOUT optimizer.step().
#
# No parameter update occurs.
# ============================================================

import gc
import random
import torch


# ------------------------------------------------------------
# A. Required state from failed Baseline A1 run
# ------------------------------------------------------------

required_globals = [
    "model",
    "training_examples",
    "optimizer",
]

missing = [
    name
    for name in required_globals
    if name not in globals()
]

assert not missing, (
    f"Required objects missing from runtime: {missing}. "
    "Do not continue with this diagnostic in a restarted runtime."
)


assert len(training_examples) == 795


# ------------------------------------------------------------
# B. Inspect current scaler + trainable parameter dtypes
# ------------------------------------------------------------

print("=" * 78)
print("GRADIENT FAILURE DIAGNOSTIC")
print("=" * 78)

if "scaler" in globals():
    try:
        print(
            "\nPrevious GradScaler scale:",
            scaler.get_scale(),
        )
    except Exception:
        print(
            "\nPrevious GradScaler scale: unavailable"
        )


trainable_dtype_counts = {}

for parameter in model.parameters():

    if parameter.requires_grad:

        dtype_name = str(parameter.dtype)

        trainable_dtype_counts[
            dtype_name
        ] = (
            trainable_dtype_counts.get(
                dtype_name,
                0,
            )
            + parameter.numel()
        )


print(
    "\nTrainable parameter dtypes:"
)

for dtype_name, count in sorted(
    trainable_dtype_counts.items()
):
    print(
        f"  {dtype_name}: {count:,}"
    )


# ------------------------------------------------------------
# C. Reproduce exact first epoch permutation
# ------------------------------------------------------------

generator = torch.Generator(
    device="cpu"
)

generator.manual_seed(42)

order = torch.randperm(
    len(training_examples),
    generator=generator,
).tolist()

first_window = order[:16]

print(
    "\nFirst accumulation window IDs:"
)

for index in first_window:
    example = training_examples[index]

    print(
        " ",
        example["id"],
        "|",
        example["label_name"],
        "| tokens=",
        example["tokens"],
        "| supervised=",
        example["supervised_tokens"],
    )


# ------------------------------------------------------------
# D. Reset failed gradients completely
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

model.zero_grad(
    set_to_none=True
)

gc.collect()
torch.cuda.empty_cache()

model.train()
model.config.use_cache = False


# ------------------------------------------------------------
# E. Replay all 16 micro-examples WITHOUT GradScaler
# ------------------------------------------------------------

window_size = len(
    first_window
)

losses = []


for micro_step, index in enumerate(
    first_window,
    1,
):

    example = training_examples[
        index
    ]

    input_ids = (
        example["input_ids"]
        .unsqueeze(0)
        .to("cuda:0")
    )

    attention_mask = (
        example["attention_mask"]
        .unsqueeze(0)
        .to("cuda:0")
    )

    labels = (
        example["labels"]
        .unsqueeze(0)
        .to("cuda:0")
    )


    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

        raw_loss = outputs.loss


    assert torch.isfinite(
        raw_loss
    ), (
        f"Non-finite raw loss on "
        f"{example['id']}"
    )


    loss_value = float(
        raw_loss.detach().cpu()
    )

    losses.append(
        loss_value
    )


    scaled_for_accumulation = (
        raw_loss
        / window_size
    )

    # IMPORTANT:
    # Normal backward — intentionally NO GradScaler.
    scaled_for_accumulation.backward()


    print(
        f"micro={micro_step:02d}/16 | "
        f"id={example['id']} | "
        f"loss={loss_value:.6f}"
    )


    del outputs
    del raw_loss
    del scaled_for_accumulation
    del input_ids
    del attention_mask
    del labels


# ------------------------------------------------------------
# F. Inspect every trainable gradient
# ------------------------------------------------------------

finite_parameter_grads = 0
nonfinite_parameter_grads = []

missing_parameter_grads = []

largest_gradients = []


for name, parameter in model.named_parameters():

    if not parameter.requires_grad:
        continue

    grad = parameter.grad

    if grad is None:

        missing_parameter_grads.append(
            name
        )

        continue


    finite = bool(
        torch.isfinite(
            grad
        ).all().item()
    )


    if finite:

        finite_parameter_grads += 1

        max_abs = float(
            grad.detach()
            .abs()
            .max()
            .cpu()
        )

        largest_gradients.append(
            (
                max_abs,
                name,
                str(grad.dtype),
            )
        )

    else:

        nan_count = int(
            torch.isnan(
                grad
            ).sum().item()
        )

        inf_count = int(
            torch.isinf(
                grad
            ).sum().item()
        )

        nonfinite_parameter_grads.append({
            "name": name,
            "dtype": str(
                grad.dtype
            ),
            "nan": nan_count,
            "inf": inf_count,
        })


largest_gradients.sort(
    reverse=True,
    key=lambda x: x[0],
)


# ------------------------------------------------------------
# G. Compute norm only if gradients are finite
# ------------------------------------------------------------

if not nonfinite_parameter_grads:

    total_grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            [
                p
                for p in model.parameters()
                if (
                    p.requires_grad
                    and p.grad is not None
                )
            ],
            max_norm=float("inf"),
        )
    )

    total_grad_norm_value = float(
        total_grad_norm.detach().cpu()
    )

else:

    total_grad_norm_value = None


# ------------------------------------------------------------
# H. Report
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("DIAGNOSTIC RESULT")
print("-" * 78)

print(
    "Mean raw loss:",
    round(
        sum(losses)
        / len(losses),
        6,
    ),
)

print(
    "Finite trainable parameter gradients:",
    finite_parameter_grads,
)

print(
    "Trainable parameters with no gradient:",
    len(
        missing_parameter_grads
    ),
)

print(
    "Trainable parameters with NaN/Inf gradients:",
    len(
        nonfinite_parameter_grads
    ),
)


if total_grad_norm_value is not None:

    print(
        "Unscaled total gradient norm:",
        total_grad_norm_value,
    )


print(
    "\nLargest finite gradients:"
)

for max_abs, name, dtype_name in (
    largest_gradients[:10]
):

    print(
        f"  {max_abs:.8f} | "
        f"{dtype_name} | "
        f"{name}"
    )


if nonfinite_parameter_grads:

    print(
        "\nFirst non-finite gradient tensors:"
    )

    for item in (
        nonfinite_parameter_grads[:20]
    ):

        print(
            " ",
            item,
        )


print("\nInterpretation")

if not nonfinite_parameter_grads:

    print(
        "  PASS: the exact first accumulation window "
        "has finite gradients without GradScaler."
    )

    print(
        "  This strongly indicates the previous failure "
        "was FP16 loss-scaling overflow rather than "
        "intrinsic model-gradient instability."
    )

else:

    print(
        "  FAIL: gradients are non-finite even without "
        "GradScaler. We must diagnose the offending "
        "module/example before training."
    )


# ------------------------------------------------------------
# I. Clean diagnostic gradients
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

model.zero_grad(
    set_to_none=True
)

gc.collect()
torch.cuda.empty_cache()


print("\nOptimizer step executed:", False)
print("Parameter update executed:", False)
print("Diagnostic gradients cleared:", True)

In [ ]:
# ============================================================
# STEP 7B — BASELINE A1 CORRECTED TRAINING
#
# Fix from failed attempt:
#   - NO GradScaler
#
# Everything scientific remains frozen:
#   Gemma 2 2B IT
#   Dataset V2
#   Prompt V1
#   max_length=256
#   NF4 QLoRA
#   LoRA r=8 / alpha=16 / dropout=0.05
#   physical batch=1
#   gradient accumulation up to 16
#   effective batch ~=16
#   LR=2e-4
#   cosine schedule
#   2 epochs
#   seed=42
#
# Official organizer validation is NOT used.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import time

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen experiment configuration
# ------------------------------------------------------------

EXPERIMENT_NAME = "baseline_a1_gemma2_2b"

# Separate directory because the original attempt failed
# before its first optimizer update.
RUN_NAME = "baseline_a1_gemma2_2b_retry1_noscaler"

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")

FIT_PATH = WORK_ROOT / "train_fit_v2.jsonl"
DEV_PATH = WORK_ROOT / "internal_dev_v2.jsonl"

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

RUN_DIR = WORK_ROOT / RUN_NAME

SEED = 42

MAX_LENGTH = 256

EPOCHS = 2
PHYSICAL_BATCH = 1
GRAD_ACCUM = 16

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# ------------------------------------------------------------
# B. Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line {line_number}: {exc}"
                )

    return rows


def directory_size_mb(path):

    total = 0

    for file in Path(path).rglob("*"):
        if file.is_file():
            total += file.stat().st_size

    return total / (1024 ** 2)


# ------------------------------------------------------------
# D. Verify frozen Dataset V2
# ------------------------------------------------------------

assert FIT_PATH.exists()
assert DEV_PATH.exists()

assert sha256_file(FIT_PATH) == EXPECTED_FIT_SHA256
assert sha256_file(DEV_PATH) == EXPECTED_DEV_SHA256

fit_rows = read_jsonl(FIT_PATH)
dev_rows = read_jsonl(DEV_PATH)

assert len(fit_rows) == 795
assert len(dev_rows) == 140

assert Counter(
    row["label"]
    for row in fit_rows
) == Counter({
    "SUPPORTS": 295,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})

assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


# ------------------------------------------------------------
# E. Do not overwrite a previous retry
# ------------------------------------------------------------

if RUN_DIR.exists():
    raise RuntimeError(
        f"{RUN_DIR} already exists. "
        "Do not overwrite an experiment."
    )

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ------------------------------------------------------------
# F. Hugging Face authentication
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:

    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is unavailable."
    )


# ------------------------------------------------------------
# G. Fully release diagnostic / failed model state
# ------------------------------------------------------------

for object_name in [
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "outputs",
]:

    if object_name in globals():

        try:
            del globals()[object_name]

        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()


print("=" * 78)
print("BASELINE A1 RETRY — CLEAN GPU STATE")
print("=" * 78)

print(
    "GPU 0 allocated before fresh load:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1 allocated:",
    round(
        (
            torch.cuda.memory_allocated(1)
            if torch.cuda.device_count() > 1
            else 0
        )
        / 1024**3,
        3,
    ),
    "GB",
)


# ------------------------------------------------------------
# H. Tokenizer + frozen Prompt V1
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(row),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


def render_full(row):

    return tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": build_user_content(row),
            },
            {
                "role": "assistant",
                "content": row["label"],
            },
        ],
        tokenize=False,
        add_generation_prompt=False,
    )


# ------------------------------------------------------------
# I. Pre-tokenize completion-only training examples
# ------------------------------------------------------------

training_examples = []


for row in fit_rows:

    prompt_text = render_prompt(row)
    full_text = render_full(row)

    assert full_text.startswith(prompt_text)

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]

    assert (
        full_ids[:len(prompt_ids)]
        == prompt_ids
    )

    assert len(full_ids) <= MAX_LENGTH

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    supervised_tokens = sum(
        token != -100
        for token in labels
    )

    assert supervised_tokens in {
        4,
        8,
    }

    training_examples.append({
        "id": row["id"],
        "label_name": row["label"],
        "input_ids": torch.tensor(
            full_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.ones(
            len(full_ids),
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
        "tokens": len(full_ids),
        "supervised_tokens": supervised_tokens,
    })


assert len(training_examples) == 795

assert max(
    example["tokens"]
    for example in training_examples
) <= MAX_LENGTH


# ------------------------------------------------------------
# J. Load completely FRESH 4-bit Gemma
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("BASELINE A1 RETRY — FRESH MODEL LOAD")
print("=" * 78)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

assert model.is_loaded_in_4bit


model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False,
)

model.config.use_cache = False


# ------------------------------------------------------------
# K. Fresh LoRA adapter
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(
    model,
    lora_config,
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert trainable_params == 10_383_360


trainable_dtypes = Counter(
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
)

assert trainable_dtypes == Counter({
    "torch.float32": 364
}) or all(
    p.dtype == torch.float32
    for p in model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# L. Modern non-reentrant gradient checkpointing
# ------------------------------------------------------------

def enable_training_checkpointing():

    model.config.use_cache = False

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()


def disable_training_checkpointing():

    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass


enable_training_checkpointing()

assert model.is_gradient_checkpointing


# ------------------------------------------------------------
# M. Optimizer + scheduler
#
# IMPORTANT:
# No GradScaler is created.
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


optimizer_steps_per_epoch = math.ceil(
    len(training_examples)
    / GRAD_ACCUM
)

total_optimizer_steps = (
    optimizer_steps_per_epoch
    * EPOCHS
)

warmup_steps = max(
    1,
    round(
        total_optimizer_steps
        * WARMUP_RATIO
    ),
)


scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)


assert total_optimizer_steps == 100
assert warmup_steps == 5


# ------------------------------------------------------------
# N. Strict deterministic internal-dev evaluation
# ------------------------------------------------------------

end_of_turn_id = tokenizer.convert_tokens_to_ids(
    "<end_of_turn>"
)

assert isinstance(
    end_of_turn_id,
    int,
)

assert end_of_turn_id >= 0


def parse_prediction(decoded):

    cleaned = decoded.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_internal_dev(tag):

    disable_training_checkpointing()

    model.eval()
    model.config.use_cache = True

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    gold = []
    predictions = []
    raw_outputs = []

    evaluation_start = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(dev_rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = dev_rows[
                start:
                start + EVAL_BATCH_SIZE
            ]

            prompts = [
                render_prompt(row)
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )

            input_ids = encoded[
                "input_ids"
            ].to("cuda:0")

            attention_mask = encoded[
                "attention_mask"
            ].to("cuda:0")

            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_decoded = tokenizer.decode(
                    token_ids,
                    skip_special_tokens=True,
                )

                prediction = parse_prediction(
                    raw_decoded
                )

                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )

                raw_outputs.append({
                    "id": row["id"],
                    "gold": row["label"],
                    "prediction": prediction,
                    "raw": raw_decoded,
                })


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )

        model.config.use_cache = False

        enable_training_checkpointing()

        model.train()


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - evaluation_start
    )


    result = {
        "tag": tag,
        "rows": len(gold),
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid_count": int(
            invalid_count
        ),
        "invalid_rate": float(
            invalid_count / len(gold)
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "confusion_matrix": (
            matrix.tolist()
        ),
        "seconds": float(
            elapsed
        ),
        "invalid_examples": [
            item
            for item in raw_outputs
            if item["prediction"] is None
        ][:10],
    }


    print("\n" + "-" * 78)
    print(
        f"INTERNAL DEV — {tag}"
    )
    print("-" * 78)

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
        "/",
        len(gold),
    )

    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )

    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print("Per-class F1:")

    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    if invalid_count:

        print(
            "First invalid raw outputs:"
        )

        for item in result[
            "invalid_examples"
        ]:

            print(
                " ",
                item["id"],
                "| gold=",
                item["gold"],
                "| raw=",
                repr(item["raw"]),
            )


    print(
        "Evaluation seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return result


# ------------------------------------------------------------
# O. Save exact run configuration before training
# ------------------------------------------------------------

run_config = {
    "experiment": EXPERIMENT_NAME,
    "technical_attempt": RUN_NAME,
    "technical_fix": (
        "Removed GradScaler after diagnostic showed "
        "finite unscaled FP32 LoRA gradients and "
        "overflow with initial scale 65536."
    ),
    "seed": SEED,
    "model": MODEL_ID,
    "dataset": {
        "fit": FIT_PATH.name,
        "fit_sha256": EXPECTED_FIT_SHA256,
        "fit_rows": len(fit_rows),
        "internal_dev": DEV_PATH.name,
        "internal_dev_sha256": EXPECTED_DEV_SHA256,
        "internal_dev_rows": len(dev_rows),
        "official_validation_used": False,
    },
    "prompt": (
        "v1_minimal_organizer_semantics"
    ),
    "max_length": MAX_LENGTH,
    "quantization": {
        "bits": 4,
        "type": "NF4",
        "double_quant": True,
        "compute_dtype": "float16",
    },
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "targets": TARGET_MODULES,
        "trainable_parameters": (
            trainable_params
        ),
    },
    "training": {
        "epochs": EPOCHS,
        "physical_batch": PHYSICAL_BATCH,
        "gradient_accumulation_max": (
            GRAD_ACCUM
        ),
        "target_effective_batch": 16,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "cosine",
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "optimizer": (
            "torch.optim.AdamW"
        ),
        "max_grad_norm": (
            MAX_GRAD_NORM
        ),
        "gradient_checkpointing": True,
        "gradient_checkpointing_use_reentrant": False,
        "autocast": "fp16",
        "grad_scaler": False,
        "loss": (
            "completion-only; "
            "prompt tokens masked; "
            "each micro-example equally weighted"
        ),
        "optimizer_steps_per_epoch": (
            optimizer_steps_per_epoch
        ),
        "total_optimizer_steps": (
            total_optimizer_steps
        ),
    },
}


(
    RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# P. Pre-training internal-dev checkpoint
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BASELINE A1 RETRY — PRE-TRAIN EVALUATION")
print("=" * 78)


all_metrics = []


pretrain_metrics = evaluate_internal_dev(
    "pretrain"
)

all_metrics.append(
    pretrain_metrics
)


# ------------------------------------------------------------
# Q. Training loop — NO GradScaler
# ------------------------------------------------------------

global_optimizer_step = 0

epoch_training_summaries = []


for epoch_index in range(EPOCHS):

    epoch_number = (
        epoch_index + 1
    )

    enable_training_checkpointing()

    model.train()


    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        SEED + epoch_index
    )

    order = torch.randperm(
        len(training_examples),
        generator=generator,
    ).tolist()


    epoch_loss_sum = 0.0
    epoch_example_count = 0

    epoch_start = (
        time.perf_counter()
    )


    print("\n" + "=" * 78)
    print(
        f"TRAINING EPOCH {epoch_number}/{EPOCHS}"
    )
    print("=" * 78)


    for window_start in range(
        0,
        len(order),
        GRAD_ACCUM,
    ):

        window_indices = order[
            window_start:
            window_start + GRAD_ACCUM
        ]

        window_size = len(
            window_indices
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        window_loss_sum = 0.0


        for index in window_indices:

            example = training_examples[
                index
            ]


            input_ids = (
                example["input_ids"]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )

            attention_mask = (
                example["attention_mask"]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )

            labels = (
                example["labels"]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )

                raw_loss = outputs.loss


            if not torch.isfinite(
                raw_loss
            ):

                raise RuntimeError(
                    f"Non-finite loss at epoch "
                    f"{epoch_number}, "
                    f"example {example['id']}"
                )


            raw_loss_value = float(
                raw_loss.detach().cpu()
            )


            epoch_loss_sum += (
                raw_loss_value
            )

            window_loss_sum += (
                raw_loss_value
            )

            epoch_example_count += 1


            # Correctly normalize the final partial
            # accumulation window as well.
            accumulation_loss = (
                raw_loss
                / window_size
            )


            # NO GradScaler.
            accumulation_loss.backward()


            del outputs
            del raw_loss
            del accumulation_loss
            del input_ids
            del attention_mask
            del labels


        # ----------------------------------------------------
        # Verify every gradient is finite BEFORE clipping/step
        # ----------------------------------------------------

        nonfinite_gradient_names = []

        for name, parameter in (
            model.named_parameters()
        ):

            if (
                parameter.requires_grad
                and parameter.grad is not None
            ):

                if not torch.isfinite(
                    parameter.grad
                ).all():

                    nonfinite_gradient_names.append(
                        name
                    )


        if nonfinite_gradient_names:

            raise RuntimeError(
                "Non-finite gradients before optimizer step. "
                f"First tensors: "
                f"{nonfinite_gradient_names[:10]}"
            )


        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter in model.parameters()
                    if (
                        parameter.requires_grad
                        and parameter.grad is not None
                    )
                ],
                MAX_GRAD_NORM,
            )
        )


        if not torch.isfinite(
            grad_norm
        ):

            raise RuntimeError(
                f"Non-finite gradient norm at "
                f"epoch {epoch_number}, "
                f"optimizer step "
                f"{global_optimizer_step + 1}"
            )


        optimizer.step()
        scheduler.step()


        global_optimizer_step += 1


        if (
            global_optimizer_step % 10 == 0
            or window_start == 0
            or (
                window_start
                + window_size
                >= len(order)
            )
        ):

            current_lr = (
                scheduler
                .get_last_lr()[0]
            )

            mean_epoch_loss = (
                epoch_loss_sum
                / epoch_example_count
            )

            mean_window_loss = (
                window_loss_sum
                / window_size
            )


            print(
                f"epoch={epoch_number} | "
                f"step={global_optimizer_step:3d}/"
                f"{total_optimizer_steps} | "
                f"examples={epoch_example_count:3d}/"
                f"{len(training_examples)} | "
                f"window_loss={mean_window_loss:.4f} | "
                f"epoch_loss={mean_epoch_loss:.4f} | "
                f"lr={current_lr:.7f} | "
                f"grad_norm={float(grad_norm):.4f}"
            )


    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )


    epoch_mean_loss = (
        epoch_loss_sum
        / epoch_example_count
    )


    assert epoch_example_count == 795


    epoch_summary = {
        "epoch": epoch_number,
        "mean_example_loss": float(
            epoch_mean_loss
        ),
        "seconds": float(
            epoch_seconds
        ),
        "optimizer_step_end": int(
            global_optimizer_step
        ),
    }


    epoch_training_summaries.append(
        epoch_summary
    )


    print(
        f"\nEpoch {epoch_number} training complete"
    )

    print(
        "  Mean example loss:",
        f"{epoch_mean_loss:.4f}",
    )

    print(
        "  Seconds:",
        round(
            epoch_seconds,
            1,
        ),
    )


    # --------------------------------------------------------
    # Save adapter BEFORE evaluation
    # --------------------------------------------------------

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )


    model.save_pretrained(
        checkpoint_dir,
        safe_serialization=True,
    )

    tokenizer.save_pretrained(
        checkpoint_dir
    )


    print(
        "  Adapter saved:",
        checkpoint_dir,
    )

    print(
        "  Adapter size:",
        f"{directory_size_mb(checkpoint_dir):.1f} MB",
    )


    # --------------------------------------------------------
    # Predetermined internal-dev evaluation
    # --------------------------------------------------------

    epoch_metrics = evaluate_internal_dev(
        f"epoch_{epoch_number}"
    )


    epoch_metrics[
        "mean_training_loss"
    ] = float(
        epoch_mean_loss
    )

    epoch_metrics[
        "checkpoint"
    ] = str(
        checkpoint_dir
    )


    all_metrics.append(
        epoch_metrics
    )


# ------------------------------------------------------------
# R. Save complete experiment ledger
# ------------------------------------------------------------

assert global_optimizer_step == 100


results = {
    "experiment": EXPERIMENT_NAME,
    "technical_attempt": RUN_NAME,
    "config": run_config,
    "training_summaries": (
        epoch_training_summaries
    ),
    "internal_dev_metrics": (
        all_metrics
    ),
}


RESULTS_PATH = (
    RUN_DIR
    / "results.json"
)


RESULTS_PATH.write_text(
    json.dumps(
        results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# S. Rank trained checkpoints on internal dev only
# ------------------------------------------------------------

trained_results = [
    result
    for result in all_metrics
    if result["tag"].startswith(
        "epoch_"
    )
]


ranked = sorted(
    trained_results,
    key=lambda result: (
        result["macro_f1"],
        result["accuracy"],
        -result["invalid_count"],
    ),
    reverse=True,
)


best_internal = ranked[0]


# ------------------------------------------------------------
# T. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BASELINE A1 — CORRECTED RESULT")
print("=" * 78)


print("\nConfiguration")

print(
    "  Model:",
    MODEL_ID,
)

print(
    "  Fit rows:",
    len(fit_rows),
)

print(
    "  Internal dev rows:",
    len(dev_rows),
)

print(
    "  Official validation used:",
    False,
)

print(
    "  Prompt:",
    "v1_minimal_organizer_semantics",
)

print(
    "  max_length:",
    MAX_LENGTH,
)

print(
    "  LoRA:",
    f"r={LORA_R}, alpha={LORA_ALPHA}, "
    f"dropout={LORA_DROPOUT}",
)

print(
    "  Physical batch:",
    PHYSICAL_BATCH,
)

print(
    "  Gradient accumulation:",
    GRAD_ACCUM,
)

print(
    "  GradScaler:",
    False,
)

print(
    "  Epochs:",
    EPOCHS,
)

print(
    "  LR:",
    LEARNING_RATE,
)

print(
    "  Optimizer steps:",
    total_optimizer_steps,
)

print(
    "  Warmup steps:",
    warmup_steps,
)


print("\nInternal-dev checkpoints")

for result in all_metrics:

    print(
        f"  {result['tag']:8} | "
        f"Accuracy={result['accuracy']:.4f} | "
        f"Macro-F1={result['macro_f1']:.4f} | "
        f"Invalid={result['invalid_count']}"
    )


print("\nPer-class F1 by trained checkpoint")

for result in trained_results:

    print(
        f"  {result['tag']}"
    )

    for label in LABELS:

        print(
            f"    {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


print("\nTraining loss")

for summary in epoch_training_summaries:

    print(
        f"  epoch_{summary['epoch']} | "
        f"loss={summary['mean_example_loss']:.4f} | "
        f"{summary['seconds']:.1f}s"
    )


print("\nProvisional best by INTERNAL DEV only")

print(
    "  Checkpoint:",
    best_internal["tag"],
)

print(
    "  Accuracy:",
    f"{best_internal['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{best_internal['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    best_internal["invalid_count"],
)


print("\nArtifacts")

for epoch_number in range(
    1,
    EPOCHS + 1,
):

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )

    print(
        f"  epoch_{epoch_number}:",
        checkpoint_dir,
        f"({directory_size_mb(checkpoint_dir):.1f} MB)",
    )


print(
    "  Config:",
    RUN_DIR / "run_config.json",
)

print(
    "  Results:",
    RESULTS_PATH,
)


print("\nGPU")

print(
    "  GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "  GPU 1 allocated:",
    round(
        (
            torch.cuda.memory_allocated(1)
            if torch.cuda.device_count() > 1
            else 0
        )
        / 1024**3,
        3,
    ),
    "GB",
)


print("\nBaseline A1 corrected training completed.")
print("Official organizer validation was not evaluated.")

In [ ]:
# ============================================================
# STEP 8 — FRESH ADAPTER RELOAD + OFFICIAL VALIDATION #1
#
# Selected checkpoint:
#   Baseline A1 / epoch 2
#
# Sequence:
#   1. verify saved adapter files/config
#   2. release in-memory training model
#   3. fresh-load 4-bit Gemma base
#   4. reproduce k-bit numerical preparation
#   5. attach saved epoch-2 adapter
#   6. assert finite logits
#   7. reproduce internal-dev metrics
#   8. ONLY THEN evaluate untouched organizer validation
#
# No training occurs.
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

RUN_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

ADAPTER_DIR = (
    RUN_DIR
    / "epoch_2_adapter"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(
        filename
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):
            matches.append(path)

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one "
            f"{filename} with organizer "
            f"hash; found: {matches}"
        )

    return matches[0]


# ------------------------------------------------------------
# C. Verify saved epoch-2 adapter
# ------------------------------------------------------------

print("=" * 78)
print("A. SAVED ADAPTER INTEGRITY")
print("=" * 78)


assert ADAPTER_DIR.exists()

adapter_model_path = (
    ADAPTER_DIR
    / "adapter_model.safetensors"
)

adapter_config_path = (
    ADAPTER_DIR
    / "adapter_config.json"
)


assert adapter_model_path.exists()
assert adapter_config_path.exists()


adapter_config = json.loads(
    adapter_config_path.read_text(
        encoding="utf-8"
    )
)


print(
    "Adapter directory:",
    ADAPTER_DIR,
)

print(
    "Adapter model exists:",
    adapter_model_path.exists(),
)

print(
    "Adapter model size:",
    round(
        adapter_model_path.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "Adapter SHA-256:",
    sha256_file(
        adapter_model_path
    ),
)

print(
    "base_model_name_or_path:",
    adapter_config.get(
        "base_model_name_or_path"
    ),
)

print(
    "r:",
    adapter_config.get("r"),
)

print(
    "lora_alpha:",
    adapter_config.get(
        "lora_alpha"
    ),
)

print(
    "target_modules:",
    adapter_config.get(
        "target_modules"
    ),
)


assert adapter_config.get(
    "base_model_name_or_path"
) == MODEL_ID

assert adapter_config.get(
    "r"
) == 8

assert adapter_config.get(
    "lora_alpha"
) == 16


# ------------------------------------------------------------
# D. Verify internal-dev + organizer validation
# ------------------------------------------------------------

assert sha256_file(
    DEV_PATH
) == EXPECTED_DEV_SHA256

DEV_ROWS = read_jsonl(
    DEV_PATH
)

assert len(DEV_ROWS) == 140


VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

assert sha256_file(
    VAL_PATH
) == EXPECTED_VAL_SHA256

VAL_ROWS = read_jsonl(
    VAL_PATH
)

assert len(VAL_ROWS) == 300


# ------------------------------------------------------------
# E. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:

    try:
        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Remove ALL in-memory training objects
# ------------------------------------------------------------

for object_name in [
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "outputs",
    "lora_config",
    "quant_config",
]:

    if object_name in globals():

        try:
            del globals()[
                object_name
            ]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):
    torch.cuda.ipc_collect()


print("\n" + "=" * 78)
print("B. FRESH-RELOAD GPU STATE")
print("=" * 78)

print(
    "GPU 0 allocated before reload:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1 allocated:",
    round(
        torch.cuda.memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)


# ------------------------------------------------------------
# G. Load tokenizer FROM SAVED ARTIFACT
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer.from_pretrained(
        ADAPTER_DIR,
        local_files_only=True,
    )
)

assert (
    tokenizer.chat_template
    is not None
)

assert (
    tokenizer.pad_token_id
    is not None
)


# ------------------------------------------------------------
# H. Frozen Prompt V1
# ------------------------------------------------------------

SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(
                row
            ),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


# ------------------------------------------------------------
# I. Fresh-load 4-bit BASE
# ------------------------------------------------------------

fresh_quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


print("\n" + "=" * 78)
print("C. FRESH BASE + ADAPTER RELOAD")
print("=" * 78)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            fresh_quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert base_model.is_loaded_in_4bit


# Reproduce the numerical preparation
# used before the adapter was trained.
base_model = (
    prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=False,
    )
)

base_model.config.use_cache = True


model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)


model.eval()
model.config.use_cache = True


trainable_after_reload = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print(
    "Model class:",
    model.__class__.__name__,
)

print(
    "Adapter loaded:",
    True,
)

print(
    "Trainable parameters "
    "during inference:",
    trainable_after_reload,
)

print(
    "use_cache:",
    model.config.use_cache,
)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


assert trainable_after_reload == 0


# ------------------------------------------------------------
# J. Numerical preflight on saved adapter
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("D. SAVED-ADAPTER NUMERICAL PREFLIGHT")
print("=" * 78)


preflight_rows = [
    DEV_ROWS[0],
    DEV_ROWS[
        len(DEV_ROWS) // 2
    ],
    DEV_ROWS[-1],
]


for row in preflight_rows:

    prompt = render_prompt(row)

    encoded = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    input_ids = encoded[
        "input_ids"
    ].to("cuda:0")

    attention_mask = encoded[
        "attention_mask"
    ].to("cuda:0")


    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            attention_mask=(
                attention_mask
            ),
            use_cache=True,
        )


    final_logits = (
        outputs.logits[
            0,
            -1,
            :
        ]
    )


    finite = bool(
        torch.isfinite(
            final_logits
        ).all().item()
    )


    print(
        row["id"],
        "| finite logits:",
        finite,
        "| dtype:",
        final_logits.dtype,
    )


    assert finite


    del encoded
    del input_ids
    del attention_mask
    del outputs
    del final_logits


torch.cuda.empty_cache()


# ------------------------------------------------------------
# K. Strict deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)

assert isinstance(
    end_of_turn_id,
    int,
)

assert end_of_turn_id >= 0


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_rows(
    rows,
    tag,
):

    old_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    gold = []
    predictions = []
    records = []

    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]

            prompts = [
                render_prompt(row)
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = encoded[
                "input_ids"
            ].to("cuda:0")

            attention_mask = encoded[
                "attention_mask"
            ].to("cuda:0")

            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_output = (
                    tokenizer.decode(
                        token_ids,
                        skip_special_tokens=True,
                    )
                )

                prediction = (
                    parse_prediction(
                        raw_output
                    )
                )

                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )

                records.append({
                    "id": row["id"],
                    "gold": row["label"],
                    "prediction": (
                        prediction
                    ),
                    "raw_output": (
                        raw_output
                    ),
                })


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            old_padding_side
        )


    invalid_count = sum(
        prediction is None
        for prediction
        in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction
        in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,
        "rows": len(rows),
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid_count": int(
            invalid_count
        ),
        "invalid_rate": float(
            invalid_count
            / len(rows)
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "confusion_matrix": (
            matrix.tolist()
        ),
        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)

    print(
        "Rows:",
        len(rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )


    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  "
        "NOT_ENOUGH_INFO"
    )

    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print("Per-class metrics:")

    for label in LABELS:

        metrics = (
            result["per_class"][
                label
            ]
        )

        print(
            f"  {label:15} | "
            f"P={metrics['precision']:.4f} | "
            f"R={metrics['recall']:.4f} | "
            f"F1={metrics['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return (
        result,
        records,
    )


# ------------------------------------------------------------
# L. FIRST: reproduce internal-dev checkpoint
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("E. FRESH-RELOAD REPRODUCTION CHECK")
print("=" * 78)


(
    reload_dev_result,
    reload_dev_records,
) = evaluate_rows(
    DEV_ROWS,
    "INTERNAL DEV — FRESH RELOAD EPOCH 2",
)


# Must reproduce the previously observed epoch-2 result
# before organizer validation is touched.
assert (
    round(
        reload_dev_result[
            "accuracy"
        ],
        4,
    )
    == 0.8214
)

assert (
    round(
        reload_dev_result[
            "macro_f1"
        ],
        4,
    )
    == 0.8249
)

assert (
    reload_dev_result[
        "invalid_count"
    ]
    == 0
)


print(
    "\nFresh reload reproduces "
    "epoch-2 internal-dev result: PASS"
)


# ------------------------------------------------------------
# M. FIRST controlled organizer-validation evaluation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("F. OFFICIAL VALIDATION — EVALUATION #1")
print("=" * 78)


validation_hash_before = (
    sha256_file(
        VAL_PATH
    )
)


(
    official_result,
    official_records,
) = evaluate_rows(
    VAL_ROWS,
    "ORGANIZER VALIDATION — BASELINE A1 EPOCH 2",
)


validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)


assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# N. Save predictions + metrics
# ------------------------------------------------------------

validation_predictions_path = (
    RUN_DIR
    / "official_validation_eval1_epoch2.csv"
)

validation_metrics_path = (
    RUN_DIR
    / "official_validation_eval1_epoch2.json"
)


prediction_df = pd.DataFrame([
    {
        "id": record["id"],
        "gold": record["gold"],
        "prediction": (
            record["prediction"]
        ),
        "correct": (
            record["gold"]
            == record["prediction"]
        ),
        "raw_output": (
            record["raw_output"]
        ),
    }
    for record in official_records
])


prediction_df.to_csv(
    validation_predictions_path,
    index=False,
)


validation_payload = {
    "evaluation_number": 1,
    "selection_basis": (
        "epoch_2 selected before organizer-validation "
        "evaluation because it had the highest frozen "
        "internal-dev Macro-F1"
    ),
    "model": MODEL_ID,
    "checkpoint": str(
        ADAPTER_DIR
    ),
    "adapter_sha256": (
        sha256_file(
            adapter_model_path
        )
    ),
    "validation_sha256": (
        EXPECTED_VAL_SHA256
    ),
    "fresh_reload_internal_dev": (
        reload_dev_result
    ),
    "official_validation": (
        official_result
    ),
}


validation_metrics_path.write_text(
    json.dumps(
        validation_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# O. Patch experiment ledger
# ------------------------------------------------------------

results_path = (
    RUN_DIR
    / "results.json"
)


if results_path.exists():

    ledger = json.loads(
        results_path.read_text(
            encoding="utf-8"
        )
    )

else:

    ledger = {}


ledger[
    "fresh_reload_epoch2"
] = {
    "adapter_sha256": (
        sha256_file(
            adapter_model_path
        )
    ),
    "internal_dev_reproduced": True,
    "internal_dev_metrics": (
        reload_dev_result
    ),
    "finite_logits": True,
}


ledger[
    "official_validation_evaluations"
] = [
    {
        "evaluation_number": 1,
        "checkpoint": "epoch_2",
        "selection_was_predeclared": True,
        "metrics": official_result,
    }
]


results_path.write_text(
    json.dumps(
        ledger,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# P. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BASELINE A1 — OFFICIAL VALIDATION RESULT")
print("=" * 78)


print("\nSaved-adapter portability")

print(
    "  Fresh reload:",
    "PASS",
)

print(
    "  Finite logits:",
    "PASS",
)

print(
    "  Internal-dev reproduction:",
    "PASS",
)

print(
    "  Adapter SHA-256:",
    sha256_file(
        adapter_model_path
    ),
)


print("\nInternal dev — fresh reload")

print(
    "  Accuracy:",
    f"{reload_dev_result['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{reload_dev_result['macro_f1']:.4f}",
)


print(
    "\nOrganizer validation "
    "— evaluation #1"
)

print(
    "  Rows:",
    official_result["rows"],
)

print(
    "  Accuracy:",
    f"{official_result['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{official_result['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    official_result[
        "invalid_count"
    ],
)


print("\nPer-class F1")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{official_result['per_class'][label]['f1']:.4f}"
    )


print("\nValidation integrity")

print(
    "  Modified:",
    False,
)

print(
    "  SHA-256:",
    validation_hash_after,
)


print("\nSaved")

print(
    " ",
    validation_predictions_path,
)

print(
    " ",
    validation_metrics_path,
)

print(
    " ",
    results_path,
)


print(
    "\nNo training occurred during this cell."
)

print(
    "Official validation has now been "
    "evaluated exactly once in our controlled workflow."
)

In [ ]:
# ============================================================
# STEP 9 — BASELINE A1 OFFICIAL-VALIDATION ERROR ANALYSIS
#
# Reads:
#   - saved official-validation predictions
#   - immutable organizer validation.jsonl
#
# Produces:
#   - confusion-pair summary
#   - heuristic reasoning/error categories
#   - category × confusion analysis
#   - concise representative examples
#   - derived CSV/JSON reports
#
# NO training.
# NO validation modification.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import re
import unicodedata

import pandas as pd


# ------------------------------------------------------------
# A. Frozen paths / hashes
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

RUN_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

PREDICTIONS_PATH = (
    RUN_DIR
    / "official_validation_eval1_epoch2.csv"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

ERROR_CSV_PATH = (
    RUN_DIR
    / "official_validation_eval1_error_analysis.csv"
)

ERROR_JSON_PATH = (
    RUN_DIR
    / "official_validation_eval1_error_analysis.json"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(filename):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):
            matches.append(path)

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one {filename} "
            f"with organizer hash. Found: {matches}"
        )

    return matches[0]


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def normalize_text(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def combined_text(row):

    return normalize_text(
        row["claim"]
        + " "
        + " ".join(
            row["evidence"]
        )
    ).lower()


def contains_pattern(
    text,
    patterns,
):

    return any(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
        for pattern in patterns
    )


def short(text, n=180):

    text = normalize_text(text)

    if len(text) <= n:
        return text

    return (
        text[:n - 3]
        + "..."
    )


# ------------------------------------------------------------
# C. Load immutable organizer validation + predictions
# ------------------------------------------------------------

assert PREDICTIONS_PATH.exists(), (
    f"Missing prediction file: {PREDICTIONS_PATH}"
)

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = sha256_file(
    VAL_PATH
)

assert (
    validation_hash_before
    == EXPECTED_VAL_SHA256
)

validation_rows = read_jsonl(
    VAL_PATH
)

assert len(validation_rows) == 300


predictions = pd.read_csv(
    PREDICTIONS_PATH
)

required_prediction_columns = {
    "id",
    "gold",
    "prediction",
    "correct",
    "raw_output",
}

assert required_prediction_columns.issubset(
    predictions.columns
)

assert len(predictions) == 300

assert predictions["id"].is_unique


validation_by_id = {
    row["id"]: row
    for row in validation_rows
}

assert set(
    predictions["id"]
) == set(
    validation_by_id
)


# ------------------------------------------------------------
# D. Build error records
# ------------------------------------------------------------

errors = []


for _, pred_row in predictions.iterrows():

    if bool(pred_row["correct"]):
        continue

    example_id = pred_row["id"]

    source = validation_by_id[
        example_id
    ]

    gold = pred_row["gold"]
    prediction = pred_row["prediction"]

    record = {
        "id": example_id,
        "gold": gold,
        "prediction": prediction,
        "confusion": (
            f"{gold} -> {prediction}"
        ),
        "claim": source["claim"],
        "evidence": source["evidence"],
        "passage_count": len(
            source["evidence"]
        ),
        "raw_output": pred_row[
            "raw_output"
        ],
    }

    errors.append(record)


assert len(errors) == 54, (
    f"Expected 54 errors from Baseline A1, "
    f"found {len(errors)}"
)


# ------------------------------------------------------------
# E. Heuristic category definitions
#
# IMPORTANT:
# Categories may overlap.
# They are diagnostic tags, not ground-truth causes.
# ------------------------------------------------------------

MONTHS = (
    "january|february|march|april|may|june|"
    "july|august|september|october|november|december"
)


CATEGORY_PATTERNS = {

    "date_or_time": [
        rf"\b({MONTHS})\b",
        r"\b20\d{2}\b",
        r"\b\d{1,2}\s+("
        + MONTHS
        + r")\b",
        r"\b(before|after|during|ended|began|effective date|term)\b",
    ],

    "exact_number_or_quantity": [
        r"\bexactly\b",
        r"\b\d+(?:\.\d+)?\b",
        r"\b(percent|percentage|%|employees?|people|parcels?|"
        r"megalitres?|litres?|units?|tonnes?|kilometres?|km|"
        r"meters?|metres?)\b",
    ],

    "comparison_or_ordering": [
        r"\b(more than|less than|greater than|fewer than|"
        r"higher than|lower than|larger than|smaller than|"
        r"before|after|earlier|later|minimum|maximum|"
        r"highest|lowest|increased|decreased|exceeded)\b",
    ],

    "range_or_boundary": [
        r"\bbetween\b",
        r"\bfrom\b.*\bto\b",
        r"\bat least\b",
        r"\bat most\b",
        r"\bminimum\b",
        r"\bmaximum\b",
        r"\brange\b",
        r"\bmargin of error\b",
        r"\bbounds?\b",
    ],

    "negation_or_polarity": [
        r"\bnot\b",
        r"\bno\b",
        r"\bnever\b",
        r"\bneither\b",
        r"\bwithout\b",
        r"\bfailed\b",
        r"\bfail\b",
        r"\bpassed\b",
        r"\bpass\b",
        r"\bfalse\b",
    ],

    "entity_role_or_attribute": [
        r"\b(director|manager|chair|leader|head|officer|"
        r"unit|division|department|site|station|observatory|"
        r"authority|works|archive|project|mission|facility|"
        r"depot|service|library|counter)\b",
    ],

    "completeness_or_closed_world": [
        r"\bonly\b",
        r"\bcomplete\b",
        r"\ball other\b",
        r"\bevery\b",
        r"\bthe only\b",
        r"\bcomplete register\b",
        r"\bcomplete list\b",
        r"\bthese were the only\b",
    ],

    "distractor_or_boilerplate": [
        r"archive was digitized after a routine records review",
        r"duplicate paper copy",
        r"municipal archive",
        r"index groups related entries",
        r"routine records review",
        r"retained by the .* archive",
    ],
}


# ------------------------------------------------------------
# F. Additional structural heuristics
# ------------------------------------------------------------

def lexical_tokens(text):

    return {
        token
        for token in re.findall(
            r"[a-z0-9]+",
            text.lower(),
        )
        if len(token) >= 3
    }


def multi_passage_signal(
    claim,
    evidence,
):

    claim_tokens = lexical_tokens(
        claim
    )

    if not claim_tokens:
        return False

    passage_overlaps = []

    for passage in evidence:

        passage_tokens = lexical_tokens(
            passage
        )

        overlap = len(
            claim_tokens
            & passage_tokens
        )

        passage_overlaps.append(
            overlap
        )

    positive_passages = sum(
        overlap >= 2
        for overlap in passage_overlaps
    )

    return positive_passages >= 2


def low_single_passage_overlap(
    claim,
    evidence,
):

    claim_tokens = lexical_tokens(
        claim
    )

    if not claim_tokens:
        return False

    best_fraction = 0.0

    for passage in evidence:

        passage_tokens = lexical_tokens(
            passage
        )

        overlap = len(
            claim_tokens
            & passage_tokens
        )

        fraction = (
            overlap
            / max(
                1,
                len(claim_tokens),
            )
        )

        best_fraction = max(
            best_fraction,
            fraction,
        )

    return best_fraction < 0.35


# ------------------------------------------------------------
# G. Assign diagnostic categories
# ------------------------------------------------------------

category_counts = Counter()

confusion_category_counts = defaultdict(
    Counter
)


for record in errors:

    text = combined_text(
        record
    )

    categories = []


    for category, patterns in (
        CATEGORY_PATTERNS.items()
    ):

        if contains_pattern(
            text,
            patterns,
        ):

            categories.append(
                category
            )


    if record["passage_count"] >= 4:

        categories.append(
            "four_plus_passages"
        )


    if multi_passage_signal(
        record["claim"],
        record["evidence"],
    ):

        categories.append(
            "multi_passage_reasoning"
        )


    if low_single_passage_overlap(
        record["claim"],
        record["evidence"],
    ):

        categories.append(
            "low_direct_lexical_overlap"
        )


    if not categories:

        categories.append(
            "other_uncategorized"
        )


    # Deduplicate while preserving order.
    categories = list(
        dict.fromkeys(
            categories
        )
    )


    record["categories"] = (
        categories
    )


    for category in categories:

        category_counts[
            category
        ] += 1

        confusion_category_counts[
            record["confusion"]
        ][category] += 1


# ------------------------------------------------------------
# H. Confusion-pair summary
# ------------------------------------------------------------

confusion_counts = Counter(
    record["confusion"]
    for record in errors
)


# ------------------------------------------------------------
# I. Build compact derived dataframe
# ------------------------------------------------------------

error_df = pd.DataFrame([
    {
        "id": record["id"],
        "gold": record["gold"],
        "prediction": (
            record["prediction"]
        ),
        "confusion": (
            record["confusion"]
        ),
        "categories": (
            " | ".join(
                record["categories"]
            )
        ),
        "passage_count": (
            record["passage_count"]
        ),
        "claim": (
            normalize_text(
                record["claim"]
            )
        ),
        "evidence": (
            " || ".join(
                normalize_text(p)
                for p in record["evidence"]
            )
        ),
    }
    for record in errors
])


error_df.to_csv(
    ERROR_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# J. Create structured JSON report
# ------------------------------------------------------------

report = {
    "experiment": (
        "baseline_a1_gemma2_2b"
    ),
    "checkpoint": "epoch_2",
    "official_validation_evaluation": 1,
    "validation_sha256": (
        EXPECTED_VAL_SHA256
    ),
    "total_validation_rows": 300,
    "total_errors": len(errors),
    "error_rate": (
        len(errors) / 300
    ),
    "confusion_counts": dict(
        confusion_counts
    ),
    "category_counts": dict(
        category_counts.most_common()
    ),
    "confusion_category_counts": {
        confusion: dict(
            counts.most_common()
        )
        for confusion, counts in (
            confusion_category_counts.items()
        )
    },
    "note": (
        "Categories are overlapping heuristic diagnostic "
        "tags and must not be treated as ground-truth "
        "causes of model failure."
    ),
}


ERROR_JSON_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# K. Concise report
# ------------------------------------------------------------

print("=" * 78)
print("BASELINE A1 — ERROR ANALYSIS")
print("=" * 78)


print("\nValidation")

print(
    "  Rows:",
    300,
)

print(
    "  Correct:",
    300 - len(errors),
)

print(
    "  Errors:",
    len(errors),
)

print(
    "  Error rate:",
    f"{len(errors) / 300:.2%}",
)


print("\nConfusion pairs")

for confusion, count in (
    confusion_counts.most_common()
):

    print(
        f"  {confusion:40} "
        f"{count:2d}"
    )


print("\nHeuristic error categories")
print(
    "  NOTE: categories overlap; "
    "counts do not sum to 54."
)

for category, count in (
    category_counts.most_common()
):

    print(
        f"  {category:32} "
        f"{count:2d} / 54 "
        f"({count / 54:.1%})"
    )


# ------------------------------------------------------------
# L. Category breakdown by confusion direction
# ------------------------------------------------------------

print("\n" + "-" * 78)
print("CATEGORY BREAKDOWN BY CONFUSION")
print("-" * 78)


for confusion, total in (
    confusion_counts.most_common()
):

    print(
        f"\n{confusion} "
        f"(n={total})"
    )

    counts = (
        confusion_category_counts[
            confusion
        ]
    )

    for category, count in (
        counts.most_common()
    ):

        print(
            f"  {category:32} "
            f"{count:2d} "
            f"({count / total:.1%})"
        )


# ------------------------------------------------------------
# M. Representative examples per major category
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("REPRESENTATIVE ERROR EXAMPLES")
print("=" * 78)


MAJOR_CATEGORY_LIMIT = 10
EXAMPLES_PER_CATEGORY = 3


major_categories = [
    category
    for category, _ in (
        category_counts.most_common(
            MAJOR_CATEGORY_LIMIT
        )
    )
]


for category in major_categories:

    matching = [
        record
        for record in errors
        if category in record[
            "categories"
        ]
    ]


    # Prefer examples from different confusion directions.
    selected = []
    seen_confusions = set()


    for record in matching:

        if (
            record["confusion"]
            not in seen_confusions
        ):

            selected.append(
                record
            )

            seen_confusions.add(
                record["confusion"]
            )

        if (
            len(selected)
            >= EXAMPLES_PER_CATEGORY
        ):
            break


    if (
        len(selected)
        < EXAMPLES_PER_CATEGORY
    ):

        for record in matching:

            if record not in selected:

                selected.append(
                    record
                )

            if (
                len(selected)
                >= EXAMPLES_PER_CATEGORY
            ):
                break


    print(
        "\n" + "-" * 78
    )

    print(
        f"{category} "
        f"(n={len(matching)})"
    )

    print(
        "-" * 78
    )


    for record in selected:

        print(
            "\nID:",
            record["id"],
        )

        print(
            "Gold -> Pred:",
            record["confusion"],
        )

        print(
            "Claim:",
            short(
                record["claim"],
                220,
            ),
        )

        print(
            "Evidence:"
        )

        for i, passage in enumerate(
            record["evidence"],
            1,
        ):

            print(
                f"  [{i}]",
                short(
                    passage,
                    190,
                ),
            )


# ------------------------------------------------------------
# N. Focus specifically on decision-boundary failures
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("DECISION-BOUNDARY SUMMARY")
print("=" * 78)


boundary_groups = {
    "SUPPORTS_vs_REFUTES": {
        "SUPPORTS -> REFUTES",
        "REFUTES -> SUPPORTS",
    },

    "REFUTES_vs_NEI": {
        "REFUTES -> NOT_ENOUGH_INFO",
        "NOT_ENOUGH_INFO -> REFUTES",
    },

    "SUPPORTS_vs_NEI": {
        "SUPPORTS -> NOT_ENOUGH_INFO",
        "NOT_ENOUGH_INFO -> SUPPORTS",
    },
}


boundary_summary = {}


for boundary_name, directions in (
    boundary_groups.items()
):

    subset = [
        record
        for record in errors
        if record["confusion"]
        in directions
    ]

    counts = Counter()

    for record in subset:

        counts.update(
            record["categories"]
        )


    boundary_summary[
        boundary_name
    ] = {
        "errors": len(subset),
        "top_categories": (
            counts.most_common(8)
        ),
    }


    print(
        f"\n{boundary_name}"
    )

    print(
        "  Errors:",
        len(subset),
    )

    print(
        "  Top categories:"
    )

    for category, count in (
        counts.most_common(8)
    ):

        print(
            f"    {category:30} "
            f"{count:2d}"
        )


# ------------------------------------------------------------
# O. Final integrity check
# ------------------------------------------------------------

validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


print("\n" + "=" * 78)
print("ERROR ANALYSIS — RESULT")
print("=" * 78)

print(
    "\nValidation rows analyzed:",
    300,
)

print(
    "Errors analyzed:",
    len(errors),
)

print(
    "Validation modified:",
    False,
)

print(
    "Validation SHA-256:",
    validation_hash_after,
)


print("\nSaved derived reports")

print(
    " ",
    ERROR_CSV_PATH,
)

print(
    " ",
    ERROR_JSON_PATH,
)


print(
    "\nNo training occurred."
)

print(
    "No augmentation was created."
)

print(
    "These categories are diagnostic only; "
    "we will use them to define A2 before "
    "generating any synthetic data."
)

In [ ]:
# ============================================================
# STEP 9B — REFINED TASK-FAMILY ERROR ANALYSIS
#
# Goal:
#   Replace overly broad number/date heuristics with meaningful
#   claim-verification task families.
#
# Compares:
#   - Baseline A1 performance on each validation family
#   - Training representation of each family
#   - confusion directions within each family
#
# NO training.
# NO augmentation.
# NO organizer-validation modification.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import re

import pandas as pd


# ------------------------------------------------------------
# A. Frozen inputs
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

RUN_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

PRED_PATH = (
    RUN_DIR
    / "official_validation_eval1_epoch2.csv"
)

FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

OUT_CSV = (
    RUN_DIR
    / "official_validation_eval1_task_family_analysis.csv"
)

OUT_JSON = (
    RUN_DIR
    / "official_validation_eval1_task_family_analysis.json"
)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))

            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(
        filename
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):

            matches.append(path)


    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one {filename} "
            f"with frozen organizer hash. "
            f"Found: {matches}"
        )

    return matches[0]


def clean(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def joined_evidence(row):

    return clean(
        " ".join(
            row.get(
                "evidence",
                [],
            )
        )
    ).lower()


# ------------------------------------------------------------
# C. Meaningful task-family classifier
#
# Classification is primarily based on CLAIM semantics,
# not merely the presence of a year or arbitrary number.
#
# Families are mutually exclusive.
# ------------------------------------------------------------

def classify_family(row):

    claim = clean(
        row["claim"]
    ).lower()

    evidence = joined_evidence(
        row
    )


    # --------------------------------------------------------
    # Revenue / percentage calculation
    # --------------------------------------------------------

    if (
        "year-on-year revenue growth"
        in claim
        or (
            "revenue growth"
            in claim
            and "%"
            in claim
        )
    ):

        return "revenue_percentage_calculation"


    # --------------------------------------------------------
    # Sum of two sites / components
    # --------------------------------------------------------

    if (
        "two named sites jointly made"
        in claim
        or (
            "jointly"
            in claim
            and "units"
            in claim
        )
    ):

        return "two_component_sum"


    # --------------------------------------------------------
    # Depot comparison
    # --------------------------------------------------------

    if (
        "depot was busier than"
        in claim
        or (
            "depot"
            in claim
            and (
                "more parcels than"
                in claim
                or "busier"
                in claim
            )
        )
    ):

        return "paired_comparison"


    # --------------------------------------------------------
    # Interval / sensor bound reasoning
    # --------------------------------------------------------

    if (
        "reached a minimum of"
        in claim
        or "reached a maximum of"
        in claim
        or (
            "megalitres"
            in claim
            and (
                "between"
                in evidence
                or "bounds"
                in evidence
                or "margin of error"
                in evidence
            )
        )
    ):

        return "range_or_measurement_bounds"


    # --------------------------------------------------------
    # Office holder / leadership on a date
    # --------------------------------------------------------

    if (
        "office-holder directing"
        in claim
        or (
            "director of"
            in claim
            and re.search(
                r"\bon\s+\d{1,2}\s+"
                r"(january|february|march|april|may|june|"
                r"july|august|september|october|november|december)",
                claim,
            )
        )
    ):

        return "temporal_office_holder"


    # --------------------------------------------------------
    # Project / site / geography mapping
    # --------------------------------------------------------

    if (
        "project"
        in claim
        and (
            "based in"
            in claim
            or "carried out"
            in claim
            or "located in"
            in claim
        )
    ):

        return "project_location_mapping"


    # --------------------------------------------------------
    # Mission payload / equipment presence or omission
    # --------------------------------------------------------

    if (
        "mission payload"
        in claim
        or "left out the"
        in claim
        or (
            (
                "magnetometer"
                in claim
                or "particle detector"
                in claim
                or "laser altimeter"
                in claim
                or "radio beacon"
                in claim
                or "radar mapper"
                in claim
            )
            and (
                "mission"
                in claim
                or "aboard"
                in claim
            )
        )
    ):

        return "complete_manifest_membership"


    # --------------------------------------------------------
    # Audit pass / fail status
    # --------------------------------------------------------

    if (
        "safety audit"
        in claim
        or (
            "audit"
            in claim
            and (
                "failed"
                in claim
                or "passed"
                in claim
            )
        )
    ):

        return "audit_status"


    # --------------------------------------------------------
    # Exact employee / staffing count
    # --------------------------------------------------------

    if (
        "employed exactly"
        in claim
        or (
            "employees"
            in claim
            and "exactly"
            in claim
        )
    ):

        return "exact_staffing_count"


    # --------------------------------------------------------
    # Service availability / full-year status
    # --------------------------------------------------------

    if (
        "full-year service"
        in claim
        or (
            "service"
            in claim
            and (
                "seasonal"
                in claim
                or "every month"
                in claim
            )
        )
    ):

        return "service_availability"


    # --------------------------------------------------------
    # Generic exact-value task
    # --------------------------------------------------------

    if "exactly" in claim:

        return "other_exact_value"


    # --------------------------------------------------------
    # Generic comparative task
    # --------------------------------------------------------

    if re.search(
        r"\b("
        r"more than|less than|greater than|fewer than|"
        r"higher than|lower than|earlier than|later than"
        r")\b",
        claim,
    ):

        return "other_comparison"


    return "other"


# ------------------------------------------------------------
# D. Reasoning primitives
#
# These may overlap and complement the exclusive task family.
# ------------------------------------------------------------

def reasoning_tags(row):

    claim = clean(
        row["claim"]
    ).lower()

    evidence = joined_evidence(
        row
    )

    tags = []


    # arithmetic
    if (
        "year-on-year revenue growth"
        in claim
        or "jointly made"
        in claim
        or re.search(
            r"\b(total|sum|percentage|percent)\b",
            claim,
        )
    ):

        tags.append(
            "arithmetic"
        )


    # comparison
    if re.search(
        r"\b("
        r"more than|less than|greater than|fewer than|"
        r"higher than|lower than|busier than|"
        r"minimum|maximum"
        r")\b",
        claim,
    ):

        tags.append(
            "comparison_or_threshold"
        )


    # temporal transition
    if (
        "appointment took effect"
        in evidence
        or "effective date"
        in evidence
        or "term ended"
        in evidence
    ):

        tags.append(
            "temporal_transition"
        )


    # range uncertainty
    if (
        "between"
        in evidence
        or "bounds"
        in evidence
        or "margin of error"
        in evidence
    ):

        tags.append(
            "range_uncertainty"
        )


    # closed-world / complete list
    if (
        "manifest is marked complete"
        in evidence
        or "only three"
        in evidence
        or "complete register"
        in evidence
        or "these were the only"
        in evidence
    ):

        tags.append(
            "closed_world_completion"
        )


    # distractor boilerplate
    boilerplate = [
        "archive was digitized after a routine records review",
        "duplicate paper copy",
        "municipal archive",
        "independent editors checked the transcription",
        "index groups related entries",
        "report uses the naming convention",
    ]

    if any(
        phrase in evidence
        for phrase in boilerplate
    ):

        tags.append(
            "distractor_passage"
        )


    # multi-passage synthesis
    if len(
        row["evidence"]
    ) >= 3:

        tags.append(
            "multi_passage"
        )


    return tags


# ------------------------------------------------------------
# E. Load frozen data
# ------------------------------------------------------------

assert FIT_PATH.exists()
assert PRED_PATH.exists()

assert (
    sha256_file(FIT_PATH)
    == EXPECTED_FIT_SHA256
)

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = (
    sha256_file(
        VAL_PATH
    )
)

fit_rows = read_jsonl(
    FIT_PATH
)

val_rows = read_jsonl(
    VAL_PATH
)

predictions = pd.read_csv(
    PRED_PATH
)

assert len(fit_rows) == 795
assert len(val_rows) == 300
assert len(predictions) == 300

assert predictions["id"].is_unique


val_by_id = {
    row["id"]: row
    for row in val_rows
}

pred_by_id = {
    row["id"]: row
    for row in predictions.to_dict(
        orient="records"
    )
}

assert (
    set(val_by_id)
    == set(pred_by_id)
)


# ------------------------------------------------------------
# F. Classify TRAIN representation
# ------------------------------------------------------------

fit_family_counts = Counter()

fit_family_label_counts = defaultdict(
    Counter
)

fit_reasoning_counts = Counter()


for row in fit_rows:

    family = classify_family(
        row
    )

    fit_family_counts[
        family
    ] += 1

    fit_family_label_counts[
        family
    ][row["label"]] += 1

    fit_reasoning_counts.update(
        reasoning_tags(row)
    )


# ------------------------------------------------------------
# G. Classify all VALIDATION examples + performance
# ------------------------------------------------------------

validation_records = []

family_stats = defaultdict(
    lambda: {
        "total": 0,
        "correct": 0,
        "errors": 0,
        "gold": Counter(),
        "pred": Counter(),
        "confusions": Counter(),
    }
)

reasoning_stats = defaultdict(
    lambda: {
        "total": 0,
        "errors": 0,
    }
)


for row in val_rows:

    prediction_row = (
        pred_by_id[
            row["id"]
        ]
    )

    gold = row["label"]

    prediction = (
        prediction_row[
            "prediction"
        ]
    )

    correct = (
        gold
        == prediction
    )

    family = classify_family(
        row
    )

    tags = reasoning_tags(
        row
    )


    stats = family_stats[
        family
    ]

    stats["total"] += 1

    stats["gold"][
        gold
    ] += 1

    stats["pred"][
        prediction
    ] += 1


    if correct:

        stats["correct"] += 1

    else:

        stats["errors"] += 1

        stats["confusions"][
            f"{gold} -> {prediction}"
        ] += 1


    for tag in tags:

        reasoning_stats[
            tag
        ]["total"] += 1

        if not correct:

            reasoning_stats[
                tag
            ]["errors"] += 1


    validation_records.append({
        "id": row["id"],
        "family": family,
        "reasoning_tags": (
            " | ".join(tags)
        ),
        "gold": gold,
        "prediction": prediction,
        "correct": correct,
        "claim": clean(
            row["claim"]
        ),
        "evidence": " || ".join(
            clean(passage)
            for passage in row[
                "evidence"
            ]
        ),
    })


# ------------------------------------------------------------
# H. Compute family metrics
# ------------------------------------------------------------

family_table = []


for family in sorted(
    family_stats
):

    stats = family_stats[
        family
    ]

    validation_total = (
        stats["total"]
    )

    errors = stats["errors"]

    error_rate = (
        errors
        / validation_total
    )

    train_count = (
        fit_family_counts[
            family
        ]
    )


    family_table.append({
        "family": family,
        "train_examples": (
            train_count
        ),
        "validation_examples": (
            validation_total
        ),
        "validation_correct": (
            stats["correct"]
        ),
        "validation_errors": (
            errors
        ),
        "validation_error_rate": (
            error_rate
        ),
        "train_per_validation_example": (
            train_count
            / validation_total
        ),
        "confusions": dict(
            stats["confusions"]
        ),
        "train_label_distribution": dict(
            fit_family_label_counts[
                family
            ]
        ),
    })


family_table = sorted(
    family_table,
    key=lambda item: (
        item[
            "validation_error_rate"
        ],
        item[
            "validation_errors"
        ],
    ),
    reverse=True,
)


# ------------------------------------------------------------
# I. Augmentation-priority heuristic
#
# This does NOT decide how many examples to generate.
# It simply identifies:
#   - enough validation examples to be meaningful
#   - elevated error rate
#   - limited training representation
# ------------------------------------------------------------

priority_candidates = []


for item in family_table:

    if (
        item[
            "validation_examples"
        ] >= 5
        and item[
            "validation_errors"
        ] >= 3
    ):

        # Higher = potentially more useful augmentation target.
        score = (
            item[
                "validation_error_rate"
            ]
            * item[
                "validation_errors"
            ]
            / max(
                1,
                math.sqrt(
                    item[
                        "train_examples"
                    ]
                ),
            )
        )

        priority_candidates.append({
            **item,
            "priority_score": score,
        })


priority_candidates.sort(
    key=lambda item: item[
        "priority_score"
    ],
    reverse=True,
)


# ------------------------------------------------------------
# J. Reasoning primitive error rates
# ------------------------------------------------------------

reasoning_table = []


for tag, stats in (
    reasoning_stats.items()
):

    total = stats[
        "total"
    ]

    errors = stats[
        "errors"
    ]

    reasoning_table.append({
        "tag": tag,
        "validation_examples": (
            total
        ),
        "validation_errors": (
            errors
        ),
        "error_rate": (
            errors / total
        ),
        "fit_examples_with_tag": (
            fit_reasoning_counts[
                tag
            ]
        ),
    })


reasoning_table.sort(
    key=lambda item: (
        item["error_rate"],
        item["validation_errors"],
    ),
    reverse=True,
)


# ------------------------------------------------------------
# K. Save detailed reports
# ------------------------------------------------------------

analysis_df = pd.DataFrame(
    validation_records
)

analysis_df.to_csv(
    OUT_CSV,
    index=False,
)


report = {
    "experiment": (
        "baseline_a1_gemma2_2b"
    ),
    "checkpoint": "epoch_2",
    "official_validation_evaluation": 1,
    "family_analysis": family_table,
    "reasoning_analysis": reasoning_table,
    "augmentation_priority_candidates": (
        priority_candidates
    ),
    "notes": [
        (
            "Task families are diagnostic categories "
            "derived primarily from claim semantics."
        ),
        (
            "No augmentation decision should be based "
            "on an individual validation ID."
        ),
        (
            "Validation remains unchanged."
        ),
    ],
}


OUT_JSON.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# L. Console report
# ------------------------------------------------------------

print("=" * 78)
print("REFINED TASK-FAMILY ANALYSIS")
print("=" * 78)


print("\nOverall")

print(
    "  Fit examples:",
    len(fit_rows),
)

print(
    "  Validation examples:",
    len(val_rows),
)

print(
    "  Validation errors:",
    sum(
        not record["correct"]
        for record in validation_records
    ),
)


print("\n" + "-" * 78)
print("TASK-FAMILY PERFORMANCE")
print("-" * 78)

print(
    f"{'Family':34} "
    f"{'Train':>6} "
    f"{'Val':>5} "
    f"{'Err':>5} "
    f"{'ErrRate':>9}"
)


for item in family_table:

    print(
        f"{item['family']:34} "
        f"{item['train_examples']:6d} "
        f"{item['validation_examples']:5d} "
        f"{item['validation_errors']:5d} "
        f"{item['validation_error_rate']:8.1%}"
    )


print("\n" + "-" * 78)
print("CONFUSIONS WITHIN EACH FAMILY")
print("-" * 78)


for item in family_table:

    if not item[
        "validation_errors"
    ]:
        continue

    print(
        f"\n{item['family']}"
    )

    print(
        "  train labels:",
        item[
            "train_label_distribution"
        ],
    )

    for confusion, count in sorted(
        item["confusions"].items(),
        key=lambda pair: pair[1],
        reverse=True,
    ):

        print(
            f"  {confusion:38} "
            f"{count}"
        )


print("\n" + "-" * 78)
print("REASONING-PRIMITIVE PERFORMANCE")
print("-" * 78)


for item in reasoning_table:

    print(
        f"{item['tag']:28} | "
        f"fit={item['fit_examples_with_tag']:3d} | "
        f"val={item['validation_examples']:3d} | "
        f"errors={item['validation_errors']:2d} | "
        f"error_rate={item['error_rate']:.1%}"
    )


print("\n" + "=" * 78)
print("POTENTIAL A2 AUGMENTATION PRIORITIES")
print("=" * 78)


if not priority_candidates:

    print(
        "No family met the minimum "
        "priority criteria."
    )

else:

    for rank, item in enumerate(
        priority_candidates[:10],
        1,
    ):

        print(
            f"\n#{rank} {item['family']}"
        )

        print(
            "  Fit examples:",
            item[
                "train_examples"
            ],
        )

        print(
            "  Validation examples:",
            item[
                "validation_examples"
            ],
        )

        print(
            "  Validation errors:",
            item[
                "validation_errors"
            ],
        )

        print(
            "  Error rate:",
            f"{item['validation_error_rate']:.1%}",
        )

        print(
            "  Training labels:",
            item[
                "train_label_distribution"
            ],
        )

        print(
            "  Main confusions:",
            item[
                "confusions"
            ],
        )

        print(
            "  Diagnostic priority score:",
            round(
                item[
                    "priority_score"
                ],
                4,
            ),
        )


# ------------------------------------------------------------
# M. Representative WRONG examples by top family
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("REPRESENTATIVE ERRORS BY TOP TASK FAMILY")
print("=" * 78)


top_families = [
    item["family"]
    for item in family_table
    if item[
        "validation_errors"
    ] > 0
][:8]


for family in top_families:

    family_errors = [
        record
        for record in validation_records
        if (
            record["family"]
            == family
            and not record[
                "correct"
            ]
        )
    ]


    print(
        "\n" + "-" * 78
    )

    print(
        family,
        f"(errors={len(family_errors)})",
    )

    print(
        "-" * 78
    )


    seen_confusions = set()
    shown = 0


    for record in family_errors:

        confusion = (
            f"{record['gold']} -> "
            f"{record['prediction']}"
        )

        if (
            confusion
            in seen_confusions
            and shown >= 2
        ):
            continue


        print(
            "\nID:",
            record["id"],
        )

        print(
            "Gold -> Pred:",
            confusion,
        )

        print(
            "Claim:",
            record["claim"],
        )

        print(
            "Reasoning:",
            record[
                "reasoning_tags"
            ],
        )

        seen_confusions.add(
            confusion
        )

        shown += 1


        if shown >= 3:
            break


# ------------------------------------------------------------
# N. Integrity
# ------------------------------------------------------------

validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


print("\n" + "=" * 78)
print("REFINED ANALYSIS — RESULT")
print("=" * 78)

print(
    "Validation modified:",
    False,
)

print(
    "Validation SHA-256:",
    validation_hash_after,
)

print(
    "\nSaved:",
    OUT_CSV,
)

print(
    "Saved:",
    OUT_JSON,
)

print(
    "\nNo training occurred."
)

print(
    "No synthetic examples were created."
)

In [ ]:
# ============================================================
# STEP 10 — CREATE CONTROLLED A2 SYNTHETIC AUGMENTATION
#
# A2 changes ONE principal variable:
#   training data only
#
# Frozen:
#   - Gemma 2 2B IT
#   - Prompt V1
#   - max_length=256
#   - LoRA / optimizer / LR / epochs
#   - internal dev set
#   - organizer validation
#
# Synthetic TRAIN additions:
#   revenue percentage       60 = 20/class
#   two-component sum        45 = 15/class
#   temporal office holder   36 = 12/class
#   paired comparison        24 =  8/class
#   measurement bounds       15 =  5/class
#   --------------------------------------
#   TOTAL                   180 = 60/class
#
# Separate synthetic sanity set:
#   60 examples, NEVER used for training.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import random
import re

from transformers import AutoTokenizer


# ------------------------------------------------------------
# A. Frozen paths
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

BASE_FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

A1_ADAPTER_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
    / "epoch_2_adapter"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

SYNTH_TRAIN_PATH = (
    WORK_ROOT
    / "synthetic_a2_v1_train.jsonl"
)

SYNTH_HOLDOUT_PATH = (
    WORK_ROOT
    / "synthetic_a2_v1_holdout.jsonl"
)

AUGMENTED_FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2_plus_a2_v1.jsonl"
)

MANIFEST_PATH = (
    WORK_ROOT
    / "synthetic_a2_v1_manifest.json"
)

SEED = 20260827

LABELS = (
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
)

MAX_LENGTH = 256


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(
                    json.loads(line)
                )

            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def write_jsonl(path, rows):

    with Path(path).open(
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(
        filename
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):

            matches.append(path)

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected one {filename} with "
            f"organizer hash. Found: {matches}"
        )

    return matches[0]


def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def content_fingerprint(row):

    payload = {
        "claim": normalize(
            row["claim"]
        ),
        "evidence": sorted(
            normalize(p)
            for p in row["evidence"]
        ),
    }

    encoded = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


# ------------------------------------------------------------
# C. Verify immutable source data
# ------------------------------------------------------------

assert (
    sha256_file(BASE_FIT_PATH)
    == EXPECTED_FIT_SHA256
)

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = (
    sha256_file(VAL_PATH)
)

base_fit = read_jsonl(
    BASE_FIT_PATH
)

internal_dev = read_jsonl(
    DEV_PATH
)

organizer_validation = read_jsonl(
    VAL_PATH
)

assert len(base_fit) == 795
assert len(internal_dev) == 140
assert len(organizer_validation) == 300


# ------------------------------------------------------------
# D. Deterministic generation state
# ------------------------------------------------------------

rng = random.Random(SEED)

synthetic_train = []
synthetic_holdout = []

metadata = {}

serial = 0


DISTRACTORS = [
    (
        "A duplicate paper copy is retained "
        "in the regional records archive."
    ),
    (
        "Independent editors checked the "
        "transcription before publication."
    ),
    (
        "The index groups related entries "
        "by reporting period."
    ),
    (
        "The archive was digitized after "
        "a routine records review."
    ),
    (
        "The register follows the naming "
        "convention used in the current edition."
    ),
]


ADJECTIVES = [
    "Silver",
    "Cedar",
    "Juniper",
    "Willow",
    "Amber",
    "Granite",
    "Linden",
    "Maple",
    "Coral",
    "Birch",
    "Saffron",
    "Indigo",
    "Copper",
    "Hazel",
    "Raven",
    "Aspen",
]


FIRST_NAMES = [
    "Amina",
    "Owen",
    "Mira",
    "Kamal",
    "Leena",
    "Jonah",
    "Nadia",
    "Tariq",
    "Elena",
    "Ravi",
    "Maya",
    "Idris",
    "Sofia",
    "Noah",
    "Priya",
    "Elias",
]


LAST_NAMES = [
    "Vale",
    "Hart",
    "Chen",
    "Brooks",
    "Sayeed",
    "Morgan",
    "Farrow",
    "Nordin",
    "Diaz",
    "Reed",
    "Khan",
    "Ibrahim",
    "Voss",
    "Shaw",
    "Nolan",
    "Okafor",
]


MONTHS = [
    "February",
    "March",
    "April",
    "May",
    "June",
    "September",
    "October",
    "November",
]


def new_id(
    family,
    split,
):

    global serial

    serial += 1

    return (
        f"a2syn_{split}_"
        f"{family[:8]}_"
        f"{serial:04d}"
    )


def natural_org(
    kind,
    i,
):

    adjective = ADJECTIVES[
        i % len(ADJECTIVES)
    ]

    number = (
        200
        + (i * 7) % 700
    )

    return (
        f"{adjective} {kind} {number}"
    )


def person_pair(i):

    old = (
        FIRST_NAMES[
            i % len(FIRST_NAMES)
        ]
        + " "
        + LAST_NAMES[
            (i * 3) % len(LAST_NAMES)
        ]
    )

    new = (
        FIRST_NAMES[
            (i + 7) % len(FIRST_NAMES)
        ]
        + " "
        + LAST_NAMES[
            (i * 5 + 2)
            % len(LAST_NAMES)
        ]
    )

    if old == new:
        new += " Jr"

    return old, new


def shuffle_evidence(items):

    items = list(items)

    rng.shuffle(items)

    return items


def add_example(
    collection,
    split,
    family,
    label,
    claim,
    evidence,
    logic,
):

    example_id = new_id(
        family,
        split,
    )

    row = {
        "id": example_id,
        "claim": normalize(claim),
        "evidence": [
            normalize(p)
            for p in evidence
        ],
        "label": label,
    }

    assert label in LABELS
    assert row["claim"]
    assert len(row["evidence"]) >= 3
    assert all(
        row["evidence"]
    )

    collection.append(row)

    metadata[
        example_id
    ] = {
        "split": split,
        "family": family,
        "label": label,
        "logic": logic,
    }


# ------------------------------------------------------------
# E. Family 1 — exact revenue growth
# ------------------------------------------------------------

def make_revenue(
    i,
    label,
    split,
):

    year = 2022 + (i % 4)

    org = natural_org(
        "Research Cooperative",
        i + 10,
    )

    true_pct = [
        10,
        15,
        20,
        25,
        30,
        40,
    ][i % 6]

    base = (
        200
        + 20 * (i % 21)
    )

    current = (
        base
        * (100 + true_pct)
        // 100
    )

    # All chosen base values make
    # these percentages exact integers.
    assert (
        current * 100
        == base * (100 + true_pct)
    )

    if label == "SUPPORTS":

        claimed_pct = true_pct

        evidence = [
            (
                f"The audited statement records revenue "
                f"of {base} million credits in {year - 1}."
            ),
            (
                f"The audited statement records revenue "
                f"of {current} million credits in {year}."
            ),
            DISTRACTORS[
                i % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"({current}-{base})/{base} "
            f"= {true_pct}% exactly."
        )

    elif label == "REFUTES":

        delta = [
            5,
            10,
            15,
        ][i % 3]

        claimed_pct = (
            true_pct + delta
        )

        if claimed_pct == true_pct:
            claimed_pct += 5

        evidence = [
            (
                f"Audited revenue for {year - 1} "
                f"was {base} million credits."
            ),
            (
                f"Audited revenue for {year} "
                f"was {current} million credits."
            ),
            DISTRACTORS[
                (i + 1)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"True growth is {true_pct}%, "
            f"not {claimed_pct}%."
        )

    else:

        claimed_pct = true_pct

        evidence = [
            (
                f"The audited statement records revenue "
                f"of {base} million credits in {year - 1}."
            ),
            (
                f"The archive lists the {year} audited "
                f"revenue statement as not included in "
                f"this evidence extract."
            ),
            DISTRACTORS[
                (i + 2)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            "Current-year audited revenue is absent, "
            "so exact growth cannot be established "
            "or contradicted."
        )


    variant = (
        i
        + (0 if split == "train" else 1)
    ) % 3

    if variant == 0:

        claim = (
            f"The audited year-on-year revenue growth "
            f"at {org} was exactly {claimed_pct}% "
            f"in {year}."
        )

    elif variant == 1:

        claim = (
            f"According to audited figures, "
            f"{org}'s revenue grew by exactly "
            f"{claimed_pct}% from {year - 1} "
            f"to {year}."
        )

    else:

        claim = (
            f"{org} recorded an exact audited "
            f"revenue increase of {claimed_pct}% "
            f"in {year} compared with {year - 1}."
        )


    return (
        claim,
        shuffle_evidence(
            evidence
        ),
        logic,
    )


# ------------------------------------------------------------
# F. Family 2 — sum of two components
# ------------------------------------------------------------

def make_sum(
    i,
    label,
    split,
):

    org = natural_org(
        "Archive",
        i + 100,
    )

    year = 2023 + (i % 3)
    quarter = 1 + (i % 4)

    north = (
        70 + (i * 7) % 120
    )

    south = (
        65 + (i * 11) % 130
    )

    true_total = (
        north + south
    )

    if label == "SUPPORTS":

        claimed_total = true_total

        evidence = [
            (
                f"The North site reported {north} "
                f"units for Q{quarter} {year}."
            ),
            (
                f"The South site reported {south} "
                f"units for Q{quarter} {year}."
            ),
            DISTRACTORS[
                i % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"{north}+{south}="
            f"{true_total}."
        )

    elif label == "REFUTES":

        offset = [
            7,
            11,
            13,
            17,
        ][i % 4]

        claimed_total = (
            true_total + offset
        )

        evidence = [
            (
                f"For Q{quarter} {year}, "
                f"the North site produced "
                f"{north} units."
            ),
            (
                f"For Q{quarter} {year}, "
                f"the South site produced "
                f"{south} units."
            ),
            DISTRACTORS[
                (i + 1)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"Actual sum {true_total}; "
            f"claim says {claimed_total}."
        )

    else:

        claimed_total = true_total

        previous_year = year - 1

        evidence = [
            (
                f"The North site reported {north} "
                f"units for Q{quarter} {year}."
            ),
            (
                f"The South site reported {south} "
                f"units for Q{quarter} "
                f"{previous_year}, not {year}."
            ),
            DISTRACTORS[
                (i + 2)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            "One current-period component is missing; "
            "joint current-period total is unknown."
        )


    variant = (
        i
        + (0 if split == "train" else 2)
    ) % 3

    if variant == 0:

        claim = (
            f"{org}'s two named sites jointly "
            f"made {claimed_total} units during "
            f"quarter {quarter} of {year}."
        )

    elif variant == 1:

        claim = (
            f"Together, the North and South sites "
            f"of {org} produced exactly "
            f"{claimed_total} units in "
            f"Q{quarter} {year}."
        )

    else:

        claim = (
            f"The combined Q{quarter} {year} "
            f"output of {org}'s North and South "
            f"sites was {claimed_total} units."
        )


    return (
        claim,
        shuffle_evidence(
            evidence
        ),
        logic,
    )


# ------------------------------------------------------------
# G. Family 3 — uncertain office-holder transition
# ------------------------------------------------------------

def make_temporal(
    i,
    label,
    split,
):

    org = natural_org(
        "Institute",
        i + 200,
    )

    old_person, new_person = (
        person_pair(
            i + 20
        )
    )

    year = 2024 + (i % 2)

    month = MONTHS[
        i % len(MONTHS)
    ]

    start_day = (
        18 + (i % 3)
    )

    end_day = (
        start_day + 2
    )

    before_day = (
        start_day - 3
    )

    inside_day = (
        start_day + 1
    )

    after_day = (
        end_day + 3
    )

    use_before = (
        i % 2 == 0
    )


    if label == "SUPPORTS":

        if use_before:

            claim_day = before_day
            claim_person = old_person

            logic = (
                "Claim date is before the earliest "
                "possible transition, so old holder "
                "is definitely still in office."
            )

        else:

            claim_day = after_day
            claim_person = new_person

            logic = (
                "Claim date is after the latest "
                "possible transition, so new holder "
                "is definitely in office."
            )

    elif label == "REFUTES":

        if use_before:

            claim_day = before_day
            claim_person = new_person

            logic = (
                "Claim assigns new holder before "
                "the earliest possible effective date."
            )

        else:

            claim_day = after_day
            claim_person = old_person

            logic = (
                "Claim assigns old holder after "
                "the latest possible transition date."
            )

    else:

        claim_day = inside_day

        claim_person = (
            old_person
            if i % 2 == 0
            else new_person
        )

        logic = (
            "Claim date lies inside the unresolved "
            "effective-date interval, so either "
            "office-holder may have held the role."
        )


    evidence = [
        (
            f"{old_person}'s term ended when "
            f"{new_person}'s appointment took effect."
        ),
        (
            f"The transition resolution places the "
            f"effective date between {start_day} "
            f"and {end_day} {month} {year}."
        ),
        DISTRACTORS[
            i % len(DISTRACTORS)
        ],
    ]


    variant = (
        i
        + (0 if split == "train" else 1)
    ) % 3

    if variant == 0:

        claim = (
            f"The office-holder directing {org} "
            f"on {claim_day} {month} {year} "
            f"was {claim_person}."
        )

    elif variant == 1:

        claim = (
            f"On {claim_day} {month} {year}, "
            f"{claim_person} was the director "
            f"of {org}."
        )

    else:

        claim = (
            f"{claim_person} held the leadership "
            f"post at {org} on "
            f"{claim_day} {month} {year}."
        )


    return (
        claim,
        shuffle_evidence(
            evidence
        ),
        logic,
    )


# ------------------------------------------------------------
# H. Family 4 — paired comparison
# ------------------------------------------------------------

def make_comparison(
    i,
    label,
    split,
):

    org = natural_org(
        "Survey Office",
        i + 300,
    )

    year = 2023 + (i % 3)

    direction = (
        "East"
        if i % 2 == 0
        else "West"
    )

    other = (
        "West"
        if direction == "East"
        else "East"
    )

    high = (
        420 + (i * 9) % 160
    )

    low = (
        high - 20 - (i % 25)
    )


    if label == "SUPPORTS":

        values = {
            direction: high,
            other: low,
        }

        evidence = [
            (
                f"The {direction} depot ledger "
                f"records {values[direction]} parcels "
                f"for {year}."
            ),
            (
                f"The {other} depot ledger records "
                f"{values[other]} parcels for {year}."
            ),
            DISTRACTORS[
                i % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"{direction}={high} > "
            f"{other}={low}."
        )

    elif label == "REFUTES":

        values = {
            direction: low,
            other: high,
        }

        evidence = [
            (
                f"The {direction} depot ledger "
                f"records {values[direction]} parcels "
                f"for {year}."
            ),
            (
                f"The {other} depot ledger records "
                f"{values[other]} parcels for {year}."
            ),
            DISTRACTORS[
                (i + 1)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            f"{direction}={low} < "
            f"{other}={high}; claim is opposite."
        )

    else:

        evidence = [
            (
                f"The {direction} depot ledger "
                f"records {high} parcels for {year}."
            ),
            (
                f"The {other} depot ledger records "
                f"{low} parcels for {year - 1}; "
                f"the {year} figure is not supplied."
            ),
            DISTRACTORS[
                (i + 2)
                % len(DISTRACTORS)
            ],
        ]

        logic = (
            "One side of the same-year comparison "
            "is missing."
        )


    if (
        (i + (1 if split == "holdout" else 0))
        % 2
        == 0
    ):

        claim = (
            f"The {direction} depot was busier "
            f"than the {other} depot at {org} "
            f"during {year}."
        )

    else:

        claim = (
            f"During {year}, {org}'s "
            f"{direction} depot handled more "
            f"parcels than its {other} depot."
        )


    return (
        claim,
        shuffle_evidence(
            evidence
        ),
        logic,
    )


# ------------------------------------------------------------
# I. Family 5 — bounded measurements / thresholds
# ------------------------------------------------------------

def make_bounds(
    i,
    label,
    split,
):

    org = natural_org(
        "Transit Authority",
        i + 400,
    )

    year = 2024 + (i % 2)

    threshold = (
        700
        + 25 * (i % 12)
    )


    if label == "SUPPORTS":

        lower = (
            threshold + 10
        )

        upper = (
            lower + 40
        )

        logic = (
            f"Entire interval [{lower},{upper}] "
            f"is at least {threshold}."
        )

    elif label == "REFUTES":

        upper = (
            threshold - 10
        )

        lower = (
            upper - 40
        )

        logic = (
            f"Entire interval [{lower},{upper}] "
            f"is below {threshold}."
        )

    else:

        lower = (
            threshold - 30
        )

        upper = (
            threshold + 30
        )

        logic = (
            f"Interval [{lower},{upper}] straddles "
            f"{threshold}, so threshold claim is "
            f"neither established nor contradicted."
        )


    evidence = [
        (
            f"Instrument calibration bounds place "
            f"the 15 July reading between {lower} "
            f"and {upper} megalitres."
        ),
        (
            "The stated interval already accounts "
            "for the instrument's margin of error."
        ),
        DISTRACTORS[
            i % len(DISTRACTORS)
        ],
    ]


    if (
        (i + (1 if split == "holdout" else 0))
        % 2
        == 0
    ):

        claim = (
            f"Storage at {org} was at least "
            f"{threshold} megalitres on "
            f"15 July {year}."
        )

    else:

        claim = (
            f"Storage at {org} reached a minimum "
            f"of {threshold} megalitres on "
            f"15 July {year}."
        )


    return (
        claim,
        shuffle_evidence(
            evidence
        ),
        logic,
    )


GENERATORS = {
    "revenue_percentage_calculation": make_revenue,
    "two_component_sum": make_sum,
    "temporal_office_holder": make_temporal,
    "paired_comparison": make_comparison,
    "range_or_measurement_bounds": make_bounds,
}


# ------------------------------------------------------------
# J. Predeclared A2 training allocation
# ------------------------------------------------------------

TRAIN_PER_LABEL = {
    "revenue_percentage_calculation": 20,
    "two_component_sum": 15,
    "temporal_office_holder": 12,
    "paired_comparison": 8,
    "range_or_measurement_bounds": 5,
}


# ------------------------------------------------------------
# K. Generate 180 TRAIN examples
# ------------------------------------------------------------

family_counter = 0

for family, per_label in (
    TRAIN_PER_LABEL.items()
):

    generator_fn = (
        GENERATORS[family]
    )

    for label in LABELS:

        for local_i in range(
            per_label
        ):

            family_counter += 1

            i = (
                family_counter * 17
                + local_i
            )

            (
                claim,
                evidence,
                logic,
            ) = generator_fn(
                i,
                label,
                "train",
            )

            add_example(
                synthetic_train,
                "train",
                family,
                label,
                claim,
                evidence,
                logic,
            )


# ------------------------------------------------------------
# L. Generate 60 NEVER-TRAINED sanity examples
#
# 5 families × 3 labels × 4 = 60
# ------------------------------------------------------------

holdout_counter = 10000

for family, generator_fn in (
    GENERATORS.items()
):

    for label in LABELS:

        for local_i in range(4):

            holdout_counter += 19

            i = (
                holdout_counter
                + local_i
            )

            (
                claim,
                evidence,
                logic,
            ) = generator_fn(
                i,
                label,
                "holdout",
            )

            add_example(
                synthetic_holdout,
                "holdout",
                family,
                label,
                claim,
                evidence,
                logic,
            )


# ------------------------------------------------------------
# M. Exact count assertions
# ------------------------------------------------------------

assert len(
    synthetic_train
) == 180

assert len(
    synthetic_holdout
) == 60


train_label_counts = Counter(
    row["label"]
    for row in synthetic_train
)

holdout_label_counts = Counter(
    row["label"]
    for row in synthetic_holdout
)

assert train_label_counts == Counter({
    "SUPPORTS": 60,
    "REFUTES": 60,
    "NOT_ENOUGH_INFO": 60,
})

assert holdout_label_counts == Counter({
    "SUPPORTS": 20,
    "REFUTES": 20,
    "NOT_ENOUGH_INFO": 20,
})


train_family_counts = Counter(
    metadata[row["id"]]["family"]
    for row in synthetic_train
)

holdout_family_counts = Counter(
    metadata[row["id"]]["family"]
    for row in synthetic_holdout
)

assert train_family_counts == Counter({
    "revenue_percentage_calculation": 60,
    "two_component_sum": 45,
    "temporal_office_holder": 36,
    "paired_comparison": 24,
    "range_or_measurement_bounds": 15,
})

assert all(
    count == 12
    for count in (
        holdout_family_counts.values()
    )
)


# ------------------------------------------------------------
# N. Leakage / duplicate protection
# ------------------------------------------------------------

base_fit_fp = {
    content_fingerprint(row)
    for row in base_fit
}

dev_fp = {
    content_fingerprint(row)
    for row in internal_dev
}

validation_fp = {
    content_fingerprint(row)
    for row in organizer_validation
}

synthetic_train_fp = [
    content_fingerprint(row)
    for row in synthetic_train
]

synthetic_holdout_fp = [
    content_fingerprint(row)
    for row in synthetic_holdout
]


assert (
    len(synthetic_train_fp)
    == len(set(synthetic_train_fp))
)

assert (
    len(synthetic_holdout_fp)
    == len(set(synthetic_holdout_fp))
)


assert not (
    set(synthetic_train_fp)
    & base_fit_fp
)

assert not (
    set(synthetic_train_fp)
    & dev_fp
)

assert not (
    set(synthetic_train_fp)
    & validation_fp
)

assert not (
    set(synthetic_holdout_fp)
    & base_fit_fp
)

assert not (
    set(synthetic_holdout_fp)
    & dev_fp
)

assert not (
    set(synthetic_holdout_fp)
    & validation_fp
)

assert not (
    set(synthetic_train_fp)
    & set(synthetic_holdout_fp)
)


# ------------------------------------------------------------
# O. Build augmented fit set
#
# Internal dev remains completely unchanged.
# ------------------------------------------------------------

augmented_fit = (
    list(base_fit)
    + list(synthetic_train)
)

assert len(augmented_fit) == 975


augmented_distribution = Counter(
    row["label"]
    for row in augmented_fit
)

assert augmented_distribution == Counter({
    "SUPPORTS": 355,
    "REFUTES": 326,
    "NOT_ENOUGH_INFO": 294,
})


# ------------------------------------------------------------
# P. Frozen Prompt V1 token-length audit
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    A1_ADAPTER_DIR,
    local_files_only=True,
)


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def full_training_length(row):

    rendered = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": build_user_content(
                    row
                ),
            },
            {
                "role": "assistant",
                "content": row["label"],
            },
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    return len(
        tokenizer(
            rendered,
            add_special_tokens=False,
        )["input_ids"]
    )


synthetic_train_lengths = [
    full_training_length(row)
    for row in synthetic_train
]

synthetic_holdout_lengths = [
    full_training_length(row)
    for row in synthetic_holdout
]


assert max(
    synthetic_train_lengths
) <= MAX_LENGTH

assert max(
    synthetic_holdout_lengths
) <= MAX_LENGTH


# ------------------------------------------------------------
# Q. Save derived A2 data
# ------------------------------------------------------------

write_jsonl(
    SYNTH_TRAIN_PATH,
    synthetic_train,
)

write_jsonl(
    SYNTH_HOLDOUT_PATH,
    synthetic_holdout,
)

write_jsonl(
    AUGMENTED_FIT_PATH,
    augmented_fit,
)


manifest = {
    "version": "synthetic_a2_v1",
    "seed": SEED,

    "hypothesis": (
        "A1 weaknesses are concentrated in arithmetic, "
        "temporal transition, paired-comparison and "
        "range-bound reasoning rather than simple "
        "class imbalance."
    ),

    "scientific_control": {
        "principal_variable_changed": (
            "training data augmentation only"
        ),
        "prompt_changed": False,
        "internal_dev_changed": False,
        "organizer_validation_changed": False,
        "model_or_lora_changed": False,
    },

    "source_fit": {
        "path": str(
            BASE_FIT_PATH
        ),
        "sha256": (
            EXPECTED_FIT_SHA256
        ),
        "rows": len(base_fit),
    },

    "synthetic_training": {
        "rows": len(
            synthetic_train
        ),
        "labels": dict(
            train_label_counts
        ),
        "families": dict(
            train_family_counts
        ),
    },

    "synthetic_holdout": {
        "rows": len(
            synthetic_holdout
        ),
        "used_for_training": False,
        "labels": dict(
            holdout_label_counts
        ),
        "families": dict(
            holdout_family_counts
        ),
    },

    "augmented_fit": {
        "rows": len(
            augmented_fit
        ),
        "distribution": dict(
            augmented_distribution
        ),
    },

    "token_lengths": {
        "synthetic_train_min": min(
            synthetic_train_lengths
        ),
        "synthetic_train_max": max(
            synthetic_train_lengths
        ),
        "synthetic_holdout_min": min(
            synthetic_holdout_lengths
        ),
        "synthetic_holdout_max": max(
            synthetic_holdout_lengths
        ),
        "max_length": MAX_LENGTH,
        "truncation_required": False,
    },

    "overlap_checks": {
        "synthetic_train_vs_base_fit": 0,
        "synthetic_train_vs_internal_dev": 0,
        "synthetic_train_vs_official_validation": 0,
        "synthetic_holdout_vs_training": 0,
        "synthetic_holdout_vs_internal_dev": 0,
        "synthetic_holdout_vs_official_validation": 0,
    },

    "example_metadata": metadata,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# R. Verify validation still immutable
# ------------------------------------------------------------

validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# S. Output hashes
# ------------------------------------------------------------

synth_train_hash = (
    sha256_file(
        SYNTH_TRAIN_PATH
    )
)

holdout_hash = (
    sha256_file(
        SYNTH_HOLDOUT_PATH
    )
)

augmented_hash = (
    sha256_file(
        AUGMENTED_FIT_PATH
    )
)


# ------------------------------------------------------------
# T. Concise report
# ------------------------------------------------------------

print("=" * 78)
print("A2 SYNTHETIC AUGMENTATION — RESULT")
print("=" * 78)


print("\nScientific control")
print(
    "  Changed:",
    "training data only",
)
print(
    "  Prompt changed:",
    False,
)
print(
    "  Internal dev changed:",
    False,
)
print(
    "  Organizer validation changed:",
    False,
)


print("\nSynthetic TRAIN")

print(
    "  Rows:",
    len(synthetic_train),
)

print(
    "  Labels:",
    dict(train_label_counts),
)

print(
    "  Families:"
)

for family, count in (
    train_family_counts.items()
):

    print(
        f"    {family:32} "
        f"{count}"
    )


print("\nSynthetic NEVER-TRAINED holdout")

print(
    "  Rows:",
    len(synthetic_holdout),
)

print(
    "  Labels:",
    dict(holdout_label_counts),
)

print(
    "  Families:",
    dict(holdout_family_counts),
)


print("\nAugmented fit")

print(
    "  Original rows:",
    len(base_fit),
)

print(
    "  Added rows:",
    len(synthetic_train),
)

print(
    "  Final rows:",
    len(augmented_fit),
)

print(
    "  Distribution:",
    dict(augmented_distribution),
)


print("\nLeakage / duplication")

print(
    "  Synthetic-train internal duplicates:",
    (
        len(synthetic_train_fp)
        - len(
            set(
                synthetic_train_fp
            )
        )
    ),
)

print(
    "  Synthetic-holdout internal duplicates:",
    (
        len(synthetic_holdout_fp)
        - len(
            set(
                synthetic_holdout_fp
            )
        )
    ),
)

print(
    "  Synthetic train ↔ base fit:",
    len(
        set(synthetic_train_fp)
        & base_fit_fp
    ),
)

print(
    "  Synthetic train ↔ internal dev:",
    len(
        set(synthetic_train_fp)
        & dev_fp
    ),
)

print(
    "  Synthetic train ↔ official validation:",
    len(
        set(synthetic_train_fp)
        & validation_fp
    ),
)

print(
    "  Synthetic holdout ↔ any training:",
    len(
        set(synthetic_holdout_fp)
        & (
            base_fit_fp
            | set(synthetic_train_fp)
        )
    ),
)


print("\nToken lengths")

print(
    "  Synthetic train min/max:",
    min(synthetic_train_lengths),
    "/",
    max(synthetic_train_lengths),
)

print(
    "  Holdout min/max:",
    min(synthetic_holdout_lengths),
    "/",
    max(synthetic_holdout_lengths),
)

print(
    "  max_length:",
    MAX_LENGTH,
)

print(
    "  Truncation required:",
    False,
)


print("\nImmutable organizer validation")

print(
    "  Modified:",
    False,
)

print(
    "  SHA-256:",
    validation_hash_after,
)


print("\nSaved")

print(
    " ",
    SYNTH_TRAIN_PATH,
)

print(
    "  SHA-256:",
    synth_train_hash,
)

print(
    " ",
    SYNTH_HOLDOUT_PATH,
)

print(
    "  SHA-256:",
    holdout_hash,
)

print(
    " ",
    AUGMENTED_FIT_PATH,
)

print(
    "  SHA-256:",
    augmented_hash,
)

print(
    " ",
    MANIFEST_PATH,
)


print("\nNo training occurred.")
print(
    "A2 augmentation is now frozen "
    "pending inspection of this output."
)

In [ ]:
# ============================================================
# STEP 11 — EXPERIMENT A2: TARGETED AUGMENTATION
#
# Principal experimental change vs A1:
#   +180 frozen synthetic training examples
#
# Everything else remains frozen:
#   Gemma 2 2B IT
#   Prompt V1
#   max_length = 256
#   4-bit NF4 QLoRA
#   LoRA r=8 / alpha=16 / dropout=0.05
#   physical batch=1
#   gradient accumulation up to 16
#   LR=2e-4
#   cosine schedule
#   2 epochs
#   seed=42
#   NO GradScaler
#
# Evaluation:
#   - unchanged 140-row INTERNAL DEV
#   - 60-row synthetic holdout (sanity only)
#
# Organizer validation is NOT evaluated.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import time

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen A2 configuration
# ------------------------------------------------------------

RUN_NAME = "a2_gemma2_2b_targeted_augmentation_v1"

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")

AUGMENTED_FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2_plus_a2_v1.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

SYNTH_HOLDOUT_PATH = (
    WORK_ROOT
    / "synthetic_a2_v1_holdout.jsonl"
)

EXPECTED_AUGMENTED_SHA256 = (
    "6dae1152d9fc310f564ee2841b44b39a"
    "ee1a3dbee559bb040a3cfc330806602f"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_SYNTH_HOLDOUT_SHA256 = (
    "c86905e26c4926b935969d2c41d2049b"
    "22b7d8ce902e092dbf6a403d08aff13b"
)

RUN_DIR = (
    WORK_ROOT
    / RUN_NAME
)

SEED = 42

MAX_LENGTH = 256

EPOCHS = 2

PHYSICAL_BATCH = 1
GRAD_ACCUM = 16

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10

# A1 reference — frozen before A2.
A1_INTERNAL_DEV_ACCURACY = 0.8214
A1_INTERNAL_DEV_MACRO_F1 = 0.8249


# ------------------------------------------------------------
# B. Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available()


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def directory_size_mb(path):

    total = 0

    for file in Path(path).rglob("*"):

        if file.is_file():

            total += (
                file.stat().st_size
            )

    return (
        total
        / (1024 ** 2)
    )


# ------------------------------------------------------------
# D. Verify frozen experiment data
# ------------------------------------------------------------

assert (
    sha256_file(
        AUGMENTED_FIT_PATH
    )
    == EXPECTED_AUGMENTED_SHA256
)

assert (
    sha256_file(
        DEV_PATH
    )
    == EXPECTED_DEV_SHA256
)

assert (
    sha256_file(
        SYNTH_HOLDOUT_PATH
    )
    == EXPECTED_SYNTH_HOLDOUT_SHA256
)


fit_rows = read_jsonl(
    AUGMENTED_FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

synthetic_holdout_rows = (
    read_jsonl(
        SYNTH_HOLDOUT_PATH
    )
)


assert len(fit_rows) == 975
assert len(dev_rows) == 140
assert len(synthetic_holdout_rows) == 60


assert Counter(
    row["label"]
    for row in fit_rows
) == Counter({
    "SUPPORTS": 355,
    "REFUTES": 326,
    "NOT_ENOUGH_INFO": 294,
})


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


assert Counter(
    row["label"]
    for row in synthetic_holdout_rows
) == Counter({
    "SUPPORTS": 20,
    "REFUTES": 20,
    "NOT_ENOUGH_INFO": 20,
})


# ------------------------------------------------------------
# E. Protect previous experiment
# ------------------------------------------------------------

if RUN_DIR.exists():

    raise RuntimeError(
        f"{RUN_DIR} already exists. "
        "Do not overwrite a previous A2 run."
    )


RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ------------------------------------------------------------
# F. Hugging Face authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# G. Fully release previous A1 model state
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "fresh_quant_config",
]:

    if object_name in globals():

        try:

            del globals()[
                object_name
            ]

        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):
    torch.cuda.ipc_collect()


print("=" * 78)
print("A2 — CLEAN GPU STATE")
print("=" * 78)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

if torch.cuda.device_count() > 1:

    print(
        "GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ------------------------------------------------------------
# H. Frozen tokenizer + Prompt V1
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)


assert (
    tokenizer.chat_template
    is not None
)

assert (
    tokenizer.pad_token_id
    is not None
)


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return (
        tokenizer
        .apply_chat_template(
            [{
                "role": "user",
                "content": (
                    build_user_content(
                        row
                    )
                ),
            }],
            tokenize=False,
            add_generation_prompt=True,
        )
    )


def render_full(row):

    return (
        tokenizer
        .apply_chat_template(
            [
                {
                    "role": "user",
                    "content": (
                        build_user_content(
                            row
                        )
                    ),
                },
                {
                    "role": "assistant",
                    "content": (
                        row["label"]
                    ),
                },
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
    )


# ------------------------------------------------------------
# I. Pre-tokenize completion-only training data
# ------------------------------------------------------------

training_examples = []


for row in fit_rows:

    prompt_text = (
        render_prompt(row)
    )

    full_text = (
        render_full(row)
    )

    assert (
        full_text.startswith(
            prompt_text
        )
    )


    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]


    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]


    assert (
        full_ids[
            :len(prompt_ids)
        ]
        == prompt_ids
    )

    assert (
        len(full_ids)
        <= MAX_LENGTH
    )


    labels = (
        [-100]
        * len(prompt_ids)
        + full_ids[
            len(prompt_ids):
        ]
    )


    supervised_tokens = sum(
        token != -100
        for token in labels
    )


    assert supervised_tokens in {
        4,
        8,
    }


    training_examples.append({
        "id": row["id"],
        "input_ids": torch.tensor(
            full_ids,
            dtype=torch.long,
        ),
        "attention_mask": (
            torch.ones(
                len(full_ids),
                dtype=torch.long,
            )
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
        "tokens": len(
            full_ids
        ),
    })


assert len(
    training_examples
) == 975


print(
    "\nTraining rows:",
    len(training_examples),
)

print(
    "Longest sequence:",
    max(
        example["tokens"]
        for example
        in training_examples
    ),
)


# ------------------------------------------------------------
# J. Load FRESH Gemma base
# ------------------------------------------------------------

quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


print("\n" + "=" * 78)
print("A2 — FRESH MODEL LOAD")
print("=" * 78)


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert model.is_loaded_in_4bit


model = (
    prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=False,
    )
)


model.config.use_cache = False


# ------------------------------------------------------------
# K. Fresh LoRA
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(
    model,
    lora_config,
)


trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


assert (
    trainable_params
    == 10_383_360
)


# ------------------------------------------------------------
# L. Gradient checkpointing
# ------------------------------------------------------------

def enable_training_checkpointing():

    model.config.use_cache = False

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

    if hasattr(
        model,
        "enable_input_require_grads",
    ):

        model.enable_input_require_grads()


def disable_training_checkpointing():

    try:

        model.gradient_checkpointing_disable()

    except Exception:
        pass


enable_training_checkpointing()


# ------------------------------------------------------------
# M. Optimizer + schedule
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


optimizer_steps_per_epoch = math.ceil(
    len(training_examples)
    / GRAD_ACCUM
)


total_optimizer_steps = (
    optimizer_steps_per_epoch
    * EPOCHS
)


warmup_steps = max(
    1,
    round(
        total_optimizer_steps
        * WARMUP_RATIO
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=(
            warmup_steps
        ),
        num_training_steps=(
            total_optimizer_steps
        ),
    )
)


assert (
    optimizer_steps_per_epoch
    == 61
)

assert (
    total_optimizer_steps
    == 122
)

assert (
    warmup_steps
    == 6
)


# ------------------------------------------------------------
# N. Deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer
    .convert_tokens_to_ids(
        "<end_of_turn>"
    )
)


assert isinstance(
    end_of_turn_id,
    int,
)


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_rows(
    rows,
    tag,
):

    disable_training_checkpointing()

    model.eval()

    model.config.use_cache = True


    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    gold = []
    predictions = []

    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(row)
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded[
                    "input_ids"
                ]
                .to("cuda:0")
            )

            attention_mask = (
                encoded[
                    "attention_mask"
                ]
                .to("cuda:0")
            )


            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_text = (
                    tokenizer.decode(
                        token_ids,
                        skip_special_tokens=True,
                    )
                )


                prediction = (
                    parse_prediction(
                        raw_text
                    )
                )


                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )

        model.config.use_cache = False

        enable_training_checkpointing()

        model.train()


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            gold,
            metric_predictions,
            labels=LABELS,
            zero_division=0,
        )
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,
        "rows": len(rows),
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid_count": int(
            invalid_count
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "confusion_matrix": (
            matrix.tolist()
        ),
        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)

    print(
        "Rows:",
        len(rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )


    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  "
        "NOT_ENOUGH_INFO"
    )


    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print(
        "Per-class F1:"
    )


    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return result


# ------------------------------------------------------------
# O. Save experiment config
# ------------------------------------------------------------

run_config = {
    "experiment": RUN_NAME,

    "hypothesis": (
        "Targeted balanced augmentation of arithmetic, "
        "temporal-transition, paired-comparison and "
        "range-bound reasoning will improve generalization "
        "relative to A1."
    ),

    "model": MODEL_ID,

    "dataset": {
        "augmented_fit_rows": (
            len(fit_rows)
        ),
        "augmented_fit_sha256": (
            EXPECTED_AUGMENTED_SHA256
        ),
        "internal_dev_rows": (
            len(dev_rows)
        ),
        "internal_dev_sha256": (
            EXPECTED_DEV_SHA256
        ),
        "synthetic_holdout_rows": (
            len(
                synthetic_holdout_rows
            )
        ),
        "synthetic_holdout_sha256": (
            EXPECTED_SYNTH_HOLDOUT_SHA256
        ),
        "official_validation_used": False,
    },

    "scientific_change_vs_a1": (
        "training data only"
    ),

    "prompt": (
        "v1_minimal_organizer_semantics"
    ),

    "max_length": MAX_LENGTH,

    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "targets": TARGET_MODULES,
    },

    "training": {
        "epochs": EPOCHS,
        "physical_batch": (
            PHYSICAL_BATCH
        ),
        "gradient_accumulation": (
            GRAD_ACCUM
        ),
        "learning_rate": (
            LEARNING_RATE
        ),
        "weight_decay": (
            WEIGHT_DECAY
        ),
        "scheduler": "cosine",
        "warmup_steps": (
            warmup_steps
        ),
        "optimizer_steps_per_epoch": (
            optimizer_steps_per_epoch
        ),
        "total_optimizer_steps": (
            total_optimizer_steps
        ),
        "grad_scaler": False,
        "seed": SEED,
    },

    "a1_reference": {
        "internal_dev_accuracy": (
            A1_INTERNAL_DEV_ACCURACY
        ),
        "internal_dev_macro_f1": (
            A1_INTERNAL_DEV_MACRO_F1
        ),
    },
}


(
    RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# P. Predetermined pre-training evaluations
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A2 — PRE-TRAIN EVALUATION")
print("=" * 78)


all_dev_metrics = []
all_synth_metrics = []


pretrain_dev = evaluate_rows(
    dev_rows,
    "INTERNAL DEV — PRETRAIN",
)

all_dev_metrics.append(
    pretrain_dev
)


pretrain_synth = evaluate_rows(
    synthetic_holdout_rows,
    "SYNTHETIC HOLDOUT — PRETRAIN",
)

all_synth_metrics.append(
    pretrain_synth
)


# ------------------------------------------------------------
# Q. Training
# ------------------------------------------------------------

global_optimizer_step = 0

training_summaries = []


for epoch_index in range(
    EPOCHS
):

    epoch_number = (
        epoch_index + 1
    )


    enable_training_checkpointing()

    model.train()


    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        SEED + epoch_index
    )


    order = torch.randperm(
        len(training_examples),
        generator=generator,
    ).tolist()


    epoch_loss_sum = 0.0

    epoch_examples = 0

    epoch_start = (
        time.perf_counter()
    )


    print("\n" + "=" * 78)
    print(
        f"A2 TRAINING EPOCH "
        f"{epoch_number}/{EPOCHS}"
    )
    print("=" * 78)


    for window_start in range(
        0,
        len(order),
        GRAD_ACCUM,
    ):

        window_indices = order[
            window_start:
            window_start
            + GRAD_ACCUM
        ]

        window_size = len(
            window_indices
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        window_loss_sum = 0.0


        for index in window_indices:

            example = (
                training_examples[
                    index
                ]
            )


            input_ids = (
                example["input_ids"]
                .unsqueeze(0)
                .to("cuda:0")
            )


            attention_mask = (
                example[
                    "attention_mask"
                ]
                .unsqueeze(0)
                .to("cuda:0")
            )


            labels = (
                example["labels"]
                .unsqueeze(0)
                .to("cuda:0")
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    labels=labels,
                    use_cache=False,
                )

                raw_loss = (
                    outputs.loss
                )


            if not torch.isfinite(
                raw_loss
            ):

                raise RuntimeError(
                    f"Non-finite loss "
                    f"at epoch "
                    f"{epoch_number}, "
                    f"example "
                    f"{example['id']}"
                )


            loss_value = float(
                raw_loss
                .detach()
                .cpu()
            )


            epoch_loss_sum += (
                loss_value
            )

            window_loss_sum += (
                loss_value
            )

            epoch_examples += 1


            accumulation_loss = (
                raw_loss
                / window_size
            )


            accumulation_loss.backward()


            del outputs
            del raw_loss
            del accumulation_loss
            del input_ids
            del attention_mask
            del labels


        # ----------------------------------------------------
        # Finite-gradient check
        # ----------------------------------------------------

        nonfinite_gradients = []


        for name, parameter in (
            model.named_parameters()
        ):

            if (
                parameter.requires_grad
                and parameter.grad
                is not None
            ):

                if not torch.isfinite(
                    parameter.grad
                ).all():

                    nonfinite_gradients.append(
                        name
                    )


        if nonfinite_gradients:

            raise RuntimeError(
                "Non-finite gradients. "
                f"First: "
                f"{nonfinite_gradients[:10]}"
            )


        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                [
                    parameter
                    for parameter
                    in model.parameters()
                    if (
                        parameter.requires_grad
                        and parameter.grad
                        is not None
                    )
                ],
                MAX_GRAD_NORM,
            )
        )


        if not torch.isfinite(
            grad_norm
        ):

            raise RuntimeError(
                "Non-finite gradient norm."
            )


        optimizer.step()

        scheduler.step()


        global_optimizer_step += 1


        if (
            global_optimizer_step % 10
            == 0
            or window_start == 0
            or (
                window_start
                + window_size
                >= len(order)
            )
        ):

            current_lr = (
                scheduler
                .get_last_lr()[0]
            )


            print(
                f"epoch={epoch_number} | "
                f"step="
                f"{global_optimizer_step:3d}/"
                f"{total_optimizer_steps} | "
                f"examples="
                f"{epoch_examples:3d}/"
                f"{len(training_examples)} | "
                f"window_loss="
                f"{window_loss_sum/window_size:.4f} | "
                f"epoch_loss="
                f"{epoch_loss_sum/epoch_examples:.4f} | "
                f"lr={current_lr:.7f} | "
                f"grad_norm="
                f"{float(grad_norm):.4f}"
            )


    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )


    mean_epoch_loss = (
        epoch_loss_sum
        / epoch_examples
    )


    assert (
        epoch_examples
        == 975
    )


    training_summaries.append({
        "epoch": epoch_number,
        "mean_example_loss": (
            float(
                mean_epoch_loss
            )
        ),
        "seconds": float(
            epoch_seconds
        ),
        "optimizer_step_end": (
            global_optimizer_step
        ),
    })


    print(
        f"\nEpoch {epoch_number} "
        f"training complete"
    )

    print(
        "  Mean example loss:",
        f"{mean_epoch_loss:.4f}",
    )

    print(
        "  Seconds:",
        round(
            epoch_seconds,
            1,
        ),
    )


    # --------------------------------------------------------
    # Save epoch adapter
    # --------------------------------------------------------

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )


    model.save_pretrained(
        checkpoint_dir,
        safe_serialization=True,
    )


    tokenizer.save_pretrained(
        checkpoint_dir
    )


    print(
        "  Adapter saved:",
        checkpoint_dir,
    )

    print(
        "  Adapter size:",
        f"{directory_size_mb(checkpoint_dir):.1f} MB",
    )


    # --------------------------------------------------------
    # Predetermined unchanged internal-dev evaluation
    # --------------------------------------------------------

    dev_result = evaluate_rows(
        dev_rows,
        (
            f"INTERNAL DEV — "
            f"EPOCH {epoch_number}"
        ),
    )


    dev_result[
        "checkpoint"
    ] = str(
        checkpoint_dir
    )

    dev_result[
        "training_loss"
    ] = float(
        mean_epoch_loss
    )


    all_dev_metrics.append(
        dev_result
    )


    # --------------------------------------------------------
    # Synthetic holdout sanity evaluation
    # --------------------------------------------------------

    synth_result = evaluate_rows(
        synthetic_holdout_rows,
        (
            f"SYNTHETIC HOLDOUT — "
            f"EPOCH {epoch_number}"
        ),
    )


    synth_result[
        "checkpoint"
    ] = str(
        checkpoint_dir
    )


    all_synth_metrics.append(
        synth_result
    )


# ------------------------------------------------------------
# R. Integrity
# ------------------------------------------------------------

assert (
    global_optimizer_step
    == 122
)


# ------------------------------------------------------------
# S. Rank A2 checkpoints ONLY by real internal dev
# ------------------------------------------------------------

trained_dev_results = [
    result
    for result in all_dev_metrics
    if "EPOCH" in result["tag"]
]


ranked = sorted(
    trained_dev_results,
    key=lambda result: (
        result["macro_f1"],
        result["accuracy"],
        -result["invalid_count"],
    ),
    reverse=True,
)


best_a2 = ranked[0]


# ------------------------------------------------------------
# T. Save full results
# ------------------------------------------------------------

results = {
    "experiment": RUN_NAME,
    "config": run_config,
    "training_summaries": (
        training_summaries
    ),
    "internal_dev_metrics": (
        all_dev_metrics
    ),
    "synthetic_holdout_metrics": (
        all_synth_metrics
    ),
    "checkpoint_selection_rule": (
        "Select using unchanged real internal-dev "
        "Macro-F1, not synthetic holdout."
    ),
    "best_a2_internal_dev": (
        best_a2
    ),
}


RESULTS_PATH = (
    RUN_DIR
    / "results.json"
)


RESULTS_PATH.write_text(
    json.dumps(
        results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# U. Comparison against A1
# ------------------------------------------------------------

delta_macro_f1 = (
    best_a2["macro_f1"]
    - A1_INTERNAL_DEV_MACRO_F1
)

delta_accuracy = (
    best_a2["accuracy"]
    - A1_INTERNAL_DEV_ACCURACY
)


# ------------------------------------------------------------
# V. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A2 — RESULT")
print("=" * 78)


print("\nScientific comparison")

print(
    "  Principal change vs A1:",
    "training data only",
)

print(
    "  Original A1 fit rows:",
    795,
)

print(
    "  A2 fit rows:",
    len(fit_rows),
)

print(
    "  Synthetic added:",
    180,
)

print(
    "  Internal dev unchanged:",
    True,
)

print(
    "  Organizer validation used:",
    False,
)


print("\nA1 reference")

print(
    "  Accuracy:",
    f"{A1_INTERNAL_DEV_ACCURACY:.4f}",
)

print(
    "  Macro-F1:",
    f"{A1_INTERNAL_DEV_MACRO_F1:.4f}",
)


print("\nA2 internal-dev checkpoints")

for result in all_dev_metrics:

    print(
        f"  {result['tag']:25} | "
        f"Accuracy="
        f"{result['accuracy']:.4f} | "
        f"Macro-F1="
        f"{result['macro_f1']:.4f} | "
        f"Invalid="
        f"{result['invalid_count']}"
    )


print("\nA2 synthetic-holdout checkpoints")
print(
    "  NOTE: sanity diagnostic only; "
    "not model-selection evidence."
)

for result in all_synth_metrics:

    print(
        f"  {result['tag']:31} | "
        f"Accuracy="
        f"{result['accuracy']:.4f} | "
        f"Macro-F1="
        f"{result['macro_f1']:.4f} | "
        f"Invalid="
        f"{result['invalid_count']}"
    )


print("\nTraining loss")

for summary in training_summaries:

    print(
        f"  epoch_{summary['epoch']} | "
        f"loss="
        f"{summary['mean_example_loss']:.4f} | "
        f"{summary['seconds']:.1f}s"
    )


print(
    "\nBest A2 checkpoint "
    "by REAL INTERNAL DEV"
)

print(
    "  Checkpoint:",
    best_a2["tag"],
)

print(
    "  Accuracy:",
    f"{best_a2['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{best_a2['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    best_a2["invalid_count"],
)


print("\nA2 minus A1")

print(
    "  Accuracy delta:",
    f"{delta_accuracy:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{delta_macro_f1:+.4f}",
)


print("\nPer-class F1 — best A2")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{best_a2['per_class'][label]['f1']:.4f}"
    )


print("\nArtifacts")

for epoch_number in range(
    1,
    EPOCHS + 1,
):

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )

    print(
        f"  epoch_{epoch_number}:",
        checkpoint_dir,
        f"({directory_size_mb(checkpoint_dir):.1f} MB)",
    )


print(
    "  Results:",
    RESULTS_PATH,
)


print("\nGPU")

print(
    "  GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

if torch.cuda.device_count() > 1:

    print(
        "  GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


print("\nA2 training completed.")
print(
    "Organizer validation was NOT evaluated."
)

In [ ]:
# ============================================================
# STEP 12 — A2 FRESH RELOAD + ADAPTIVE VALIDATION EVALUATION #2
#
# Selected checkpoint:
#   A2 epoch 2
#
# Sequence:
#   1. verify saved A2 adapter
#   2. fresh reload base + adapter
#   3. finite-logit check
#   4. reproduce unchanged internal-dev result
#   5. evaluate organizer validation
#   6. compare A1 vs A2 prediction-by-prediction
#   7. compare A1 vs A2 by task family
#
# IMPORTANT:
#   This organizer-validation evaluation is ADAPTIVE because
#   its earlier error analysis informed A2 augmentation.
#
# NO TRAINING.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen paths
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

A2_RUN_DIR = (
    WORK_ROOT
    / "a2_gemma2_2b_targeted_augmentation_v1"
)

A2_ADAPTER_DIR = (
    A2_RUN_DIR
    / "epoch_2_adapter"
)

A1_RUN_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

A1_PREDICTIONS_PATH = (
    A1_RUN_DIR
    / "official_validation_eval1_epoch2.csv"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

EXPECTED_A2_DEV_ACCURACY = 0.8286
EXPECTED_A2_DEV_MACRO_F1 = 0.8334

A1_OFFICIAL_ACCURACY = 0.8200
A1_OFFICIAL_MACRO_F1 = 0.8207

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:

                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(filename):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):

            matches.append(path)

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one "
            f"{filename} with frozen hash. "
            f"Found: {matches}"
        )

    return matches[0]


def clean(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


# ------------------------------------------------------------
# C. Verify saved A2 adapter
# ------------------------------------------------------------

print("=" * 78)
print("A. A2 SAVED-ADAPTER INTEGRITY")
print("=" * 78)


assert A2_ADAPTER_DIR.exists()

adapter_model_path = (
    A2_ADAPTER_DIR
    / "adapter_model.safetensors"
)

adapter_config_path = (
    A2_ADAPTER_DIR
    / "adapter_config.json"
)

assert adapter_model_path.exists()
assert adapter_config_path.exists()


adapter_config = json.loads(
    adapter_config_path.read_text(
        encoding="utf-8"
    )
)


adapter_hash = sha256_file(
    adapter_model_path
)


print(
    "Adapter:",
    A2_ADAPTER_DIR,
)

print(
    "Adapter size:",
    round(
        adapter_model_path.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "Adapter SHA-256:",
    adapter_hash,
)

print(
    "Base:",
    adapter_config.get(
        "base_model_name_or_path"
    ),
)

print(
    "r:",
    adapter_config.get("r"),
)

print(
    "alpha:",
    adapter_config.get(
        "lora_alpha"
    ),
)


assert (
    adapter_config.get(
        "base_model_name_or_path"
    )
    == MODEL_ID
)

assert adapter_config.get("r") == 8
assert adapter_config.get("lora_alpha") == 16


# ------------------------------------------------------------
# D. Verify immutable evaluation data
# ------------------------------------------------------------

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

DEV_ROWS = read_jsonl(
    DEV_PATH
)

assert len(DEV_ROWS) == 140


VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = (
    sha256_file(
        VAL_PATH
    )
)

VAL_ROWS = read_jsonl(
    VAL_PATH
)

assert len(VAL_ROWS) == 300


# ------------------------------------------------------------
# E. Load A1 prediction record
# ------------------------------------------------------------

assert A1_PREDICTIONS_PATH.exists()

a1_df = pd.read_csv(
    A1_PREDICTIONS_PATH
)

assert len(a1_df) == 300
assert a1_df["id"].is_unique


a1_pred_by_id = {
    row["id"]: row["prediction"]
    for row in a1_df.to_dict(
        orient="records"
    )
}


# ------------------------------------------------------------
# F. HF authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# G. Release current training model
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "lora_config",
]:

    if object_name in globals():

        try:

            del globals()[
                object_name
            ]

        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


print("\n" + "=" * 78)
print("B. CLEAN GPU STATE")
print("=" * 78)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

if torch.cuda.device_count() > 1:

    print(
        "GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ------------------------------------------------------------
# H. Tokenizer FROM SAVED A2 ARTIFACT
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        A2_ADAPTER_DIR,
        local_files_only=True,
    )
)

assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


SYSTEM_INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"
    "SUPPORTS: the evidence establishes the claim.\n"
    "REFUTES: the evidence contradicts the claim.\n"
    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"
    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(
                row
            ),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


# ------------------------------------------------------------
# I. Fresh base + saved A2 adapter
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.float16
    ),
)


print("\n" + "=" * 78)
print("C. FRESH A2 RELOAD")
print("=" * 78)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert base_model.is_loaded_in_4bit


base_model = (
    prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=False,
    )
)

base_model.config.use_cache = True


model = PeftModel.from_pretrained(
    base_model,
    A2_ADAPTER_DIR,
    is_trainable=False,
)


model.eval()
model.config.use_cache = True


trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert trainable_params == 0


print(
    "Adapter loaded:",
    True,
)

print(
    "Inference trainable parameters:",
    trainable_params,
)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


# ------------------------------------------------------------
# J. Numerical preflight
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("D. A2 NUMERICAL PREFLIGHT")
print("=" * 78)


for row in [
    DEV_ROWS[0],
    DEV_ROWS[
        len(DEV_ROWS) // 2
    ],
    DEV_ROWS[-1],
]:

    encoded = tokenizer(
        render_prompt(row),
        add_special_tokens=False,
        return_tensors="pt",
    )

    input_ids = encoded[
        "input_ids"
    ].to("cuda:0")

    attention_mask = encoded[
        "attention_mask"
    ].to("cuda:0")


    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            attention_mask=(
                attention_mask
            ),
            use_cache=True,
        )


    final_logits = (
        outputs.logits[
            0,
            -1,
            :
        ]
    )

    finite = bool(
        torch.isfinite(
            final_logits
        ).all().item()
    )

    print(
        row["id"],
        "| finite:",
        finite,
        "| dtype:",
        final_logits.dtype,
    )

    assert finite


    del encoded
    del input_ids
    del attention_mask
    del outputs
    del final_logits


torch.cuda.empty_cache()


# ------------------------------------------------------------
# K. Deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_rows(
    rows,
    tag,
):

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    gold = []
    predictions = []
    records = []

    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(row)
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded["input_ids"]
                .to("cuda:0")
            )

            attention_mask = (
                encoded["attention_mask"]
                .to("cuda:0")
            )

            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_output = (
                    tokenizer.decode(
                        token_ids,
                        skip_special_tokens=True,
                    )
                )

                prediction = (
                    parse_prediction(
                        raw_output
                    )
                )

                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )

                records.append({
                    "id": row["id"],
                    "gold": row["label"],
                    "prediction": prediction,
                    "raw_output": raw_output,
                })


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            gold,
            metric_predictions,
            labels=LABELS,
            zero_division=0,
        )
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,
        "rows": len(rows),
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid_count": int(
            invalid_count
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "confusion_matrix": (
            matrix.tolist()
        ),
        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)

    print(
        "Rows:",
        len(rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )

    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )

    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print("Per-class F1:")

    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return (
        result,
        records,
    )


# ------------------------------------------------------------
# L. Reproduce A2 internal dev first
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("E. A2 FRESH-RELOAD REPRODUCTION")
print("=" * 78)


(
    dev_result,
    dev_records,
) = evaluate_rows(
    DEV_ROWS,
    "INTERNAL DEV — A2 EPOCH 2 FRESH RELOAD",
)


assert (
    round(
        dev_result["accuracy"],
        4,
    )
    == EXPECTED_A2_DEV_ACCURACY
)

assert (
    round(
        dev_result["macro_f1"],
        4,
    )
    == EXPECTED_A2_DEV_MACRO_F1
)

assert (
    dev_result[
        "invalid_count"
    ]
    == 0
)


print(
    "\nA2 internal-dev reproduction: PASS"
)


# ------------------------------------------------------------
# M. Adaptive organizer-validation evaluation #2
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("F. ORGANIZER VALIDATION — ADAPTIVE EVALUATION #2")
print("=" * 78)


(
    official_result,
    official_records,
) = evaluate_rows(
    VAL_ROWS,
    "ORGANIZER VALIDATION — A2 EPOCH 2",
)


validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# N. Task-family classifier
# ------------------------------------------------------------

def joined_evidence(row):

    return clean(
        " ".join(
            row["evidence"]
        )
    ).lower()


def classify_family(row):

    claim = clean(
        row["claim"]
    ).lower()

    evidence = joined_evidence(
        row
    )


    if (
        "year-on-year revenue growth"
        in claim
        or (
            "revenue growth"
            in claim
            and "%"
            in claim
        )
    ):

        return (
            "revenue_percentage_calculation"
        )


    if (
        "two named sites jointly made"
        in claim
        or (
            "jointly"
            in claim
            and "units"
            in claim
        )
    ):

        return "two_component_sum"


    if (
        "depot was busier than"
        in claim
        or (
            "depot"
            in claim
            and (
                "more parcels than"
                in claim
                or "busier"
                in claim
            )
        )
    ):

        return "paired_comparison"


    if (
        "reached a minimum of"
        in claim
        or "reached a maximum of"
        in claim
        or (
            "megalitres"
            in claim
            and (
                "between"
                in evidence
                or "bounds"
                in evidence
                or "margin of error"
                in evidence
            )
        )
    ):

        return (
            "range_or_measurement_bounds"
        )


    if (
        "office-holder directing"
        in claim
    ):

        return "temporal_office_holder"


    if (
        "project"
        in claim
        and (
            "based in"
            in claim
            or "carried out"
            in claim
            or "located in"
            in claim
        )
    ):

        return "project_location_mapping"


    if (
        "mission payload"
        in claim
        or "left out the"
        in claim
    ):

        return (
            "complete_manifest_membership"
        )


    if (
        "safety audit"
        in claim
        or (
            "audit"
            in claim
            and (
                "failed"
                in claim
                or "passed"
                in claim
            )
        )
    ):

        return "audit_status"


    if (
        "employed exactly"
        in claim
        or (
            "employees"
            in claim
            and "exactly"
            in claim
        )
    ):

        return "exact_staffing_count"


    if (
        "full-year service"
        in claim
    ):

        return "service_availability"


    if "exactly" in claim:

        return "other_exact_value"


    return "other"


# ------------------------------------------------------------
# O. A1 vs A2 paired comparison
# ------------------------------------------------------------

a2_pred_by_id = {
    record["id"]: (
        record["prediction"]
    )
    for record in official_records
}


comparison_rows = []

family_stats = defaultdict(
    lambda: {
        "total": 0,
        "a1_correct": 0,
        "a2_correct": 0,
        "fixed_by_a2": 0,
        "broken_by_a2": 0,
        "both_wrong": 0,
    }
)


overall_transition = Counter()


for row in VAL_ROWS:

    example_id = row["id"]

    gold = row["label"]

    a1_prediction = (
        a1_pred_by_id[
            example_id
        ]
    )

    a2_prediction = (
        a2_pred_by_id[
            example_id
        ]
    )

    family = classify_family(
        row
    )


    a1_correct = (
        a1_prediction == gold
    )

    a2_correct = (
        a2_prediction == gold
    )


    if (
        not a1_correct
        and a2_correct
    ):

        transition = "A2_FIXED"

    elif (
        a1_correct
        and not a2_correct
    ):

        transition = "A2_BROKE"

    elif (
        a1_correct
        and a2_correct
    ):

        transition = "BOTH_CORRECT"

    else:

        transition = "BOTH_WRONG"


    overall_transition[
        transition
    ] += 1


    stats = family_stats[
        family
    ]

    stats["total"] += 1

    stats["a1_correct"] += int(
        a1_correct
    )

    stats["a2_correct"] += int(
        a2_correct
    )

    stats["fixed_by_a2"] += int(
        transition == "A2_FIXED"
    )

    stats["broken_by_a2"] += int(
        transition == "A2_BROKE"
    )

    stats["both_wrong"] += int(
        transition == "BOTH_WRONG"
    )


    comparison_rows.append({
        "id": example_id,
        "family": family,
        "gold": gold,
        "a1_prediction": (
            a1_prediction
        ),
        "a2_prediction": (
            a2_prediction
        ),
        "a1_correct": (
            a1_correct
        ),
        "a2_correct": (
            a2_correct
        ),
        "transition": transition,
        "claim": row["claim"],
    })


# ------------------------------------------------------------
# P. Family comparison table
# ------------------------------------------------------------

family_results = []


for family, stats in (
    family_stats.items()
):

    total = stats[
        "total"
    ]

    a1_accuracy = (
        stats["a1_correct"]
        / total
    )

    a2_accuracy = (
        stats["a2_correct"]
        / total
    )


    family_results.append({
        "family": family,
        "total": total,
        "a1_correct": (
            stats["a1_correct"]
        ),
        "a2_correct": (
            stats["a2_correct"]
        ),
        "a1_accuracy": (
            a1_accuracy
        ),
        "a2_accuracy": (
            a2_accuracy
        ),
        "accuracy_delta": (
            a2_accuracy
            - a1_accuracy
        ),
        "fixed_by_a2": (
            stats["fixed_by_a2"]
        ),
        "broken_by_a2": (
            stats["broken_by_a2"]
        ),
        "both_wrong": (
            stats["both_wrong"]
        ),
    })


family_results.sort(
    key=lambda item: (
        item["accuracy_delta"],
        item["total"],
    ),
    reverse=True,
)


# ------------------------------------------------------------
# Q. Save derived results
# ------------------------------------------------------------

A2_PREDICTIONS_PATH = (
    A2_RUN_DIR
    / "official_validation_eval2_epoch2.csv"
)

PAIRWISE_PATH = (
    A2_RUN_DIR
    / "a1_vs_a2_official_validation.csv"
)

METRICS_PATH = (
    A2_RUN_DIR
    / "official_validation_eval2_epoch2.json"
)


pd.DataFrame([
    {
        "id": record["id"],
        "gold": record["gold"],
        "prediction": (
            record["prediction"]
        ),
        "correct": (
            record["gold"]
            == record["prediction"]
        ),
        "raw_output": (
            record["raw_output"]
        ),
    }
    for record in official_records
]).to_csv(
    A2_PREDICTIONS_PATH,
    index=False,
)


pd.DataFrame(
    comparison_rows
).to_csv(
    PAIRWISE_PATH,
    index=False,
)


payload = {
    "evaluation_number": 2,

    "adaptive": True,

    "adaptive_reason": (
        "Organizer-validation error analysis from "
        "A1 informed the design of A2 augmentation."
    ),

    "checkpoint": (
        "A2 epoch_2"
    ),

    "adapter_sha256": (
        adapter_hash
    ),

    "internal_dev_reproduction": (
        dev_result
    ),

    "official_validation": (
        official_result
    ),

    "a1_reference": {
        "accuracy": (
            A1_OFFICIAL_ACCURACY
        ),
        "macro_f1": (
            A1_OFFICIAL_MACRO_F1
        ),
    },

    "paired_transitions": dict(
        overall_transition
    ),

    "family_comparison": (
        family_results
    ),
}


METRICS_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# R. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A2 — ADAPTIVE VALIDATION RESULT")
print("=" * 78)


print("\nFresh reload")

print(
    "  Adapter portability:",
    "PASS",
)

print(
    "  Finite logits:",
    "PASS",
)

print(
    "  Internal-dev reproduction:",
    "PASS",
)


print("\nInternal dev")

print(
    "  A1 Macro-F1:",
    "0.8249",
)

print(
    "  A2 Macro-F1:",
    f"{dev_result['macro_f1']:.4f}",
)

print(
    "  Delta:",
    f"{dev_result['macro_f1'] - 0.8249:+.4f}",
)


print("\nOrganizer validation")

print(
    "  IMPORTANT:",
    "adaptive evaluation",
)

print(
    "  A1 Accuracy:",
    f"{A1_OFFICIAL_ACCURACY:.4f}",
)

print(
    "  A2 Accuracy:",
    f"{official_result['accuracy']:.4f}",
)

print(
    "  Accuracy delta:",
    f"{official_result['accuracy'] - A1_OFFICIAL_ACCURACY:+.4f}",
)

print(
    "  A1 Macro-F1:",
    f"{A1_OFFICIAL_MACRO_F1:.4f}",
)

print(
    "  A2 Macro-F1:",
    f"{official_result['macro_f1']:.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{official_result['macro_f1'] - A1_OFFICIAL_MACRO_F1:+.4f}",
)


print("\nA2 organizer-validation per-class F1")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{official_result['per_class'][label]['f1']:.4f}"
    )


print("\nPrediction transitions")

for key in [
    "A2_FIXED",
    "A2_BROKE",
    "BOTH_CORRECT",
    "BOTH_WRONG",
]:

    print(
        f"  {key:14}",
        overall_transition[
            key
        ],
    )


print("\nTask-family comparison")
print(
    f"{'Family':34} "
    f"{'N':>3} "
    f"{'A1':>7} "
    f"{'A2':>7} "
    f"{'Delta':>8} "
    f"{'Fix':>4} "
    f"{'Break':>5}"
)


for item in family_results:

    print(
        f"{item['family']:34} "
        f"{item['total']:3d} "
        f"{item['a1_accuracy']:7.1%} "
        f"{item['a2_accuracy']:7.1%} "
        f"{item['accuracy_delta']:+7.1%} "
        f"{item['fixed_by_a2']:4d} "
        f"{item['broken_by_a2']:5d}"
    )


print("\nValidation integrity")

print(
    "  Modified:",
    False,
)

print(
    "  SHA-256:",
    validation_hash_after,
)


print("\nSaved")

print(
    " ",
    A2_PREDICTIONS_PATH,
)

print(
    " ",
    PAIRWISE_PATH,
)

print(
    " ",
    METRICS_PATH,
)


print("\nNo training occurred.")
print(
    "This was organizer-validation "
    "evaluation #2 and is explicitly "
    "recorded as adaptive."
)

In [ ]:
# ============================================================
# STEP 13 — A3 PROMPT V2 TOKEN / STRUCTURE PREFLIGHT
#
# Experiment A3:
#   Data: original frozen Dataset V2 fit (795)
#   Model: google/gemma-2-2b-it
#   Prompt: NEW Prompt V2
#   Synthetic augmentation: NONE
#
# This cell:
#   - freezes Prompt V2
#   - verifies chat-template structure
#   - measures exact token lengths
#   - recommends the smallest safe max_length
#
# NO training.
# NO dataset modification.
# NO model evaluation.
# ============================================================

from pathlib import Path
import hashlib
import json
import math

from transformers import AutoTokenizer


# ------------------------------------------------------------
# A. Frozen paths / hashes
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

A1_TOKENIZER_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
    / "epoch_2_adapter"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

MODEL_ID = "google/gemma-2-2b-it"

PROMPT_VERSION = (
    "v2_precise_reasoning_rules"
)

CANDIDATE_MAX_LENGTHS = [
    256,
    288,
    320,
    384,
    512,
]

MIN_SAFETY_MARGIN = 24


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(
            f,
            1,
        ):

            if not line.strip():
                continue

            try:
                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(
        filename
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):

            matches.append(path)

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one {filename} "
            f"with frozen hash. Found: {matches}"
        )

    return matches[0]


def percentile(
    values,
    p,
):

    values = sorted(values)

    position = (
        (len(values) - 1)
        * p
    )

    low = math.floor(position)
    high = math.ceil(position)

    if low == high:
        return values[low]

    fraction = (
        position - low
    )

    return (
        values[low]
        * (1 - fraction)
        + values[high]
        * fraction
    )


# ------------------------------------------------------------
# C. Verify frozen datasets
# ------------------------------------------------------------

assert (
    sha256_file(FIT_PATH)
    == EXPECTED_FIT_SHA256
)

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = (
    sha256_file(
        VAL_PATH
    )
)

fit_rows = read_jsonl(
    FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

val_rows = read_jsonl(
    VAL_PATH
)

assert len(fit_rows) == 795
assert len(dev_rows) == 140
assert len(val_rows) == 300


# ------------------------------------------------------------
# D. Load frozen Gemma tokenizer locally
# ------------------------------------------------------------

assert A1_TOKENIZER_DIR.exists()

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        A1_TOKENIZER_DIR,
        local_files_only=True,
    )
)

assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# E. Freeze Prompt V2
# ------------------------------------------------------------

SYSTEM_INSTRUCTION_V2 = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


print("=" * 78)
print("A3 PROMPT V2")
print("=" * 78)

print(
    SYSTEM_INSTRUCTION_V2
)


# ------------------------------------------------------------
# F. Rendering functions
# ------------------------------------------------------------

def build_user_content_v2(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION_V2
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt_v2(row):

    return (
        tokenizer
        .apply_chat_template(
            [{
                "role": "user",
                "content": (
                    build_user_content_v2(
                        row
                    )
                ),
            }],
            tokenize=False,
            add_generation_prompt=True,
        )
    )


def render_full_v2(row):

    return (
        tokenizer
        .apply_chat_template(
            [
                {
                    "role": "user",
                    "content": (
                        build_user_content_v2(
                            row
                        )
                    ),
                },
                {
                    "role": "assistant",
                    "content": (
                        row["label"]
                    ),
                },
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
    )


def token_ids(text):

    return tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]


# ------------------------------------------------------------
# G. Structural test
# ------------------------------------------------------------

sample = fit_rows[0]

sample_prompt = render_prompt_v2(
    sample
)

sample_full = render_full_v2(
    sample
)

assert sample_full.startswith(
    sample_prompt
)

sample_prompt_ids = token_ids(
    sample_prompt
)

sample_full_ids = token_ids(
    sample_full
)

assert (
    sample_full_ids[
        :len(sample_prompt_ids)
    ]
    == sample_prompt_ids
)

completion_ids = (
    sample_full_ids[
        len(sample_prompt_ids):
    ]
)

completion_text = tokenizer.decode(
    completion_ids,
    skip_special_tokens=False,
)

assert sample["label"] in completion_text


# ------------------------------------------------------------
# H. Measure all splits
# ------------------------------------------------------------

records = []


for split_name, rows in [
    ("fit", fit_rows),
    ("internal_dev", dev_rows),
    ("official_validation", val_rows),
]:

    for row in rows:

        prompt_text = (
            render_prompt_v2(row)
        )

        full_text = (
            render_full_v2(row)
        )

        assert full_text.startswith(
            prompt_text
        )

        prompt_ids = token_ids(
            prompt_text
        )

        full_ids = token_ids(
            full_text
        )

        assert (
            full_ids[
                :len(prompt_ids)
            ]
            == prompt_ids
        )

        completion_length = (
            len(full_ids)
            - len(prompt_ids)
        )

        assert completion_length in {
            4,
            8,
        }

        records.append({
            "split": split_name,
            "id": row["id"],
            "label": row["label"],
            "passages": len(
                row["evidence"]
            ),
            "prompt_tokens": len(
                prompt_ids
            ),
            "full_tokens": len(
                full_ids
            ),
            "completion_tokens": (
                completion_length
            ),
        })


assert len(records) == 1235


prompt_lengths = [
    row["prompt_tokens"]
    for row in records
]

full_lengths = [
    row["full_tokens"]
    for row in records
]


longest = max(
    records,
    key=lambda row: (
        row["full_tokens"]
    ),
)


top_10 = sorted(
    records,
    key=lambda row: (
        row["full_tokens"]
    ),
    reverse=True,
)[:10]


# ------------------------------------------------------------
# I. Select smallest safe context length
# ------------------------------------------------------------

observed_max = max(
    full_lengths
)

selected_max_length = None


for candidate in (
    CANDIDATE_MAX_LENGTHS
):

    if (
        candidate
        - observed_max
        >= MIN_SAFETY_MARGIN
    ):

        selected_max_length = (
            candidate
        )

        break


if selected_max_length is None:

    raise RuntimeError(
        "No candidate max_length provides "
        "the required safety margin."
    )


assert (
    selected_max_length
    >= observed_max
)


# ------------------------------------------------------------
# J. Compare Prompt V2 size with old V1 result
# ------------------------------------------------------------

OLD_V1_MAX = 216

extra_tokens_at_max = (
    observed_max
    - OLD_V1_MAX
)


# ------------------------------------------------------------
# K. Verify validation unchanged
# ------------------------------------------------------------

validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)

assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# L. Save A3 prompt preflight
# ------------------------------------------------------------

A3_PREFLIGHT_PATH = (
    WORK_ROOT
    / "a3_prompt_v2_preflight.json"
)


payload = {
    "experiment": "A3",
    "model": MODEL_ID,

    "principal_change_vs_a1": (
        "prompt only"
    ),

    "data": {
        "fit_rows": 795,
        "fit_sha256": (
            EXPECTED_FIT_SHA256
        ),
        "internal_dev_rows": 140,
        "internal_dev_sha256": (
            EXPECTED_DEV_SHA256
        ),
        "synthetic_training_examples": 0,
    },

    "prompt_version": (
        PROMPT_VERSION
    ),

    "prompt_text": (
        SYSTEM_INSTRUCTION_V2
    ),

    "token_lengths": {
        "rows_measured": len(
            records
        ),
        "prompt_min": min(
            prompt_lengths
        ),
        "prompt_p50": percentile(
            prompt_lengths,
            0.50,
        ),
        "prompt_p95": percentile(
            prompt_lengths,
            0.95,
        ),
        "prompt_p99": percentile(
            prompt_lengths,
            0.99,
        ),
        "prompt_max": max(
            prompt_lengths
        ),
        "full_min": min(
            full_lengths
        ),
        "full_p50": percentile(
            full_lengths,
            0.50,
        ),
        "full_p90": percentile(
            full_lengths,
            0.90,
        ),
        "full_p95": percentile(
            full_lengths,
            0.95,
        ),
        "full_p99": percentile(
            full_lengths,
            0.99,
        ),
        "full_max": observed_max,
        "old_v1_full_max": OLD_V1_MAX,
        "extra_tokens_vs_v1_max": (
            extra_tokens_at_max
        ),
    },

    "context_decision": {
        "selected_max_length": (
            selected_max_length
        ),
        "minimum_safety_margin": (
            MIN_SAFETY_MARGIN
        ),
        "actual_safety_margin": (
            selected_max_length
            - observed_max
        ),
        "truncation_required": False,
    },

    "longest_example": (
        longest
    ),

    "official_validation_modified": (
        False
    ),
}


A3_PREFLIGHT_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# M. Report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A3 PROMPT V2 PREFLIGHT — RESULT")
print("=" * 78)


print("\nScientific control")

print(
    "  Dataset:",
    "original V2 fit only",
)

print(
    "  Fit rows:",
    len(fit_rows),
)

print(
    "  Synthetic examples:",
    0,
)

print(
    "  Model changed:",
    False,
)

print(
    "  Prompt changed:",
    True,
)


print("\nPrompt tokens")

print(
    "  min:",
    min(prompt_lengths),
)

print(
    "  p50:",
    round(
        percentile(
            prompt_lengths,
            0.50,
        ),
        1,
    ),
)

print(
    "  p95:",
    round(
        percentile(
            prompt_lengths,
            0.95,
        ),
        1,
    ),
)

print(
    "  p99:",
    round(
        percentile(
            prompt_lengths,
            0.99,
        ),
        1,
    ),
)

print(
    "  max:",
    max(prompt_lengths),
)


print("\nFull training tokens")

print(
    "  min:",
    min(full_lengths),
)

print(
    "  p50:",
    round(
        percentile(
            full_lengths,
            0.50,
        ),
        1,
    ),
)

print(
    "  p90:",
    round(
        percentile(
            full_lengths,
            0.90,
        ),
        1,
    ),
)

print(
    "  p95:",
    round(
        percentile(
            full_lengths,
            0.95,
        ),
        1,
    ),
)

print(
    "  p99:",
    round(
        percentile(
            full_lengths,
            0.99,
        ),
        1,
    ),
)

print(
    "  max:",
    observed_max,
)


print("\nLongest example")

print(
    " ",
    longest,
)


print("\nTop 10 longest")

for row in top_10:

    print(
        " ",
        row["split"],
        "|",
        row["id"],
        "|",
        row["label"],
        "| full=",
        row["full_tokens"],
        "| prompt=",
        row["prompt_tokens"],
        "| passages=",
        row["passages"],
    )


print("\nPrompt V1 -> V2 context effect")

print(
    "  V1 observed max:",
    OLD_V1_MAX,
)

print(
    "  V2 observed max:",
    observed_max,
)

print(
    "  Increase:",
    f"{extra_tokens_at_max:+d}",
    "tokens",
)


print("\nContext decision")

print(
    "  Candidate lengths:",
    CANDIDATE_MAX_LENGTHS,
)

print(
    "  Required safety margin:",
    MIN_SAFETY_MARGIN,
)

print(
    "  Selected max_length:",
    selected_max_length,
)

print(
    "  Actual safety margin:",
    selected_max_length
    - observed_max,
)

print(
    "  Truncation required:",
    False,
)


print("\nCompletion-only structure")

print(
    "  Sample ID:",
    sample["id"],
)

print(
    "  Prompt tokens:",
    len(sample_prompt_ids),
)

print(
    "  Full tokens:",
    len(sample_full_ids),
)

print(
    "  Completion:",
    repr(
        completion_text
    ),
)


print("\nValidation integrity")

print(
    "  Modified:",
    False,
)

print(
    "  SHA-256:",
    validation_hash_after,
)


print("\nSaved")

print(
    " ",
    A3_PREFLIGHT_PATH,
)


print("\nNo training occurred.")
print(
    "Prompt V2 is frozen pending "
    "review of this output."
)

In [ ]:
# ============================================================
# STEP 14 — EXPERIMENT A3: PROMPT V2
#
# Principal change vs A1:
#   Prompt V1 -> Prompt V2
#
# Frozen from A1:
#   - original Dataset V2 fit = 795
#   - unchanged internal dev = 140
#   - google/gemma-2-2b-it
#   - 4-bit NF4
#   - LoRA r=8 / alpha=16 / dropout=0.05
#   - same 182 target modules
#   - physical batch = 1
#   - grad accumulation = 16
#   - LR = 2e-4
#   - weight decay = 0.01
#   - cosine scheduler
#   - warmup ratio = 5%
#   - 2 epochs
#   - seed = 42
#   - NO GradScaler
#
# Necessary context adjustment:
#   max_length 256 -> 384
#   because Prompt V2 is longer.
#
# Organizer validation is NOT evaluated here.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import time

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen A3 configuration
# ------------------------------------------------------------

RUN_NAME = (
    "a3_gemma2_2b_prompt_v2"
)

MODEL_ID = (
    "google/gemma-2-2b-it"
)

WORK_ROOT = Path(
    "/kaggle/working"
)

FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

PREFLIGHT_PATH = (
    WORK_ROOT
    / "a3_prompt_v2_preflight.json"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

RUN_DIR = (
    WORK_ROOT
    / RUN_NAME
)


SEED = 42

MAX_LENGTH = 384

EPOCHS = 2

PHYSICAL_BATCH = 1
GRAD_ACCUM = 16

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0


LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(
    LABELS
)


EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# Frozen reference results
A1_DEV_ACCURACY = 0.8214
A1_DEV_MACRO_F1 = 0.8249

A2_DEV_ACCURACY = 0.8286
A2_DEV_MACRO_F1 = 0.8334


# ------------------------------------------------------------
# B. Reproducibility
# ------------------------------------------------------------

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

torch.cuda.manual_seed_all(
    SEED
)


assert torch.cuda.is_available()


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            1,
        ):

            if not line.strip():
                continue

            try:

                rows.append(
                    json.loads(
                        line
                    )
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def directory_size_mb(path):

    total = 0

    for file in Path(
        path
    ).rglob("*"):

        if file.is_file():

            total += (
                file.stat().st_size
            )

    return (
        total
        / (1024 ** 2)
    )


# ------------------------------------------------------------
# D. Verify frozen A3 inputs
# ------------------------------------------------------------

assert FIT_PATH.exists()
assert DEV_PATH.exists()
assert PREFLIGHT_PATH.exists()


assert (
    sha256_file(
        FIT_PATH
    )
    == EXPECTED_FIT_SHA256
)

assert (
    sha256_file(
        DEV_PATH
    )
    == EXPECTED_DEV_SHA256
)


fit_rows = read_jsonl(
    FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)


assert len(
    fit_rows
) == 795

assert len(
    dev_rows
) == 140


assert Counter(
    row["label"]
    for row in fit_rows
) == Counter({
    "SUPPORTS": 295,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


preflight = json.loads(
    PREFLIGHT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    preflight[
        "context_decision"
    ][
        "selected_max_length"
    ]
    == MAX_LENGTH
)

assert (
    preflight[
        "token_lengths"
    ][
        "full_max"
    ]
    == 318
)


# ------------------------------------------------------------
# E. Protect previous experiments
# ------------------------------------------------------------

if RUN_DIR.exists():

    raise RuntimeError(
        f"{RUN_DIR} already exists. "
        "Do not overwrite an existing A3 run."
    )


RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ------------------------------------------------------------
# F. Hugging Face authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# G. Completely release previous A2/A1 model state
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "lora_config",
]:

    if object_name in globals():

        try:

            del globals()[
                object_name
            ]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


print("=" * 78)
print("A3 — CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ------------------------------------------------------------
# H. Tokenizer
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)


assert (
    tokenizer.chat_template
    is not None
)

assert (
    tokenizer.pad_token_id
    is not None
)


# ------------------------------------------------------------
# I. FROZEN PROMPT V2
# ------------------------------------------------------------

SYSTEM_INSTRUCTION_V2 = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        SYSTEM_INSTRUCTION_V2
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": (
                build_user_content(
                    row
                )
            ),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


def render_full(row):

    return tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": (
                    build_user_content(
                        row
                    )
                ),
            },
            {
                "role": "assistant",
                "content": (
                    row["label"]
                ),
            },
        ],
        tokenize=False,
        add_generation_prompt=False,
    )


# ------------------------------------------------------------
# J. Pre-tokenize completion-only training data
# ------------------------------------------------------------

training_examples = []


for row in fit_rows:

    prompt_text = (
        render_prompt(
            row
        )
    )

    full_text = (
        render_full(
            row
        )
    )


    assert full_text.startswith(
        prompt_text
    )


    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    assert (
        full_ids[
            :len(prompt_ids)
        ]
        == prompt_ids
    )


    assert (
        len(full_ids)
        <= MAX_LENGTH
    )


    labels = (
        [-100]
        * len(prompt_ids)
        + full_ids[
            len(prompt_ids):
        ]
    )


    supervised_tokens = sum(
        token != -100
        for token in labels
    )


    assert supervised_tokens in {
        4,
        8,
    }


    training_examples.append({
        "id": row["id"],

        "input_ids": torch.tensor(
            full_ids,
            dtype=torch.long,
        ),

        "attention_mask": torch.ones(
            len(full_ids),
            dtype=torch.long,
        ),

        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),

        "tokens": len(
            full_ids
        ),

        "supervised_tokens": (
            supervised_tokens
        ),
    })


assert len(
    training_examples
) == 795


longest_training_sequence = max(
    example["tokens"]
    for example in training_examples
)


assert (
    longest_training_sequence
    == 318
)


print(
    "\nTraining rows:",
    len(
        training_examples
    ),
)

print(
    "Longest training sequence:",
    longest_training_sequence,
)

print(
    "max_length:",
    MAX_LENGTH,
)

print(
    "Safety margin:",
    MAX_LENGTH
    - longest_training_sequence,
)


# ------------------------------------------------------------
# K. Load completely FRESH Gemma base
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.float16
    ),
)


print("\n" + "=" * 78)
print("A3 — FRESH MODEL LOAD")
print("=" * 78)


model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert model.is_loaded_in_4bit


model = (
    prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=False,
    )
)


model.config.use_cache = False


# ------------------------------------------------------------
# L. Attach fresh LoRA
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(
    model,
    lora_config,
)


trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


assert (
    trainable_params
    == 10_383_360
)


print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)


# ------------------------------------------------------------
# M. Gradient checkpointing
# ------------------------------------------------------------

def enable_training_checkpointing():

    model.config.use_cache = False

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

    if hasattr(
        model,
        "enable_input_require_grads",
    ):

        model.enable_input_require_grads()


def disable_training_checkpointing():

    try:

        model.gradient_checkpointing_disable()

    except Exception:
        pass


enable_training_checkpointing()


assert (
    model.is_gradient_checkpointing
)


# ------------------------------------------------------------
# N. Optimizer + SAME A1 schedule
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


optimizer_steps_per_epoch = math.ceil(
    len(training_examples)
    / GRAD_ACCUM
)


total_optimizer_steps = (
    optimizer_steps_per_epoch
    * EPOCHS
)


warmup_steps = max(
    1,
    round(
        total_optimizer_steps
        * WARMUP_RATIO
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=(
            warmup_steps
        ),
        num_training_steps=(
            total_optimizer_steps
        ),
    )
)


# Must exactly match A1.
assert (
    optimizer_steps_per_epoch
    == 50
)

assert (
    total_optimizer_steps
    == 100
)

assert (
    warmup_steps
    == 5
)


# ------------------------------------------------------------
# O. Deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)


assert isinstance(
    end_of_turn_id,
    int,
)


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:

        return cleaned

    return None


def evaluate_rows(
    rows,
    tag,
):

    disable_training_checkpointing()

    model.eval()

    model.config.use_cache = True


    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    gold = []
    predictions = []

    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(
                    row
                )
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded[
                    "input_ids"
                ]
                .to("cuda:0")
            )


            attention_mask = (
                encoded[
                    "attention_mask"
                ]
                .to("cuda:0")
            )


            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_text = (
                    tokenizer.decode(
                        token_ids,
                        skip_special_tokens=True,
                    )
                )


                prediction = (
                    parse_prediction(
                        raw_text
                    )
                )


                gold.append(
                    row["label"]
                )


                predictions.append(
                    prediction
                )


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )

        model.config.use_cache = False

        enable_training_checkpointing()

        model.train()


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(
            f1
        )
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,

        "rows": len(
            rows
        ),

        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            macro_f1
        ),

        "invalid_count": int(
            invalid_count
        ),

        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },

        "confusion_matrix": (
            matrix.tolist()
        ),

        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)


    print(
        "Rows:",
        len(rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )


    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  "
        "NOT_ENOUGH_INFO"
    )


    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print(
        "Per-class F1:"
    )


    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return result


# ------------------------------------------------------------
# P. Save exact A3 configuration
# ------------------------------------------------------------

run_config = {
    "experiment": RUN_NAME,

    "hypothesis": (
        "Explicit reasoning instructions for exact numerical, "
        "date, comparison, range, identity, and closed-world "
        "cases improve claim-verification generalization."
    ),

    "principal_change_vs_a1": (
        "Prompt V1 -> Prompt V2"
    ),

    "data": {
        "fit_rows": 795,
        "fit_sha256": (
            EXPECTED_FIT_SHA256
        ),
        "internal_dev_rows": 140,
        "internal_dev_sha256": (
            EXPECTED_DEV_SHA256
        ),
        "synthetic_examples": 0,
        "official_validation_used": False,
    },

    "prompt_version": (
        "v2_precise_reasoning_rules"
    ),

    "prompt_text": (
        SYSTEM_INSTRUCTION_V2
    ),

    "model": MODEL_ID,

    "max_length": MAX_LENGTH,

    "observed_max_sequence": (
        longest_training_sequence
    ),

    "quantization": {
        "bits": 4,
        "type": "NF4",
        "double_quant": True,
        "compute_dtype": "float16",
    },

    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "targets": (
            TARGET_MODULES
        ),
        "trainable_parameters": (
            trainable_params
        ),
    },

    "training": {
        "epochs": EPOCHS,
        "physical_batch": (
            PHYSICAL_BATCH
        ),
        "gradient_accumulation": (
            GRAD_ACCUM
        ),
        "learning_rate": (
            LEARNING_RATE
        ),
        "weight_decay": (
            WEIGHT_DECAY
        ),
        "warmup_ratio": (
            WARMUP_RATIO
        ),
        "warmup_steps": (
            warmup_steps
        ),
        "scheduler": "cosine",
        "total_optimizer_steps": (
            total_optimizer_steps
        ),
        "grad_scaler": False,
        "gradient_checkpointing": True,
        "gradient_checkpointing_use_reentrant": False,
        "seed": SEED,
    },

    "references": {
        "A1_dev_accuracy": (
            A1_DEV_ACCURACY
        ),
        "A1_dev_macro_f1": (
            A1_DEV_MACRO_F1
        ),
        "A2_dev_accuracy": (
            A2_DEV_ACCURACY
        ),
        "A2_dev_macro_f1": (
            A2_DEV_MACRO_F1
        ),
    },
}


(
    RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Q. PRETRAIN evaluation using Prompt V2
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A3 — PRE-TRAIN EVALUATION")
print("=" * 78)


all_metrics = []


pretrain_result = evaluate_rows(
    dev_rows,
    "INTERNAL DEV — A3 PRETRAIN",
)


all_metrics.append(
    pretrain_result
)


# ------------------------------------------------------------
# R. Training loop — identical optimization to A1
# ------------------------------------------------------------

global_optimizer_step = 0

training_summaries = []


for epoch_index in range(
    EPOCHS
):

    epoch_number = (
        epoch_index + 1
    )


    enable_training_checkpointing()

    model.train()


    generator = torch.Generator(
        device="cpu"
    )


    generator.manual_seed(
        SEED + epoch_index
    )


    order = torch.randperm(
        len(training_examples),
        generator=generator,
    ).tolist()


    epoch_loss_sum = 0.0
    epoch_example_count = 0


    epoch_start = (
        time.perf_counter()
    )


    print("\n" + "=" * 78)
    print(
        f"A3 TRAINING EPOCH "
        f"{epoch_number}/{EPOCHS}"
    )
    print("=" * 78)


    for window_start in range(
        0,
        len(order),
        GRAD_ACCUM,
    ):

        window_indices = order[
            window_start:
            window_start + GRAD_ACCUM
        ]


        window_size = len(
            window_indices
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        window_loss_sum = 0.0


        for index in window_indices:

            example = (
                training_examples[
                    index
                ]
            )


            input_ids = (
                example[
                    "input_ids"
                ]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )


            attention_mask = (
                example[
                    "attention_mask"
                ]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )


            labels = (
                example[
                    "labels"
                ]
                .unsqueeze(0)
                .to(
                    "cuda:0",
                    non_blocking=True,
                )
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    labels=labels,
                    use_cache=False,
                )

                raw_loss = (
                    outputs.loss
                )


            if not torch.isfinite(
                raw_loss
            ):

                raise RuntimeError(
                    f"Non-finite loss "
                    f"at epoch {epoch_number}, "
                    f"example {example['id']}"
                )


            raw_loss_value = float(
                raw_loss
                .detach()
                .cpu()
            )


            epoch_loss_sum += (
                raw_loss_value
            )

            window_loss_sum += (
                raw_loss_value
            )

            epoch_example_count += 1


            accumulation_loss = (
                raw_loss
                / window_size
            )


            accumulation_loss.backward()


            del outputs
            del raw_loss
            del accumulation_loss
            del input_ids
            del attention_mask
            del labels


        # ----------------------------------------------------
        # Finite-gradient check
        # ----------------------------------------------------

        nonfinite_gradient_names = []


        for name, parameter in (
            model.named_parameters()
        ):

            if (
                parameter.requires_grad
                and parameter.grad
                is not None
            ):

                if not torch.isfinite(
                    parameter.grad
                ).all():

                    nonfinite_gradient_names.append(
                        name
                    )


        if nonfinite_gradient_names:

            raise RuntimeError(
                "Non-finite gradients before "
                "optimizer step. First tensors: "
                f"{nonfinite_gradient_names[:10]}"
            )


        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                [
                    parameter
                    for parameter
                    in model.parameters()
                    if (
                        parameter.requires_grad
                        and parameter.grad
                        is not None
                    )
                ],
                MAX_GRAD_NORM,
            )
        )


        if not torch.isfinite(
            grad_norm
        ):

            raise RuntimeError(
                f"Non-finite gradient norm "
                f"at epoch {epoch_number}, "
                f"step "
                f"{global_optimizer_step + 1}"
            )


        optimizer.step()

        scheduler.step()


        global_optimizer_step += 1


        if (
            global_optimizer_step % 10
            == 0
            or window_start == 0
            or (
                window_start
                + window_size
                >= len(order)
            )
        ):

            current_lr = (
                scheduler
                .get_last_lr()[0]
            )


            mean_epoch_loss = (
                epoch_loss_sum
                / epoch_example_count
            )


            mean_window_loss = (
                window_loss_sum
                / window_size
            )


            print(
                f"epoch={epoch_number} | "
                f"step="
                f"{global_optimizer_step:3d}/"
                f"{total_optimizer_steps} | "
                f"examples="
                f"{epoch_example_count:3d}/"
                f"{len(training_examples)} | "
                f"window_loss="
                f"{mean_window_loss:.4f} | "
                f"epoch_loss="
                f"{mean_epoch_loss:.4f} | "
                f"lr="
                f"{current_lr:.7f} | "
                f"grad_norm="
                f"{float(grad_norm):.4f}"
            )


    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )


    epoch_mean_loss = (
        epoch_loss_sum
        / epoch_example_count
    )


    assert (
        epoch_example_count
        == 795
    )


    training_summaries.append({
        "epoch": epoch_number,

        "mean_example_loss": float(
            epoch_mean_loss
        ),

        "seconds": float(
            epoch_seconds
        ),

        "optimizer_step_end": int(
            global_optimizer_step
        ),
    })


    print(
        f"\nEpoch {epoch_number} "
        f"training complete"
    )

    print(
        "  Mean example loss:",
        f"{epoch_mean_loss:.4f}",
    )

    print(
        "  Seconds:",
        round(
            epoch_seconds,
            1,
        ),
    )


    # --------------------------------------------------------
    # Save checkpoint BEFORE evaluation
    # --------------------------------------------------------

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )


    model.save_pretrained(
        checkpoint_dir,
        safe_serialization=True,
    )


    tokenizer.save_pretrained(
        checkpoint_dir
    )


    print(
        "  Adapter saved:",
        checkpoint_dir,
    )

    print(
        "  Adapter size:",
        f"{directory_size_mb(checkpoint_dir):.1f} MB",
    )


    # --------------------------------------------------------
    # Predetermined real internal-dev evaluation
    # --------------------------------------------------------

    dev_result = evaluate_rows(
        dev_rows,
        (
            f"INTERNAL DEV — "
            f"A3 EPOCH {epoch_number}"
        ),
    )


    dev_result[
        "checkpoint"
    ] = str(
        checkpoint_dir
    )


    dev_result[
        "training_loss"
    ] = float(
        epoch_mean_loss
    )


    all_metrics.append(
        dev_result
    )


# ------------------------------------------------------------
# S. Verify complete run
# ------------------------------------------------------------

assert (
    global_optimizer_step
    == 100
)


# ------------------------------------------------------------
# T. Select checkpoint using INTERNAL DEV only
# ------------------------------------------------------------

trained_results = [
    result
    for result in all_metrics
    if "EPOCH" in result[
        "tag"
    ]
]


ranked = sorted(
    trained_results,
    key=lambda result: (
        result["macro_f1"],
        result["accuracy"],
        -result["invalid_count"],
    ),
    reverse=True,
)


best_a3 = ranked[0]


# ------------------------------------------------------------
# U. Save results
# ------------------------------------------------------------

results = {
    "experiment": RUN_NAME,

    "config": run_config,

    "training_summaries": (
        training_summaries
    ),

    "internal_dev_metrics": (
        all_metrics
    ),

    "selection_rule": (
        "Highest unchanged internal-dev Macro-F1; "
        "accuracy then invalid count as tie-breakers."
    ),

    "best_a3": (
        best_a3
    ),
}


RESULTS_PATH = (
    RUN_DIR
    / "results.json"
)


RESULTS_PATH.write_text(
    json.dumps(
        results,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# V. Deltas
# ------------------------------------------------------------

a3_vs_a1_macro_delta = (
    best_a3["macro_f1"]
    - A1_DEV_MACRO_F1
)

a3_vs_a1_accuracy_delta = (
    best_a3["accuracy"]
    - A1_DEV_ACCURACY
)


a3_vs_a2_macro_delta = (
    best_a3["macro_f1"]
    - A2_DEV_MACRO_F1
)


# ------------------------------------------------------------
# W. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A3 — RESULT")
print("=" * 78)


print("\nScientific comparison")

print(
    "  Principal change vs A1:",
    "Prompt V1 -> Prompt V2",
)

print(
    "  Fit rows:",
    len(fit_rows),
)

print(
    "  Synthetic examples:",
    0,
)

print(
    "  Internal dev unchanged:",
    True,
)

print(
    "  Model changed:",
    False,
)

print(
    "  LoRA changed:",
    False,
)

print(
    "  Training hyperparameters changed:",
    False,
)

print(
    "  max_length:",
    MAX_LENGTH,
    "(required by longer prompt)",
)

print(
    "  Organizer validation used:",
    False,
)


print("\nReference results")

print(
    f"  A1 | Accuracy="
    f"{A1_DEV_ACCURACY:.4f} | "
    f"Macro-F1="
    f"{A1_DEV_MACRO_F1:.4f}"
)

print(
    f"  A2 | Accuracy="
    f"{A2_DEV_ACCURACY:.4f} | "
    f"Macro-F1="
    f"{A2_DEV_MACRO_F1:.4f}"
)


print("\nA3 internal-dev checkpoints")

for result in all_metrics:

    print(
        f"  {result['tag']:28} | "
        f"Accuracy="
        f"{result['accuracy']:.4f} | "
        f"Macro-F1="
        f"{result['macro_f1']:.4f} | "
        f"Invalid="
        f"{result['invalid_count']}"
    )


print("\nTraining loss")

for summary in training_summaries:

    print(
        f"  epoch_{summary['epoch']} | "
        f"loss="
        f"{summary['mean_example_loss']:.4f} | "
        f"{summary['seconds']:.1f}s"
    )


print(
    "\nBest A3 checkpoint "
    "by unchanged INTERNAL DEV"
)

print(
    "  Checkpoint:",
    best_a3["tag"],
)

print(
    "  Accuracy:",
    f"{best_a3['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{best_a3['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    best_a3[
        "invalid_count"
    ],
)


print("\nA3 minus A1")

print(
    "  Accuracy delta:",
    f"{a3_vs_a1_accuracy_delta:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{a3_vs_a1_macro_delta:+.4f}",
)


print("\nA3 minus A2 internal-dev Macro-F1")

print(
    "  Delta:",
    f"{a3_vs_a2_macro_delta:+.4f}",
)


print("\nPer-class F1 — best A3")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{best_a3['per_class'][label]['f1']:.4f}"
    )


print("\nArtifacts")

for epoch_number in range(
    1,
    EPOCHS + 1,
):

    checkpoint_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )

    print(
        f"  epoch_{epoch_number}:",
        checkpoint_dir,
        f"({directory_size_mb(checkpoint_dir):.1f} MB)",
    )


print(
    "  Config:",
    RUN_DIR
    / "run_config.json",
)

print(
    "  Results:",
    RESULTS_PATH,
)


print("\nGPU")

print(
    "  GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


if torch.cuda.device_count() > 1:

    print(
        "  GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


print("\nA3 training completed.")
print(
    "Organizer validation was NOT evaluated."
)

In [ ]:
# ============================================================
# POST-A3 DIAGNOSTIC — PROMPT COMPARISON
#
# Compare three prompts:
#   V1       = original baseline prompt
#   V2_SHORT = concise reasoning prompt
#   V2_LONG  = long prompt used by A3
#
# Evaluate each on:
#   1. Fresh base Gemma
#   2. Saved A1 epoch-2 adapter
#
# INTERNAL DEV ONLY.
# NO training.
# NO organizer validation.
#
# Purpose:
#   Determine whether prompt engineering is genuinely useful
#   before deciding on A4 / model change.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import time

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Paths / frozen references
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

A1_ADAPTER_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
    / "epoch_2_adapter"
)

OUTPUT_PATH = (
    WORK_ROOT
    / "prompt_comparison_post_a3.json"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))

            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


# ------------------------------------------------------------
# C. Verify unchanged internal dev
# ------------------------------------------------------------

assert DEV_PATH.exists()

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

dev_rows = read_jsonl(
    DEV_PATH
)

assert len(dev_rows) == 140

assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


# ------------------------------------------------------------
# D. Prompt candidates
# ------------------------------------------------------------

PROMPT_V1 = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


PROMPT_V2_SHORT = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence establishes that the claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither. "
    "Missing support is not REFUTES.\n\n"

    "For numerical, date, range, or comparison claims, "
    "check the exact values and relations before deciding. "
    "Match the correct entity and time period. "
    "If the evidence allows both possibilities, "
    "use NOT_ENOUGH_INFO.\n\n"

    "Output only SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


PROMPT_V2_LONG = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


PROMPTS = {
    "V1": PROMPT_V1,
    "V2_SHORT": PROMPT_V2_SHORT,
    "V2_LONG": PROMPT_V2_LONG,
}


# ------------------------------------------------------------
# E. HF authentication
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:

    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Release A3 training objects now that A3 is finished
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "lora_config",
]:

    if object_name in globals():

        try:
            del globals()[object_name]

        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()


print("=" * 78)
print("PROMPT COMPARISON — CLEAN GPU STATE")
print("=" * 78)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


# ------------------------------------------------------------
# G. Tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    A1_ADAPTER_DIR,
    local_files_only=True,
)

assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# H. Prompt rendering
# ------------------------------------------------------------

def build_user_content(
    row,
    instruction,
):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        instruction
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(
    row,
    instruction,
):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(
                row,
                instruction,
            ),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


# ------------------------------------------------------------
# I. Exact token-length comparison
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A. PROMPT LENGTH COMPARISON")
print("=" * 78)


prompt_length_stats = {}


for prompt_name, instruction in PROMPTS.items():

    lengths = []

    for row in dev_rows:

        rendered = render_prompt(
            row,
            instruction,
        )

        ids = tokenizer(
            rendered,
            add_special_tokens=False,
        )["input_ids"]

        lengths.append(
            len(ids)
        )


    stats = {
        "min": min(lengths),
        "mean": float(
            np.mean(lengths)
        ),
        "median": float(
            np.median(lengths)
        ),
        "max": max(lengths),
    }

    prompt_length_stats[
        prompt_name
    ] = stats


    print(
        f"{prompt_name:10} | "
        f"min={stats['min']:3d} | "
        f"median={stats['median']:.1f} | "
        f"mean={stats['mean']:.1f} | "
        f"max={stats['max']:3d}"
    )


# ------------------------------------------------------------
# J. Deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)

assert isinstance(
    end_of_turn_id,
    int,
)


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_prompt(
    model,
    instruction,
    prompt_name,
    model_name,
):

    model.eval()
    model.config.use_cache = True

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"

    gold = []
    predictions = []

    start_time = time.perf_counter()


    try:

        for start in range(
            0,
            len(dev_rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = dev_rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(
                    row,
                    instruction,
                )
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded["input_ids"]
                .to("cuda:0")
            )

            attention_mask = (
                encoded["attention_mask"]
                .to("cuda:0")
            )

            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                decoded = tokenizer.decode(
                    token_ids,
                    skip_special_tokens=True,
                )

                prediction = (
                    parse_prediction(
                        decoded
                    )
                )

                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "model": model_name,
        "prompt": prompt_name,
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid": int(
            invalid_count
        ),
        "predictions": predictions,
        "gold": gold,
        "confusion_matrix": (
            matrix.tolist()
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)

    print(
        f"{model_name} | {prompt_name}"
    )

    print("-" * 78)

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )


    print(
        "Confusion matrix:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )

    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print("Per-class F1:")

    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    return result


# ------------------------------------------------------------
# K. Fresh 4-bit Gemma base
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.float16
    ),
)


print("\n" + "=" * 78)
print("B. FRESH BASE MODEL PROMPT TEST")
print("=" * 78)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert base_model.is_loaded_in_4bit


base_model = (
    prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=False,
    )
)


base_model.eval()
base_model.config.use_cache = True


base_results = {}


for prompt_name, instruction in (
    PROMPTS.items()
):

    base_results[
        prompt_name
    ] = evaluate_prompt(
        base_model,
        instruction,
        prompt_name,
        "BASE_GEMMA",
    )


# V1 must reproduce our known baseline behavior closely/exactly.
assert (
    round(
        base_results["V1"][
            "accuracy"
        ],
        4,
    )
    == 0.4214
)

assert (
    round(
        base_results["V1"][
            "macro_f1"
        ],
        4,
    )
    == 0.3615
)


print(
    "\nBase V1 reproduction: PASS"
)


# ------------------------------------------------------------
# L. Attach frozen A1 adapter
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("C. A1 ADAPTER PROMPT-SENSITIVITY TEST")
print("=" * 78)


model = PeftModel.from_pretrained(
    base_model,
    A1_ADAPTER_DIR,
    is_trainable=False,
)


model.eval()
model.config.use_cache = True


assert sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
) == 0


a1_results = {}


for prompt_name, instruction in (
    PROMPTS.items()
):

    a1_results[
        prompt_name
    ] = evaluate_prompt(
        model,
        instruction,
        prompt_name,
        "A1_ADAPTER",
    )


# Critical reproduction check.
assert (
    round(
        a1_results["V1"][
            "accuracy"
        ],
        4,
    )
    == 0.8214
)

assert (
    round(
        a1_results["V1"][
            "macro_f1"
        ],
        4,
    )
    == 0.8249
)


print(
    "\nA1 + V1 reproduction: PASS"
)


# ------------------------------------------------------------
# M. Pairwise internal-dev changes relative to V1
# ------------------------------------------------------------

def paired_changes(
    reference,
    candidate,
):

    fixed = 0
    broken = 0
    unchanged_correct = 0
    unchanged_wrong = 0


    for gold, ref_pred, cand_pred in zip(
        reference["gold"],
        reference["predictions"],
        candidate["predictions"],
    ):

        ref_correct = (
            ref_pred == gold
        )

        cand_correct = (
            cand_pred == gold
        )


        if (
            not ref_correct
            and cand_correct
        ):
            fixed += 1

        elif (
            ref_correct
            and not cand_correct
        ):
            broken += 1

        elif (
            ref_correct
            and cand_correct
        ):
            unchanged_correct += 1

        else:
            unchanged_wrong += 1


    return {
        "fixed": fixed,
        "broken": broken,
        "unchanged_correct": (
            unchanged_correct
        ),
        "unchanged_wrong": (
            unchanged_wrong
        ),
    }


a1_short_changes = paired_changes(
    a1_results["V1"],
    a1_results["V2_SHORT"],
)

a1_long_changes = paired_changes(
    a1_results["V1"],
    a1_results["V2_LONG"],
)


# ------------------------------------------------------------
# N. Final comparison
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PROMPT COMPARISON — RESULT")
print("=" * 78)


print("\nPrompt lengths")

for prompt_name in PROMPTS:

    stats = (
        prompt_length_stats[
            prompt_name
        ]
    )

    print(
        f"  {prompt_name:10} | "
        f"median={stats['median']:.1f} | "
        f"max={stats['max']}"
    )


print("\nFresh base Gemma")

for prompt_name in PROMPTS:

    result = (
        base_results[
            prompt_name
        ]
    )

    print(
        f"  {prompt_name:10} | "
        f"Accuracy="
        f"{result['accuracy']:.4f} | "
        f"Macro-F1="
        f"{result['macro_f1']:.4f}"
    )


print("\nFrozen A1 adapter")

for prompt_name in PROMPTS:

    result = (
        a1_results[
            prompt_name
        ]
    )

    print(
        f"  {prompt_name:10} | "
        f"Accuracy="
        f"{result['accuracy']:.4f} | "
        f"Macro-F1="
        f"{result['macro_f1']:.4f} | "
        f"SUP="
        f"{result['per_class']['SUPPORTS']['f1']:.4f} | "
        f"REF="
        f"{result['per_class']['REFUTES']['f1']:.4f} | "
        f"NEI="
        f"{result['per_class']['NOT_ENOUGH_INFO']['f1']:.4f}"
    )


print(
    "\nA1 prediction changes vs V1"
)

print(
    "  V2_SHORT:",
    a1_short_changes,
)

print(
    "  V2_LONG :",
    a1_long_changes,
)


# ------------------------------------------------------------
# O. Rank prompts diagnostically
# ------------------------------------------------------------

base_ranking = sorted(
    base_results.values(),
    key=lambda result: (
        result["macro_f1"],
        result["accuracy"],
    ),
    reverse=True,
)


a1_ranking = sorted(
    a1_results.values(),
    key=lambda result: (
        result["macro_f1"],
        result["accuracy"],
    ),
    reverse=True,
)


print("\nDiagnostic ranking")

print(
    "  Best prompt on BASE:",
    base_ranking[0]["prompt"],
    "| Macro-F1=",
    f"{base_ranking[0]['macro_f1']:.4f}",
)

print(
    "  Best prompt on A1:",
    a1_ranking[0]["prompt"],
    "| Macro-F1=",
    f"{a1_ranking[0]['macro_f1']:.4f}",
)


# ------------------------------------------------------------
# P. Save report
# ------------------------------------------------------------

def strip_predictions(result):

    return {
        key: value
        for key, value in result.items()
        if key not in {
            "predictions",
            "gold",
        }
    }


payload = {
    "purpose": (
        "Post-A3 prompt comparison on unchanged "
        "internal dev only."
    ),

    "organizer_validation_used": False,

    "prompt_texts": PROMPTS,

    "prompt_length_stats": (
        prompt_length_stats
    ),

    "base_results": {
        name: strip_predictions(
            result
        )
        for name, result in (
            base_results.items()
        )
    },

    "a1_results": {
        name: strip_predictions(
            result
        )
        for name, result in (
            a1_results.items()
        )
    },

    "a1_pairwise_changes": {
        "V2_SHORT_vs_V1": (
            a1_short_changes
        ),
        "V2_LONG_vs_V1": (
            a1_long_changes
        ),
    },
}


OUTPUT_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("\nSaved")

print(
    " ",
    OUTPUT_PATH,
)


print("\nNo training occurred.")
print(
    "Organizer validation was NOT used."
)

In [ ]:
# ============================================================
# STEP 15 — A3 FRESH RELOAD + VALIDATION EVALUATION #3
#           + A1 / A2 / A3 CONSISTENT COMPARISON
#
# Purpose:
#   1. Verify saved A3 epoch-2 adapter integrity
#   2. Reload A3 from a fresh Gemma base
#   3. Run finite-logit numerical preflight
#   4. Reproduce A3 internal-dev score exactly
#   5. Evaluate organizer validation
#   6. Compare A1 vs A2 vs A3 prediction-by-prediction
#   7. Compare task-family performance
#   8. Quantify paired A3-vs-A1 / A3-vs-A2 differences
#
# IMPORTANT:
#   Organizer validation evaluation #3 is ADAPTIVE.
#   It is not a pristine unseen estimate anymore.
#
# NO TRAINING.
# NO DATA MODIFICATION.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import gc
import hashlib
import json
import math
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen experiment paths
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")


# A1
A1_RUN_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

A1_PRED_PATH = (
    A1_RUN_DIR
    / "official_validation_eval1_epoch2.csv"
)


# A2
A2_RUN_DIR = (
    WORK_ROOT
    / "a2_gemma2_2b_targeted_augmentation_v1"
)

A2_PRED_PATH = (
    A2_RUN_DIR
    / "official_validation_eval2_epoch2.csv"
)


# A3
A3_RUN_DIR = (
    WORK_ROOT
    / "a3_gemma2_2b_prompt_v2"
)

A3_ADAPTER_DIR = (
    A3_RUN_DIR
    / "epoch_2_adapter"
)


# Frozen dev
DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)


EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)


# Expected A3 result selected BEFORE validation
EXPECTED_A3_DEV_ACCURACY = 0.8571
EXPECTED_A3_DEV_MACRO_F1 = 0.8580


# Previously frozen organizer-validation results
A1_VAL_ACCURACY = 0.8200
A1_VAL_MACRO_F1 = 0.8207

A2_VAL_ACCURACY = 0.8033
A2_VAL_MACRO_F1 = 0.8051


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(
            f,
            1,
        ):

            if not line.strip():
                continue

            try:
                rows.append(
                    json.loads(line)
                )

            except Exception as exc:
                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def find_by_hash(
    filename,
    expected_hash,
):

    matches = []

    for path in INPUT_ROOT.rglob(
        filename
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == expected_hash
        ):
            matches.append(path)


    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one "
            f"{filename} with frozen hash. "
            f"Found: {matches}"
        )

    return matches[0]


def clean(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def exact_mcnemar_pvalue(
    fixed,
    broken,
):

    """
    Exact two-sided paired binomial / McNemar p-value.

    fixed:
        reference wrong -> candidate correct

    broken:
        reference correct -> candidate wrong
    """

    n = fixed + broken

    if n == 0:
        return 1.0

    smaller = min(
        fixed,
        broken,
    )

    tail = sum(
        math.comb(n, k)
        for k in range(
            smaller + 1
        )
    ) / (2 ** n)

    return min(
        1.0,
        2.0 * tail,
    )


# ------------------------------------------------------------
# C. Verify saved A3 artifact
# ------------------------------------------------------------

print("=" * 78)
print("A. A3 SAVED-ADAPTER INTEGRITY")
print("=" * 78)


assert A3_ADAPTER_DIR.exists()

adapter_model_path = (
    A3_ADAPTER_DIR
    / "adapter_model.safetensors"
)

adapter_config_path = (
    A3_ADAPTER_DIR
    / "adapter_config.json"
)

assert adapter_model_path.exists()
assert adapter_config_path.exists()


adapter_config = json.loads(
    adapter_config_path.read_text(
        encoding="utf-8"
    )
)


adapter_hash = sha256_file(
    adapter_model_path
)


print(
    "Adapter:",
    A3_ADAPTER_DIR,
)

print(
    "Adapter model size:",
    round(
        adapter_model_path.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "Adapter SHA-256:",
    adapter_hash,
)

print(
    "Base:",
    adapter_config.get(
        "base_model_name_or_path"
    ),
)

print(
    "LoRA r:",
    adapter_config.get("r"),
)

print(
    "LoRA alpha:",
    adapter_config.get(
        "lora_alpha"
    ),
)


assert (
    adapter_config.get(
        "base_model_name_or_path"
    )
    == MODEL_ID
)

assert adapter_config.get("r") == 8

assert (
    adapter_config.get(
        "lora_alpha"
    )
    == 16
)


# ------------------------------------------------------------
# D. Verify frozen evaluation data
# ------------------------------------------------------------

assert DEV_PATH.exists()

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)


dev_rows = read_jsonl(
    DEV_PATH
)

assert len(dev_rows) == 140


VAL_PATH = find_by_hash(
    "validation.jsonl",
    EXPECTED_VAL_SHA256,
)

validation_hash_before = (
    sha256_file(
        VAL_PATH
    )
)

val_rows = read_jsonl(
    VAL_PATH
)

assert len(val_rows) == 300


# ------------------------------------------------------------
# E. Load frozen A1/A2 prediction records
# ------------------------------------------------------------

assert A1_PRED_PATH.exists()
assert A2_PRED_PATH.exists()


a1_df = pd.read_csv(
    A1_PRED_PATH
)

a2_df = pd.read_csv(
    A2_PRED_PATH
)


assert len(a1_df) == 300
assert len(a2_df) == 300

assert a1_df["id"].is_unique
assert a2_df["id"].is_unique


a1_pred_by_id = {
    row["id"]: row["prediction"]
    for row in a1_df.to_dict(
        orient="records"
    )
}

a2_pred_by_id = {
    row["id"]: row["prediction"]
    for row in a2_df.to_dict(
        orient="records"
    )
}


validation_ids = {
    row["id"]
    for row in val_rows
}


assert (
    set(a1_pred_by_id)
    == validation_ids
)

assert (
    set(a2_pred_by_id)
    == validation_ids
)


# ------------------------------------------------------------
# F. HF authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# G. Release A3 training model from memory
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "lora_config",
]:

    if object_name in globals():

        try:
            del globals()[object_name]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


print("\n" + "=" * 78)
print("B. CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU 1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ------------------------------------------------------------
# H. Tokenizer from SAVED A3 artifact
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        A3_ADAPTER_DIR,
        local_files_only=True,
    )
)


assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# I. Exact frozen A3 Prompt V2
# ------------------------------------------------------------

PROMPT_V2_LONG = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


def build_user_content(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    return (
        PROMPT_V2_LONG
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(row):

    return tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": build_user_content(
                row
            ),
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


# ------------------------------------------------------------
# J. Fresh Gemma base + saved A3 adapter
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(
        torch.float16
    ),
)


print("\n" + "=" * 78)
print("C. FRESH A3 RELOAD")
print("=" * 78)


base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


assert base_model.is_loaded_in_4bit


base_model = (
    prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=False,
    )
)


base_model.config.use_cache = True


model = PeftModel.from_pretrained(
    base_model,
    A3_ADAPTER_DIR,
    is_trainable=False,
)


model.eval()
model.config.use_cache = True


trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


assert trainable_params == 0


print(
    "Adapter loaded:",
    True,
)

print(
    "Inference trainable params:",
    trainable_params,
)

print(
    "GPU 0 allocated:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


# ------------------------------------------------------------
# K. Numerical preflight
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("D. A3 NUMERICAL PREFLIGHT")
print("=" * 78)


preflight_rows = [
    dev_rows[0],
    dev_rows[
        len(dev_rows) // 2
    ],
    dev_rows[-1],
]


for row in preflight_rows:

    encoded = tokenizer(
        render_prompt(row),
        add_special_tokens=False,
        return_tensors="pt",
    )


    input_ids = (
        encoded["input_ids"]
        .to("cuda:0")
    )

    attention_mask = (
        encoded["attention_mask"]
        .to("cuda:0")
    )


    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=True,
        )


    final_logits = (
        outputs.logits[
            0,
            -1,
            :
        ]
    )


    finite = bool(
        torch.isfinite(
            final_logits
        ).all().item()
    )


    print(
        row["id"],
        "| finite:",
        finite,
        "| dtype:",
        final_logits.dtype,
    )


    assert finite


    del encoded
    del input_ids
    del attention_mask
    del outputs
    del final_logits


torch.cuda.empty_cache()


# ------------------------------------------------------------
# L. Deterministic evaluator
# ------------------------------------------------------------

end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)


assert isinstance(
    end_of_turn_id,
    int,
)


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


def evaluate_rows(
    rows,
    tag,
):

    model.eval()
    model.config.use_cache = True


    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    gold = []
    predictions = []
    records = []


    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(row)
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded["input_ids"]
                .to("cuda:0")
            )


            attention_mask = (
                encoded["attention_mask"]
                .to("cuda:0")
            )


            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for row, token_ids in zip(
                batch_rows,
                new_tokens,
            ):

                raw_output = (
                    tokenizer.decode(
                        token_ids,
                        skip_special_tokens=True,
                    )
                )


                prediction = (
                    parse_prediction(
                        raw_output
                    )
                )


                gold.append(
                    row["label"]
                )

                predictions.append(
                    prediction
                )


                records.append({
                    "id": row["id"],
                    "gold": row["label"],
                    "prediction": prediction,
                    "raw_output": raw_output,
                })


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )


    invalid_count = sum(
        prediction is None
        for prediction in predictions
    )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            gold,
            metric_predictions,
            labels=LABELS,
            zero_division=0,
        )
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,
        "rows": len(rows),
        "accuracy": float(
            accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "invalid_count": int(
            invalid_count
        ),
        "confusion_matrix": (
            matrix.tolist()
        ),
        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)

    print(
        "Rows:",
        len(rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid_count,
    )


    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )


    for label, values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            values,
        )


    print("Per-class F1:")


    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return (
        result,
        records,
    )


# ------------------------------------------------------------
# M. Fresh-reload INTERNAL DEV reproduction
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("E. A3 FRESH-RELOAD REPRODUCTION")
print("=" * 78)


(
    dev_result,
    dev_records,
) = evaluate_rows(
    dev_rows,
    "INTERNAL DEV — A3 EPOCH 2 FRESH RELOAD",
)


assert (
    round(
        dev_result["accuracy"],
        4,
    )
    == EXPECTED_A3_DEV_ACCURACY
)


assert (
    round(
        dev_result["macro_f1"],
        4,
    )
    == EXPECTED_A3_DEV_MACRO_F1
)


assert (
    dev_result[
        "invalid_count"
    ]
    == 0
)


print(
    "\nA3 internal-dev reproduction: PASS"
)


# ------------------------------------------------------------
# N. Organizer validation evaluation #3
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("F. ORGANIZER VALIDATION — ADAPTIVE EVALUATION #3")
print("=" * 78)


(
    val_result,
    val_records,
) = evaluate_rows(
    val_rows,
    "ORGANIZER VALIDATION — A3 EPOCH 2",
)


validation_hash_after = (
    sha256_file(
        VAL_PATH
    )
)


assert (
    validation_hash_before
    == validation_hash_after
    == EXPECTED_VAL_SHA256
)


# ------------------------------------------------------------
# O. Task-family classifier
# ------------------------------------------------------------

def joined_evidence(row):

    return clean(
        " ".join(
            row["evidence"]
        )
    ).lower()


def classify_family(row):

    claim = clean(
        row["claim"]
    ).lower()

    evidence = joined_evidence(
        row
    )


    if (
        "year-on-year revenue growth"
        in claim
        or (
            "revenue growth"
            in claim
            and "%"
            in claim
        )
    ):

        return (
            "revenue_percentage_calculation"
        )


    if (
        "two named sites jointly made"
        in claim
        or (
            "jointly"
            in claim
            and "units"
            in claim
        )
    ):

        return "two_component_sum"


    if (
        "depot was busier than"
        in claim
        or (
            "depot"
            in claim
            and (
                "more parcels than"
                in claim
                or "busier"
                in claim
            )
        )
    ):

        return "paired_comparison"


    if (
        "reached a minimum of"
        in claim
        or "reached a maximum of"
        in claim
        or (
            "megalitres"
            in claim
            and (
                "between"
                in evidence
                or "bounds"
                in evidence
                or "margin of error"
                in evidence
            )
        )
    ):

        return (
            "range_or_measurement_bounds"
        )


    if (
        "office-holder directing"
        in claim
    ):

        return "temporal_office_holder"


    if (
        "project"
        in claim
        and (
            "based in"
            in claim
            or "carried out"
            in claim
            or "located in"
            in claim
        )
    ):

        return "project_location_mapping"


    if (
        "mission payload"
        in claim
        or "left out the"
        in claim
    ):

        return (
            "complete_manifest_membership"
        )


    if (
        "safety audit"
        in claim
        or (
            "audit"
            in claim
            and (
                "failed"
                in claim
                or "passed"
                in claim
            )
        )
    ):

        return "audit_status"


    if (
        "employed exactly"
        in claim
        or (
            "employees"
            in claim
            and "exactly"
            in claim
        )
    ):

        return "exact_staffing_count"


    if (
        "full-year service"
        in claim
    ):

        return "service_availability"


    if "exactly" in claim:

        return "other_exact_value"


    return "other"


# ------------------------------------------------------------
# P. Build three-model prediction maps
# ------------------------------------------------------------

a3_pred_by_id = {
    record["id"]: (
        record["prediction"]
    )
    for record in val_records
}


assert (
    set(a3_pred_by_id)
    == validation_ids
)


# ------------------------------------------------------------
# Q. Full A1/A2/A3 paired comparison
# ------------------------------------------------------------

comparison_rows = []

family_stats = defaultdict(
    lambda: {
        "total": 0,
        "a1_correct": 0,
        "a2_correct": 0,
        "a3_correct": 0,
        "a3_fixed_a1": 0,
        "a3_broke_a1": 0,
        "a3_fixed_a2": 0,
        "a3_broke_a2": 0,
    }
)


a3_vs_a1 = Counter()
a3_vs_a2 = Counter()

all_three_correct = 0
all_three_wrong = 0

a1_only_correct = 0
a2_only_correct = 0
a3_only_correct = 0


for row in val_rows:

    example_id = row["id"]
    gold = row["label"]

    a1_pred = (
        a1_pred_by_id[
            example_id
        ]
    )

    a2_pred = (
        a2_pred_by_id[
            example_id
        ]
    )

    a3_pred = (
        a3_pred_by_id[
            example_id
        ]
    )


    a1_correct = (
        a1_pred == gold
    )

    a2_correct = (
        a2_pred == gold
    )

    a3_correct = (
        a3_pred == gold
    )


    # A3 vs A1
    if (
        not a1_correct
        and a3_correct
    ):
        a3_vs_a1[
            "FIXED"
        ] += 1

    elif (
        a1_correct
        and not a3_correct
    ):
        a3_vs_a1[
            "BROKE"
        ] += 1

    elif (
        a1_correct
        and a3_correct
    ):
        a3_vs_a1[
            "BOTH_CORRECT"
        ] += 1

    else:
        a3_vs_a1[
            "BOTH_WRONG"
        ] += 1


    # A3 vs A2
    if (
        not a2_correct
        and a3_correct
    ):
        a3_vs_a2[
            "FIXED"
        ] += 1

    elif (
        a2_correct
        and not a3_correct
    ):
        a3_vs_a2[
            "BROKE"
        ] += 1

    elif (
        a2_correct
        and a3_correct
    ):
        a3_vs_a2[
            "BOTH_CORRECT"
        ] += 1

    else:
        a3_vs_a2[
            "BOTH_WRONG"
        ] += 1


    if (
        a1_correct
        and a2_correct
        and a3_correct
    ):
        all_three_correct += 1


    if (
        not a1_correct
        and not a2_correct
        and not a3_correct
    ):
        all_three_wrong += 1


    if (
        a1_correct
        and not a2_correct
        and not a3_correct
    ):
        a1_only_correct += 1


    if (
        a2_correct
        and not a1_correct
        and not a3_correct
    ):
        a2_only_correct += 1


    if (
        a3_correct
        and not a1_correct
        and not a2_correct
    ):
        a3_only_correct += 1


    family = classify_family(
        row
    )

    stats = family_stats[
        family
    ]

    stats["total"] += 1

    stats["a1_correct"] += int(
        a1_correct
    )

    stats["a2_correct"] += int(
        a2_correct
    )

    stats["a3_correct"] += int(
        a3_correct
    )


    stats["a3_fixed_a1"] += int(
        (
            not a1_correct
            and a3_correct
        )
    )

    stats["a3_broke_a1"] += int(
        (
            a1_correct
            and not a3_correct
        )
    )


    stats["a3_fixed_a2"] += int(
        (
            not a2_correct
            and a3_correct
        )
    )

    stats["a3_broke_a2"] += int(
        (
            a2_correct
            and not a3_correct
        )
    )


    comparison_rows.append({
        "id": example_id,
        "family": family,
        "gold": gold,

        "a1_prediction": a1_pred,
        "a2_prediction": a2_pred,
        "a3_prediction": a3_pred,

        "a1_correct": a1_correct,
        "a2_correct": a2_correct,
        "a3_correct": a3_correct,

        "claim": row["claim"],
    })


# ------------------------------------------------------------
# R. Task-family table
# ------------------------------------------------------------

family_results = []


for family, stats in (
    family_stats.items()
):

    n = stats[
        "total"
    ]


    family_results.append({
        "family": family,
        "n": n,

        "a1_accuracy": (
            stats["a1_correct"]
            / n
        ),

        "a2_accuracy": (
            stats["a2_correct"]
            / n
        ),

        "a3_accuracy": (
            stats["a3_correct"]
            / n
        ),

        "a3_vs_a1_delta": (
            (
                stats["a3_correct"]
                - stats["a1_correct"]
            )
            / n
        ),

        "a3_vs_a2_delta": (
            (
                stats["a3_correct"]
                - stats["a2_correct"]
            )
            / n
        ),

        "a3_fixed_a1": (
            stats["a3_fixed_a1"]
        ),

        "a3_broke_a1": (
            stats["a3_broke_a1"]
        ),

        "a3_fixed_a2": (
            stats["a3_fixed_a2"]
        ),

        "a3_broke_a2": (
            stats["a3_broke_a2"]
        ),
    })


family_results.sort(
    key=lambda item: (
        item["a3_vs_a1_delta"],
        item["n"],
    ),
    reverse=True,
)


# ------------------------------------------------------------
# S. Paired exact significance diagnostics
# ------------------------------------------------------------

a3_vs_a1_p = (
    exact_mcnemar_pvalue(
        a3_vs_a1["FIXED"],
        a3_vs_a1["BROKE"],
    )
)


a3_vs_a2_p = (
    exact_mcnemar_pvalue(
        a3_vs_a2["FIXED"],
        a3_vs_a2["BROKE"],
    )
)


# ------------------------------------------------------------
# T. Save A3 predictions and consolidated comparison
# ------------------------------------------------------------

A3_PRED_PATH = (
    A3_RUN_DIR
    / "official_validation_eval3_epoch2.csv"
)

COMPARISON_PATH = (
    A3_RUN_DIR
    / "a1_a2_a3_official_validation_comparison.csv"
)

SUMMARY_PATH = (
    A3_RUN_DIR
    / "official_validation_eval3_epoch2_summary.json"
)


pd.DataFrame([
    {
        "id": record["id"],
        "gold": record["gold"],
        "prediction": (
            record["prediction"]
        ),
        "correct": (
            record["gold"]
            == record["prediction"]
        ),
        "raw_output": (
            record["raw_output"]
        ),
    }
    for record in val_records
]).to_csv(
    A3_PRED_PATH,
    index=False,
)


pd.DataFrame(
    comparison_rows
).to_csv(
    COMPARISON_PATH,
    index=False,
)


summary_payload = {
    "evaluation_number": 3,

    "adaptive": True,

    "adaptive_reason": (
        "Earlier organizer-validation results and "
        "error analyses informed subsequent "
        "experiment design."
    ),

    "a3_adapter_sha256": (
        adapter_hash
    ),

    "fresh_reload_internal_dev": (
        dev_result
    ),

    "a3_official_validation": (
        val_result
    ),

    "previous_official_validation": {
        "A1": {
            "accuracy": (
                A1_VAL_ACCURACY
            ),
            "macro_f1": (
                A1_VAL_MACRO_F1
            ),
        },
        "A2": {
            "accuracy": (
                A2_VAL_ACCURACY
            ),
            "macro_f1": (
                A2_VAL_MACRO_F1
            ),
        },
    },

    "paired_a3_vs_a1": {
        **dict(
            a3_vs_a1
        ),
        "exact_mcnemar_p": (
            a3_vs_a1_p
        ),
    },

    "paired_a3_vs_a2": {
        **dict(
            a3_vs_a2
        ),
        "exact_mcnemar_p": (
            a3_vs_a2_p
        ),
    },

    "three_model_overlap": {
        "all_three_correct": (
            all_three_correct
        ),
        "all_three_wrong": (
            all_three_wrong
        ),
        "a1_only_correct": (
            a1_only_correct
        ),
        "a2_only_correct": (
            a2_only_correct
        ),
        "a3_only_correct": (
            a3_only_correct
        ),
    },

    "task_family_comparison": (
        family_results
    ),

    "validation_sha256": (
        validation_hash_after
    ),
}


SUMMARY_PATH.write_text(
    json.dumps(
        summary_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# U. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("A3 — FINAL VALIDATION RESULT")
print("=" * 78)


print("\nFresh reload")

print(
    "  Adapter portability:",
    "PASS",
)

print(
    "  Finite logits:",
    "PASS",
)

print(
    "  Internal-dev reproduction:",
    "PASS",
)

print(
    "  Adapter SHA-256:",
    adapter_hash,
)


print("\nInternal dev")

print(
    "  A1 Macro-F1:",
    "0.8249",
)

print(
    "  A2 Macro-F1:",
    "0.8334",
)

print(
    "  A3 Macro-F1:",
    f"{dev_result['macro_f1']:.4f}",
)


print("\nOrganizer validation")
print(
    "  NOTE:",
    "evaluation #3 is adaptive",
)

print(
    f"  A1 | Accuracy="
    f"{A1_VAL_ACCURACY:.4f} | "
    f"Macro-F1="
    f"{A1_VAL_MACRO_F1:.4f}"
)

print(
    f"  A2 | Accuracy="
    f"{A2_VAL_ACCURACY:.4f} | "
    f"Macro-F1="
    f"{A2_VAL_MACRO_F1:.4f}"
)

print(
    f"  A3 | Accuracy="
    f"{val_result['accuracy']:.4f} | "
    f"Macro-F1="
    f"{val_result['macro_f1']:.4f}"
)


print("\nA3 minus A1")

print(
    "  Accuracy delta:",
    f"{val_result['accuracy'] - A1_VAL_ACCURACY:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{val_result['macro_f1'] - A1_VAL_MACRO_F1:+.4f}",
)


print("\nA3 organizer-validation per-class F1")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{val_result['per_class'][label]['f1']:.4f}"
    )


print("\nA3 vs A1 paired transitions")

print(
    "  A3 fixed A1 errors:",
    a3_vs_a1["FIXED"],
)

print(
    "  A3 broke A1 correct:",
    a3_vs_a1["BROKE"],
)

print(
    "  Both correct:",
    a3_vs_a1["BOTH_CORRECT"],
)

print(
    "  Both wrong:",
    a3_vs_a1["BOTH_WRONG"],
)

print(
    "  Exact paired p-value:",
    f"{a3_vs_a1_p:.6f}",
)


print("\nA3 vs A2 paired transitions")

print(
    "  A3 fixed A2 errors:",
    a3_vs_a2["FIXED"],
)

print(
    "  A3 broke A2 correct:",
    a3_vs_a2["BROKE"],
)

print(
    "  Both correct:",
    a3_vs_a2["BOTH_CORRECT"],
)

print(
    "  Both wrong:",
    a3_vs_a2["BOTH_WRONG"],
)

print(
    "  Exact paired p-value:",
    f"{a3_vs_a2_p:.6f}",
)


print("\nThree-model overlap")

print(
    "  All three correct:",
    all_three_correct,
)

print(
    "  All three wrong:",
    all_three_wrong,
)

print(
    "  A1 only correct:",
    a1_only_correct,
)

print(
    "  A2 only correct:",
    a2_only_correct,
)

print(
    "  A3 only correct:",
    a3_only_correct,
)


print("\nTask-family comparison")

print(
    f"{'Family':34} "
    f"{'N':>3} "
    f"{'A1':>7} "
    f"{'A2':>7} "
    f"{'A3':>7} "
    f"{'A3-A1':>8} "
    f"{'Fix':>4} "
    f"{'Break':>5}"
)


for item in family_results:

    print(
        f"{item['family']:34} "
        f"{item['n']:3d} "
        f"{item['a1_accuracy']:7.1%} "
        f"{item['a2_accuracy']:7.1%} "
        f"{item['a3_accuracy']:7.1%} "
        f"{item['a3_vs_a1_delta']:+7.1%} "
        f"{item['a3_fixed_a1']:4d} "
        f"{item['a3_broke_a1']:5d}"
    )


print("\nValidation integrity")

print(
    "  Modified:",
    False,
)

print(
    "  SHA-256:",
    validation_hash_after,
)


print("\nSaved")

print(
    " ",
    A3_PRED_PATH,
)

print(
    " ",
    COMPARISON_PATH,
)

print(
    " ",
    SUMMARY_PATH,
)


print("\nNo training occurred.")
print(
    "A3 is now fully evaluated and "
    "fresh-reload verified."
)

In [ ]:
# ============================================================
# STEP 16 — CLOSE GEMMA-2-2B EXPERIMENT PHASE
#
# Purpose:
#   - Freeze A1/A2/A3 conclusions
#   - Verify critical saved adapters/results
#   - Declare A1 as current Gemma-2-2B champion
#   - Create portable backup of champion adapter
#   - Save consolidated experiment report
#
# NO training.
# NO inference.
# NO organizer validation access.
# ============================================================

from pathlib import Path
import hashlib
import json
import shutil
import zipfile


# ------------------------------------------------------------
# A. Paths
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")

A1_DIR = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
)

A2_DIR = (
    WORK_ROOT
    / "a2_gemma2_2b_targeted_augmentation_v1"
)

A3_DIR = (
    WORK_ROOT
    / "a3_gemma2_2b_prompt_v2"
)


A1_ADAPTER = (
    A1_DIR
    / "epoch_2_adapter"
)

A2_ADAPTER = (
    A2_DIR
    / "epoch_2_adapter"
)

A3_ADAPTER = (
    A3_DIR
    / "epoch_2_adapter"
)


EXPECTED_A1_ADAPTER_SHA256 = (
    "b975567893982e0cf445d4b1df87f52f"
    "99972c49f6058e5e8172926663c4a01e"
)

EXPECTED_A2_ADAPTER_SHA256 = (
    "580d76cfd8dfc9076b9ac99ef9aa0ef64"
    "f373226e4ddbeff5bc398abe6f46478"
)

EXPECTED_A3_ADAPTER_SHA256 = (
    "178ed3d2d8308d3beaef84f530a5c0c717"
    "a7c3810be2c1be582237cb9e86643e"
)


CHAMPION_DIR = (
    WORK_ROOT
    / "gemma2_2b_champion_a1"
)

CHAMPION_ZIP = (
    WORK_ROOT
    / "gemma2_2b_champion_a1.zip"
)

SUMMARY_JSON = (
    WORK_ROOT
    / "gemma2_2b_experiment_closure.json"
)

SUMMARY_MD = (
    WORK_ROOT
    / "gemma2_2b_experiment_closure.md"
)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def directory_size_mb(path):

    total = 0

    for file in Path(path).rglob("*"):

        if file.is_file():

            total += file.stat().st_size

    return total / (1024 ** 2)


def verify_adapter(
    adapter_dir,
    expected_sha,
    experiment_name,
):

    assert adapter_dir.exists()

    model_path = (
        adapter_dir
        / "adapter_model.safetensors"
    )

    config_path = (
        adapter_dir
        / "adapter_config.json"
    )

    assert model_path.exists()
    assert config_path.exists()

    actual_sha = sha256_file(
        model_path
    )

    assert actual_sha == expected_sha, (
        f"{experiment_name} adapter hash mismatch.\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}"
    )

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        config[
            "base_model_name_or_path"
        ]
        == "google/gemma-2-2b-it"
    )

    assert config["r"] == 8
    assert config["lora_alpha"] == 16

    return {
        "adapter_sha256": actual_sha,
        "adapter_model_mb": (
            model_path.stat().st_size
            / 1024**2
        ),
        "directory_mb": (
            directory_size_mb(
                adapter_dir
            )
        ),
    }


# ------------------------------------------------------------
# C. Verify all three adapters
# ------------------------------------------------------------

print("=" * 78)
print("GEMMA-2-2B EXPERIMENT CLOSURE")
print("=" * 78)


a1_integrity = verify_adapter(
    A1_ADAPTER,
    EXPECTED_A1_ADAPTER_SHA256,
    "A1",
)

a2_integrity = verify_adapter(
    A2_ADAPTER,
    EXPECTED_A2_ADAPTER_SHA256,
    "A2",
)

a3_integrity = verify_adapter(
    A3_ADAPTER,
    EXPECTED_A3_ADAPTER_SHA256,
    "A3",
)


print("\nAdapter integrity")

for name, info in [
    ("A1", a1_integrity),
    ("A2", a2_integrity),
    ("A3", a3_integrity),
]:

    print(
        f"  {name} | PASS | "
        f"SHA={info['adapter_sha256']} | "
        f"model={info['adapter_model_mb']:.2f} MB"
    )


# ------------------------------------------------------------
# D. Verify critical result files
# ------------------------------------------------------------

critical_files = {
    "A1 run config": (
        A1_DIR
        / "run_config.json"
    ),

    "A1 results": (
        A1_DIR
        / "results.json"
    ),

    "A1 official predictions": (
        A1_DIR
        / "official_validation_eval1_epoch2.csv"
    ),

    "A2 run config": (
        A2_DIR
        / "run_config.json"
    ),

    "A2 results": (
        A2_DIR
        / "results.json"
    ),

    "A2 official predictions": (
        A2_DIR
        / "official_validation_eval2_epoch2.csv"
    ),

    "A3 run config": (
        A3_DIR
        / "run_config.json"
    ),

    "A3 results": (
        A3_DIR
        / "results.json"
    ),

    "A3 official predictions": (
        A3_DIR
        / "official_validation_eval3_epoch2.csv"
    ),

    "A1/A2/A3 comparison": (
        A3_DIR
        / "a1_a2_a3_official_validation_comparison.csv"
    ),

    "A3 validation summary": (
        A3_DIR
        / "official_validation_eval3_epoch2_summary.json"
    ),

    "Prompt diagnostic": (
        WORK_ROOT
        / "prompt_comparison_post_a3.json"
    ),
}


print("\nCritical experiment records")


critical_hashes = {}


for name, path in (
    critical_files.items()
):

    assert path.exists(), (
        f"Missing critical artifact: {path}"
    )

    file_hash = sha256_file(
        path
    )

    critical_hashes[
        name
    ] = {
        "path": str(path),
        "sha256": file_hash,
    }

    print(
        "  PASS |",
        name,
    )


# ------------------------------------------------------------
# E. Frozen experimental conclusions
# ------------------------------------------------------------

experiments = {
    "A1": {
        "name": (
            "baseline_a1_gemma2_2b"
        ),

        "change": (
            "Baseline: cleaned real training "
            "data + Prompt V1"
        ),

        "fit_rows": 795,

        "synthetic_rows": 0,

        "prompt": "V1",

        "max_length": 256,

        "internal_dev": {
            "accuracy": 0.8214,
            "macro_f1": 0.8249,
            "class_f1": {
                "SUPPORTS": 0.8257,
                "REFUTES": 0.7660,
                "NOT_ENOUGH_INFO": 0.8831,
            },
        },

        "organizer_validation": {
            "accuracy": 0.8200,
            "macro_f1": 0.8207,
            "class_f1": {
                "SUPPORTS": 0.8141,
                "REFUTES": 0.7923,
                "NOT_ENOUGH_INFO": 0.8557,
            },
            "adaptive": False,
        },

        "conclusion": (
            "Best observed Gemma-2-2B "
            "generalization; retained champion."
        ),
    },


    "A2": {
        "name": (
            "targeted_augmentation_v1"
        ),

        "change": (
            "Added 180 balanced synthetic "
            "reasoning examples"
        ),

        "fit_rows": 975,

        "synthetic_rows": 180,

        "prompt": "V1",

        "max_length": 256,

        "internal_dev": {
            "accuracy": 0.8286,
            "macro_f1": 0.8334,
            "class_f1": {
                "SUPPORTS": 0.8224,
                "REFUTES": 0.7527,
                "NOT_ENOUGH_INFO": 0.9250,
            },
        },

        "organizer_validation": {
            "accuracy": 0.8033,
            "macro_f1": 0.8051,
            "class_f1": {
                "SUPPORTS": 0.8103,
                "REFUTES": 0.7570,
                "NOT_ENOUGH_INFO": 0.8482,
            },
            "adaptive": True,
        },

        "paired_vs_a1": {
            "fixed": 25,
            "broken": 30,
        },

        "conclusion": (
            "Targeted synthetic augmentation "
            "was not supported as an improvement."
        ),
    },


    "A3": {
        "name": (
            "prompt_v2_precise_reasoning"
        ),

        "change": (
            "Prompt V1 -> long reasoning Prompt V2"
        ),

        "fit_rows": 795,

        "synthetic_rows": 0,

        "prompt": "V2_LONG",

        "max_length": 384,

        "internal_dev": {
            "accuracy": 0.8571,
            "macro_f1": 0.8580,
            "class_f1": {
                "SUPPORTS": 0.8673,
                "REFUTES": 0.7816,
                "NOT_ENOUGH_INFO": 0.9250,
            },
        },

        "organizer_validation": {
            "accuracy": 0.8067,
            "macro_f1": 0.8077,
            "class_f1": {
                "SUPPORTS": 0.8020,
                "REFUTES": 0.7670,
                "NOT_ENOUGH_INFO": 0.8542,
            },
            "adaptive": True,
        },

        "paired_vs_a1": {
            "fixed": 11,
            "broken": 15,
            "exact_mcnemar_p": (
                0.557197
            ),
        },

        "family_effects_vs_a1": {
            "revenue_percentage_calculation": (
                "+8.3 percentage points"
            ),
            "range_or_measurement_bounds": (
                "+7.4 percentage points"
            ),
            "paired_comparison": (
                "-14.8 percentage points"
            ),
            "temporal_office_holder": (
                "-4.2 percentage points"
            ),
        },

        "conclusion": (
            "Reasoning prompt strongly improved "
            "internal dev and some specific families, "
            "but did not improve overall organizer "
            "validation generalization."
        ),
    },
}


# ------------------------------------------------------------
# F. Freeze 2B champion
# ------------------------------------------------------------

champion = {
    "experiment": "A1",

    "base_model": (
        "google/gemma-2-2b-it"
    ),

    "adapter": str(
        A1_ADAPTER
    ),

    "adapter_sha256": (
        EXPECTED_A1_ADAPTER_SHA256
    ),

    "prompt": "V1",

    "max_length": 256,

    "reason": (
        "Highest organizer-validation "
        "Macro-F1 among A1/A2/A3 and "
        "smallest dev-to-validation discrepancy."
    ),

    "internal_dev_macro_f1": (
        0.8249
    ),

    "organizer_validation_macro_f1": (
        0.8207
    ),
}


# ------------------------------------------------------------
# G. Create clean champion backup directory
# ------------------------------------------------------------

if CHAMPION_DIR.exists():

    shutil.rmtree(
        CHAMPION_DIR
    )


shutil.copytree(
    A1_ADAPTER,
    CHAMPION_DIR,
)


# Add explicit champion metadata.
champion_metadata_path = (
    CHAMPION_DIR
    / "champion_metadata.json"
)


champion_metadata_path.write_text(
    json.dumps(
        champion,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# Verify copied model hash.
copied_adapter_hash = sha256_file(
    CHAMPION_DIR
    / "adapter_model.safetensors"
)


assert (
    copied_adapter_hash
    == EXPECTED_A1_ADAPTER_SHA256
)


# ------------------------------------------------------------
# H. Create ZIP backup
# ------------------------------------------------------------

if CHAMPION_ZIP.exists():

    CHAMPION_ZIP.unlink()


with zipfile.ZipFile(
    CHAMPION_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:

    for file in sorted(
        CHAMPION_DIR.rglob("*")
    ):

        if file.is_file():

            archive.write(
                file,
                arcname=(
                    Path(
                        CHAMPION_DIR.name
                    )
                    / file.relative_to(
                        CHAMPION_DIR
                    )
                ),
            )


champion_zip_sha256 = (
    sha256_file(
        CHAMPION_ZIP
    )
)


# ------------------------------------------------------------
# I. Final consolidated report
# ------------------------------------------------------------

closure = {
    "phase": (
        "Gemma-2-2B experimentation"
    ),

    "status": "CLOSED",

    "champion": champion,

    "experiments": experiments,

    "adapter_integrity": {
        "A1": a1_integrity,
        "A2": a2_integrity,
        "A3": a3_integrity,
    },

    "critical_artifacts": (
        critical_hashes
    ),

    "methodological_lessons": [
        (
            "Internal-dev improvements must "
            "not automatically be treated as "
            "generalization improvements."
        ),

        (
            "Targeted synthetic augmentation "
            "changed decision boundaries but "
            "did not improve overall validation."
        ),

        (
            "Explicit reasoning instructions "
            "helped some arithmetic/range families "
            "but hurt other relation/comparison families."
        ),

        (
            "A1 remains the safest Gemma-2-2B "
            "checkpoint for independent testing."
        ),

        (
            "Further organizer-validation-driven "
            "tuning should be avoided; future "
            "experiments should primarily use "
            "internal development data."
        ),
    ],

    "next_phase": (
        "Evaluate whether a stronger Gemma base "
        "model provides better latent reasoning "
        "while preserving the established "
        "training methodology."
    ),

    "champion_backup": {
        "directory": str(
            CHAMPION_DIR
        ),
        "zip": str(
            CHAMPION_ZIP
        ),
        "zip_sha256": (
            champion_zip_sha256
        ),
    },
}


SUMMARY_JSON.write_text(
    json.dumps(
        closure,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# J. Human-readable Markdown record
# ------------------------------------------------------------

markdown = f"""# Gemma 2 2B Experiment Closure

## Status

Gemma-2-2B experimental phase is **closed**.

Current 2B champion: **A1**.

## Results

| Experiment | Change | Internal Dev Macro-F1 | Organizer Validation Macro-F1 |
|---|---|---:|---:|
| A1 | Baseline | 0.8249 | **0.8207** |
| A2 | +180 targeted synthetic examples | 0.8334 | 0.8051 |
| A3 | Long reasoning prompt V2 | **0.8580** | 0.8077 |

## Champion

- Base model: `google/gemma-2-2b-it`
- Adapter: A1 epoch 2
- Prompt: V1
- max_length: 256
- Adapter SHA-256: `{EXPECTED_A1_ADAPTER_SHA256}`

## Interpretation

A2 did not establish that targeted synthetic augmentation improves
generalization.

A3 strongly improved the internal development split and improved some
specific reasoning families, especially revenue arithmetic and bounded
measurement reasoning. However, it reduced performance on other
families and did not improve overall organizer-validation performance.

Therefore A1 remains the most reliable Gemma-2-2B checkpoint.

## Important Methodological Note

Organizer validation has now been used adaptively in later experiments.
Future model-development decisions should avoid repeatedly optimizing
against this validation set. The hidden competition evaluation remains
the important independent test.

## Champion Backup

ZIP:

`{CHAMPION_ZIP}`

SHA-256:

`{champion_zip_sha256}`
"""


SUMMARY_MD.write_text(
    markdown,
    encoding="utf-8",
)


# ------------------------------------------------------------
# K. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("GEMMA-2-2B PHASE — FINAL RESULT")
print("=" * 78)


print("\nExperiments")

print(
    "  A1 | dev F1=0.8249 | "
    "validation F1=0.8207 | CHAMPION"
)

print(
    "  A2 | dev F1=0.8334 | "
    "validation F1=0.8051 | REJECTED"
)

print(
    "  A3 | dev F1=0.8580 | "
    "validation F1=0.8077 | REJECTED AS CHAMPION"
)


print("\nChampion")

print(
    "  Experiment:",
    champion["experiment"],
)

print(
    "  Model:",
    champion["base_model"],
)

print(
    "  Prompt:",
    champion["prompt"],
)

print(
    "  max_length:",
    champion["max_length"],
)

print(
    "  Adapter SHA-256:",
    champion["adapter_sha256"],
)


print("\nChampion backup")

print(
    "  Directory:",
    CHAMPION_DIR,
)

print(
    "  ZIP:",
    CHAMPION_ZIP,
)

print(
    "  ZIP size:",
    round(
        CHAMPION_ZIP.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "  ZIP SHA-256:",
    champion_zip_sha256,
)


print("\nExperiment records")

print(
    " ",
    SUMMARY_JSON,
)

print(
    " ",
    SUMMARY_MD,
)


print("\nFinal status")

print(
    "  A1/A2/A3 complete:",
    True,
)

print(
    "  Champion frozen:",
    True,
)

print(
    "  Further 2B validation tuning:",
    False,
)

print(
    "  Ready for next model phase:",
    True,
)


print("\nNo training occurred.")
print(
    "No organizer-validation inference occurred."
)

In [ ]:
# ============================================================
# EXTERNAL STRESS SET — FREEZE + AUDIT
#
# 30 LLM-generated external examples supplied by user.
#
# Purpose:
#   - preserve examples exactly
#   - assign stable IDs
#   - audit class distribution
#   - check exact normalized overlap against:
#       fit
#       internal dev
#       organizer validation
#   - record the set BEFORE A1/A2/A3 comparison
#
# NO model inference.
# NO training.
# NO modification of competition data.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import re


WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

OUT_PATH = (
    WORK_ROOT
    / "external_stress_30_v1.jsonl"
)

AUDIT_PATH = (
    WORK_ROOT
    / "external_stress_30_v1_audit.json"
)


EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_VAL_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)


# ------------------------------------------------------------
# A. Exact user-supplied examples
# ------------------------------------------------------------

raw_examples = [
{
"claim": "BioGreen Innovations experienced a revenue growth of $14 million between 2022 and 2023.",
"evidence": "In 2023, BioGreen Innovations reported annual revenues of $84 million, an increase compared to its 2022 revenue of $70 million. Meanwhile, operational expenses rose from $40 million to $52 million over the same period.",
"label": "SUPPORTS"
},
{
"claim": "The Helios Deep-Space Probe began orbiting Asteroid 101955 Bennu more than three years after its initial launch.",
"evidence": "The Helios Deep-Space Probe was launched on March 14, 2018. Following a 42-month cruise through the inner Solar System, it successfully entered orbit around Asteroid 101955 Bennu in September 2021.",
"label": "SUPPORTS"
},
{
"claim": "Dr. Elena Rostova has held a professional position at an organization located in Gothenburg.",
"evidence": "Dr. Elena Rostova completed her doctoral residency at the Karolinska Institute in Stockholm. In 2019, she was appointed Chief Medical Officer at Vinter Health, a private biotechnology firm headquartered in Gothenburg.",
"label": "SUPPORTS"
},
{
"claim": "The cargo vessel MV Arctic Dawn complies with the 2022 Clean Shipping Directive while operating in maritime zone Alpha.",
"evidence": "The 2022 Clean Shipping Directive mandates that all commercial vessels exceeding 5,000 gross tonnage operating in maritime zone Alpha must either utilize liquefied natural gas (LNG) or achieve at least a 30% reduction in sulfur emissions via exhaust scrubbers. The cargo vessel MV Arctic Dawn operates in zone Alpha with a gross tonnage of 12,400 and does not use LNG, but it operates with certified scrubbers reducing sulfur emissions by 35%.",
"label": "SUPPORTS"
},
{
"claim": "The governing coalition formed after the 2024 Valdoria regional elections controls a majority of 85 seats in the 160-seat parliament.",
"evidence": "During the 2024 regional elections in Valdoria, the Progress Alliance secured 42 parliamentary seats, the Civic Union won 28 seats, and the Green Coalition took 15 seats. The three parties subsequently formed a joint governing coalition in the 160-seat parliament.",
"label": "SUPPORTS"
},
{
"claim": "Ridership on Metro Line 4 increased by more than 25% from 2021 to 2022.",
"evidence": "According to the municipal transit audit, Metro Line 4 recorded 14.2 million passenger journeys in 2021 and 17.04 million passenger journeys in 2022, representing a ridership increase of exactly 20%.",
"label": "REFUTES"
},
{
"claim": "The Siege of Dunhaven began before the ratification of the Treaty of Aldermoor.",
"evidence": "The Treaty of Aldermoor was formally ratified by King Charles IV on August 12, 1684. Historical military chronicles document that the Siege of Dunhaven commenced three weeks later, on September 3, 1684.",
"label": "REFUTES"
},
{
"claim": "Mira Thorne co-designed the architectural blueprints for the Horizon Tower in 2011.",
"evidence": "The architectural blueprints for the Horizon Tower were drafted exclusively by Samuel Vance of Vance & Partners in 2011. Mira Thorne joined the firm as a senior partner in 2015 and had no involvement in the tower's design.",
"label": "REFUTES"
},
{
"claim": "Every member of the 18-person Mount Rainier expedition team reached the final summit ridge.",
"evidence": "All 18 members of the expedition team reached the high-altitude base camp on Mount Rainier; however, three climbers—Marcus, Leigh, and Davis—declined to attempt the final summit ridge due to severe frostbite warnings, while the remaining 15 successfully completed the ascent.",
"label": "REFUTES"
},
{
"claim": "Drug X-7 caused a statistically significant decrease in resting heart rate during the 500-participant trial.",
"evidence": "A randomized clinical trial of 500 participants showed that Drug X-7 reduced systolic blood pressure by an average of 8 mmHg, but demonstrated no statistically significant difference in resting heart rate compared to the placebo group (p = 0.64).",
"label": "REFUTES"
},
{
"claim": "The SolarFlare-8 smartphone can charge from 0% to 50% in under fifteen minutes.",
"evidence": "The SolarFlare-8 smartphone was released in November 2023 featuring an OLED display, a 5,000 mAh battery, and an IP68 water resistance rating. Industry reviews consistently praised the device's rapid charging capabilities and display quality.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Traffic accidents on Highway 101 decreased following the speed limit reduction in June 2020.",
"evidence": "In June 2020, the regional transport authority enacted a pilot program lowering the maximum speed limit from 65 mph to 55 mph along Highway 101. Traffic volume remained stable at approximately 45,000 vehicles per day throughout the following six months.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Professor Liam Gallagher's £1.2 million grant was awarded specifically for research into quantum annealing algorithms.",
"evidence": "Professor Liam Gallagher authored four peer-reviewed papers on quantum annealing in 2021 while affiliated with the Department of Physics at the University of Edinburgh. In late 2022, he received a £1.2 million research award from the European Research Council.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Dr. Tariq Mansoor was appointed Chief Technology Officer of AstraCore Technologies after the January 2022 acquisition.",
"evidence": "AstraCore Technologies acquired NeuroWave Systems in January 2022 for $120 million. Prior to the acquisition, NeuroWave Systems had patented a non-invasive neural interface device developed by its lead engineer, Dr. Tariq Mansoor.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "The Grand Canal Aqueduct was fully completed and opened for public service by April 2022.",
"evidence": "Construction of the Grand Canal Aqueduct began in April 2017 with an estimated five-year project timeline and an initial budget of €340 million. By December 2021, municipal contractors reported that 80% of the primary structural masonry was in place.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "The Meridian Bridge in Port Callum was completed in 1987.",
"evidence": "Construction of the Meridian Bridge in Port Callum finished in 1987, connecting the northern and southern districts of the city.",
"label": "SUPPORTS"
},
{
"claim": "Nova Textiles reported a profit in the third quarter of 2023.",
"evidence": "Nova Textiles posted a net loss of $4.2 million in Q3 2023, its third consecutive quarterly loss.",
"label": "REFUTES"
},
{
"claim": "Dr. Elena Vasquez was the first woman to lead the Astrophysics Institute.",
"evidence": "Dr. Elena Vasquez became director of the Astrophysics Institute in 2015, overseeing a major expansion of the observatory program.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Enrollment at Cedarbrook University grew by more than 20% between 2018 and 2023.",
"evidence": "Cedarbrook University enrolled 8,000 students in 2018. By 2023, enrollment had risen to 9,800 students.",
"label": "SUPPORTS"
},
{
"claim": "The unemployment rate in Kestrel County fell by half between 2019 and 2022.",
"evidence": "Kestrel County's unemployment rate was 9% in 2019 and stood at 6% in 2022.",
"label": "REFUTES"
},
{
"claim": "It took less than a decade for the Solvig Dam project to move from approval to completion.",
"evidence": "The Solvig Dam was approved by parliament in March 2011 and officially opened in November 2019.",
"label": "SUPPORTS"
},
{
"claim": "The Halden Tunnel took over 15 years to build.",
"evidence": "Excavation of the Halden Tunnel began in June 2005, and the tunnel opened to traffic in September 2014.",
"label": "REFUTES"
},
{
"claim": "Renna Corp's CEO Marcus Ohlin previously worked at Delacroix Systems.",
"evidence": "Marcus Ohlin was appointed CEO of Renna Corp in 2020. Before joining Renna, he spent twelve years in the technology sector, including senior roles at two Fortune 500 firms.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "The novel 'Wintersong' by Adaeze Nwosu won the Corvallis Prize for Fiction in 2021.",
"evidence": "The Corvallis Prize for Fiction is awarded annually to an outstanding work of literary fiction. In 2021, the jury selected 'Wintersong,' a debut novel by Nigerian-born author Adaeze Nwosu, as the winner.",
"label": "SUPPORTS"
},
{
"claim": "Harrow Industries acquired Bellwood Manufacturing in 2020.",
"evidence": "In 2020, Harrow Industries acquired Fenwick Manufacturing for $340 million. A separate deal, in which rival firm Talbot Group acquired Bellwood Manufacturing, closed the following year.",
"label": "REFUTES"
},
{
"claim": "The Pemberton Art Museum's new wing was designed by architect Sofia Ricci.",
"evidence": "The Pemberton Art Museum unveiled plans for a new eastern wing in 2022, intended to house its growing collection of modern sculpture. The museum's original 1930s building was designed by architect Henry Talbot.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Fewer than half of the survey respondents supported the new zoning proposal.",
"evidence": "Of the 1,200 residents surveyed about the zoning proposal, 540 said they supported it, while the rest opposed or were undecided.",
"label": "SUPPORTS"
},
{
"claim": "More than 60% of Larkspur High School's graduating class enrolled in a four-year college.",
"evidence": "Of Larkspur High School's 300 graduates last year, 165 enrolled directly in four-year colleges, with the remainder pursuing community college, employment, or other paths.",
"label": "REFUTES"
},
{
"claim": "Global sales of the Orion X7 smartphone exceeded 10 million units in its first year.",
"evidence": "The Orion X7 launched in select European markets in early 2022 and sold 2.3 million units in its first six months there, outperforming its predecessor.",
"label": "NOT_ENOUGH_INFO"
},
{
"claim": "Senator Priya Malhotra voted against the Coastal Infrastructure Bill in 2023.",
"evidence": "The Coastal Infrastructure Bill passed the senate 52-48 in October 2023. Senator Priya Malhotra was among the 48 senators who voted no, citing concerns over funding allocation.",
"label": "SUPPORTS"
},
]


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():
                rows.append(
                    json.loads(line)
                )

    return rows


def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def fingerprint(
    claim,
    evidence,
):

    if isinstance(evidence, list):

        evidence_text = " ".join(
            normalize(x)
            for x in evidence
        )

    else:

        evidence_text = normalize(
            evidence
        )


    payload = {
        "claim": normalize(
            claim
        ).lower(),
        "evidence": (
            evidence_text.lower()
        ),
    }


    encoded = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")


    return hashlib.sha256(
        encoded
    ).hexdigest()


def find_validation():

    for path in INPUT_ROOT.rglob(
        "validation.jsonl"
    ):

        if (
            path.is_file()
            and sha256_file(path)
            == EXPECTED_VAL_SHA256
        ):

            return path


    raise RuntimeError(
        "Frozen validation file not found."
    )


# ------------------------------------------------------------
# C. Verify competition datasets
# ------------------------------------------------------------

assert (
    sha256_file(FIT_PATH)
    == EXPECTED_FIT_SHA256
)

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)


VAL_PATH = find_validation()


assert (
    sha256_file(VAL_PATH)
    == EXPECTED_VAL_SHA256
)


fit_rows = read_jsonl(
    FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

val_rows = read_jsonl(
    VAL_PATH
)


# ------------------------------------------------------------
# D. Freeze external examples
#
# Convert single evidence paragraph to one-item list so it
# matches our normal inference schema.
# ------------------------------------------------------------

external_rows = []


for index, example in enumerate(
    raw_examples,
    1,
):

    row = {
        "id": (
            f"external_stress_v1_{index:02d}"
        ),

        "claim": normalize(
            example["claim"]
        ),

        "evidence": [
            normalize(
                example["evidence"]
            )
        ],

        "label": (
            example["label"]
        ),
    }


    assert row["label"] in {
        "SUPPORTS",
        "REFUTES",
        "NOT_ENOUGH_INFO",
    }

    external_rows.append(
        row
    )


assert len(
    external_rows
) == 30


# ------------------------------------------------------------
# E. Distribution
# ------------------------------------------------------------

label_distribution = Counter(
    row["label"]
    for row in external_rows
)


assert label_distribution == Counter({
    "SUPPORTS": 11,
    "REFUTES": 10,
    "NOT_ENOUGH_INFO": 9,
})


# ------------------------------------------------------------
# F. Duplicate checks
# ------------------------------------------------------------

external_fp = [
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in external_rows
]


assert (
    len(external_fp)
    == len(set(external_fp))
)


fit_fp = {
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in fit_rows
}


dev_fp = {
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in dev_rows
}


val_fp = {
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in val_rows
}


overlap_fit = (
    set(external_fp)
    & fit_fp
)

overlap_dev = (
    set(external_fp)
    & dev_fp
)

overlap_val = (
    set(external_fp)
    & val_fp
)


assert len(overlap_fit) == 0
assert len(overlap_dev) == 0
assert len(overlap_val) == 0


# ------------------------------------------------------------
# G. Freeze known provenance / caveats
# ------------------------------------------------------------

flags = {
    "external_stress_v1_25": {
        "type": "label_ambiguity_review",

        "note": (
            "Supplied gold label is REFUTES. "
            "Evidence says Harrow acquired Fenwick "
            "in 2020 and Talbot acquired Bellwood "
            "in 2021. This strongly suggests the "
            "intended contradiction, but sequential "
            "ownership would make strict logical "
            "refutation somewhat less airtight."
        ),

        "gold_label_preserved": (
            "REFUTES"
        ),
    }
}


provenance = {
    "name": (
        "external_stress_30_v1"
    ),

    "rows": 30,

    "source": (
        "LLM-generated examples supplied "
        "externally by user"
    ),

    "competition_data": False,

    "used_for_training": False,

    "previously_seen_model_results": {
        "A1_vs_A2": True,

        "note": (
            "User reported previously observing "
            "A2 outperform A1 on these examples. "
            "Therefore this set is not a pristine "
            "blind holdout for A1-vs-A2."
        ),
    },

    "intended_use": (
        "Secondary external reasoning stress test"
    ),

    "gold_label_review": {
        "clearly_acceptable": 29,
        "flagged_for_ambiguity": 1,
        "labels_modified": 0,
    },
}


# ------------------------------------------------------------
# H. Save frozen JSONL
# ------------------------------------------------------------

with OUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    for row in external_rows:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )


external_sha256 = (
    sha256_file(
        OUT_PATH
    )
)


# ------------------------------------------------------------
# I. Save audit
# ------------------------------------------------------------

audit = {
    "dataset": provenance,

    "sha256": (
        external_sha256
    ),

    "label_distribution": dict(
        label_distribution
    ),

    "exact_duplicate_checks": {
        "internal_external_duplicates": 0,
        "vs_fit": len(
            overlap_fit
        ),
        "vs_internal_dev": len(
            overlap_dev
        ),
        "vs_organizer_validation": len(
            overlap_val
        ),
    },

    "manual_review_flags": (
        flags
    ),

    "competition_data_hashes": {
        "fit": (
            EXPECTED_FIT_SHA256
        ),
        "internal_dev": (
            EXPECTED_DEV_SHA256
        ),
        "organizer_validation": (
            EXPECTED_VAL_SHA256
        ),
    },
}


AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# J. Report
# ------------------------------------------------------------

print("=" * 78)
print("EXTERNAL STRESS SET — FREEZE RESULT")
print("=" * 78)


print("\nRows")

print(
    "  Total:",
    len(external_rows),
)


print("\nLabels")

for label in [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]:

    print(
        f"  {label:15}",
        label_distribution[
            label
        ],
    )


print("\nExact overlap")

print(
    "  vs fit:",
    len(overlap_fit),
)

print(
    "  vs internal dev:",
    len(overlap_dev),
)

print(
    "  vs organizer validation:",
    len(overlap_val),
)


print("\nManual label review")

print(
    "  Clearly acceptable:",
    29,
)

print(
    "  Flagged:",
    1,
)

print(
    "  Gold labels changed:",
    0,
)

print(
    "  Flagged ID:",
    "external_stress_v1_25",
)


print("\nMethodological status")

print(
    "  Competition data:",
    False,
)

print(
    "  Training data:",
    False,
)

print(
    "  External synthetic stress test:",
    True,
)

print(
    "  Blind for prior A1-vs-A2 comparison:",
    False,
)


print("\nFrozen file")

print(
    " ",
    OUT_PATH,
)

print(
    "  SHA-256:",
    external_sha256,
)


print("\nAudit")

print(
    " ",
    AUDIT_PATH,
)


print("\nNo model inference occurred.")
print(
    "No competition dataset was modified."
)

In [ ]:
# ============================================================
# EXTERNAL STRESS TEST — A1 vs A2 vs A3
#
# Frozen external set:
#   /kaggle/working/external_stress_30_v1.jsonl
#
# Models:
#   A1 epoch 2 + Prompt V1
#   A2 epoch 2 + Prompt V1
#   A3 epoch 2 + Prompt V2_LONG
#
# This cell:
#   - verifies all hashes
#   - fresh-reloads each adapter
#   - runs deterministic inference on the same 30 examples
#   - reports Accuracy / Macro-F1 / class F1
#   - reports paired fixed/broken examples
#   - reports a 29-example sensitivity analysis excluding
#     the single manually flagged label-ambiguity case
#   - saves prediction-level comparison
#
# NO training.
# NO competition validation access.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# A. Frozen paths / hashes
# ------------------------------------------------------------

MODEL_ID = "google/gemma-2-2b-it"

WORK_ROOT = Path("/kaggle/working")

STRESS_PATH = (
    WORK_ROOT
    / "external_stress_30_v1.jsonl"
)

EXPECTED_STRESS_SHA256 = (
    "05624200845a79228ac4e05540b013ea"
    "0836d049802fcd3f9dcc60ada8aecedc"
)


A1_ADAPTER = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
    / "epoch_2_adapter"
)

A2_ADAPTER = (
    WORK_ROOT
    / "a2_gemma2_2b_targeted_augmentation_v1"
    / "epoch_2_adapter"
)

A3_ADAPTER = (
    WORK_ROOT
    / "a3_gemma2_2b_prompt_v2"
    / "epoch_2_adapter"
)


EXPECTED_ADAPTER_HASHES = {
    "A1": (
        "b975567893982e0cf445d4b1df87f52f"
        "99972c49f6058e5e8172926663c4a01e"
    ),
    "A2": (
        "580d76cfd8dfc9076b9ac99ef9aa0ef"
        "64f373226e4ddbeff5bc398abe6f46478"
    ),
    "A3": (
        "178ed3d2d8308d3beaef84f530a5c0c"
        "717a7c3810be2c1be582237cb9e86643e"
    ),
}


ADAPTER_DIRS = {
    "A1": A1_ADAPTER,
    "A2": A2_ADAPTER,
    "A3": A3_ADAPTER,
}


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)

FLAGGED_ID = "external_stress_v1_25"

EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 10


OUT_CSV = (
    WORK_ROOT
    / "external_stress_30_a1_a2_a3_predictions.csv"
)

OUT_JSON = (
    WORK_ROOT
    / "external_stress_30_a1_a2_a3_results.json"
)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def clear_gpu():

    for name in [
        "model",
        "base_model",
        "quant_config",
    ]:

        if name in globals():

            try:
                del globals()[name]

            except Exception:
                pass

    gc.collect()
    torch.cuda.empty_cache()

    if hasattr(
        torch.cuda,
        "ipc_collect",
    ):
        torch.cuda.ipc_collect()


# ------------------------------------------------------------
# C. Verify frozen stress set
# ------------------------------------------------------------

assert STRESS_PATH.exists()

assert (
    sha256_file(STRESS_PATH)
    == EXPECTED_STRESS_SHA256
)


rows = read_jsonl(
    STRESS_PATH
)


assert len(rows) == 30


assert Counter(
    row["label"]
    for row in rows
) == Counter({
    "SUPPORTS": 11,
    "REFUTES": 10,
    "NOT_ENOUGH_INFO": 9,
})


assert FLAGGED_ID in {
    row["id"]
    for row in rows
}


# ------------------------------------------------------------
# D. Verify all adapter artifacts
# ------------------------------------------------------------

print("=" * 78)
print("EXTERNAL STRESS TEST — ARTIFACT INTEGRITY")
print("=" * 78)


for experiment_name, adapter_dir in (
    ADAPTER_DIRS.items()
):

    assert adapter_dir.exists()

    model_path = (
        adapter_dir
        / "adapter_model.safetensors"
    )

    config_path = (
        adapter_dir
        / "adapter_config.json"
    )

    assert model_path.exists()
    assert config_path.exists()

    actual_hash = sha256_file(
        model_path
    )

    assert (
        actual_hash
        == EXPECTED_ADAPTER_HASHES[
            experiment_name
        ]
    )


    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )


    assert (
        config[
            "base_model_name_or_path"
        ]
        == MODEL_ID
    )

    assert config["r"] == 8
    assert config["lora_alpha"] == 16


    print(
        f"{experiment_name} | PASS | "
        f"{actual_hash}"
    )


# ------------------------------------------------------------
# E. Exact prompts used by each trained experiment
# ------------------------------------------------------------

PROMPT_V1 = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


PROMPT_V2_LONG = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


EXPERIMENT_PROMPTS = {
    "A1": PROMPT_V1,
    "A2": PROMPT_V1,
    "A3": PROMPT_V2_LONG,
}


# ------------------------------------------------------------
# F. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# G. Shared tokenizer
#
# A1/A2/A3 all use same Gemma-2 tokenizer.
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        A1_ADAPTER,
        local_files_only=True,
    )
)


assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


end_of_turn_id = (
    tokenizer.convert_tokens_to_ids(
        "<end_of_turn>"
    )
)


# ------------------------------------------------------------
# H. Prompt renderer
# ------------------------------------------------------------

def build_user_content(
    row,
    instruction,
):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    return (
        instruction
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def render_prompt(
    row,
    instruction,
):

    return (
        tokenizer
        .apply_chat_template(
            [{
                "role": "user",
                "content": (
                    build_user_content(
                        row,
                        instruction,
                    )
                ),
            }],
            tokenize=False,
            add_generation_prompt=True,
        )
    )


def parse_prediction(text):

    cleaned = text.strip()

    if cleaned in LABEL_SET:
        return cleaned

    return None


# ------------------------------------------------------------
# I. Metric helper
# ------------------------------------------------------------

def calculate_metrics(
    gold,
    predictions,
):

    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            gold,
            metric_predictions,
            labels=LABELS,
            zero_division=0,
        )
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    return {
        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            np.mean(f1)
        ),

        "invalid": int(
            sum(
                prediction is None
                for prediction
                in predictions
            )
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }
            for i, label in enumerate(
                LABELS
            )
        },
    }


# ------------------------------------------------------------
# J. Fresh-load and evaluate ONE experiment
# ------------------------------------------------------------

def evaluate_experiment(
    experiment_name,
):

    clear_gpu()


    adapter_dir = (
        ADAPTER_DIRS[
            experiment_name
        ]
    )

    instruction = (
        EXPERIMENT_PROMPTS[
            experiment_name
        ]
    )


    quant_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(
                torch.float16
            ),
        )
    )


    print("\n" + "=" * 78)
    print(
        f"{experiment_name} — FRESH RELOAD"
    )
    print("=" * 78)


    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
            quantization_config=(
                quant_config
            ),
            device_map={"": 0},
            dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
    )


    assert base_model.is_loaded_in_4bit


    base_model = (
        prepare_model_for_kbit_training(
            base_model,
            use_gradient_checkpointing=False,
        )
    )


    base_model.config.use_cache = True


    model = (
        PeftModel
        .from_pretrained(
            base_model,
            adapter_dir,
            is_trainable=False,
        )
    )


    model.eval()
    model.config.use_cache = True


    assert sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) == 0


    # --------------------------------------------------------
    # Finite-logit check
    # --------------------------------------------------------

    sample = rows[0]


    encoded = tokenizer(
        render_prompt(
            sample,
            instruction,
        ),
        add_special_tokens=False,
        return_tensors="pt",
    )


    with torch.inference_mode():

        preflight = model(
            input_ids=(
                encoded["input_ids"]
                .to("cuda:0")
            ),
            attention_mask=(
                encoded["attention_mask"]
                .to("cuda:0")
            ),
            use_cache=True,
        )


    assert torch.isfinite(
        preflight.logits[
            0,
            -1,
            :
        ]
    ).all()


    del encoded
    del preflight

    torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Deterministic generation
    # --------------------------------------------------------

    previous_padding_side = (
        tokenizer.padding_side
    )

    tokenizer.padding_side = "left"


    predictions = []
    raw_outputs = []

    start_time = (
        time.perf_counter()
    )


    try:

        for start in range(
            0,
            len(rows),
            EVAL_BATCH_SIZE,
        ):

            batch_rows = rows[
                start:
                start + EVAL_BATCH_SIZE
            ]


            prompts = [
                render_prompt(
                    row,
                    instruction,
                )
                for row in batch_rows
            ]


            encoded = tokenizer(
                prompts,
                add_special_tokens=False,
                padding=True,
                return_tensors="pt",
            )


            input_ids = (
                encoded[
                    "input_ids"
                ]
                .to("cuda:0")
            )

            attention_mask = (
                encoded[
                    "attention_mask"
                ]
                .to("cuda:0")
            )


            input_width = (
                input_ids.shape[1]
            )


            with torch.inference_mode():

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        MAX_NEW_TOKENS
                    ),
                    eos_token_id=[
                        tokenizer.eos_token_id,
                        end_of_turn_id,
                    ],
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                    use_cache=True,
                )


            new_tokens = generated[
                :,
                input_width:
            ]


            for token_ids in new_tokens:

                raw = tokenizer.decode(
                    token_ids,
                    skip_special_tokens=True,
                )


                raw_outputs.append(
                    raw
                )

                predictions.append(
                    parse_prediction(
                        raw
                    )
                )


            del encoded
            del input_ids
            del attention_mask
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        tokenizer.padding_side = (
            previous_padding_side
        )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    gold = [
        row["label"]
        for row in rows
    ]


    metrics_all = (
        calculate_metrics(
            gold,
            predictions,
        )
    )


    # --------------------------------------------------------
    # Sensitivity analysis excluding flagged item
    # --------------------------------------------------------

    filtered_indices = [
        i
        for i, row in enumerate(
            rows
        )
        if row["id"] != FLAGGED_ID
    ]


    filtered_gold = [
        gold[i]
        for i in filtered_indices
    ]


    filtered_predictions = [
        predictions[i]
        for i in filtered_indices
    ]


    metrics_29 = (
        calculate_metrics(
            filtered_gold,
            filtered_predictions,
        )
    )


    print(
        "\nAll 30"
    )

    print(
        "  Accuracy:",
        f"{metrics_all['accuracy']:.4f}",
    )

    print(
        "  Macro-F1:",
        f"{metrics_all['macro_f1']:.4f}",
    )

    print(
        "  Invalid:",
        metrics_all["invalid"],
    )


    print(
        "  Confusion matrix:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )


    for label, values in zip(
        LABELS,
        metrics_all[
            "confusion_matrix"
        ],
    ):

        print(
            f"  {label:15}",
            values,
        )


    print(
        "  Per-class F1:"
    )

    for label in LABELS:

        print(
            f"    {label:15} "
            f"{metrics_all['per_class'][label]['f1']:.4f}"
        )


    print(
        "\n29-example sensitivity "
        "(excluding flagged #25)"
    )

    print(
        "  Accuracy:",
        f"{metrics_29['accuracy']:.4f}",
    )

    print(
        "  Macro-F1:",
        f"{metrics_29['macro_f1']:.4f}",
    )


    print(
        "  Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    result = {
        "experiment": (
            experiment_name
        ),

        "prompt": (
            "V1"
            if experiment_name
            in {"A1", "A2"}
            else "V2_LONG"
        ),

        "all_30": (
            metrics_all
        ),

        "excluding_flagged_29": (
            metrics_29
        ),

        "predictions": (
            predictions
        ),

        "raw_outputs": (
            raw_outputs
        ),

        "seconds": float(
            elapsed
        ),
    }


    del model
    del base_model
    del quant_config

    clear_gpu()


    return result


# ------------------------------------------------------------
# K. Evaluate A1, A2, A3
# ------------------------------------------------------------

results = {}


for experiment_name in [
    "A1",
    "A2",
    "A3",
]:

    results[
        experiment_name
    ] = evaluate_experiment(
        experiment_name
    )


# ------------------------------------------------------------
# L. Build prediction-level table
# ------------------------------------------------------------

comparison_rows = []


for i, row in enumerate(
    rows
):

    record = {
        "id": row["id"],
        "gold": row["label"],
        "claim": row["claim"],
        "flagged_label_review": (
            row["id"]
            == FLAGGED_ID
        ),
    }


    for experiment_name in [
        "A1",
        "A2",
        "A3",
    ]:

        prediction = (
            results[
                experiment_name
            ][
                "predictions"
            ][i]
        )

        record[
            f"{experiment_name}_prediction"
        ] = prediction

        record[
            f"{experiment_name}_correct"
        ] = (
            prediction
            == row["label"]
        )


    comparison_rows.append(
        record
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


# ------------------------------------------------------------
# M. Paired comparison helper
# ------------------------------------------------------------

def paired_summary(
    reference,
    candidate,
):

    fixed_ids = []
    broken_ids = []
    both_correct_ids = []
    both_wrong_ids = []


    for record in comparison_rows:

        ref_correct = (
            record[
                f"{reference}_correct"
            ]
        )

        cand_correct = (
            record[
                f"{candidate}_correct"
            ]
        )


        if (
            not ref_correct
            and cand_correct
        ):

            fixed_ids.append(
                record["id"]
            )

        elif (
            ref_correct
            and not cand_correct
        ):

            broken_ids.append(
                record["id"]
            )

        elif (
            ref_correct
            and cand_correct
        ):

            both_correct_ids.append(
                record["id"]
            )

        else:

            both_wrong_ids.append(
                record["id"]
            )


    return {
        "fixed": len(
            fixed_ids
        ),

        "broken": len(
            broken_ids
        ),

        "both_correct": len(
            both_correct_ids
        ),

        "both_wrong": len(
            both_wrong_ids
        ),

        "fixed_ids": (
            fixed_ids
        ),

        "broken_ids": (
            broken_ids
        ),
    }


a2_vs_a1 = paired_summary(
    "A1",
    "A2",
)

a3_vs_a1 = paired_summary(
    "A1",
    "A3",
)

a3_vs_a2 = paired_summary(
    "A2",
    "A3",
)


# ------------------------------------------------------------
# N. Per-example disagreement report
# ------------------------------------------------------------

disagreements = []


for record in comparison_rows:

    predictions = {
        record["A1_prediction"],
        record["A2_prediction"],
        record["A3_prediction"],
    }


    if len(predictions) > 1:

        disagreements.append(
            record
        )


# ------------------------------------------------------------
# O. Save results
# ------------------------------------------------------------

comparison_df.to_csv(
    OUT_CSV,
    index=False,
)


json_payload = {
    "dataset": {
        "path": str(
            STRESS_PATH
        ),
        "sha256": (
            EXPECTED_STRESS_SHA256
        ),
        "rows": 30,
        "flagged_id": (
            FLAGGED_ID
        ),
        "blind_for_prior_A1_vs_A2": (
            False
        ),
    },

    "results": {
        name: {
            key: value
            for key, value
            in result.items()
            if key not in {
                "predictions",
                "raw_outputs",
            }
        }
        for name, result
        in results.items()
    },

    "paired": {
        "A2_vs_A1": (
            a2_vs_a1
        ),
        "A3_vs_A1": (
            a3_vs_a1
        ),
        "A3_vs_A2": (
            a3_vs_a2
        ),
    },

    "number_of_three_model_disagreements": (
        len(disagreements)
    ),
}


OUT_JSON.write_text(
    json.dumps(
        json_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# P. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("EXTERNAL STRESS TEST — FINAL RESULT")
print("=" * 78)


print("\nAll 30 examples")

print(
    f"{'Model':6} "
    f"{'Correct':>8} "
    f"{'Accuracy':>10} "
    f"{'Macro-F1':>10} "
    f"{'SUP F1':>9} "
    f"{'REF F1':>9} "
    f"{'NEI F1':>9}"
)


for name in [
    "A1",
    "A2",
    "A3",
]:

    m = (
        results[
            name
        ][
            "all_30"
        ]
    )

    correct = int(
        round(
            m["accuracy"]
            * 30
        )
    )


    print(
        f"{name:6} "
        f"{correct:8d} "
        f"{m['accuracy']:10.4f} "
        f"{m['macro_f1']:10.4f} "
        f"{m['per_class']['SUPPORTS']['f1']:9.4f} "
        f"{m['per_class']['REFUTES']['f1']:9.4f} "
        f"{m['per_class']['NOT_ENOUGH_INFO']['f1']:9.4f}"
    )


print(
    "\n29 examples excluding "
    "flagged external_stress_v1_25"
)


for name in [
    "A1",
    "A2",
    "A3",
]:

    m = (
        results[
            name
        ][
            "excluding_flagged_29"
        ]
    )

    correct = int(
        round(
            m["accuracy"]
            * 29
        )
    )


    print(
        f"  {name} | "
        f"correct={correct:2d}/29 | "
        f"Accuracy={m['accuracy']:.4f} | "
        f"Macro-F1={m['macro_f1']:.4f}"
    )


print("\nPaired comparisons")


print(
    "  A2 vs A1 | "
    f"fixed={a2_vs_a1['fixed']} | "
    f"broken={a2_vs_a1['broken']} | "
    f"both_correct={a2_vs_a1['both_correct']} | "
    f"both_wrong={a2_vs_a1['both_wrong']}"
)


print(
    "  A3 vs A1 | "
    f"fixed={a3_vs_a1['fixed']} | "
    f"broken={a3_vs_a1['broken']} | "
    f"both_correct={a3_vs_a1['both_correct']} | "
    f"both_wrong={a3_vs_a1['both_wrong']}"
)


print(
    "  A3 vs A2 | "
    f"fixed={a3_vs_a2['fixed']} | "
    f"broken={a3_vs_a2['broken']} | "
    f"both_correct={a3_vs_a2['both_correct']} | "
    f"both_wrong={a3_vs_a2['both_wrong']}"
)


print("\nA2 fixes relative to A1")

print(
    " ",
    a2_vs_a1[
        "fixed_ids"
    ],
)


print(
    "A2 breaks relative to A1"
)

print(
    " ",
    a2_vs_a1[
        "broken_ids"
    ],
)


print("\nA3 fixes relative to A1")

print(
    " ",
    a3_vs_a1[
        "fixed_ids"
    ],
)


print(
    "A3 breaks relative to A1"
)

print(
    " ",
    a3_vs_a1[
        "broken_ids"
    ],
)


print(
    "\nThree-model disagreement examples:",
    len(disagreements),
)


for record in disagreements:

    print(
        "\n",
        record["id"],
        "| gold=",
        record["gold"],
        "| A1=",
        record["A1_prediction"],
        "| A2=",
        record["A2_prediction"],
        "| A3=",
        record["A3_prediction"],
    )

    print(
        "  Claim:",
        record["claim"],
    )


print("\nSaved")

print(
    " ",
    OUT_CSV,
)

print(
    " ",
    OUT_JSON,
)


print("\nMethodological note")

print(
    "  This is an external synthetic "
    "reasoning stress test."
)

print(
    "  It does not replace organizer "
    "validation or hidden competition testing."
)

print(
    "  A1-vs-A2 is not fully blind because "
    "their relative performance had been "
    "observed previously."
)


print("\nNo training occurred.")
print(
    "No organizer-validation inference occurred."
)

In [ ]:
# ============================================================
# FINAL 2B TEST-READY BACKUP
#
# Creates one portable ZIP containing:
#   - A1 finalist adapter
#   - A3 finalist adapter
#   - exact inference prompts/settings
#   - hashes
#   - experiment records
#   - external stress-test results
#
# This ZIP is enough to preserve the two candidate adapters.
# The base Gemma model itself is NOT included; it can be
# reloaded later from Hugging Face.
#
# NO training.
# NO inference.
# ============================================================

from pathlib import Path
import hashlib
import json
import shutil
import zipfile

from IPython.display import FileLink, display


WORK_ROOT = Path("/kaggle/working")

A1_ADAPTER = (
    WORK_ROOT
    / "baseline_a1_gemma2_2b_retry1_noscaler"
    / "epoch_2_adapter"
)

A3_ADAPTER = (
    WORK_ROOT
    / "a3_gemma2_2b_prompt_v2"
    / "epoch_2_adapter"
)


EXPECTED_A1_SHA256 = (
    "b975567893982e0cf445d4b1df87f52f"
    "99972c49f6058e5e8172926663c4a01e"
)

EXPECTED_A3_SHA256 = (
    "178ed3d2d8308d3beaef84f530a5c0c"
    "717a7c3810be2c1be582237cb9e86643e"
)


BUNDLE_DIR = (
    WORK_ROOT
    / "FINAL_TEST_READY_A1_A3"
)

BUNDLE_ZIP = (
    WORK_ROOT
    / "FINAL_TEST_READY_A1_A3.zip"
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Verify candidate adapters
# ------------------------------------------------------------

for path in [
    A1_ADAPTER,
    A3_ADAPTER,
]:

    assert path.exists()


a1_model = (
    A1_ADAPTER
    / "adapter_model.safetensors"
)

a3_model = (
    A3_ADAPTER
    / "adapter_model.safetensors"
)


assert sha256_file(
    a1_model
) == EXPECTED_A1_SHA256

assert sha256_file(
    a3_model
) == EXPECTED_A3_SHA256


# ------------------------------------------------------------
# Rebuild bundle cleanly
# ------------------------------------------------------------

if BUNDLE_DIR.exists():

    shutil.rmtree(
        BUNDLE_DIR
    )


BUNDLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Copy full adapter directories
# ------------------------------------------------------------

shutil.copytree(
    A1_ADAPTER,
    BUNDLE_DIR / "A1_adapter",
)

shutil.copytree(
    A3_ADAPTER,
    BUNDLE_DIR / "A3_adapter",
)


# ------------------------------------------------------------
# Exact frozen prompts
# ------------------------------------------------------------

PROMPT_V1 = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


PROMPT_V2_LONG = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "Reason carefully before deciding. For numerical, date, range, "
    "and comparison claims, compute or check the exact relation first. "
    "Exact claims require matching the correct value, entity, date, "
    "and time period; nearby values do not count. "

    "If a stated range permits both the claim and its alternative, "
    "use NOT_ENOUGH_INFO. "

    "Infer that something is absent from a list only when the evidence "
    "establishes that the list is complete.\n\n"

    "Return exactly one label and nothing else: "
    "SUPPORTS, REFUTES, or NOT_ENOUGH_INFO."
)


# ------------------------------------------------------------
# Test-time manifest
# ------------------------------------------------------------

manifest = {
    "purpose": (
        "Portable backup of the two retained "
        "Gemma-2-2B competition candidates."
    ),

    "base_model": (
        "google/gemma-2-2b-it"
    ),

    "labels": [
        "SUPPORTS",
        "REFUTES",
        "NOT_ENOUGH_INFO",
    ],

    "generation": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 10,
    },

    "candidates": {
        "A1": {
            "status": (
                "primary 2B organizer-distribution champion"
            ),

            "adapter_directory": (
                "A1_adapter"
            ),

            "adapter_model_sha256": (
                EXPECTED_A1_SHA256
            ),

            "prompt_name": "V1",

            "prompt": PROMPT_V1,

            "max_length": 256,

            "internal_dev": {
                "accuracy": 0.8214,
                "macro_f1": 0.8249,
            },

            "organizer_validation": {
                "accuracy": 0.8200,
                "macro_f1": 0.8207,
            },

            "external_stress_30": {
                "accuracy": 0.5667,
                "macro_f1": 0.5260,
            },
        },

        "A3": {
            "status": (
                "2B external-reasoning robustness candidate"
            ),

            "adapter_directory": (
                "A3_adapter"
            ),

            "adapter_model_sha256": (
                EXPECTED_A3_SHA256
            ),

            "prompt_name": "V2_LONG",

            "prompt": PROMPT_V2_LONG,

            "max_length": 384,

            "internal_dev": {
                "accuracy": 0.8571,
                "macro_f1": 0.8580,
            },

            "organizer_validation": {
                "accuracy": 0.8067,
                "macro_f1": 0.8077,
            },

            "external_stress_30": {
                "accuracy": 0.6333,
                "macro_f1": 0.6248,
            },
        },
    },

    "important_test_time_rule": (
        "Use the prompt paired with the adapter. "
        "Do not run A1 with Prompt V2 or A3 with Prompt V1."
    ),
}


manifest_path = (
    BUNDLE_DIR
    / "TEST_READY_MANIFEST.json"
)


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Human-readable instructions
# ------------------------------------------------------------

readme = f"""# Final Gemma-2-2B Test Candidates

## A1

Base model:

google/gemma-2-2b-it

Adapter:

A1_adapter/

Prompt:

V1

Maximum sequence length:

256

Adapter SHA-256:

{EXPECTED_A1_SHA256}

Organizer validation Macro-F1:

0.8207


## A3

Base model:

google/gemma-2-2b-it

Adapter:

A3_adapter/

Prompt:

V2_LONG

Maximum sequence length:

384

Adapter SHA-256:

{EXPECTED_A3_SHA256}

External 30-example stress Macro-F1:

0.6248


## Critical rule

Each adapter must be evaluated with the prompt it was trained with.

A1 -> Prompt V1

A3 -> Prompt V2_LONG


## Hidden test

When test.json/test.jsonl arrives, load the same base model,
attach the desired adapter, render its matching prompt, generate
deterministically, and create the required id,label submission file.

Do not modify the adapter files.
"""


(
    BUNDLE_DIR
    / "README_TEST_TIME.md"
).write_text(
    readme,
    encoding="utf-8",
)


# ------------------------------------------------------------
# Copy useful experiment records when available
# ------------------------------------------------------------

records_dir = (
    BUNDLE_DIR
    / "experiment_records"
)

records_dir.mkdir()


important_records = [
    WORK_ROOT
    / "gemma2_2b_experiment_closure.json",

    WORK_ROOT
    / "gemma2_2b_experiment_closure.md",

    WORK_ROOT
    / "prompt_comparison_post_a3.json",

    WORK_ROOT
    / "external_stress_30_v1.jsonl",

    WORK_ROOT
    / "external_stress_30_v1_audit.json",

    WORK_ROOT
    / "external_stress_30_a1_a2_a3_predictions.csv",

    WORK_ROOT
    / "external_stress_30_a1_a2_a3_results.json",

    WORK_ROOT
    / "dataset_v2_audit.json",
]


copied_records = []


for path in important_records:

    if path.exists():

        shutil.copy2(
            path,
            records_dir / path.name,
        )

        copied_records.append(
            path.name
        )


# ------------------------------------------------------------
# Hash inventory
# ------------------------------------------------------------

inventory = {}


for file in sorted(
    BUNDLE_DIR.rglob("*")
):

    if file.is_file():

        relative = str(
            file.relative_to(
                BUNDLE_DIR
            )
        )

        inventory[relative] = {
            "size_bytes": (
                file.stat().st_size
            ),

            "sha256": (
                sha256_file(file)
            ),
        }


inventory_path = (
    BUNDLE_DIR
    / "FILE_HASHES.json"
)


inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

if BUNDLE_ZIP.exists():

    BUNDLE_ZIP.unlink()


with zipfile.ZipFile(
    BUNDLE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for file in sorted(
        BUNDLE_DIR.rglob("*")
    ):

        if file.is_file():

            zf.write(
                file,
                arcname=(
                    Path(BUNDLE_DIR.name)
                    / file.relative_to(
                        BUNDLE_DIR
                    )
                ),
            )


zip_sha = sha256_file(
    BUNDLE_ZIP
)


# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

print("=" * 78)
print("FINAL TEST-READY BACKUP — RESULT")
print("=" * 78)


print("\nCandidates included")

print(
    "  A1:",
    "PASS",
)

print(
    "     adapter SHA:",
    EXPECTED_A1_SHA256,
)

print(
    "  A3:",
    "PASS",
)

print(
    "     adapter SHA:",
    EXPECTED_A3_SHA256,
)


print("\nRecords copied")

for name in copied_records:

    print(
        " ",
        name,
    )


print("\nBundle")

print(
    "  Directory:",
    BUNDLE_DIR,
)

print(
    "  ZIP:",
    BUNDLE_ZIP,
)

print(
    "  ZIP size:",
    round(
        BUNDLE_ZIP.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "  ZIP SHA-256:",
    zip_sha,
)


print("\nIMPORTANT")

print(
    "  Download this ZIP and keep a copy "
    "outside Kaggle:"
)

print(
    " ",
    BUNDLE_ZIP,
)


print("\nClickable download link:")

display(
    FileLink(
        str(BUNDLE_ZIP)
    )
)


print("\nNo model inference occurred.")
print("No training occurred.")

In [ ]:
# ============================================================
# B0 — GEMMA 4 E4B IT
#      HARDWARE + ARCHITECTURE + 4-BIT LOAD PREFLIGHT
#
# MODEL:
#   google/gemma-4-E4B-it
#
# PURPOSE:
#   1. Confirm Kaggle software support
#   2. Load Gemma 4 E4B IT in NF4 4-bit on GPU0 ONLY
#   3. Measure real T4 VRAM usage
#   4. Inspect architecture / layer count
#   5. Discover the ACTUAL quantized projection modules
#   6. Identify large non-4bit tensors / embeddings
#   7. Verify tokenizer/chat template + thinking controls
#   8. Run one finite-logit forward pass
#
# NO TRAINING.
# NO LoRA attached yet.
# NO internal-dev evaluation.
# NO organizer-validation access.
# NO external-stress evaluation.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import json
import os
import platform
import time

import torch
import transformers
import bitsandbytes as bnb

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)


# ------------------------------------------------------------
# A. Configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

OUT_PATH = Path(
    "/kaggle/working/"
    "b0_gemma4_e4b_hardware_architecture_preflight.json"
)


# ------------------------------------------------------------
# B. Environment
# ------------------------------------------------------------

print("=" * 78)
print("B0 — ENVIRONMENT")
print("=" * 78)


print(
    "Python:",
    platform.python_version(),
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "Transformers:",
    transformers.__version__,
)

print(
    "bitsandbytes:",
    bnb.__version__,
)

print(
    "CUDA available:",
    torch.cuda.is_available(),
)

print(
    "GPU count:",
    torch.cuda.device_count(),
)


assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1


for index in range(
    torch.cuda.device_count()
):

    props = torch.cuda.get_device_properties(
        index
    )

    print(
        f"GPU {index}:",
        props.name,
        "| VRAM:",
        round(
            props.total_memory
            / 1024**3,
            2,
        ),
        "GB",
        "| capability:",
        f"{props.major}.{props.minor}",
    )


# We deliberately use ONLY GPU0 for this preflight.
torch.cuda.set_device(0)


# ------------------------------------------------------------
# C. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable. "
        "Gemma access requires your accepted "
        "Hugging Face access token."
    )


# ------------------------------------------------------------
# D. Clean GPU
# ------------------------------------------------------------

for object_name in [
    "model",
    "processor",
    "inputs",
    "outputs",
    "quant_config",
]:

    if object_name in globals():

        try:
            del globals()[
                object_name
            ]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


torch.cuda.reset_peak_memory_stats(
    0
)


gpu_total_gb = (
    torch.cuda.get_device_properties(
        0
    ).total_memory
    / 1024**3
)


initial_allocated_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)

initial_reserved_gb = (
    torch.cuda.memory_reserved(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("B0 — CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU0 total:",
    round(
        gpu_total_gb,
        3,
    ),
    "GB",
)

print(
    "GPU0 allocated:",
    round(
        initial_allocated_gb,
        3,
    ),
    "GB",
)

print(
    "GPU0 reserved:",
    round(
        initial_reserved_gb,
        3,
    ),
    "GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        round(
            torch.cuda.memory_allocated(1)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ------------------------------------------------------------
# E. Processor / tokenizer
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — PROCESSOR / TOKENIZER")
print("=" * 78)


processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)


tokenizer = processor.tokenizer


print(
    "Processor class:",
    type(processor).__name__,
)

print(
    "Tokenizer class:",
    type(tokenizer).__name__,
)

print(
    "Vocabulary size:",
    len(tokenizer),
)

print(
    "BOS:",
    tokenizer.bos_token,
    tokenizer.bos_token_id,
)

print(
    "EOS:",
    tokenizer.eos_token,
    tokenizer.eos_token_id,
)

print(
    "PAD:",
    tokenizer.pad_token,
    tokenizer.pad_token_id,
)

print(
    "Chat template present:",
    tokenizer.chat_template
    is not None,
)


assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# F. Confirm thinking / non-thinking template support
# ------------------------------------------------------------

template_probe_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Classify this claim using only "
                    "the supplied evidence.\n\n"
                    "Claim: 9,800 is more than 20% "
                    "greater than 8,000.\n\n"
                    "Evidence: The values are "
                    "8,000 and 9,800."
                ),
            }
        ],
    }
]


thinking_template_supported = True


try:

    rendered_no_thinking = (
        processor
        .apply_chat_template(
            template_probe_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )


    rendered_thinking = (
        processor
        .apply_chat_template(
            template_probe_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    )


except TypeError:

    thinking_template_supported = False

    rendered_no_thinking = (
        processor
        .apply_chat_template(
            template_probe_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    rendered_thinking = None


print(
    "enable_thinking template argument:",
    thinking_template_supported,
)


no_thinking_tokens = tokenizer(
    rendered_no_thinking,
    add_special_tokens=False,
)["input_ids"]


print(
    "No-thinking probe tokens:",
    len(
        no_thinking_tokens
    ),
)


if rendered_thinking is not None:

    thinking_tokens = tokenizer(
        rendered_thinking,
        add_special_tokens=False,
    )["input_ids"]

    print(
        "Thinking probe tokens:",
        len(
            thinking_tokens
        ),
    )

else:

    thinking_tokens = None


# ------------------------------------------------------------
# G. 4-bit configuration
#
# T4:
#   - FP16 compute
#   - NF4
#   - double quantization
# ------------------------------------------------------------

quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


# ------------------------------------------------------------
# H. Load Gemma 4 on GPU0 ONLY
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — GEMMA 4 E4B 4-BIT LOAD")
print("=" * 78)


load_start = time.perf_counter()


try:

    model = (
        AutoModelForMultimodalLM
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
            quantization_config=(
                quant_config
            ),

            # IMPORTANT:
            # GPU0 only.
            device_map={
                "": 0
            },

            # T4 does not provide native BF16
            # execution, so keep nonquantized
            # floating modules in FP16.
            dtype=torch.float16,

            low_cpu_mem_usage=True,
        )
    )


except torch.cuda.OutOfMemoryError:

    print(
        "\nRESULT: GPU0 OOM DURING MODEL LOAD."
    )

    print(
        "One 14.56-GB T4 is NOT sufficient "
        "for this exact 4-bit loading setup."
    )

    raise


load_seconds = (
    time.perf_counter()
    - load_start
)


torch.cuda.synchronize(0)


load_allocated_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)

load_reserved_gb = (
    torch.cuda.memory_reserved(0)
    / 1024**3
)

load_peak_gb = (
    torch.cuda.max_memory_allocated(0)
    / 1024**3
)


print(
    "Model class:",
    type(model).__name__,
)

print(
    "Load seconds:",
    round(
        load_seconds,
        1,
    ),
)

print(
    "Loaded in 4-bit:",
    getattr(
        model,
        "is_loaded_in_4bit",
        False,
    ),
)

print(
    "GPU0 allocated:",
    round(
        load_allocated_gb,
        3,
    ),
    "GB",
)

print(
    "GPU0 reserved:",
    round(
        load_reserved_gb,
        3,
    ),
    "GB",
)

print(
    "GPU0 peak allocated:",
    round(
        load_peak_gb,
        3,
    ),
    "GB",
)

print(
    "GPU0 remaining from total:",
    round(
        gpu_total_gb
        - load_allocated_gb,
        3,
    ),
    "GB",
)


assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


# ------------------------------------------------------------
# I. Parameter / architecture inspection
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — ARCHITECTURE")
print("=" * 78)


config = model.config

text_config = getattr(
    config,
    "text_config",
    None,
)


print(
    "Config model_type:",
    getattr(
        config,
        "model_type",
        None,
    ),
)

print(
    "Architecture:",
    getattr(
        config,
        "architectures",
        None,
    ),
)


if text_config is not None:

    print(
        "Text model_type:",
        getattr(
            text_config,
            "model_type",
            None,
        ),
    )

    print(
        "Decoder layers:",
        getattr(
            text_config,
            "num_hidden_layers",
            None,
        ),
    )

    print(
        "Hidden size:",
        getattr(
            text_config,
            "hidden_size",
            None,
        ),
    )

    print(
        "Intermediate size:",
        getattr(
            text_config,
            "intermediate_size",
            None,
        ),
    )

    print(
        "Attention heads:",
        getattr(
            text_config,
            "num_attention_heads",
            None,
        ),
    )

    print(
        "KV heads:",
        getattr(
            text_config,
            "num_key_value_heads",
            None,
        ),
    )

    print(
        "Context length:",
        getattr(
            text_config,
            "max_position_embeddings",
            None,
        ),
    )

    print(
        "Sliding window:",
        getattr(
            text_config,
            "sliding_window",
            None,
        ),
    )

    print(
        "Vocabulary:",
        getattr(
            text_config,
            "vocab_size",
            None,
        ),
    )


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "\nTotal parameter elements:",
    f"{total_params:,}",
)

print(
    "Currently trainable:",
    f"{trainable_params:,}",
)


if hasattr(
    model,
    "get_memory_footprint",
):

    footprint_gb = (
        model.get_memory_footprint()
        / 1024**3
    )

    print(
        "Transformers model footprint:",
        round(
            footprint_gb,
            3,
        ),
        "GB",
    )

else:

    footprint_gb = None


# ------------------------------------------------------------
# J. Discover actual 4-bit Linear module names
#
# This replaces any assumption carried over
# from Gemma 2.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — 4-BIT MODULE DISCOVERY")
print("=" * 78)


linear4bit_modules = []


for name, module in (
    model.named_modules()
):

    if isinstance(
        module,
        bnb.nn.Linear4bit,
    ):

        linear4bit_modules.append(
            name
        )


leaf_counts = Counter(
    name.split(".")[-1]
    for name in linear4bit_modules
)


print(
    "Total Linear4bit modules:",
    len(
        linear4bit_modules
    ),
)


print(
    "\nLinear4bit module suffix counts:"
)


for leaf_name, count in sorted(
    leaf_counts.items(),
    key=lambda item: (
        -item[1],
        item[0],
    ),
):

    print(
        f"  {leaf_name:30} "
        f"{count:4d}"
    )


# Likely language projection candidates,
# but we DISCOVER rather than assume.
known_projection_names = {
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
}


projection_counts = {
    name: leaf_counts.get(
        name,
        0,
    )
    for name in sorted(
        known_projection_names
    )
}


print(
    "\nClassic language projection counts:"
)


for name, count in (
    projection_counts.items()
):

    print(
        f"  {name:15}",
        count,
    )


# ------------------------------------------------------------
# K. Largest parameter tensors
#
# Especially useful for Gemma 4 E4B because PLE
# embeddings contribute heavily to total parameters
# and may not be bitsandbytes Linear4bit tensors.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — LARGEST PARAMETER TENSORS")
print("=" * 78)


largest_parameters = sorted(
    [
        (
            name,
            parameter.numel(),
            str(parameter.dtype),
            bool(
                parameter.requires_grad
            ),
        )
        for name, parameter
        in model.named_parameters()
    ],
    key=lambda x: x[1],
    reverse=True,
)[:15]


for (
    name,
    numel,
    dtype,
    requires_grad,
) in largest_parameters:

    print(
        f"{numel / 1e6:10.2f} M | "
        f"{dtype:20} | "
        f"grad={str(requires_grad):5} | "
        f"{name}"
    )


# ------------------------------------------------------------
# L. One text-only finite-logit preflight
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — TEXT-ONLY NUMERICAL PREFLIGHT")
print("=" * 78)


model.eval()

model.config.use_cache = False


inputs = processor(
    text=rendered_no_thinking,
    return_tensors="pt",
)


# Move every tensor input to GPU0.
inputs = {
    key: value.to(
        "cuda:0"
    )
    if torch.is_tensor(value)
    else value

    for key, value
    in inputs.items()
}


print(
    "Input token length:",
    inputs[
        "input_ids"
    ].shape[-1],
)


torch.cuda.reset_peak_memory_stats(
    0
)


forward_start = (
    time.perf_counter()
)


with torch.inference_mode():

    outputs = model(
        **inputs,
        use_cache=False,
    )


torch.cuda.synchronize(0)


forward_seconds = (
    time.perf_counter()
    - forward_start
)


final_logits = (
    outputs.logits[
        0,
        -1,
        :
    ]
)


finite_logits = bool(
    torch.isfinite(
        final_logits
    ).all().item()
)


forward_peak_gb = (
    torch.cuda.max_memory_allocated(
        0
    )
    / 1024**3
)


print(
    "Final logits finite:",
    finite_logits,
)

print(
    "Logits dtype:",
    final_logits.dtype,
)

print(
    "Logits shape:",
    tuple(
        final_logits.shape
    ),
)

print(
    "Forward seconds:",
    round(
        forward_seconds,
        3,
    ),
)

print(
    "Forward peak GPU0 allocated:",
    round(
        forward_peak_gb,
        3,
    ),
    "GB",
)


assert finite_logits


# ------------------------------------------------------------
# M. Preliminary hardware classification
#
# This is ONLY an inference/load classification.
# Training safety still requires a QLoRA backward
# memory probe in the next step.
# ------------------------------------------------------------

remaining_after_load = (
    gpu_total_gb
    - load_allocated_gb
)


if (
    load_allocated_gb
    < 10.0
    and forward_peak_gb
    < 11.0
):

    inference_status = (
        "COMFORTABLE_FOR_4BIT_INFERENCE"
    )

elif (
    load_allocated_gb
    < 12.5
    and forward_peak_gb
    < 13.5
):

    inference_status = (
        "FITS_FOR_4BIT_INFERENCE_BUT_TRAINING_NEEDS_CAREFUL_PROBE"
    )

else:

    inference_status = (
        "VERY_TIGHT_ON_ONE_T4"
    )


# ------------------------------------------------------------
# N. Save preflight report
# ------------------------------------------------------------

report = {
    "model_id": MODEL_ID,

    "purpose": (
        "Gemma 4 E4B IT hardware and "
        "architecture preflight only."
    ),

    "training_occurred": False,

    "datasets_evaluated": False,

    "software": {
        "python": (
            platform.python_version()
        ),
        "torch": (
            torch.__version__
        ),
        "transformers": (
            transformers.__version__
        ),
        "bitsandbytes": (
            bnb.__version__
        ),
    },

    "gpu": {
        "name": (
            torch.cuda.get_device_properties(
                0
            ).name
        ),

        "total_gb": (
            gpu_total_gb
        ),

        "initial_allocated_gb": (
            initial_allocated_gb
        ),

        "load_allocated_gb": (
            load_allocated_gb
        ),

        "load_reserved_gb": (
            load_reserved_gb
        ),

        "load_peak_gb": (
            load_peak_gb
        ),

        "forward_peak_gb": (
            forward_peak_gb
        ),

        "remaining_after_load_gb": (
            remaining_after_load
        ),
    },

    "model": {
        "class": (
            type(model).__name__
        ),

        "model_type": (
            getattr(
                config,
                "model_type",
                None,
            )
        ),

        "loaded_in_4bit": (
            bool(
                getattr(
                    model,
                    "is_loaded_in_4bit",
                    False,
                )
            )
        ),

        "total_parameter_elements": (
            total_params
        ),

        "memory_footprint_gb": (
            footprint_gb
        ),

        "decoder_layers": (
            getattr(
                text_config,
                "num_hidden_layers",
                None,
            )
            if text_config
            is not None
            else None
        ),

        "context_length": (
            getattr(
                text_config,
                "max_position_embeddings",
                None,
            )
            if text_config
            is not None
            else None
        ),
    },

    "processor": {
        "class": (
            type(processor).__name__
        ),

        "tokenizer_class": (
            type(tokenizer).__name__
        ),

        "vocab_size": (
            len(tokenizer)
        ),

        "thinking_template_supported": (
            thinking_template_supported
        ),
    },

    "quantized_modules": {
        "total_linear4bit": (
            len(
                linear4bit_modules
            )
        ),

        "leaf_counts": dict(
            leaf_counts
        ),

        "classic_projection_counts": (
            projection_counts
        ),
    },

    "finite_preflight": {
        "input_tokens": int(
            inputs[
                "input_ids"
            ].shape[-1]
        ),

        "finite_logits": (
            finite_logits
        ),

        "logits_dtype": (
            str(
                final_logits.dtype
            )
        ),

        "forward_seconds": (
            forward_seconds
        ),
    },

    "preliminary_status": (
        inference_status
    ),
}


OUT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# O. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0 — FINAL PREFLIGHT RESULT")
print("=" * 78)


print(
    "Model:",
    MODEL_ID,
)

print(
    "Model class:",
    type(model).__name__,
)

print(
    "4-bit load:",
    "PASS",
)

print(
    "Finite logits:",
    "PASS",
)

print(
    "Thinking template support:",
    thinking_template_supported,
)


print("\nArchitecture")

print(
    "  Decoder layers:",
    getattr(
        text_config,
        "num_hidden_layers",
        None,
    )
    if text_config is not None
    else None,
)

print(
    "  Total parameter elements:",
    f"{total_params:,}",
)

print(
    "  Linear4bit modules:",
    len(
        linear4bit_modules
    ),
)


print("\nMemory")

print(
    "  GPU0 total:",
    f"{gpu_total_gb:.3f} GB",
)

print(
    "  Model loaded allocation:",
    f"{load_allocated_gb:.3f} GB",
)

print(
    "  Forward peak:",
    f"{forward_peak_gb:.3f} GB",
)

print(
    "  Remaining after model load:",
    f"{remaining_after_load:.3f} GB",
)


print(
    "\nPreliminary one-T4 status:"
)

print(
    " ",
    inference_status,
)


print(
    "\nIMPORTANT:"
)

print(
    "  This proves only 4-bit inference/load safety."
)

print(
    "  It does NOT yet prove QLoRA training fits."
)

print(
    "  We will run an actual backward-memory probe "
    "before any training."
)


print(
    "\nSaved:"
)

print(
    " ",
    OUT_PATH,
)


print("\nNo training occurred.")
print("No dataset evaluation occurred.")
print("GPU1 was not intentionally used.")

In [ ]:
# ============================================================
# B0.2 — GEMMA 4 E4B RAW REASONING DIAGNOSTIC
#
# IMPORTANT:
#   Run after restarting the Kaggle KERNEL following the
#   failed Gemma4ForCausalLM load.
#
# Official model:
#   google/gemma-4-E4B-it
#
# Compare:
#   1. Thinking OFF
#   2. Thinking ON
#
# Dataset:
#   Frozen 30-example EXTERNAL synthetic stress test ONLY.
#
# We specifically inspect:
#   - arithmetic / percentages
#   - elapsed-time reasoning
#   - logical contradiction
#   - NOT_ENOUGH_INFO boundaries
#
# NO TRAINING.
# NO LoRA.
# NO organizer validation.
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

STRESS_PATH = (
    WORK_ROOT
    / "external_stress_30_v1.jsonl"
)

EXPECTED_STRESS_SHA256 = (
    "05624200845a79228ac4e05540b013ea"
    "0836d049802fcd3f9dcc60ada8aecedc"
)

OLD_PRED_PATH = (
    WORK_ROOT
    / "external_stress_30_a1_a2_a3_predictions.csv"
)

OUT_CSV = (
    WORK_ROOT
    / "b0_2_gemma4_e4b_raw_reasoning_predictions.csv"
)

OUT_JSON = (
    WORK_ROOT
    / "b0_2_gemma4_e4b_raw_reasoning_results.json"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# C. Verify frozen external set
# ------------------------------------------------------------

assert STRESS_PATH.exists()

assert (
    sha256_file(STRESS_PATH)
    == EXPECTED_STRESS_SHA256
)


rows = read_jsonl(
    STRESS_PATH
)


assert len(rows) == 30


assert Counter(
    row["label"]
    for row in rows
) == Counter({
    "SUPPORTS": 11,
    "REFUTES": 10,
    "NOT_ENOUGH_INFO": 9,
})


# ------------------------------------------------------------
# D. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# E. Clean GPU
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()


torch.cuda.set_device(0)
torch.cuda.reset_peak_memory_stats(0)


start_allocated_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)

gpu_total_gb = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    / 1024**3
)


print("=" * 78)
print("B0.2 — CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU0 total:",
    f"{gpu_total_gb:.3f} GB",
)

print(
    "GPU0 allocated:",
    f"{start_allocated_gb:.3f} GB",
)

if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


# After kernel restart this should be small.
if start_allocated_gb > 3.0:

    raise RuntimeError(
        "GPU0 still has >3 GB allocated. "
        "Restart the Kaggle kernel before running B0.2."
    )


# ------------------------------------------------------------
# F. Processor
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = processor.tokenizer


assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# G. Load OFFICIAL full Gemma 4 in 4-bit
#
# We already established that this is the compatible
# checkpoint/model pairing.
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("B0.2 — OFFICIAL GEMMA 4 LOAD")
print("=" * 78)


load_start = time.perf_counter()


model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


torch.cuda.synchronize(0)


load_seconds = (
    time.perf_counter()
    - load_start
)


loaded_allocated_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print(
    "Model class:",
    type(model).__name__,
)

print(
    "4-bit:",
    getattr(
        model,
        "is_loaded_in_4bit",
        False,
    ),
)

print(
    "Load seconds:",
    round(load_seconds, 1),
)

print(
    "GPU0 allocated:",
    f"{loaded_allocated_gb:.3f} GB",
)

print(
    "Physical remaining:",
    f"{gpu_total_gb - loaded_allocated_gb:.3f} GB",
)


assert (
    type(model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


# ------------------------------------------------------------
# H. Freeze base for inference
#
# No dtype conversion.
# No PEFT preparation.
# ------------------------------------------------------------

for parameter in model.parameters():
    parameter.requires_grad = False


assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0


model.eval()


# ------------------------------------------------------------
# I. Frozen diagnostic prompt
#
# This is deliberately concise.
#
# Thinking mode itself is the experimental variable.
# ------------------------------------------------------------

INSTRUCTION = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "For numerical, percentage, date, duration, range, or comparison "
    "claims, check the exact values and relations before deciding.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


def make_messages(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )

    content = (
        INSTRUCTION
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": content,
                }
            ],
        }
    ]


# ------------------------------------------------------------
# J. Robust final-label parser
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_label(text):

    matches = list(
        FINAL_RE.finditer(text)
    )

    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    # Diagnostic fallback only.
    fallback = list(
        LABEL_RE.finditer(text)
    )

    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


# ------------------------------------------------------------
# K. Metrics
# ------------------------------------------------------------

def calculate_metrics(
    gold,
    predictions,
):

    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"
        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    return {
        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            np.mean(f1)
        ),

        "invalid": int(
            sum(
                p is None
                for p in predictions
            )
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class_f1": {
            LABELS[i]: float(
                f1[i]
            )
            for i in range(
                len(LABELS)
            )
        },
    }


# ------------------------------------------------------------
# L. Evaluate one thinking mode
#
# One example at a time intentionally:
#   - safer VRAM
#   - thinking responses have variable length
# ------------------------------------------------------------

def evaluate_mode(
    mode_name,
    enable_thinking,
):

    print("\n" + "=" * 78)
    print(
        f"B0.2 — {mode_name}"
    )
    print("=" * 78)


    predictions = []
    raw_outputs = []
    token_counts = []


    start_time = time.perf_counter()


    for index, row in enumerate(
        rows,
        1,
    ):

        messages = make_messages(
            row
        )


        inputs = (
            processor
            .apply_chat_template(
                messages,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=(
                    enable_thinking
                ),
            )
        )


        inputs = {
            key: (
                value.to("cuda:0")
                if torch.is_tensor(
                    value
                )
                else value
            )
            for key, value
            in inputs.items()
        }


        input_len = (
            inputs["input_ids"]
            .shape[-1]
        )


        # Thinking needs enough room for reasoning.
        max_new_tokens = (
            256
            if enable_thinking
            else 32
        )


        with torch.inference_mode():

            output = model.generate(
                **inputs,
                do_sample=False,
                num_beams=1,
                max_new_tokens=(
                    max_new_tokens
                ),
                pad_token_id=(
                    tokenizer.pad_token_id
                ),
            )


        generated_tokens = output[
            0,
            input_len:
        ]


        raw = processor.decode(
            generated_tokens,
            skip_special_tokens=False,
        )


        prediction = parse_label(
            raw
        )


        predictions.append(
            prediction
        )

        raw_outputs.append(
            raw
        )

        token_counts.append(
            int(
                generated_tokens.numel()
            )
        )


        print(
            f"{row['id']} | "
            f"gold={row['label']:16} | "
            f"pred={str(prediction):16} | "
            f"tokens={token_counts[-1]:3d}"
        )


        del inputs
        del output
        del generated_tokens

        torch.cuda.empty_cache()


    elapsed = (
        time.perf_counter()
        - start_time
    )


    gold = [
        row["label"]
        for row in rows
    ]


    metrics = calculate_metrics(
        gold,
        predictions,
    )


    print("\nResult")

    print(
        "  Accuracy:",
        f"{metrics['accuracy']:.4f}",
    )

    print(
        "  Macro-F1:",
        f"{metrics['macro_f1']:.4f}",
    )

    print(
        "  Invalid:",
        metrics["invalid"],
    )

    print(
        "  Mean generated tokens:",
        f"{np.mean(token_counts):.1f}",
    )

    print(
        "  Max generated tokens:",
        max(token_counts),
    )

    print(
        "  Seconds:",
        round(elapsed, 1),
    )


    print(
        "  Confusion matrix:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )

    for label, matrix_row in zip(
        LABELS,
        metrics[
            "confusion_matrix"
        ],
    ):

        print(
            f"  {label:15}",
            matrix_row,
        )


    print(
        "  Per-class F1:"
    )

    for label in LABELS:

        print(
            f"    {label:15} "
            f"{metrics['per_class_f1'][label]:.4f}"
        )


    return {
        "mode": mode_name,

        "enable_thinking": (
            enable_thinking
        ),

        "metrics": metrics,

        "predictions": predictions,

        "raw_outputs": raw_outputs,

        "generated_token_counts": (
            token_counts
        ),

        "seconds": float(
            elapsed
        ),
    }


# ------------------------------------------------------------
# M. Run both modes
# ------------------------------------------------------------

results = {}


results["NO_THINKING"] = evaluate_mode(
    "NO THINKING",
    False,
)


results["THINKING"] = evaluate_mode(
    "THINKING",
    True,
)


# ------------------------------------------------------------
# N. Reasoning-family diagnostic groups
#
# IDs correspond to the already-frozen external set.
# These are diagnostic groups only.
# ------------------------------------------------------------

GROUPS = {
    "arithmetic_percentage": {
        "external_stress_v1_01",
        "external_stress_v1_05",
        "external_stress_v1_06",
        "external_stress_v1_19",
        "external_stress_v1_20",
        "external_stress_v1_27",
        "external_stress_v1_28",
    },

    "temporal_duration": {
        "external_stress_v1_02",
        "external_stress_v1_07",
        "external_stress_v1_21",
        "external_stress_v1_22",
    },

    "nei_boundary": {
        "external_stress_v1_11",
        "external_stress_v1_12",
        "external_stress_v1_13",
        "external_stress_v1_14",
        "external_stress_v1_15",
        "external_stress_v1_18",
        "external_stress_v1_23",
        "external_stress_v1_26",
        "external_stress_v1_29",
    },

    "direct_logic_contradiction": {
        "external_stress_v1_04",
        "external_stress_v1_09",
        "external_stress_v1_10",
        "external_stress_v1_25",
        "external_stress_v1_30",
    },
}


id_to_index = {
    row["id"]: i
    for i, row in enumerate(
        rows
    )
}


group_results = {}


for group_name, ids in (
    GROUPS.items()
):

    print("\n" + "-" * 78)
    print(
        "GROUP:",
        group_name,
    )
    print("-" * 78)


    group_results[
        group_name
    ] = {}


    for mode_name in [
        "NO_THINKING",
        "THINKING",
    ]:

        predictions = (
            results[
                mode_name
            ][
                "predictions"
            ]
        )


        correct = 0


        for example_id in sorted(
            ids
        ):

            idx = id_to_index[
                example_id
            ]

            gold = rows[
                idx
            ]["label"]

            pred = predictions[
                idx
            ]

            correct += int(
                pred == gold
            )


        accuracy = (
            correct
            / len(ids)
        )


        group_results[
            group_name
        ][
            mode_name
        ] = {
            "correct": correct,
            "total": len(ids),
            "accuracy": (
                accuracy
            ),
        }


        print(
            f"{mode_name:12} | "
            f"{correct}/{len(ids)} | "
            f"{accuracy:.1%}"
        )


# ------------------------------------------------------------
# O. Compare OFF vs ON prediction transitions
# ------------------------------------------------------------

thinking_fixed = []
thinking_broke = []
both_correct = []
both_wrong = []


for i, row in enumerate(
    rows
):

    gold = row["label"]

    off_pred = (
        results[
            "NO_THINKING"
        ][
            "predictions"
        ][i]
    )

    on_pred = (
        results[
            "THINKING"
        ][
            "predictions"
        ][i]
    )


    off_correct = (
        off_pred == gold
    )

    on_correct = (
        on_pred == gold
    )


    if (
        not off_correct
        and on_correct
    ):

        thinking_fixed.append(
            row["id"]
        )

    elif (
        off_correct
        and not on_correct
    ):

        thinking_broke.append(
            row["id"]
        )

    elif (
        off_correct
        and on_correct
    ):

        both_correct.append(
            row["id"]
        )

    else:

        both_wrong.append(
            row["id"]
        )


# ------------------------------------------------------------
# P. Compare with saved A1/A3 predictions
# ------------------------------------------------------------

old_comparison = None


if OLD_PRED_PATH.exists():

    old_df = pd.read_csv(
        OLD_PRED_PATH
    )


    old_by_id = {
        row["id"]: row
        for row in old_df.to_dict(
            orient="records"
        )
    }


    old_comparison = {
        "A1": {
            "correct": 0,
        },

        "A3": {
            "correct": 0,
        },
    }


    for row in rows:

        rec = old_by_id[
            row["id"]
        ]

        old_comparison[
            "A1"
        ][
            "correct"
        ] += int(
            rec[
                "A1_prediction"
            ]
            == row["label"]
        )

        old_comparison[
            "A3"
        ][
            "correct"
        ] += int(
            rec[
                "A3_prediction"
            ]
            == row["label"]
        )


# ------------------------------------------------------------
# Q. Save detailed predictions
# ------------------------------------------------------------

prediction_rows = []


for i, row in enumerate(
    rows
):

    prediction_rows.append({
        "id": row["id"],

        "gold": row["label"],

        "claim": row["claim"],

        "gemma4_no_thinking": (
            results[
                "NO_THINKING"
            ][
                "predictions"
            ][i]
        ),

        "gemma4_thinking": (
            results[
                "THINKING"
            ][
                "predictions"
            ][i]
        ),

        "no_thinking_correct": (
            results[
                "NO_THINKING"
            ][
                "predictions"
            ][i]
            == row["label"]
        ),

        "thinking_correct": (
            results[
                "THINKING"
            ][
                "predictions"
            ][i]
            == row["label"]
        ),

        "no_thinking_raw": (
            results[
                "NO_THINKING"
            ][
                "raw_outputs"
            ][i]
        ),

        "thinking_raw": (
            results[
                "THINKING"
            ][
                "raw_outputs"
            ][i]
        ),
    })


pd.DataFrame(
    prediction_rows
).to_csv(
    OUT_CSV,
    index=False,
)


payload = {
    "model_id": MODEL_ID,

    "dataset": {
        "path": str(
            STRESS_PATH
        ),

        "sha256": (
            EXPECTED_STRESS_SHA256
        ),

        "rows": 30,

        "organizer_validation_used": (
            False
        ),
    },

    "prompt": (
        INSTRUCTION
    ),

    "results": {
        mode: {
            "enable_thinking": (
                result[
                    "enable_thinking"
                ]
            ),

            "metrics": (
                result[
                    "metrics"
                ]
            ),

            "seconds": (
                result[
                    "seconds"
                ]
            ),

            "mean_generated_tokens": float(
                np.mean(
                    result[
                        "generated_token_counts"
                    ]
                )
            ),
        }

        for mode, result
        in results.items()
    },

    "groups": group_results,

    "thinking_vs_no_thinking": {
        "fixed": (
            thinking_fixed
        ),

        "broken": (
            thinking_broke
        ),

        "both_correct": (
            both_correct
        ),

        "both_wrong": (
            both_wrong
        ),
    },

    "old_external_reference": (
        old_comparison
    ),
}


OUT_JSON.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# R. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0.2 — FINAL RAW REASONING RESULT")
print("=" * 78)


for mode_name in [
    "NO_THINKING",
    "THINKING",
]:

    m = (
        results[
            mode_name
        ][
            "metrics"
        ]
    )


    print(
        f"{mode_name:12} | "
        f"Accuracy={m['accuracy']:.4f} | "
        f"Macro-F1={m['macro_f1']:.4f} | "
        f"SUP={m['per_class_f1']['SUPPORTS']:.4f} | "
        f"REF={m['per_class_f1']['REFUTES']:.4f} | "
        f"NEI={m['per_class_f1']['NOT_ENOUGH_INFO']:.4f}"
    )


if old_comparison is not None:

    print("\nExisting fine-tuned 2B references")

    print(
        "  A1:",
        f"{old_comparison['A1']['correct']}/30",
        "| known Macro-F1=0.5260",
    )

    print(
        "  A3:",
        f"{old_comparison['A3']['correct']}/30",
        "| known Macro-F1=0.6248",
    )


print("\nReasoning groups")

for group_name in GROUPS:

    off = (
        group_results[
            group_name
        ][
            "NO_THINKING"
        ]
    )

    on = (
        group_results[
            group_name
        ][
            "THINKING"
        ]
    )


    print(
        f"  {group_name:28} | "
        f"OFF={off['correct']}/{off['total']} "
        f"({off['accuracy']:.1%}) | "
        f"ON={on['correct']}/{on['total']} "
        f"({on['accuracy']:.1%})"
    )


print("\nThinking ON vs OFF")

print(
    "  Fixed:",
    len(thinking_fixed),
    thinking_fixed,
)

print(
    "  Broken:",
    len(thinking_broke),
    thinking_broke,
)

print(
    "  Both correct:",
    len(both_correct),
)

print(
    "  Both wrong:",
    len(both_wrong),
)


print("\nCritical arithmetic / temporal examples")


critical_ids = [
    "external_stress_v1_01",
    "external_stress_v1_02",
    "external_stress_v1_05",
    "external_stress_v1_06",
    "external_stress_v1_19",
    "external_stress_v1_20",
    "external_stress_v1_21",
    "external_stress_v1_22",
    "external_stress_v1_27",
    "external_stress_v1_28",
]


for example_id in critical_ids:

    i = id_to_index[
        example_id
    ]

    row = rows[i]

    print(
        f"  {example_id} | "
        f"gold={row['label']:16} | "
        f"OFF={str(results['NO_THINKING']['predictions'][i]):16} | "
        f"ON={str(results['THINKING']['predictions'][i]):16}"
    )


print("\nSaved")

print(
    " ",
    OUT_CSV,
)

print(
    " ",
    OUT_JSON,
)


print("\nNo training occurred.")
print("No organizer-validation inference occurred.")
print("GPU1 was not used.")

In [ ]:
# ============================================================
# B0.3 — RAW GEMMA 4 INTERNAL-DEV DIAGNOSTIC
#
# Compare two prompts on the unchanged 140-row internal dev:
#
#   BASE
#   REASONING
#
# Thinking is OFF for both.
# Output format is held constant.
#
# Purpose:
#   Determine which prompt should be used for B1 fine-tuning.
#
# NO TRAINING.
# NO LoRA.
# NO organizer validation.
# NO synthetic stress-set selection.
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

OUT_PATH = (
    WORK_ROOT
    / "b0_3_gemma4_raw_internal_dev_prompt_comparison.json"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# C. Verify unchanged internal dev
# ------------------------------------------------------------

assert DEV_PATH.exists()

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)


dev_rows = read_jsonl(
    DEV_PATH
)


assert len(dev_rows) == 140


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


print("=" * 78)
print("B0.3 — FROZEN INTERNAL DEV")
print("=" * 78)

print(
    "Rows:",
    len(dev_rows),
)

print(
    "SHA-256:",
    sha256_file(DEV_PATH),
)

print(
    "Organizer validation used:",
    False,
)


# ------------------------------------------------------------
# D. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# E. Reuse B0.2 model if available.
# Otherwise reload cleanly.
# ------------------------------------------------------------

valid_existing_model = False


if "model" in globals():

    try:

        valid_existing_model = (
            type(model).__name__
            == "Gemma4ForConditionalGeneration"
            and getattr(
                model,
                "is_loaded_in_4bit",
                False,
            )
        )

    except Exception:

        valid_existing_model = False


if "processor" in globals():

    try:

        valid_existing_processor = (
            type(processor).__name__
            == "Gemma4Processor"
        )

    except Exception:

        valid_existing_processor = False

else:

    valid_existing_processor = False


if (
    valid_existing_model
    and valid_existing_processor
):

    print("\nReusing B0.2 Gemma 4 model.")

    tokenizer = processor.tokenizer


else:

    print(
        "\nB0.2 model not available; "
        "loading official Gemma 4."
    )


    for object_name in [
        "model",
        "processor",
        "tokenizer",
        "quant_config",
    ]:

        if object_name in globals():

            try:
                del globals()[
                    object_name
                ]

            except Exception:
                pass


    gc.collect()
    torch.cuda.empty_cache()

    if hasattr(
        torch.cuda,
        "ipc_collect",
    ):
        torch.cuda.ipc_collect()


    processor = (
        AutoProcessor
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
        )
    )

    tokenizer = processor.tokenizer


    quant_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(
                torch.float16
            ),
        )
    )


    model = (
        AutoModelForMultimodalLM
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
            quantization_config=(
                quant_config
            ),
            device_map={"": 0},
            dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
    )


assert (
    type(model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


for parameter in model.parameters():

    parameter.requires_grad = False


model.eval()


print(
    "GPU0 allocation:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


# ------------------------------------------------------------
# F. Prompt candidates
#
# IMPORTANT:
# Same output format for both prompts.
# Only reasoning instructions change.
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


PROMPT_REASONING = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence is sufficient to establish the claim.\n"

    "REFUTES: the evidence is sufficient to establish that the "
    "claim is false.\n"

    "NOT_ENOUGH_INFO: the evidence establishes neither the claim "
    "nor its contradiction. Missing support is not REFUTES.\n\n"

    "For numerical, percentage, date, duration, range, or comparison "
    "claims, check the exact values and relations before deciding. "
    "Match the correct entity and time period.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


PROMPTS = {
    "BASE": PROMPT_BASE,
    "REASONING": PROMPT_REASONING,
}


# ------------------------------------------------------------
# G. Renderer
# ------------------------------------------------------------

def make_messages(
    row,
    instruction,
):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    content = (
        instruction
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": content,
                }
            ],
        }
    ]


# ------------------------------------------------------------
# H. Parser
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_label(text):

    matches = list(
        FINAL_RE.finditer(text)
    )

    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(text)
    )

    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


# ------------------------------------------------------------
# I. Metric helper
# ------------------------------------------------------------

def metrics(
    gold,
    predictions,
):

    metric_predictions = [
        p
        if p is not None
        else "__INVALID__"

        for p in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    return {
        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            np.mean(f1)
        ),

        "invalid": int(
            sum(
                p is None
                for p in predictions
            )
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),
                "recall": float(
                    recall[i]
                ),
                "f1": float(
                    f1[i]
                ),
                "support": int(
                    support[i]
                ),
            }

            for i, label in enumerate(
                LABELS
            )
        },
    }


# ------------------------------------------------------------
# J. Token-length audit
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0.3 — PROMPT LENGTHS")
print("=" * 78)


prompt_lengths = {}


for prompt_name, instruction in (
    PROMPTS.items()
):

    lengths = []


    for row in dev_rows:

        rendered = (
            processor
            .apply_chat_template(
                make_messages(
                    row,
                    instruction,
                ),
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )


        length = len(
            tokenizer(
                rendered,
                add_special_tokens=False,
            )["input_ids"]
        )


        lengths.append(
            length
        )


    prompt_lengths[
        prompt_name
    ] = {
        "min": int(
            min(lengths)
        ),

        "median": float(
            np.median(lengths)
        ),

        "p95": float(
            np.percentile(
                lengths,
                95,
            )
        ),

        "max": int(
            max(lengths)
        ),
    }


    print(
        f"{prompt_name:10} | "
        f"min={min(lengths):3d} | "
        f"median={np.median(lengths):.1f} | "
        f"p95={np.percentile(lengths,95):.1f} | "
        f"max={max(lengths):3d}"
    )


# ------------------------------------------------------------
# K. Evaluate each prompt
#
# One example at a time = conservative VRAM.
# Thinking OFF = short generation.
# ------------------------------------------------------------

results = {}


for prompt_name, instruction in (
    PROMPTS.items()
):

    print("\n" + "=" * 78)
    print(
        f"B0.3 — {prompt_name}"
    )
    print("=" * 78)


    predictions = []
    raw_outputs = []


    start_time = (
        time.perf_counter()
    )


    for index, row in enumerate(
        dev_rows,
        1,
    ):

        inputs = (
            processor
            .apply_chat_template(
                make_messages(
                    row,
                    instruction,
                ),
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )


        inputs = {
            key: (
                value.to("cuda:0")
                if torch.is_tensor(
                    value
                )
                else value
            )

            for key, value
            in inputs.items()
        }


        input_len = (
            inputs[
                "input_ids"
            ].shape[-1]
        )


        with torch.inference_mode():

            generated = model.generate(
                **inputs,
                do_sample=False,
                num_beams=1,
                max_new_tokens=16,
                pad_token_id=(
                    tokenizer.pad_token_id
                ),
            )


        output_tokens = generated[
            0,
            input_len:
        ]


        raw = processor.decode(
            output_tokens,
            skip_special_tokens=False,
        )


        prediction = (
            parse_label(raw)
        )


        predictions.append(
            prediction
        )

        raw_outputs.append(
            raw
        )


        if (
            index % 20 == 0
            or index
            == len(dev_rows)
        ):

            print(
                f"  evaluated "
                f"{index}/{len(dev_rows)}"
            )


        del inputs
        del generated
        del output_tokens

        torch.cuda.empty_cache()


    elapsed = (
        time.perf_counter()
        - start_time
    )


    gold = [
        row["label"]
        for row in dev_rows
    ]


    result_metrics = metrics(
        gold,
        predictions,
    )


    results[
        prompt_name
    ] = {
        "metrics": (
            result_metrics
        ),

        "predictions": (
            predictions
        ),

        "raw_outputs": (
            raw_outputs
        ),

        "seconds": float(
            elapsed
        ),
    }


    print("\nResult")

    print(
        "  Accuracy:",
        f"{result_metrics['accuracy']:.4f}",
    )

    print(
        "  Macro-F1:",
        f"{result_metrics['macro_f1']:.4f}",
    )

    print(
        "  Invalid:",
        result_metrics["invalid"],
    )


    print(
        "  Confusion matrix:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )


    for label, matrix_row in zip(
        LABELS,
        result_metrics[
            "confusion_matrix"
        ],
    ):

        print(
            f"  {label:15}",
            matrix_row,
        )


    print(
        "  Per-class F1:"
    )

    for label in LABELS:

        print(
            f"    {label:15} "
            f"{result_metrics['per_class'][label]['f1']:.4f}"
        )


    print(
        "  Seconds:",
        round(
            elapsed,
            1,
        ),
    )


# ------------------------------------------------------------
# L. Paired prompt transitions
# ------------------------------------------------------------

base_predictions = (
    results[
        "BASE"
    ][
        "predictions"
    ]
)

reason_predictions = (
    results[
        "REASONING"
    ][
        "predictions"
    ]
)


fixed = []
broken = []
both_correct = []
both_wrong = []


for i, row in enumerate(
    dev_rows
):

    gold = row["label"]

    base_correct = (
        base_predictions[i]
        == gold
    )

    reason_correct = (
        reason_predictions[i]
        == gold
    )


    if (
        not base_correct
        and reason_correct
    ):

        fixed.append(
            row["id"]
        )

    elif (
        base_correct
        and not reason_correct
    ):

        broken.append(
            row["id"]
        )

    elif (
        base_correct
        and reason_correct
    ):

        both_correct.append(
            row["id"]
        )

    else:

        both_wrong.append(
            row["id"]
        )


# ------------------------------------------------------------
# M. Pick pretraining prompt candidate
#
# This is diagnostic selection before B1 training.
# ------------------------------------------------------------

ranking = sorted(
    PROMPTS.keys(),

    key=lambda name: (
        results[
            name
        ][
            "metrics"
        ][
            "macro_f1"
        ],

        results[
            name
        ][
            "metrics"
        ][
            "accuracy"
        ],
    ),

    reverse=True,
)


selected_prompt = ranking[0]


# ------------------------------------------------------------
# N. Save compact report
# ------------------------------------------------------------

payload = {
    "model_id": MODEL_ID,

    "thinking_enabled": False,

    "training_occurred": False,

    "organizer_validation_used": False,

    "internal_dev": {
        "path": str(
            DEV_PATH
        ),

        "sha256": (
            EXPECTED_DEV_SHA256
        ),

        "rows": 140,
    },

    "prompts": PROMPTS,

    "prompt_lengths": (
        prompt_lengths
    ),

    "results": {
        prompt_name: {
            "metrics": (
                result[
                    "metrics"
                ]
            ),

            "seconds": (
                result[
                    "seconds"
                ]
            ),
        }

        for prompt_name, result
        in results.items()
    },

    "reasoning_vs_base": {
        "fixed": fixed,
        "broken": broken,
        "both_correct": (
            both_correct
        ),
        "both_wrong": (
            both_wrong
        ),
    },

    "selected_prompt_for_next_preflight": (
        selected_prompt
    ),
}


OUT_PATH.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# O. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0.3 — FINAL INTERNAL-DEV RESULT")
print("=" * 78)


for prompt_name in [
    "BASE",
    "REASONING",
]:

    m = (
        results[
            prompt_name
        ][
            "metrics"
        ]
    )


    print(
        f"{prompt_name:10} | "
        f"Accuracy={m['accuracy']:.4f} | "
        f"Macro-F1={m['macro_f1']:.4f} | "
        f"SUP={m['per_class']['SUPPORTS']['f1']:.4f} | "
        f"REF={m['per_class']['REFUTES']['f1']:.4f} | "
        f"NEI={m['per_class']['NOT_ENOUGH_INFO']['f1']:.4f}"
    )


print("\nKnown fine-tuned 2B internal-dev references")

print(
    "  A1 | Accuracy=0.8214 | "
    "Macro-F1=0.8249"
)

print(
    "  A3 | Accuracy=0.8571 | "
    "Macro-F1=0.8580"
)


print("\nREASONING vs BASE paired changes")

print(
    "  Fixed:",
    len(fixed),
)

print(
    "  Broken:",
    len(broken),
)

print(
    "  Both correct:",
    len(both_correct),
)

print(
    "  Both wrong:",
    len(both_wrong),
)


print(
    "\nSelected prompt for next B-series preflight:",
    selected_prompt,
)


print("\nSaved:")

print(
    " ",
    OUT_PATH,
)


print("\nNo training occurred.")
print(
    "No organizer-validation inference occurred."
)

In [ ]:
# ============================================================
# B0.4 — GEMMA 4 E4B LOW-MEMORY QLoRA BACKWARD PREFLIGHT
#
# PURPOSE:
#   1. Reuse official 4-bit Gemma 4 E4B model
#   2. Freeze ALL base parameters WITHOUT FP32 upcast
#   3. Attach LoRA ONLY to language-model projections
#   4. Verify no vision/audio LoRA modules are targeted
#   5. Audit completion-only training construction
#   6. Measure exact fit-set token lengths
#   7. Select required max_length with safety margin
#   8. Run ONE real forward + backward pass
#   9. Measure peak VRAM and gradient finiteness
#
# IMPORTANT:
#   We deliberately DO NOT call:
#       prepare_model_for_kbit_training()
#
#   because Gemma 4 contains enormous FP16 embedding tensors
#   that standard preparation may upcast to FP32.
#
# NO optimizer step.
# NO epoch training.
# NO organizer validation.
# NO synthetic data.
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import re
import time

import numpy as np
import torch
import bitsandbytes as bnb

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

OUT_PATH = (
    WORK_ROOT
    / "b0_4_gemma4_e4b_low_memory_qlora_backward_probe.json"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


# Frozen B1 candidate prompt selected by B0.3.
PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


# LoRA baseline.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


# Require at least this much unused sequence space.
REQUIRED_MARGIN = 24


# Candidate training lengths.
LENGTH_CANDIDATES = [
    256,
    288,
    320,
    384,
    512,
]


# ------------------------------------------------------------
# B. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# C. Verify frozen fit set
# ------------------------------------------------------------

assert FIT_PATH.exists()

assert (
    sha256_file(FIT_PATH)
    == EXPECTED_FIT_SHA256
)


fit_rows = read_jsonl(
    FIT_PATH
)


assert len(fit_rows) == 795


assert Counter(
    row["label"]
    for row in fit_rows
) == Counter({
    "SUPPORTS": 295,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})


print("=" * 78)
print("B0.4 — FROZEN TRAINING DATA")
print("=" * 78)


print(
    "Rows:",
    len(fit_rows),
)

print(
    "SHA-256:",
    sha256_file(FIT_PATH),
)

print(
    "Synthetic examples:",
    0,
)

print(
    "Organizer validation used:",
    False,
)


# ------------------------------------------------------------
# D. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# E. Reuse B0.3 model if possible
# ------------------------------------------------------------

valid_model = False
valid_processor = False


if "model" in globals():

    try:

        valid_model = (
            type(model).__name__
            == "Gemma4ForConditionalGeneration"
            and getattr(
                model,
                "is_loaded_in_4bit",
                False,
            )
            and not hasattr(
                model,
                "peft_config",
            )
        )

    except Exception:

        valid_model = False


if "processor" in globals():

    try:

        valid_processor = (
            type(processor).__name__
            == "Gemma4Processor"
        )

    except Exception:

        valid_processor = False


if (
    valid_model
    and valid_processor
):

    print(
        "\nReusing B0.3 official Gemma 4 model."
    )

    tokenizer = processor.tokenizer


else:

    print(
        "\nReloading official Gemma 4 model."
    )


    for object_name in [
        "model",
        "processor",
        "tokenizer",
        "quant_config",
    ]:

        if object_name in globals():

            try:
                del globals()[object_name]

            except Exception:
                pass


    gc.collect()
    torch.cuda.empty_cache()

    if hasattr(
        torch.cuda,
        "ipc_collect",
    ):
        torch.cuda.ipc_collect()


    processor = (
        AutoProcessor
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
        )
    )

    tokenizer = (
        processor.tokenizer
    )


    quant_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(
                torch.float16
            ),
        )
    )


    model = (
        AutoModelForMultimodalLM
        .from_pretrained(
            MODEL_ID,
            token=hf_token,
            quantization_config=(
                quant_config
            ),
            device_map={"": 0},
            dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
    )


assert (
    type(model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


torch.cuda.set_device(0)

gc.collect()
torch.cuda.empty_cache()


base_loaded_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


gpu_total_gb = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    / 1024**3
)


print("\n" + "=" * 78)
print("B0.4 — BASE MODEL MEMORY")
print("=" * 78)


print(
    "GPU total:",
    f"{gpu_total_gb:.3f} GB",
)

print(
    "Base allocated:",
    f"{base_loaded_gb:.3f} GB",
)

print(
    "Physical remaining:",
    f"{gpu_total_gb - base_loaded_gb:.3f} GB",
)


# ------------------------------------------------------------
# F. Freeze EVERY base parameter
#
# CRITICAL:
#   no FP16 -> FP32 conversion.
# ------------------------------------------------------------

for parameter in model.parameters():

    parameter.requires_grad = False


base_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


assert base_trainable == 0


# ------------------------------------------------------------
# G. Discover EXACT language-model projection modules
#
# Gemma 4 full model also contains vision/audio modules.
# We MUST NOT LoRA those.
# ------------------------------------------------------------

TARGET_SUFFIXES = {
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
}


language_target_names = []


for name, module in (
    model.named_modules()
):

    if not isinstance(
        module,
        bnb.nn.Linear4bit,
    ):

        continue


    if not name.startswith(
        "model.language_model.layers."
    ):

        continue


    suffix = name.split(".")[-1]


    if suffix in TARGET_SUFFIXES:

        language_target_names.append(
            name
        )


language_target_suffix_counts = (
    Counter(
        name.split(".")[-1]
        for name
        in language_target_names
    )
)


print("\n" + "=" * 78)
print("B0.4 — LANGUAGE LoRA TARGET DISCOVERY")
print("=" * 78)


print(
    "Language projection modules:",
    len(language_target_names),
)


for suffix in [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]:

    print(
        f"  {suffix:15}",
        language_target_suffix_counts[
            suffix
        ],
    )


# Expected from B0:
#
# q/o/gate/up/down = 42 each
# k/v = 24 each
#
# Total = 258
assert (
    language_target_suffix_counts[
        "q_proj"
    ]
    == 42
)

assert (
    language_target_suffix_counts[
        "o_proj"
    ]
    == 42
)

assert (
    language_target_suffix_counts[
        "gate_proj"
    ]
    == 42
)

assert (
    language_target_suffix_counts[
        "up_proj"
    ]
    == 42
)

assert (
    language_target_suffix_counts[
        "down_proj"
    ]
    == 42
)

assert (
    language_target_suffix_counts[
        "k_proj"
    ]
    == 24
)

assert (
    language_target_suffix_counts[
        "v_proj"
    ]
    == 24
)

assert len(
    language_target_names
) == 258


# ------------------------------------------------------------
# H. Attach LoRA ONLY to language-model projections
#
# A string target_modules value is treated as a regex by PEFT.
#
# This regex cannot match the vision/audio towers.
# ------------------------------------------------------------

TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\.(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\.(?:gate_proj|up_proj|down_proj)"
    r")"
)


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_REGEX,
)


memory_before_lora_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


model = get_peft_model(
    model,
    lora_config,
)


memory_after_lora_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


# ------------------------------------------------------------
# I. Verify actual LoRA locations
# ------------------------------------------------------------

lora_module_names = []


for name, module in (
    model.named_modules()
):

    if hasattr(
        module,
        "lora_A",
    ):

        if len(
            module.lora_A
        ) > 0:

            lora_module_names.append(
                name
            )


vision_lora = [
    name
    for name
    in lora_module_names
    if "vision" in name.lower()
]


audio_lora = [
    name
    for name
    in lora_module_names
    if "audio" in name.lower()
]


language_lora = [
    name
    for name
    in lora_module_names
    if "language_model.layers"
    in name
]


assert len(
    vision_lora
) == 0

assert len(
    audio_lora
) == 0

assert len(
    language_lora
) == 258


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


print("\n" + "=" * 78)
print("B0.4 — LoRA ATTACHED")
print("=" * 78)


print(
    "LoRA modules:",
    len(lora_module_names),
)

print(
    "Language LoRA modules:",
    len(language_lora),
)

print(
    "Vision LoRA modules:",
    len(vision_lora),
)

print(
    "Audio LoRA modules:",
    len(audio_lora),
)


print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)

print(
    "Total stored parameters:",
    f"{total_params:,}",
)

print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%",
)


print(
    "GPU before LoRA:",
    f"{memory_before_lora_gb:.3f} GB",
)

print(
    "GPU after LoRA:",
    f"{memory_after_lora_gb:.3f} GB",
)

print(
    "LoRA persistent memory delta:",
    f"{memory_after_lora_gb - memory_before_lora_gb:+.3f} GB",
)


# ------------------------------------------------------------
# J. Verify the giant Gemma 4 embeddings remain frozen FP16
#
# This is a critical safety condition.
# ------------------------------------------------------------

critical_embedding_info = {}


for name, parameter in (
    model.named_parameters()
):

    if (
        "embed_tokens_per_layer.weight"
        in name
        or name.endswith(
            "language_model.embed_tokens.weight"
        )
    ):

        critical_embedding_info[
            name
        ] = {
            "numel": int(
                parameter.numel()
            ),

            "dtype": str(
                parameter.dtype
            ),

            "requires_grad": bool(
                parameter.requires_grad
            ),
        }


print("\nCritical embeddings:")


for name, info in (
    critical_embedding_info.items()
):

    print(
        f"  {name}"
    )

    print(
        "    elements:",
        f"{info['numel']:,}",
    )

    print(
        "    dtype:",
        info["dtype"],
    )

    print(
        "    trainable:",
        info[
            "requires_grad"
        ],
    )


assert len(
    critical_embedding_info
) >= 2


for info in (
    critical_embedding_info.values()
):

    assert (
        info["dtype"]
        == "torch.float16"
    )

    assert (
        info["requires_grad"]
        is False
    )


# ------------------------------------------------------------
# K. Enable gradient checkpointing
#
# Non-reentrant checkpointing is intentional.
# ------------------------------------------------------------

model.config.use_cache = False


model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    }
)


# ------------------------------------------------------------
# L. Training sequence construction
#
# User turn -> generation prompt
# Full sequence -> user + assistant FINAL label
#
# Loss applies ONLY to assistant completion tokens.
# ------------------------------------------------------------

def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    return (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",

                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        }
    ]


def full_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",

                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        },

        {
            "role": "assistant",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "FINAL: "
                        + row["label"]
                    ),
                }
            ],
        },
    ]


def get_prompt_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            user_messages(row),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


def get_full_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            full_messages(row),
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


# ------------------------------------------------------------
# M. Exact fit-set token audit
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0.4 — FIT TOKEN AUDIT")
print("=" * 78)


prompt_lengths = []
full_lengths = []

completion_lengths = []
audit_rows = []


prefix_failures = []


for row in fit_rows:

    prompt_ids = get_prompt_ids(
        row
    )

    full_ids = get_full_ids(
        row
    )


    prompt_len = len(
        prompt_ids
    )

    full_len = len(
        full_ids
    )


    # The full training sequence must begin
    # with the exact generation prompt.
    prefix_ok = (
        full_ids[
            :prompt_len
        ]
        == prompt_ids
    )


    if not prefix_ok:

        prefix_failures.append(
            row["id"]
        )


    completion_len = (
        full_len
        - prompt_len
    )


    prompt_lengths.append(
        prompt_len
    )

    full_lengths.append(
        full_len
    )

    completion_lengths.append(
        completion_len
    )


    audit_rows.append({
        "id": row["id"],
        "label": row["label"],
        "prompt_len": prompt_len,
        "full_len": full_len,
        "completion_len": (
            completion_len
        ),
    })


assert len(
    prefix_failures
) == 0


max_full = max(
    full_lengths
)


selected_max_length = None


for candidate in (
    LENGTH_CANDIDATES
):

    if (
        candidate
        - max_full
        >= REQUIRED_MARGIN
    ):

        selected_max_length = (
            candidate
        )

        break


if selected_max_length is None:

    raise RuntimeError(
        "No candidate max_length "
        "provides required margin."
    )


longest_record = max(
    audit_rows,
    key=lambda item: (
        item["full_len"]
    ),
)


print(
    "Prompt min:",
    min(prompt_lengths),
)

print(
    "Prompt median:",
    f"{np.median(prompt_lengths):.1f}",
)

print(
    "Prompt p95:",
    f"{np.percentile(prompt_lengths,95):.1f}",
)

print(
    "Prompt max:",
    max(prompt_lengths),
)


print(
    "\nFull min:",
    min(full_lengths),
)

print(
    "Full median:",
    f"{np.median(full_lengths):.1f}",
)

print(
    "Full p95:",
    f"{np.percentile(full_lengths,95):.1f}",
)

print(
    "Full max:",
    max_full,
)


print(
    "\nCompletion lengths:",
    sorted(
        set(
            completion_lengths
        )
    ),
)


print(
    "\nLongest:",
    longest_record,
)


print(
    "\nSelected max_length:",
    selected_max_length,
)

print(
    "Safety margin:",
    selected_max_length
    - max_full,
)


# ------------------------------------------------------------
# N. Completion-only masking check
# ------------------------------------------------------------

probe_row = next(
    row
    for row in fit_rows
    if row["id"]
    == longest_record["id"]
)


prompt_ids = get_prompt_ids(
    probe_row
)

full_ids = get_full_ids(
    probe_row
)


input_ids_list = list(
    full_ids
)

labels_list = (
    [-100] * len(
        prompt_ids
    )
    + full_ids[
        len(prompt_ids):
    ]
)


assert len(
    input_ids_list
) == len(
    labels_list
)


supervised_tokens = sum(
    label != -100
    for label in labels_list
)


assert (
    supervised_tokens
    == longest_record[
        "completion_len"
    ]
)


completion_text = (
    tokenizer.decode(
        full_ids[
            len(prompt_ids):
        ],
        skip_special_tokens=False,
    )
)


print("\n" + "=" * 78)
print("B0.4 — COMPLETION-ONLY MASK CHECK")
print("=" * 78)


print(
    "Probe ID:",
    probe_row["id"],
)

print(
    "Gold:",
    probe_row["label"],
)

print(
    "Prompt tokens:",
    len(prompt_ids),
)

print(
    "Full tokens:",
    len(full_ids),
)

print(
    "Supervised completion tokens:",
    supervised_tokens,
)

print(
    "Completion repr:",
    repr(
        completion_text
    ),
)


# ------------------------------------------------------------
# O. Pad to exact selected max_length
#
# This deliberately measures worst-case sequence memory.
# ------------------------------------------------------------

pad_count = (
    selected_max_length
    - len(
        input_ids_list
    )
)


assert pad_count >= 0


padded_input_ids = (
    input_ids_list
    + [
        tokenizer.pad_token_id
    ] * pad_count
)


padded_attention_mask = (
    [1] * len(
        input_ids_list
    )
    + [0] * pad_count
)


padded_labels = (
    labels_list
    + [-100] * pad_count
)


input_ids = torch.tensor(
    [
        padded_input_ids
    ],
    dtype=torch.long,
    device="cuda:0",
)


attention_mask = torch.tensor(
    [
        padded_attention_mask
    ],
    dtype=torch.long,
    device="cuda:0",
)


labels = torch.tensor(
    [
        padded_labels
    ],
    dtype=torch.long,
    device="cuda:0",
)


assert (
    input_ids.shape
    == attention_mask.shape
    == labels.shape
)


assert (
    input_ids.shape[1]
    == selected_max_length
)


# ------------------------------------------------------------
# P. Clean before backward memory probe
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()


for parameter in (
    model.parameters()
):

    if parameter.grad is not None:

        parameter.grad = None


memory_before_probe_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


torch.cuda.reset_peak_memory_stats(
    0
)


print("\n" + "=" * 78)
print("B0.4 — REAL FORWARD + BACKWARD PROBE")
print("=" * 78)


print(
    "Physical batch:",
    1,
)

print(
    "Sequence length:",
    selected_max_length,
)

print(
    "Allocated before probe:",
    f"{memory_before_probe_gb:.3f} GB",
)


# ------------------------------------------------------------
# Q. One REAL forward + backward
# ------------------------------------------------------------

probe_start = (
    time.perf_counter()
)


try:

    model.train()


    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False,
    )


    loss = outputs.loss


    loss_finite = bool(
        torch.isfinite(
            loss
        ).item()
    )


    print(
        "Forward loss:",
        float(
            loss.detach().cpu()
        ),
    )

    print(
        "Loss finite:",
        loss_finite,
    )


    assert loss_finite


    loss.backward()


    torch.cuda.synchronize(0)


except torch.cuda.OutOfMemoryError:

    print(
        "\nRESULT: OOM DURING REAL BACKWARD."
    )

    print(
        "One-T4 QLoRA is NOT safe at "
        f"sequence length {selected_max_length} "
        "with this configuration."
    )

    raise


probe_seconds = (
    time.perf_counter()
    - probe_start
)


peak_allocated_gb = (
    torch.cuda.max_memory_allocated(0)
    / 1024**3
)

after_backward_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


# ------------------------------------------------------------
# R. Gradient audit
# ------------------------------------------------------------

gradient_parameter_tensors = 0

finite_gradient_tensors = 0

nonfinite_gradient_tensors = []

zero_gradient_tensors = []

grad_sq_sum = 0.0


for name, parameter in (
    model.named_parameters()
):

    if not parameter.requires_grad:

        continue


    if parameter.grad is None:

        continue


    gradient_parameter_tensors += 1


    grad = parameter.grad


    if torch.isfinite(
        grad
    ).all():

        finite_gradient_tensors += 1

    else:

        nonfinite_gradient_tensors.append(
            name
        )


    grad_float = grad.float()


    if (
        torch.count_nonzero(
            grad_float
        ).item()
        == 0
    ):

        zero_gradient_tensors.append(
            name
        )


    grad_sq_sum += float(
        torch.sum(
            grad_float
            * grad_float
        ).item()
    )


gradient_norm = math.sqrt(
    grad_sq_sum
)


all_gradients_finite = (
    len(
        nonfinite_gradient_tensors
    )
    == 0
)


print("\nGradient audit")

print(
    "  Trainable tensors with gradients:",
    gradient_parameter_tensors,
)

print(
    "  Finite gradient tensors:",
    finite_gradient_tensors,
)

print(
    "  Non-finite gradient tensors:",
    len(
        nonfinite_gradient_tensors
    ),
)

print(
    "  Zero gradient tensors:",
    len(
        zero_gradient_tensors
    ),
)

print(
    "  Raw global grad norm:",
    gradient_norm,
)


print("\nMemory")

print(
    "  Allocated before:",
    f"{memory_before_probe_gb:.3f} GB",
)

print(
    "  Peak allocated:",
    f"{peak_allocated_gb:.3f} GB",
)

print(
    "  After backward:",
    f"{after_backward_gb:.3f} GB",
)

print(
    "  Physical headroom at peak:",
    f"{gpu_total_gb - peak_allocated_gb:.3f} GB",
)

print(
    "  Probe seconds:",
    f"{probe_seconds:.2f}",
)


# ------------------------------------------------------------
# S. Training-safety classification
# ------------------------------------------------------------

headroom_gb = (
    gpu_total_gb
    - peak_allocated_gb
)


if (
    all_gradients_finite
    and headroom_gb >= 1.5
):

    training_status = (
        "SAFE_FOR_BATCH1_QLORA_PROVISIONALLY"
    )

elif (
    all_gradients_finite
    and headroom_gb >= 0.6
):

    training_status = (
        "TIGHT_BUT_POSSIBLE_BATCH1"
    )

elif all_gradients_finite:

    training_status = (
        "TOO_TIGHT_FOR_RELIABLE_ONE_T4_TRAINING"
    )

else:

    training_status = (
        "GRADIENT_NUMERICAL_FAILURE"
    )


# ------------------------------------------------------------
# T. Save report
# ------------------------------------------------------------

report = {
    "model_id": MODEL_ID,

    "purpose": (
        "Real low-memory QLoRA "
        "forward/backward preflight."
    ),

    "training_occurred": False,

    "optimizer_step_occurred": False,

    "organizer_validation_used": False,

    "synthetic_examples_used": False,

    "prompt": {
        "name": "BASE",
        "thinking": False,
        "text": PROMPT_BASE,
    },

    "fit": {
        "rows": 795,
        "sha256": (
            EXPECTED_FIT_SHA256
        ),
    },

    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": (
            LORA_DROPOUT
        ),

        "target_regex": (
            TARGET_REGEX
        ),

        "language_modules": (
            len(
                language_lora
            )
        ),

        "vision_modules": (
            len(
                vision_lora
            )
        ),

        "audio_modules": (
            len(
                audio_lora
            )
        ),

        "trainable_parameters": (
            trainable_params
        ),
    },

    "token_audit": {
        "prompt_min": int(
            min(
                prompt_lengths
            )
        ),

        "prompt_median": float(
            np.median(
                prompt_lengths
            )
        ),

        "prompt_p95": float(
            np.percentile(
                prompt_lengths,
                95,
            )
        ),

        "prompt_max": int(
            max(
                prompt_lengths
            )
        ),

        "full_min": int(
            min(
                full_lengths
            )
        ),

        "full_median": float(
            np.median(
                full_lengths
            )
        ),

        "full_p95": float(
            np.percentile(
                full_lengths,
                95,
            )
        ),

        "full_max": int(
            max_full
        ),

        "completion_lengths": sorted(
            set(
                completion_lengths
            )
        ),

        "selected_max_length": (
            selected_max_length
        ),

        "margin": (
            selected_max_length
            - max_full
        ),
    },

    "probe": {
        "batch_size": 1,

        "sequence_length": (
            selected_max_length
        ),

        "loss": float(
            loss.detach().cpu()
        ),

        "loss_finite": (
            loss_finite
        ),

        "gradient_norm": (
            gradient_norm
        ),

        "all_gradients_finite": (
            all_gradients_finite
        ),

        "gradient_tensors": (
            gradient_parameter_tensors
        ),

        "nonfinite_gradient_tensors": (
            nonfinite_gradient_tensors
        ),

        "zero_gradient_tensor_count": (
            len(
                zero_gradient_tensors
            )
        ),

        "seconds": (
            probe_seconds
        ),
    },

    "memory": {
        "gpu_total_gb": (
            gpu_total_gb
        ),

        "base_loaded_gb": (
            base_loaded_gb
        ),

        "before_lora_gb": (
            memory_before_lora_gb
        ),

        "after_lora_gb": (
            memory_after_lora_gb
        ),

        "before_probe_gb": (
            memory_before_probe_gb
        ),

        "peak_backward_gb": (
            peak_allocated_gb
        ),

        "after_backward_gb": (
            after_backward_gb
        ),

        "peak_headroom_gb": (
            headroom_gb
        ),
    },

    "critical_embeddings": (
        critical_embedding_info
    ),

    "status": (
        training_status
    ),
}


OUT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# U. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B0.4 — FINAL QLoRA PREFLIGHT RESULT")
print("=" * 78)


print(
    "Model:",
    MODEL_ID,
)

print(
    "Prompt:",
    "BASE",
)

print(
    "Thinking:",
    False,
)


print("\nLoRA")

print(
    "  r:",
    LORA_R,
)

print(
    "  alpha:",
    LORA_ALPHA,
)

print(
    "  dropout:",
    LORA_DROPOUT,
)

print(
    "  language modules:",
    len(
        language_lora
    ),
)

print(
    "  vision modules:",
    len(
        vision_lora
    ),
)

print(
    "  audio modules:",
    len(
        audio_lora
    ),
)

print(
    "  trainable parameters:",
    f"{trainable_params:,}",
)


print("\nTokenization")

print(
    "  Max full training sequence:",
    max_full,
)

print(
    "  Selected max_length:",
    selected_max_length,
)

print(
    "  Margin:",
    selected_max_length
    - max_full,
)

print(
    "  Completion lengths:",
    sorted(
        set(
            completion_lengths
        )
    ),
)


print("\nNumerical")

print(
    "  Loss:",
    f"{float(loss.detach().cpu()):.6f}",
)

print(
    "  Loss finite:",
    loss_finite,
)

print(
    "  Gradient norm:",
    f"{gradient_norm:.6f}",
)

print(
    "  All gradients finite:",
    all_gradients_finite,
)


print("\nMemory")

print(
    "  GPU total:",
    f"{gpu_total_gb:.3f} GB",
)

print(
    "  Base loaded:",
    f"{base_loaded_gb:.3f} GB",
)

print(
    "  After LoRA:",
    f"{memory_after_lora_gb:.3f} GB",
)

print(
    "  Backward peak:",
    f"{peak_allocated_gb:.3f} GB",
)

print(
    "  Peak headroom:",
    f"{headroom_gb:.3f} GB",
)


print("\nOne-T4 QLoRA status:")

print(
    " ",
    training_status,
)


print("\nSaved:")

print(
    " ",
    OUT_PATH,
)


print("\nNO optimizer step occurred.")
print("NO epoch training occurred.")
print(
    "NO organizer-validation inference occurred."
)

In [ ]:
# ============================================================
# B1 RETRY — CLEAN-RUNTIME CHECK
#
# Run this immediately after restarting the Kaggle kernel.
#
# Do NOT run B0.2/B0.3/B0.4 first.
# ============================================================

import os

# Helps reduce CUDA allocator fragmentation.
# Set before importing torch in the restarted runtime.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch


assert torch.cuda.is_available()

torch.cuda.set_device(0)

gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()


allocated_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)

reserved_gb = (
    torch.cuda.memory_reserved(0)
    / 1024**3
)

total_gb = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / 1024**3
)


print("=" * 78)
print("B1 RETRY — CLEAN RUNTIME CHECK")
print("=" * 78)

print(
    "GPU0 total:",
    f"{total_gb:.3f} GB",
)

print(
    "GPU0 allocated:",
    f"{allocated_gb:.3f} GB",
)

print(
    "GPU0 reserved:",
    f"{reserved_gb:.3f} GB",
)

if torch.cuda.device_count() > 1:
    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


if allocated_gb <= 0.5:

    print("\nSTATUS: PASS — runtime is clean.")
    print(
        "Next: run the original B1 training cell "
        "directly. Do not run B0 cells first."
    )

else:

    print("\nSTATUS: FAIL — runtime is not clean.")

    raise RuntimeError(
        "GPU0 still has more than 0.5 GB allocated. "
        "Restart the Kaggle session/kernel again before B1."
    )

In [ ]:
# ============================================================
# B1 — GEMMA 4 E4B IT QLoRA
#
# Controlled stronger-model experiment.
#
# MODEL:
#   google/gemma-4-E4B-it
#
# DATA:
#   795 frozen REAL fit examples
#   140 frozen internal-dev examples
#   0 synthetic examples
#
# PROMPT:
#   BASE prompt selected by B0.3
#   thinking=False
#
# QLoRA:
#   NF4 + double quant
#   FP16 compute
#   r=8
#   alpha=16
#   dropout=0.05
#   language projections ONLY
#
# TRAINING:
#   physical batch = 1
#   gradient accumulation = 16
#   effective batch = 16
#   epochs = 2
#   LR = 2e-4
#   weight decay = 0.01
#   cosine schedule
#   warmup = 5%
#   max_length = 256
#   completion-only loss
#   no GradScaler
#   seed = 42
#
# IMPORTANT:
#   We DO NOT call prepare_model_for_kbit_training().
#   Gemma 4's giant FP16 embeddings remain frozen FP16.
#
# NO organizer validation.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

RUN_NAME = "b1_gemma4_e4b_real795_base_prompt"

WORK_ROOT = Path("/kaggle/working")

RUN_DIR = (
    WORK_ROOT
    / RUN_NAME
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_ID = "google/gemma-4-E4B-it"


FIT_PATH = (
    WORK_ROOT
    / "train_fit_v2.jsonl"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)


EXPECTED_FIT_SHA256 = (
    "994809512667fee82490e61380cb0bad"
    "11503163af12b0e31efe9b3446d17582"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

LABEL_SET = set(LABELS)


SEED = 42

MAX_LENGTH = 256

EPOCHS = 2

PHYSICAL_BATCH_SIZE = 1

GRAD_ACCUM_STEPS = 16

LEARNING_RATE = 2e-4

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.05

MAX_GRAD_NORM = 1.0


LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


MAX_NEW_TOKENS = 16


# ------------------------------------------------------------
# B. Frozen B1 prompt
#
# Selected over REASONING prompt by B0.3:
#
# BASE:
#   Acc 0.5786
#   Macro-F1 0.5540
#
# REASONING:
#   Acc 0.5643
#   Macro-F1 0.5267
#
# Thinking is OFF.
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(
            f,
            1,
        ):

            if not line.strip():
                continue

            try:

                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name}, line "
                    f"{line_number}: {exc}"
                )

    return rows


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ------------------------------------------------------------
# D. Verify frozen datasets
# ------------------------------------------------------------

assert FIT_PATH.exists()
assert DEV_PATH.exists()


assert (
    sha256_file(FIT_PATH)
    == EXPECTED_FIT_SHA256
)

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)


fit_rows = read_jsonl(
    FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)


assert len(fit_rows) == 795
assert len(dev_rows) == 140


assert Counter(
    row["label"]
    for row in fit_rows
) == Counter({
    "SUPPORTS": 295,
    "REFUTES": 266,
    "NOT_ENOUGH_INFO": 234,
})


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


print("=" * 78)
print("B1 — EXPERIMENT DEFINITION")
print("=" * 78)


print(
    "Model:",
    MODEL_ID,
)

print(
    "Real fit rows:",
    len(fit_rows),
)

print(
    "Internal dev rows:",
    len(dev_rows),
)

print(
    "Synthetic rows:",
    0,
)

print(
    "max_length:",
    MAX_LENGTH,
)

print(
    "Thinking:",
    False,
)

print(
    "Organizer validation used:",
    False,
)


# ------------------------------------------------------------
# E. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Release previous B0 model completely
#
# B1 must start from a FRESH base + fresh LoRA.
# ------------------------------------------------------------

for object_name in [
    "model",
    "base_model",
    "processor",
    "tokenizer",
    "optimizer",
    "scheduler",
    "outputs",
    "quant_config",
    "lora_config",
]:

    if object_name in globals():

        try:
            del globals()[object_name]

        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


torch.cuda.set_device(0)

torch.cuda.reset_peak_memory_stats(
    0
)


print("\n" + "=" * 78)
print("B1 — CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)

if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


# ------------------------------------------------------------
# G. Processor
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = processor.tokenizer


assert tokenizer.chat_template is not None
assert tokenizer.pad_token_id is not None


# ------------------------------------------------------------
# H. Fresh official Gemma 4 load
# ------------------------------------------------------------

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("B1 — FRESH MODEL LOAD")
print("=" * 78)


load_start = time.perf_counter()


model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


torch.cuda.synchronize(0)


print(
    "Model class:",
    type(model).__name__,
)

print(
    "4-bit:",
    getattr(
        model,
        "is_loaded_in_4bit",
        False,
    ),
)

print(
    "Load seconds:",
    round(
        time.perf_counter()
        - load_start,
        1,
    ),
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


assert (
    type(model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


# ------------------------------------------------------------
# I. Freeze entire base WITHOUT dtype conversion
# ------------------------------------------------------------

for parameter in model.parameters():

    parameter.requires_grad = False


assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0


# ------------------------------------------------------------
# J. Language-only LoRA
# ------------------------------------------------------------

TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\.(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\.(?:gate_proj|up_proj|down_proj)"
    r")"
)


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_REGEX,
)


# Reset RNG immediately before LoRA initialization.
set_seed(SEED)


model = get_peft_model(
    model,
    lora_config,
)


# ------------------------------------------------------------
# K. Verify LoRA placement
# ------------------------------------------------------------

lora_module_names = []


for name, module in (
    model.named_modules()
):

    if (
        hasattr(
            module,
            "lora_A",
        )
        and len(
            module.lora_A
        ) > 0
    ):

        lora_module_names.append(
            name
        )


language_lora = [
    name
    for name
    in lora_module_names
    if "language_model.layers"
    in name
]


vision_lora = [
    name
    for name
    in lora_module_names
    if "vision" in name.lower()
]


audio_lora = [
    name
    for name
    in lora_module_names
    if "audio" in name.lower()
]


assert len(language_lora) == 258
assert len(vision_lora) == 0
assert len(audio_lora) == 0


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


assert (
    trainable_params
    == 17_440_768
)


print(
    "Language LoRA modules:",
    len(language_lora),
)

print(
    "Vision LoRA modules:",
    len(vision_lora),
)

print(
    "Audio LoRA modules:",
    len(audio_lora),
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)


# ------------------------------------------------------------
# L. Critical embedding safety check
# ------------------------------------------------------------

critical_embeddings = []


for name, parameter in (
    model.named_parameters()
):

    if (
        "embed_tokens_per_layer.weight"
        in name
        or name.endswith(
            "language_model.embed_tokens.weight"
        )
    ):

        critical_embeddings.append(
            (
                name,
                str(
                    parameter.dtype
                ),
                bool(
                    parameter.requires_grad
                ),
            )
        )


assert len(
    critical_embeddings
) >= 2


for (
    name,
    dtype,
    requires_grad,
) in critical_embeddings:

    assert dtype == "torch.float16"
    assert requires_grad is False


print(
    "Critical FP16 embeddings frozen:",
    True,
)


# ------------------------------------------------------------
# M. Gradient checkpointing
# ------------------------------------------------------------

model.config.use_cache = False


model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    }
)


# ------------------------------------------------------------
# N. Prompt / training sequence helpers
# ------------------------------------------------------------

def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    return (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        }
    ]


def full_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        },

        {
            "role": "assistant",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "FINAL: "
                        + row["label"]
                    ),
                }
            ],
        },
    ]


def get_prompt_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            user_messages(row),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


def get_full_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            full_messages(row),
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


# ------------------------------------------------------------
# O. Pre-tokenize training examples
#
# Completion-only labels.
# No truncation permitted.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1 — TRAINING TOKENIZATION")
print("=" * 78)


train_examples = []


for row in fit_rows:

    prompt_ids = get_prompt_ids(
        row
    )

    full_ids = get_full_ids(
        row
    )


    assert (
        full_ids[
            :len(
                prompt_ids
            )
        ]
        == prompt_ids
    )


    assert len(
        full_ids
    ) <= MAX_LENGTH


    labels = (
        [-100] * len(
            prompt_ids
        )
        + full_ids[
            len(prompt_ids):
        ]
    )


    assert len(
        labels
    ) == len(
        full_ids
    )


    train_examples.append({
        "id": row["id"],
        "label": row["label"],
        "input_ids": full_ids,
        "labels": labels,
    })


max_train_length = max(
    len(
        example["input_ids"]
    )
    for example in train_examples
)


print(
    "Training examples:",
    len(train_examples),
)

print(
    "Longest full sequence:",
    max_train_length,
)

print(
    "max_length:",
    MAX_LENGTH,
)

print(
    "Safety margin:",
    MAX_LENGTH
    - max_train_length,
)


assert (
    max_train_length
    == 229
)


# ------------------------------------------------------------
# P. Deterministic dev evaluator
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(
            text
        )
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(
            text
        )
    )


    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


def evaluate_internal_dev(
    tag,
):

    # Save training checkpointing state.
    model.gradient_checkpointing_disable()

    model.config.use_cache = True

    model.eval()


    predictions = []

    gold = []

    records = []


    start_time = (
        time.perf_counter()
    )


    try:

        for index, row in enumerate(
            dev_rows,
            1,
        ):

            inputs = (
                processor
                .apply_chat_template(
                    user_messages(row),
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            )


            inputs = {
                key: (
                    value.to(
                        "cuda:0"
                    )
                    if torch.is_tensor(
                        value
                    )
                    else value
                )

                for key, value
                in inputs.items()
            }


            input_len = (
                inputs[
                    "input_ids"
                ].shape[-1]
            )


            with torch.inference_mode():

                generated = (
                    model.generate(
                        **inputs,
                        do_sample=False,
                        num_beams=1,
                        max_new_tokens=(
                            MAX_NEW_TOKENS
                        ),
                        pad_token_id=(
                            tokenizer.pad_token_id
                        ),
                    )
                )


            new_tokens = generated[
                0,
                input_len:
            ]


            raw = processor.decode(
                new_tokens,
                skip_special_tokens=False,
            )


            prediction = (
                parse_prediction(
                    raw
                )
            )


            predictions.append(
                prediction
            )

            gold.append(
                row["label"]
            )


            records.append({
                "id": row["id"],
                "gold": row["label"],
                "prediction": (
                    prediction
                ),
                "raw_output": raw,
            })


            del inputs
            del generated
            del new_tokens

            torch.cuda.empty_cache()


    finally:

        model.config.use_cache = False

        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False,
            }
        )


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"

        for prediction
        in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    invalid = sum(
        prediction is None
        for prediction in predictions
    )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    result = {
        "tag": tag,

        "accuracy": float(
            accuracy
        ),

        "macro_f1": (
            macro_f1
        ),

        "invalid": int(
            invalid
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class": {
            label: {
                "precision": float(
                    precision[i]
                ),

                "recall": float(
                    recall[i]
                ),

                "f1": float(
                    f1[i]
                ),

                "support": int(
                    support[i]
                ),
            }

            for i, label in enumerate(
                LABELS
            )
        },

        "seconds": float(
            elapsed
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)


    print(
        "Rows:",
        len(dev_rows),
    )

    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid,
    )


    print(
        "Confusion matrix "
        "[rows=true, cols=pred]:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )


    for (
        label,
        matrix_row,
    ) in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            matrix_row,
        )


    print(
        "Per-class F1:"
    )


    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    print(
        "Seconds:",
        round(
            elapsed,
            1,
        ),
    )


    return result, records


# ------------------------------------------------------------
# Q. Pretrain dev evaluation
#
# This should approximately reproduce B0.3 BASE.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1 — PRE-TRAIN INTERNAL DEV")
print("=" * 78)


(
    pretrain_dev_result,
    pretrain_dev_records,
) = evaluate_internal_dev(
    "INTERNAL DEV — B1 PRETRAIN",
)


# ------------------------------------------------------------
# R. Optimizer / schedule
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


optimizer_steps_per_epoch = (
    math.ceil(
        len(train_examples)
        / GRAD_ACCUM_STEPS
    )
)


total_optimizer_steps = (
    optimizer_steps_per_epoch
    * EPOCHS
)


warmup_steps = int(
    round(
        total_optimizer_steps
        * WARMUP_RATIO
    )
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=(
            warmup_steps
        ),
        num_training_steps=(
            total_optimizer_steps
        ),
    )
)


print("\n" + "=" * 78)
print("B1 — OPTIMIZATION")
print("=" * 78)


print(
    "Physical batch:",
    PHYSICAL_BATCH_SIZE,
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS,
)

print(
    "Effective batch:",
    PHYSICAL_BATCH_SIZE
    * GRAD_ACCUM_STEPS,
)

print(
    "Optimizer steps / epoch:",
    optimizer_steps_per_epoch,
)

print(
    "Total optimizer steps:",
    total_optimizer_steps,
)

print(
    "Warmup steps:",
    warmup_steps,
)

print(
    "Learning rate:",
    LEARNING_RATE,
)

print(
    "Weight decay:",
    WEIGHT_DECAY,
)

print(
    "GradScaler:",
    False,
)


# ------------------------------------------------------------
# S. Frozen deterministic epoch orders
# ------------------------------------------------------------

epoch_orders = []


for epoch_index in range(
    EPOCHS
):

    generator = random.Random(
        SEED
        + epoch_index
    )


    indices = list(
        range(
            len(
                train_examples
            )
        )
    )


    generator.shuffle(
        indices
    )


    epoch_orders.append(
        indices
    )


# ------------------------------------------------------------
# T. One-example training tensor builder
# ------------------------------------------------------------

def make_training_tensors(
    example,
):

    input_ids = torch.tensor(
        [
            example[
                "input_ids"
            ]
        ],
        dtype=torch.long,
        device="cuda:0",
    )


    attention_mask = torch.ones_like(
        input_ids
    )


    labels = torch.tensor(
        [
            example[
                "labels"
            ]
        ],
        dtype=torch.long,
        device="cuda:0",
    )


    return (
        input_ids,
        attention_mask,
        labels,
    )


# ------------------------------------------------------------
# U. Training
# ------------------------------------------------------------

training_history = []

dev_history = [
    pretrain_dev_result
]

global_optimizer_step = 0


for epoch_index in range(
    EPOCHS
):

    epoch_number = (
        epoch_index
        + 1
    )


    print("\n" + "=" * 78)
    print(
        f"B1 TRAINING EPOCH "
        f"{epoch_number}/{EPOCHS}"
    )
    print("=" * 78)


    model.train()

    model.config.use_cache = False


    epoch_start = (
        time.perf_counter()
    )


    epoch_loss_sum = 0.0

    processed_examples = 0


    optimizer.zero_grad(
        set_to_none=True
    )


    order = epoch_orders[
        epoch_index
    ]


    for position, example_index in enumerate(
        order
    ):

        example = (
            train_examples[
                example_index
            ]
        )


        # ----------------------------------------------------
        # Normalize accumulation by ACTUAL window size.
        #
        # Last window has only 11 examples:
        #   795 mod 16 = 11
        #
        # It should not be divided by 16.
        # ----------------------------------------------------

        window_start = (
            position
            // GRAD_ACCUM_STEPS
        ) * GRAD_ACCUM_STEPS


        window_end = min(
            window_start
            + GRAD_ACCUM_STEPS,

            len(order),
        )


        actual_window_size = (
            window_end
            - window_start
        )


        (
            input_ids,
            attention_mask,
            labels,
        ) = make_training_tensors(
            example
        )


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )


        raw_loss = (
            outputs.loss
        )


        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                f"Non-finite loss at "
                f"epoch={epoch_number}, "
                f"position={position}, "
                f"id={example['id']}"
            )


        epoch_loss_sum += float(
            raw_loss
            .detach()
            .cpu()
        )


        processed_examples += 1


        scaled_loss = (
            raw_loss
            / actual_window_size
        )


        scaled_loss.backward()


        end_of_window = (
            position + 1
            == window_end
        )


        if end_of_window:

            grad_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    [
                        parameter
                        for parameter
                        in model.parameters()
                        if (
                            parameter.requires_grad
                            and parameter.grad
                            is not None
                        )
                    ],

                    max_norm=(
                        MAX_GRAD_NORM
                    ),
                )
            )


            if not torch.isfinite(
                grad_norm
            ):

                raise RuntimeError(
                    "Non-finite gradient norm "
                    f"at epoch={epoch_number}, "
                    f"step="
                    f"{global_optimizer_step + 1}"
                )


            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )


            global_optimizer_step += 1


            # ------------------------------------------------
            # Progress logging
            # ------------------------------------------------

            step_in_epoch = (
                global_optimizer_step
                - (
                    epoch_index
                    * optimizer_steps_per_epoch
                )
            )


            if (
                step_in_epoch == 1
                or step_in_epoch % 10 == 0
                or step_in_epoch
                == optimizer_steps_per_epoch
            ):

                mean_epoch_loss = (
                    epoch_loss_sum
                    / processed_examples
                )


                current_lr = (
                    optimizer
                    .param_groups[0]["lr"]
                )


                print(
                    f"epoch={epoch_number} | "
                    f"step="
                    f"{global_optimizer_step:3d}/"
                    f"{total_optimizer_steps} | "
                    f"examples="
                    f"{processed_examples:3d}/"
                    f"{len(order)} | "
                    f"epoch_loss="
                    f"{mean_epoch_loss:.4f} | "
                    f"lr="
                    f"{current_lr:.7f} | "
                    f"grad_norm="
                    f"{float(grad_norm):.4f}"
                )


        del input_ids
        del attention_mask
        del labels
        del outputs
        del raw_loss
        del scaled_loss


    torch.cuda.synchronize(0)


    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )


    mean_epoch_loss = (
        epoch_loss_sum
        / processed_examples
    )


    print(
        "\nEpoch training complete"
    )

    print(
        "  Mean example loss:",
        f"{mean_epoch_loss:.4f}",
    )

    print(
        "  Seconds:",
        round(
            epoch_seconds,
            1,
        ),
    )


    # --------------------------------------------------------
    # Save adapter checkpoint BEFORE evaluation
    # --------------------------------------------------------

    epoch_adapter_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )


    model.save_pretrained(
        epoch_adapter_dir
    )


    processor.save_pretrained(
        epoch_adapter_dir
    )


    adapter_model_path = (
        epoch_adapter_dir
        / "adapter_model.safetensors"
    )


    assert (
        adapter_model_path.exists()
    )


    print(
        "  Adapter saved:",
        epoch_adapter_dir,
    )

    print(
        "  Adapter model size:",
        round(
            adapter_model_path.stat().st_size
            / 1024**2,
            2,
        ),
        "MB",
    )


    # --------------------------------------------------------
    # Internal-dev evaluation
    # --------------------------------------------------------

    (
        dev_result,
        dev_records,
    ) = evaluate_internal_dev(
        f"INTERNAL DEV — B1 EPOCH {epoch_number}"
    )


    dev_history.append(
        dev_result
    )


    prediction_path = (
        RUN_DIR
        / (
            f"internal_dev_epoch_"
            f"{epoch_number}.csv"
        )
    )


    pd.DataFrame(
        dev_records
    ).to_csv(
        prediction_path,
        index=False,
    )


    training_history.append({
        "epoch": (
            epoch_number
        ),

        "mean_example_loss": (
            mean_epoch_loss
        ),

        "seconds": (
            epoch_seconds
        ),

        "adapter_dir": str(
            epoch_adapter_dir
        ),

        "dev_accuracy": (
            dev_result[
                "accuracy"
            ]
        ),

        "dev_macro_f1": (
            dev_result[
                "macro_f1"
            ]
        ),
    })


# ------------------------------------------------------------
# V. Select best B1 checkpoint using INTERNAL DEV ONLY
# ------------------------------------------------------------

epoch_dev_results = (
    dev_history[1:]
)


best_dev_result = max(
    epoch_dev_results,

    key=lambda result: (
        result[
            "macro_f1"
        ],

        result[
            "accuracy"
        ],
    ),
)


best_epoch = int(
    re.search(
        r"EPOCH\s+(\d+)",
        best_dev_result[
            "tag"
        ],
    ).group(1)
)


best_adapter_dir = (
    RUN_DIR
    / f"epoch_{best_epoch}_adapter"
)


# ------------------------------------------------------------
# W. Save frozen run configuration
# ------------------------------------------------------------

run_config = {
    "run_name": RUN_NAME,

    "experiment_family": (
        "B-series stronger Gemma"
    ),

    "model_id": MODEL_ID,

    "principal_change_vs_A1": (
        "Gemma 2 2B IT -> Gemma 4 E4B IT"
    ),

    "fit": {
        "path": str(
            FIT_PATH
        ),

        "sha256": (
            EXPECTED_FIT_SHA256
        ),

        "rows": 795,

        "synthetic_rows": 0,
    },

    "internal_dev": {
        "path": str(
            DEV_PATH
        ),

        "sha256": (
            EXPECTED_DEV_SHA256
        ),

        "rows": 140,
    },

    "prompt": {
        "name": "BASE",
        "text": PROMPT_BASE,
        "enable_thinking": False,
    },

    "max_length": (
        MAX_LENGTH
    ),

    "quantization": {
        "load_in_4bit": True,
        "type": "nf4",
        "double_quant": True,
        "compute_dtype": (
            "float16"
        ),
    },

    "lora": {
        "r": LORA_R,
        "alpha": (
            LORA_ALPHA
        ),
        "dropout": (
            LORA_DROPOUT
        ),
        "bias": "none",
        "target_regex": (
            TARGET_REGEX
        ),
        "language_modules": 258,
        "trainable_parameters": (
            trainable_params
        ),
    },

    "optimization": {
        "epochs": EPOCHS,
        "physical_batch_size": (
            PHYSICAL_BATCH_SIZE
        ),
        "gradient_accumulation": (
            GRAD_ACCUM_STEPS
        ),
        "effective_batch_size": (
            PHYSICAL_BATCH_SIZE
            * GRAD_ACCUM_STEPS
        ),
        "optimizer": "AdamW",
        "learning_rate": (
            LEARNING_RATE
        ),
        "weight_decay": (
            WEIGHT_DECAY
        ),
        "scheduler": "cosine",
        "warmup_ratio": (
            WARMUP_RATIO
        ),
        "warmup_steps": (
            warmup_steps
        ),
        "max_grad_norm": (
            MAX_GRAD_NORM
        ),
        "grad_scaler": False,
        "seed": SEED,
        "gradient_checkpointing": True,
        "checkpoint_use_reentrant": False,
    },

    "loss": (
        "assistant completion only"
    ),

    "standard_prepare_model_for_kbit_training_used": (
        False
    ),

    "organizer_validation_used": (
        False
    ),
}


(
    RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# X. Save results
# ------------------------------------------------------------

results_payload = {
    "pretrain_internal_dev": (
        pretrain_dev_result
    ),

    "training_history": (
        training_history
    ),

    "internal_dev_history": (
        dev_history
    ),

    "best_epoch": (
        best_epoch
    ),

    "best_adapter_dir": str(
        best_adapter_dir
    ),

    "best_internal_dev": (
        best_dev_result
    ),

    "references": {
        "A1": {
            "accuracy": 0.8214,
            "macro_f1": 0.8249,
        },

        "A3": {
            "accuracy": 0.8571,
            "macro_f1": 0.8580,
        },

        "Gemma4_raw_BASE": {
            "accuracy": 0.5786,
            "macro_f1": 0.5540,
        },
    },

    "organizer_validation_used": (
        False
    ),
}


(
    RUN_DIR
    / "results.json"
).write_text(
    json.dumps(
        results_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Y. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1 — RESULT")
print("=" * 78)


print("\nScientific comparison")

print(
    "  Principal change vs A1:",
    "Gemma 2 2B -> Gemma 4 E4B",
)

print(
    "  Fit rows:",
    795,
)

print(
    "  Synthetic examples:",
    0,
)

print(
    "  Internal dev unchanged:",
    True,
)

print(
    "  Thinking:",
    False,
)

print(
    "  Organizer validation used:",
    False,
)


print("\nReference internal-dev results")

print(
    "  A1 | Accuracy=0.8214 | "
    "Macro-F1=0.8249"
)

print(
    "  A3 | Accuracy=0.8571 | "
    "Macro-F1=0.8580"
)

print(
    "  Raw Gemma4 BASE | "
    "Accuracy=0.5786 | "
    "Macro-F1=0.5540"
)


print("\nB1 checkpoints")


print(
    f"  PRETRAIN | "
    f"Accuracy="
    f"{pretrain_dev_result['accuracy']:.4f} | "
    f"Macro-F1="
    f"{pretrain_dev_result['macro_f1']:.4f}"
)


for item in (
    training_history
):

    print(
        f"  EPOCH {item['epoch']} | "
        f"loss="
        f"{item['mean_example_loss']:.4f} | "
        f"Accuracy="
        f"{item['dev_accuracy']:.4f} | "
        f"Macro-F1="
        f"{item['dev_macro_f1']:.4f} | "
        f"{item['seconds']:.1f}s"
    )


print("\nBest B1 checkpoint by INTERNAL DEV")

print(
    "  Epoch:",
    best_epoch,
)

print(
    "  Adapter:",
    best_adapter_dir,
)

print(
    "  Accuracy:",
    f"{best_dev_result['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{best_dev_result['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    best_dev_result["invalid"],
)


print("\nPer-class F1 — best B1")


for label in LABELS:

    print(
        f"  {label:15} "
        f"{best_dev_result['per_class'][label]['f1']:.4f}"
    )


print("\nB1 minus A1 internal dev")

print(
    "  Accuracy delta:",
    f"{best_dev_result['accuracy'] - 0.8214:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{best_dev_result['macro_f1'] - 0.8249:+.4f}",
)


print("\nB1 minus A3 internal dev")

print(
    "  Accuracy delta:",
    f"{best_dev_result['accuracy'] - 0.8571:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{best_dev_result['macro_f1'] - 0.8580:+.4f}",
)


print("\nArtifacts")

for epoch_number in range(
    1,
    EPOCHS + 1,
):

    print(
        f"  epoch_{epoch_number}:",
        RUN_DIR
        / f"epoch_{epoch_number}_adapter",
    )


print(
    "  Config:",
    RUN_DIR
    / "run_config.json",
)

print(
    "  Results:",
    RUN_DIR
    / "results.json",
)


print("\nGPU")

print(
    "  GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)

if torch.cuda.device_count() > 1:

    print(
        "  GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


print("\nB1 training completed.")
print(
    "Organizer validation was NOT evaluated."
)

In [ ]:
# ============================================================
# B1.1 — FRESH RELOAD + EXTERNAL REASONING CHECK
#
# RUN AFTER RESTARTING THE KAGGLE KERNEL.
#
# Purpose:
#   1. Fresh-load official Gemma 4 E4B in 4-bit
#   2. Evaluate RAW Gemma 4 using B1's exact BASE prompt
#      on frozen external_stress_30_v1
#   3. Fresh-load B1 epoch-2 adapter from disk
#   4. Reproduce saved epoch-2 internal-dev predictions EXACTLY
#   5. Evaluate B1 on frozen external stress
#   6. Compare raw BASE -> B1
#
# NO TRAINING.
# NO organizer validation.
# NO synthetic data used for training.
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel


# ------------------------------------------------------------
# A. Frozen paths / hashes
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

B1_RUN_DIR = (
    WORK_ROOT
    / "b1_gemma4_e4b_real795_base_prompt"
)

ADAPTER_DIR = (
    B1_RUN_DIR
    / "epoch_2_adapter"
)

ADAPTER_MODEL_PATH = (
    ADAPTER_DIR
    / "adapter_model.safetensors"
)

SAVED_DEV_PREDICTIONS = (
    B1_RUN_DIR
    / "internal_dev_epoch_2.csv"
)


DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)


STRESS_PATH = (
    WORK_ROOT
    / "external_stress_30_v1.jsonl"
)

EXPECTED_STRESS_SHA256 = (
    "05624200845a79228ac4e05540b013ea"
    "0836d049802fcd3f9dcc60ada8aecedc"
)


OUT_CSV = (
    B1_RUN_DIR
    / "b1_epoch2_external_stress_predictions.csv"
)

OUT_JSON = (
    B1_RUN_DIR
    / "b1_epoch2_fresh_reload_external_results.json"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


# ------------------------------------------------------------
# B. Exact frozen B1 prompt
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


MAX_NEW_TOKENS = 16


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# D. Verify artifacts
# ------------------------------------------------------------

assert ADAPTER_DIR.exists()
assert ADAPTER_MODEL_PATH.exists()
assert SAVED_DEV_PREDICTIONS.exists()

assert DEV_PATH.exists()
assert STRESS_PATH.exists()


assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

assert (
    sha256_file(STRESS_PATH)
    == EXPECTED_STRESS_SHA256
)


adapter_sha256 = sha256_file(
    ADAPTER_MODEL_PATH
)


dev_rows = read_jsonl(
    DEV_PATH
)

stress_rows = read_jsonl(
    STRESS_PATH
)


assert len(dev_rows) == 140
assert len(stress_rows) == 30


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


assert Counter(
    row["label"]
    for row in stress_rows
) == Counter({
    "SUPPORTS": 11,
    "REFUTES": 10,
    "NOT_ENOUGH_INFO": 9,
})


print("=" * 78)
print("B1.1 — FROZEN ARTIFACTS")
print("=" * 78)

print(
    "Adapter:",
    ADAPTER_DIR,
)

print(
    "Adapter size:",
    round(
        ADAPTER_MODEL_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "Adapter SHA-256:",
    adapter_sha256,
)

print(
    "Internal dev:",
    len(dev_rows),
)

print(
    "External stress:",
    len(stress_rows),
)

print(
    "Organizer validation used:",
    False,
)


# ------------------------------------------------------------
# E. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Clean runtime guard
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


torch.cuda.set_device(0)


starting_gpu_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("B1.1 — CLEAN GPU")
print("=" * 78)


print(
    "GPU0 allocated:",
    f"{starting_gpu_gb:.3f} GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


if starting_gpu_gb > 0.5:

    raise RuntimeError(
        "GPU0 is not clean. Restart the Kaggle "
        "kernel and run B1.1 directly."
    )


# ------------------------------------------------------------
# G. Processor + fresh base
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = processor.tokenizer


quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("B1.1 — FRESH BASE LOAD")
print("=" * 78)


load_start = time.perf_counter()


base_model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


torch.cuda.synchronize(0)


print(
    "Model class:",
    type(base_model).__name__,
)

print(
    "4-bit:",
    getattr(
        base_model,
        "is_loaded_in_4bit",
        False,
    ),
)

print(
    "Load seconds:",
    round(
        time.perf_counter()
        - load_start,
        1,
    ),
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


assert (
    type(base_model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    base_model,
    "is_loaded_in_4bit",
    False,
)


for parameter in base_model.parameters():

    parameter.requires_grad = False


base_model.eval()


# ------------------------------------------------------------
# H. Exact B1 rendering
# ------------------------------------------------------------

def make_messages(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    content = (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": content,
                }
            ],
        }
    ]


# ------------------------------------------------------------
# I. Output parser
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(text)
    )

    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(text)
    )

    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


# ------------------------------------------------------------
# J. Metrics
# ------------------------------------------------------------

def calculate_metrics(
    rows,
    predictions,
):

    gold = [
        row["label"]
        for row in rows
    ]


    metric_predictions = [
        p
        if p is not None
        else "__INVALID__"

        for p in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    return {
        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            np.mean(f1)
        ),

        "invalid": int(
            sum(
                p is None
                for p in predictions
            )
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class_f1": {
            LABELS[i]: float(
                f1[i]
            )

            for i in range(
                len(LABELS)
            )
        },
    }


# ------------------------------------------------------------
# K. Generic deterministic evaluator
# ------------------------------------------------------------

def evaluate(
    model_object,
    rows,
    tag,
):

    model_object.eval()


    predictions = []
    raw_outputs = []


    start_time = time.perf_counter()


    for index, row in enumerate(
        rows,
        1,
    ):

        inputs = (
            processor
            .apply_chat_template(
                make_messages(row),
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )


        inputs = {
            key: (
                value.to("cuda:0")
                if torch.is_tensor(value)
                else value
            )

            for key, value
            in inputs.items()
        }


        input_length = (
            inputs["input_ids"]
            .shape[-1]
        )


        with torch.inference_mode():

            generated = (
                model_object.generate(
                    **inputs,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=(
                        tokenizer.pad_token_id
                    ),
                )
            )


        generated_tokens = generated[
            0,
            input_length:
        ]


        raw = processor.decode(
            generated_tokens,
            skip_special_tokens=False,
        )


        prediction = parse_prediction(
            raw
        )


        predictions.append(
            prediction
        )

        raw_outputs.append(
            raw
        )


        if (
            index % 20 == 0
            or index == len(rows)
        ):

            print(
                f"  {tag}: "
                f"{index}/{len(rows)}"
            )


        del inputs
        del generated
        del generated_tokens

        torch.cuda.empty_cache()


    elapsed = (
        time.perf_counter()
        - start_time
    )


    metrics = calculate_metrics(
        rows,
        predictions,
    )


    return {
        "predictions": predictions,
        "raw_outputs": raw_outputs,
        "metrics": metrics,
        "seconds": float(
            elapsed
        ),
    }


# ------------------------------------------------------------
# L. Raw Gemma 4 + exact B1 BASE prompt
#
# This creates a FAIR prompt-matched raw baseline.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.1 — RAW GEMMA 4 / BASE PROMPT / EXTERNAL STRESS")
print("=" * 78)


raw_external = evaluate(
    base_model,
    stress_rows,
    "RAW",
)


m = raw_external["metrics"]


print("\nRAW BASE result")

print(
    "  Accuracy:",
    f"{m['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{m['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    m["invalid"],
)

print(
    "  Per-class F1:"
)


for label in LABELS:

    print(
        f"    {label:15} "
        f"{m['per_class_f1'][label]:.4f}"
    )


# ------------------------------------------------------------
# M. Attach SAVED B1 epoch-2 adapter
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.1 — FRESH B1 ADAPTER RELOAD")
print("=" * 78)


model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False,
)


model.eval()


trainable_after_reload = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Wrapper class:",
    type(model).__name__,
)

print(
    "Trainable after reload:",
    f"{trainable_after_reload:,}",
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


# In inference reload, nothing should be trainable.
assert trainable_after_reload == 0


# ------------------------------------------------------------
# N. Fresh-reload internal-dev reproduction
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.1 — INTERNAL-DEV REPRODUCTION")
print("=" * 78)


fresh_dev = evaluate(
    model,
    dev_rows,
    "DEV",
)


fresh_dev_metrics = (
    fresh_dev[
        "metrics"
    ]
)


saved_dev_df = pd.read_csv(
    SAVED_DEV_PREDICTIONS
)


saved_prediction_by_id = {
    str(row["id"]): str(
        row["prediction"]
    )

    for row in saved_dev_df.to_dict(
        orient="records"
    )
}


fresh_prediction_by_id = {
    row["id"]: prediction

    for row, prediction
    in zip(
        dev_rows,
        fresh_dev[
            "predictions"
        ],
    )
}


mismatches = []


for row in dev_rows:

    example_id = row["id"]

    saved_prediction = (
        saved_prediction_by_id[
            example_id
        ]
    )

    fresh_prediction = (
        fresh_prediction_by_id[
            example_id
        ]
    )


    if (
        saved_prediction
        != fresh_prediction
    ):

        mismatches.append({
            "id": example_id,

            "saved": (
                saved_prediction
            ),

            "fresh": (
                fresh_prediction
            ),
        })


exact_reproduction = (
    len(mismatches) == 0
)


print(
    "Fresh Accuracy:",
    f"{fresh_dev_metrics['accuracy']:.4f}",
)

print(
    "Fresh Macro-F1:",
    f"{fresh_dev_metrics['macro_f1']:.4f}",
)

print(
    "Saved-vs-fresh mismatches:",
    len(mismatches),
)

print(
    "Exact prediction reproduction:",
    exact_reproduction,
)


assert abs(
    fresh_dev_metrics[
        "accuracy"
    ]
    - 0.8857142857142857
) < 1e-12


assert abs(
    fresh_dev_metrics[
        "macro_f1"
    ]
    - 0.887999
) < 0.001


if not exact_reproduction:

    print(
        "\nFirst mismatches:"
    )

    for item in mismatches[:10]:

        print(
            " ",
            item,
        )


    raise RuntimeError(
        "Fresh B1 adapter did not reproduce "
        "the saved epoch-2 predictions exactly."
    )


# ------------------------------------------------------------
# O. B1 external stress
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.1 — B1 EPOCH 2 / EXTERNAL STRESS")
print("=" * 78)


b1_external = evaluate(
    model,
    stress_rows,
    "B1",
)


b1_m = b1_external[
    "metrics"
]


print("\nB1 external result")

print(
    "  Accuracy:",
    f"{b1_m['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{b1_m['macro_f1']:.4f}",
)

print(
    "  Invalid:",
    b1_m["invalid"],
)


print(
    "  Confusion matrix:"
)

print(
    "                "
    "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
)


for (
    label,
    row_values,
) in zip(
    LABELS,
    b1_m[
        "confusion_matrix"
    ],
):

    print(
        f"  {label:15}",
        row_values,
    )


print(
    "  Per-class F1:"
)


for label in LABELS:

    print(
        f"    {label:15} "
        f"{b1_m['per_class_f1'][label]:.4f}"
    )


# ------------------------------------------------------------
# P. Raw BASE -> B1 paired changes
# ------------------------------------------------------------

fixed = []
broken = []
both_correct = []
both_wrong = []


for index, row in enumerate(
    stress_rows
):

    gold = row["label"]

    raw_pred = (
        raw_external[
            "predictions"
        ][index]
    )

    b1_pred = (
        b1_external[
            "predictions"
        ][index]
    )


    raw_correct = (
        raw_pred == gold
    )

    b1_correct = (
        b1_pred == gold
    )


    if (
        not raw_correct
        and b1_correct
    ):

        fixed.append(
            row["id"]
        )

    elif (
        raw_correct
        and not b1_correct
    ):

        broken.append(
            row["id"]
        )

    elif (
        raw_correct
        and b1_correct
    ):

        both_correct.append(
            row["id"]
        )

    else:

        both_wrong.append(
            row["id"]
        )


# ------------------------------------------------------------
# Q. Critical arithmetic / temporal cases
# ------------------------------------------------------------

critical_ids = [
    "external_stress_v1_01",
    "external_stress_v1_02",
    "external_stress_v1_05",
    "external_stress_v1_06",
    "external_stress_v1_19",
    "external_stress_v1_20",
    "external_stress_v1_21",
    "external_stress_v1_22",
    "external_stress_v1_27",
    "external_stress_v1_28",
]


stress_index = {
    row["id"]: index

    for index, row
    in enumerate(
        stress_rows
    )
}


print("\n" + "=" * 78)
print("B1.1 — CRITICAL NUMERICAL / TEMPORAL CASES")
print("=" * 78)


for example_id in critical_ids:

    index = stress_index[
        example_id
    ]

    row = stress_rows[
        index
    ]


    print(
        f"{example_id} | "
        f"gold={row['label']:16} | "
        f"RAW={str(raw_external['predictions'][index]):16} | "
        f"B1={str(b1_external['predictions'][index]):16}"
    )


# ------------------------------------------------------------
# R. Save full prediction table
# ------------------------------------------------------------

prediction_records = []


for index, row in enumerate(
    stress_rows
):

    prediction_records.append({
        "id": row["id"],

        "gold": row["label"],

        "claim": row["claim"],

        "raw_base_prediction": (
            raw_external[
                "predictions"
            ][index]
        ),

        "b1_prediction": (
            b1_external[
                "predictions"
            ][index]
        ),

        "raw_base_correct": (
            raw_external[
                "predictions"
            ][index]
            == row["label"]
        ),

        "b1_correct": (
            b1_external[
                "predictions"
            ][index]
            == row["label"]
        ),

        "raw_base_output": (
            raw_external[
                "raw_outputs"
            ][index]
        ),

        "b1_output": (
            b1_external[
                "raw_outputs"
            ][index]
        ),
    })


pd.DataFrame(
    prediction_records
).to_csv(
    OUT_CSV,
    index=False,
)


# ------------------------------------------------------------
# S. Save result
# ------------------------------------------------------------

payload = {
    "model_id": MODEL_ID,

    "adapter": {
        "directory": str(
            ADAPTER_DIR
        ),

        "adapter_model_sha256": (
            adapter_sha256
        ),
    },

    "prompt": {
        "name": "BASE",
        "enable_thinking": False,
        "text": PROMPT_BASE,
    },

    "fresh_reload": {
        "internal_dev_exact_prediction_reproduction": (
            exact_reproduction
        ),

        "mismatch_count": (
            len(mismatches)
        ),

        "internal_dev_metrics": (
            fresh_dev_metrics
        ),
    },

    "external_stress": {
        "rows": 30,

        "sha256": (
            EXPECTED_STRESS_SHA256
        ),

        "raw_base_BASE_prompt": (
            raw_external[
                "metrics"
            ]
        ),

        "b1_epoch2_BASE_prompt": (
            b1_external[
                "metrics"
            ]
        ),

        "raw_to_b1": {
            "fixed": fixed,
            "broken": broken,
            "both_correct": (
                both_correct
            ),
            "both_wrong": (
                both_wrong
            ),
        },
    },

    "training_occurred": False,

    "organizer_validation_used": False,
}


OUT_JSON.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# T. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.1 — FINAL FRESH-RELOAD / EXTERNAL RESULT")
print("=" * 78)


print(
    "Adapter SHA-256:",
    adapter_sha256,
)


print("\nFresh reload")

print(
    "  Internal-dev exact reproduction:",
    exact_reproduction,
)

print(
    "  Accuracy:",
    f"{fresh_dev_metrics['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{fresh_dev_metrics['macro_f1']:.4f}",
)


print("\nExternal stress — exact BASE prompt")

print(
    "  RAW BASE | "
    f"Accuracy={raw_external['metrics']['accuracy']:.4f} | "
    f"Macro-F1={raw_external['metrics']['macro_f1']:.4f}"
)

print(
    "  B1       | "
    f"Accuracy={b1_m['accuracy']:.4f} | "
    f"Macro-F1={b1_m['macro_f1']:.4f}"
)


print("\nRaw BASE -> B1")

print(
    "  Fixed:",
    len(fixed),
    fixed,
)

print(
    "  Broken:",
    len(broken),
    broken,
)

print(
    "  Both correct:",
    len(both_correct),
)

print(
    "  Both wrong:",
    len(both_wrong),
)


print("\nExisting 2B external references")

print(
    "  A1 | 17/30 | Macro-F1=0.5260"
)

print(
    "  A3 | 19/30 | Macro-F1=0.6248"
)


print("\nSaved")

print(
    " ",
    OUT_CSV,
)

print(
    " ",
    OUT_JSON,
)


print("\nNO training occurred.")
print("NO organizer validation was evaluated.")

In [ ]:
# ============================================================
# B1.2 — EPOCH 1 FRESH-RELOAD + EXTERNAL ROBUSTNESS CHECK
#
# RUN AFTER RESTARTING THE KAGGLE KERNEL.
#
# Purpose:
#   1. Fresh-load official Gemma 4 E4B
#   2. Fresh-load B1 epoch-1 adapter
#   3. Reproduce epoch-1 internal-dev predictions EXACTLY
#   4. Evaluate epoch 1 on frozen external stress 30
#   5. Compare epoch 1 vs:
#        - raw Gemma 4
#        - B1 epoch 2
#
# NO TRAINING.
# NO organizer validation.
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel


# ------------------------------------------------------------
# A. Paths
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

RUN_DIR = (
    WORK_ROOT
    / "b1_gemma4_e4b_real795_base_prompt"
)

EPOCH1_ADAPTER = (
    RUN_DIR
    / "epoch_1_adapter"
)

EPOCH1_MODEL_FILE = (
    EPOCH1_ADAPTER
    / "adapter_model.safetensors"
)

SAVED_EPOCH1_DEV = (
    RUN_DIR
    / "internal_dev_epoch_1.csv"
)

DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

STRESS_PATH = (
    WORK_ROOT
    / "external_stress_30_v1.jsonl"
)

EPOCH2_RESULT_PATH = (
    RUN_DIR
    / "b1_epoch2_fresh_reload_external_results.json"
)

OUT_CSV = (
    RUN_DIR
    / "b1_epoch1_external_stress_predictions.csv"
)

OUT_JSON = (
    RUN_DIR
    / "b1_epoch1_external_robustness_results.json"
)


EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)

EXPECTED_STRESS_SHA256 = (
    "05624200845a79228ac4e05540b013ea"
    "0836d049802fcd3f9dcc60ada8aecedc"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


# ------------------------------------------------------------
# B. Exact B1 prompt
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


MAX_NEW_TOKENS = 16


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# D. Verify frozen artifacts
# ------------------------------------------------------------

assert EPOCH1_ADAPTER.exists()
assert EPOCH1_MODEL_FILE.exists()
assert SAVED_EPOCH1_DEV.exists()
assert DEV_PATH.exists()
assert STRESS_PATH.exists()


assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

assert (
    sha256_file(STRESS_PATH)
    == EXPECTED_STRESS_SHA256
)


adapter_sha = sha256_file(
    EPOCH1_MODEL_FILE
)


dev_rows = read_jsonl(
    DEV_PATH
)

stress_rows = read_jsonl(
    STRESS_PATH
)


assert len(dev_rows) == 140
assert len(stress_rows) == 30


print("=" * 78)
print("B1.2 — ARTIFACTS")
print("=" * 78)


print(
    "Epoch-1 adapter SHA-256:",
    adapter_sha,
)

print(
    "Internal dev rows:",
    len(dev_rows),
)

print(
    "External stress rows:",
    len(stress_rows),
)

print(
    "Organizer validation used:",
    False,
)


# ------------------------------------------------------------
# E. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Clean runtime guard
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):
    torch.cuda.ipc_collect()


torch.cuda.set_device(0)


start_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("B1.2 — CLEAN GPU")
print("=" * 78)


print(
    "GPU0 allocated:",
    f"{start_gb:.3f} GB",
)

if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


if start_gb > 0.5:

    raise RuntimeError(
        "Restart Kaggle kernel and run B1.2 directly."
    )


# ------------------------------------------------------------
# G. Fresh base
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = processor.tokenizer


quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("B1.2 — FRESH BASE + EPOCH-1 ADAPTER")
print("=" * 78)


base_model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


for parameter in base_model.parameters():
    parameter.requires_grad = False


model = PeftModel.from_pretrained(
    base_model,
    str(EPOCH1_ADAPTER),
    is_trainable=False,
)


model.eval()


assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0


print(
    "Base class:",
    type(base_model).__name__,
)

print(
    "PEFT class:",
    type(model).__name__,
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


# ------------------------------------------------------------
# H. Prompt renderer
# ------------------------------------------------------------

def make_messages(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    text = (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": text,
                }
            ],
        }
    ]


# ------------------------------------------------------------
# I. Parser
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(text)
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(text)
    )


    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


# ------------------------------------------------------------
# J. Evaluator
# ------------------------------------------------------------

def evaluate(rows, tag):

    predictions = []
    raw_outputs = []


    start = time.perf_counter()


    for index, row in enumerate(
        rows,
        1,
    ):

        inputs = (
            processor
            .apply_chat_template(
                make_messages(row),
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=False,
            )
        )


        inputs = {
            key: (
                value.to("cuda:0")
                if torch.is_tensor(value)
                else value
            )

            for key, value
            in inputs.items()
        }


        input_len = (
            inputs["input_ids"]
            .shape[-1]
        )


        with torch.inference_mode():

            generated = model.generate(
                **inputs,
                do_sample=False,
                num_beams=1,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=(
                    tokenizer.pad_token_id
                ),
            )


        new_tokens = generated[
            0,
            input_len:
        ]


        raw = processor.decode(
            new_tokens,
            skip_special_tokens=False,
        )


        pred = parse_prediction(
            raw
        )


        predictions.append(
            pred
        )

        raw_outputs.append(
            raw
        )


        if (
            index % 20 == 0
            or index == len(rows)
        ):

            print(
                f"  {tag}: "
                f"{index}/{len(rows)}"
            )


        del inputs
        del generated
        del new_tokens

        torch.cuda.empty_cache()


    gold = [
        row["label"]
        for row in rows
    ]


    metric_predictions = [
        p
        if p is not None
        else "__INVALID__"

        for p in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    result = {
        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            np.mean(f1)
        ),

        "invalid": int(
            sum(
                p is None
                for p in predictions
            )
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class_f1": {
            LABELS[i]: float(
                f1[i]
            )

            for i in range(
                len(LABELS)
            )
        },

        "seconds": float(
            time.perf_counter()
            - start
        ),
    }


    return (
        result,
        predictions,
        raw_outputs,
    )


# ------------------------------------------------------------
# K. Internal-dev exact reproduction
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.2 — EPOCH-1 INTERNAL-DEV REPRODUCTION")
print("=" * 78)


(
    dev_metrics,
    dev_predictions,
    dev_outputs,
) = evaluate(
    dev_rows,
    "DEV",
)


saved_df = pd.read_csv(
    SAVED_EPOCH1_DEV
)


saved_by_id = {
    str(row["id"]): str(
        row["prediction"]
    )

    for row in saved_df.to_dict(
        orient="records"
    )
}


mismatches = []


for row, prediction in zip(
    dev_rows,
    dev_predictions,
):

    saved_prediction = (
        saved_by_id[
            row["id"]
        ]
    )


    if saved_prediction != prediction:

        mismatches.append({
            "id": row["id"],
            "saved": saved_prediction,
            "fresh": prediction,
        })


exact_reproduction = (
    len(mismatches) == 0
)


print(
    "Accuracy:",
    f"{dev_metrics['accuracy']:.4f}",
)

print(
    "Macro-F1:",
    f"{dev_metrics['macro_f1']:.4f}",
)

print(
    "Prediction mismatches:",
    len(mismatches),
)

print(
    "Exact reproduction:",
    exact_reproduction,
)


assert abs(
    dev_metrics["accuracy"]
    - 0.8642857142857143
) < 1e-12


assert abs(
    dev_metrics["macro_f1"]
    - 0.8673
) < 0.001


if not exact_reproduction:

    raise RuntimeError(
        "Epoch-1 fresh reload did not reproduce "
        "saved internal-dev predictions."
    )


# ------------------------------------------------------------
# L. External stress
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.2 — EPOCH-1 EXTERNAL STRESS")
print("=" * 78)


(
    ext_metrics,
    ext_predictions,
    ext_outputs,
) = evaluate(
    stress_rows,
    "EXT",
)


print(
    "Accuracy:",
    f"{ext_metrics['accuracy']:.4f}",
)

print(
    "Macro-F1:",
    f"{ext_metrics['macro_f1']:.4f}",
)

print(
    "Invalid:",
    ext_metrics["invalid"],
)


print(
    "Confusion matrix:"
)

print(
    "                "
    "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
)


for label, matrix_row in zip(
    LABELS,
    ext_metrics[
        "confusion_matrix"
    ],
):

    print(
        f"  {label:15}",
        matrix_row,
    )


print(
    "Per-class F1:"
)


for label in LABELS:

    print(
        f"  {label:15} "
        f"{ext_metrics['per_class_f1'][label]:.4f}"
    )


# ------------------------------------------------------------
# M. Load epoch-2 result from B1.1
# ------------------------------------------------------------

epoch2_payload = None


if EPOCH2_RESULT_PATH.exists():

    epoch2_payload = json.loads(
        EPOCH2_RESULT_PATH.read_text(
            encoding="utf-8"
        )
    )


    epoch2_metrics = (
        epoch2_payload[
            "external_stress"
        ][
            "b1_epoch2_BASE_prompt"
        ]
    )

else:

    epoch2_metrics = {
        "accuracy": 0.8000,
        "macro_f1": 0.8038,
    }


# ------------------------------------------------------------
# N. Critical numerical/temporal examples
# ------------------------------------------------------------

critical_ids = [
    "external_stress_v1_01",
    "external_stress_v1_02",
    "external_stress_v1_05",
    "external_stress_v1_06",
    "external_stress_v1_19",
    "external_stress_v1_20",
    "external_stress_v1_21",
    "external_stress_v1_22",
    "external_stress_v1_27",
    "external_stress_v1_28",
]


stress_index = {
    row["id"]: i

    for i, row
    in enumerate(stress_rows)
}


print("\n" + "=" * 78)
print("B1.2 — CRITICAL NUMERICAL / TEMPORAL CASES")
print("=" * 78)


for example_id in critical_ids:

    i = stress_index[
        example_id
    ]

    row = stress_rows[i]


    print(
        f"{example_id} | "
        f"gold={row['label']:16} | "
        f"E1={str(ext_predictions[i]):16}"
    )


# ------------------------------------------------------------
# O. Save predictions
# ------------------------------------------------------------

records = []


for i, row in enumerate(
    stress_rows
):

    records.append({
        "id": row["id"],

        "gold": row["label"],

        "claim": row["claim"],

        "epoch1_prediction": (
            ext_predictions[i]
        ),

        "epoch1_correct": (
            ext_predictions[i]
            == row["label"]
        ),

        "epoch1_raw_output": (
            ext_outputs[i]
        ),
    })


pd.DataFrame(
    records
).to_csv(
    OUT_CSV,
    index=False,
)


payload = {
    "model_id": MODEL_ID,

    "adapter": {
        "epoch": 1,

        "directory": str(
            EPOCH1_ADAPTER
        ),

        "sha256": (
            adapter_sha
        ),
    },

    "fresh_reload": {
        "exact_internal_dev_reproduction": (
            exact_reproduction
        ),

        "internal_dev": (
            dev_metrics
        ),
    },

    "external_stress": {
        "epoch1": (
            ext_metrics
        ),

        "references": {
            "raw_gemma4": {
                "accuracy": 0.8667,
                "macro_f1": 0.8681,
            },

            "epoch2": {
                "accuracy": (
                    epoch2_metrics[
                        "accuracy"
                    ]
                ),

                "macro_f1": (
                    epoch2_metrics[
                        "macro_f1"
                    ]
                ),
            },

            "A1": {
                "accuracy": 0.5667,
                "macro_f1": 0.5260,
            },

            "A3": {
                "accuracy": 0.6333,
                "macro_f1": 0.6248,
            },
        },
    },

    "organizer_validation_used": False,

    "training_occurred": False,
}


OUT_JSON.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# P. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.2 — FINAL EPOCH COMPARISON")
print("=" * 78)


print(
    "Epoch 1 internal dev | "
    f"Accuracy={dev_metrics['accuracy']:.4f} | "
    f"Macro-F1={dev_metrics['macro_f1']:.4f}"
)

print(
    "Epoch 2 internal dev | "
    "Accuracy=0.8857 | "
    "Macro-F1=0.8880"
)


print("\nExternal stress")

print(
    "  Raw Gemma 4 | "
    "Accuracy=0.8667 | "
    "Macro-F1=0.8681"
)

print(
    "  B1 Epoch 1  | "
    f"Accuracy={ext_metrics['accuracy']:.4f} | "
    f"Macro-F1={ext_metrics['macro_f1']:.4f}"
)

print(
    "  B1 Epoch 2  | "
    f"Accuracy={epoch2_metrics['accuracy']:.4f} | "
    f"Macro-F1={epoch2_metrics['macro_f1']:.4f}"
)

print(
    "  A3          | "
    "Accuracy=0.6333 | "
    "Macro-F1=0.6248"
)


print("\nFresh epoch-1 reload")

print(
    "  Exact dev reproduction:",
    exact_reproduction,
)

print(
    "  Adapter SHA-256:",
    adapter_sha,
)


print("\nSaved:")

print(
    " ",
    OUT_CSV,
)

print(
    " ",
    OUT_JSON,
)


print("\nNO training occurred.")
print("NO organizer validation was evaluated.")

In [ ]:
# ============================================================
# B1.3 — ORGANIZER VALIDATION EVALUATION
#
# FINAL B1 CHECKPOINT:
#   Gemma 4 E4B IT
#   B1 epoch 2
#   BASE prompt
#   thinking=False
#
# Purpose:
#   Evaluate frozen B1 epoch-2 ONCE on organizer validation.
#
# IMPORTANT:
#   - NO training
#   - NO model selection between epoch 1 / epoch 2 here
#   - Epoch 2 was already selected using internal dev
#   - This organizer validation has been inspected previously
#     during the A-series, so this is an ADAPTIVE benchmark,
#     not a pristine test set.
#
# GPU0 only.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel


# ------------------------------------------------------------
# A. Frozen configuration
# ------------------------------------------------------------

MODEL_ID = "google/gemma-4-E4B-it"

WORK_ROOT = Path("/kaggle/working")

RUN_DIR = (
    WORK_ROOT
    / "b1_gemma4_e4b_real795_base_prompt"
)

ADAPTER_DIR = (
    RUN_DIR
    / "epoch_2_adapter"
)

ADAPTER_MODEL_PATH = (
    ADAPTER_DIR
    / "adapter_model.safetensors"
)

EXPECTED_ADAPTER_SHA256 = (
    "100f2e9640a586632f7450f8bae7a7fb"
    "63136c078185e4f16b665e3d5320213d"
)

EXPECTED_VALIDATION_SHA256 = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]

MAX_NEW_TOKENS = 16


OUT_CSV = (
    RUN_DIR
    / "official_validation_eval_b1_epoch2.csv"
)

OUT_JSON = (
    RUN_DIR
    / "official_validation_eval_b1_epoch2_summary.json"
)


# ------------------------------------------------------------
# B. Exact frozen B1 prompt
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


# ------------------------------------------------------------
# D. Verify B1 adapter
# ------------------------------------------------------------

assert ADAPTER_DIR.exists()
assert ADAPTER_MODEL_PATH.exists()

adapter_sha = sha256_file(
    ADAPTER_MODEL_PATH
)

assert (
    adapter_sha
    == EXPECTED_ADAPTER_SHA256
)


# ------------------------------------------------------------
# E. Find organizer validation by exact SHA
#
# This avoids relying on a guessed filename/path.
# ------------------------------------------------------------

print("=" * 78)
print("B1.3 — LOCATING ORGANIZER VALIDATION")
print("=" * 78)


candidate_files = []


for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:

    if not root.exists():
        continue

    for pattern in [
        "*.jsonl",
        "*.json",
    ]:

        candidate_files.extend(
            root.rglob(pattern)
        )


validation_path = None


for path in candidate_files:

    try:

        if sha256_file(path) == EXPECTED_VALIDATION_SHA256:

            validation_path = path
            break

    except Exception:

        pass


if validation_path is None:

    raise FileNotFoundError(
        "Could not find the organizer validation file "
        "with the frozen expected SHA-256."
    )


print(
    "Validation path:",
    validation_path,
)

print(
    "Validation SHA-256:",
    sha256_file(validation_path),
)


# ------------------------------------------------------------
# F. Read / verify organizer validation
# ------------------------------------------------------------

validation_rows = read_jsonl(
    validation_path
)


assert len(validation_rows) == 300


assert Counter(
    row["label"]
    for row in validation_rows
) == Counter({
    "SUPPORTS": 100,
    "REFUTES": 100,
    "NOT_ENOUGH_INFO": 100,
})


print(
    "Rows:",
    len(validation_rows),
)

print(
    "Labels:",
    Counter(
        row["label"]
        for row in validation_rows
    ),
)

print(
    "Historical status:",
    "ADAPTIVE ORGANIZER VALIDATION",
)


# ------------------------------------------------------------
# G. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# H. Clean runtime guard
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if hasattr(torch.cuda, "ipc_collect"):

    torch.cuda.ipc_collect()


torch.cuda.set_device(0)


start_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("B1.3 — CLEAN GPU")
print("=" * 78)


print(
    "GPU0 allocated:",
    f"{start_gb:.3f} GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


if start_gb > 0.5:

    raise RuntimeError(
        "GPU0 is not clean. Restart the Kaggle kernel "
        "and run B1.3 directly."
    )


# ------------------------------------------------------------
# I. Fresh processor + base model
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = processor.tokenizer


quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("\n" + "=" * 78)
print("B1.3 — FRESH B1 EPOCH-2 RELOAD")
print("=" * 78)


load_start = time.perf_counter()


base_model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


for parameter in base_model.parameters():

    parameter.requires_grad = False


model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False,
)


model.eval()


torch.cuda.synchronize(0)


print(
    "Base class:",
    type(base_model).__name__,
)

print(
    "PEFT class:",
    type(model).__name__,
)

print(
    "Adapter SHA-256:",
    adapter_sha,
)

print(
    "Load seconds:",
    round(
        time.perf_counter()
        - load_start,
        1,
    ),
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ),
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0


# ------------------------------------------------------------
# J. Prompt rendering
# ------------------------------------------------------------

def make_messages(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    text = (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": text,
                }
            ],
        }
    ]


# ------------------------------------------------------------
# K. Parser
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(text)
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(text)
    )


    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


# ------------------------------------------------------------
# L. Evaluate organizer validation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.3 — ORGANIZER VALIDATION")
print("=" * 78)


predictions = []
raw_outputs = []


evaluation_start = time.perf_counter()


for index, row in enumerate(
    validation_rows,
    1,
):

    inputs = (
        processor
        .apply_chat_template(
            make_messages(row),
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )


    inputs = {
        key: (
            value.to("cuda:0")
            if torch.is_tensor(value)
            else value
        )

        for key, value
        in inputs.items()
    }


    input_len = (
        inputs["input_ids"]
        .shape[-1]
    )


    with torch.inference_mode():

        generated = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=(
                tokenizer.pad_token_id
            ),
        )


    output_tokens = generated[
        0,
        input_len:
    ]


    raw = processor.decode(
        output_tokens,
        skip_special_tokens=False,
    )


    prediction = parse_prediction(
        raw
    )


    predictions.append(
        prediction
    )

    raw_outputs.append(
        raw
    )


    if (
        index % 25 == 0
        or index == len(validation_rows)
    ):

        print(
            f"  evaluated "
            f"{index}/{len(validation_rows)}"
        )


    del inputs
    del generated
    del output_tokens

    torch.cuda.empty_cache()


elapsed = (
    time.perf_counter()
    - evaluation_start
)


# ------------------------------------------------------------
# M. Metrics
# ------------------------------------------------------------

gold = [
    row["label"]
    for row in validation_rows
]


metric_predictions = [
    prediction
    if prediction is not None
    else "__INVALID__"

    for prediction in predictions
]


accuracy = accuracy_score(
    gold,
    metric_predictions,
)


(
    precision,
    recall,
    f1,
    support,
) = precision_recall_fscore_support(
    gold,
    metric_predictions,
    labels=LABELS,
    zero_division=0,
)


macro_f1 = float(
    np.mean(f1)
)


matrix = confusion_matrix(
    gold,
    metric_predictions,
    labels=LABELS,
)


invalid = sum(
    prediction is None
    for prediction in predictions
)


per_class_f1 = {
    LABELS[i]: float(
        f1[i]
    )

    for i in range(
        len(LABELS)
    )
}


print("\nResult")

print(
    "  Accuracy:",
    f"{accuracy:.4f}",
)

print(
    "  Macro-F1:",
    f"{macro_f1:.4f}",
)

print(
    "  Invalid:",
    invalid,
)


print(
    "  Confusion matrix:"
)

print(
    "                "
    "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
)


for (
    label,
    matrix_row,
) in zip(
    LABELS,
    matrix.tolist(),
):

    print(
        f"  {label:15}",
        matrix_row,
    )


print(
    "  Per-class F1:"
)


for label in LABELS:

    print(
        f"    {label:15} "
        f"{per_class_f1[label]:.4f}"
    )


print(
    "  Seconds:",
    round(
        elapsed,
        1,
    ),
)


# ------------------------------------------------------------
# N. Compare against frozen A-series validation results
# ------------------------------------------------------------

A1_ACC = 0.8200
A1_F1 = 0.8207

A2_ACC = 0.8033
A2_F1 = 0.8051

A3_ACC = 0.8067
A3_F1 = 0.8077


print("\n" + "=" * 78)
print("B1.3 — COMPARISON WITH A-SERIES")
print("=" * 78)


print(
    "A1 | Accuracy=0.8200 | Macro-F1=0.8207"
)

print(
    "A2 | Accuracy=0.8033 | Macro-F1=0.8051"
)

print(
    "A3 | Accuracy=0.8067 | Macro-F1=0.8077"
)

print(
    "B1 | "
    f"Accuracy={accuracy:.4f} | "
    f"Macro-F1={macro_f1:.4f}"
)


print("\nB1 minus A1")

print(
    "  Accuracy:",
    f"{accuracy - A1_ACC:+.4f}",
)

print(
    "  Macro-F1:",
    f"{macro_f1 - A1_F1:+.4f}",
)


print("\nB1 minus A3")

print(
    "  Accuracy:",
    f"{accuracy - A3_ACC:+.4f}",
)

print(
    "  Macro-F1:",
    f"{macro_f1 - A3_F1:+.4f}",
)


# ------------------------------------------------------------
# O. Save predictions
# ------------------------------------------------------------

records = []


for row, prediction, raw in zip(
    validation_rows,
    predictions,
    raw_outputs,
):

    records.append({
        "id": row["id"],
        "gold": row["label"],
        "prediction": prediction,
        "correct": (
            prediction
            == row["label"]
        ),
        "raw_output": raw,
    })


pd.DataFrame(
    records
).to_csv(
    OUT_CSV,
    index=False,
)


# ------------------------------------------------------------
# P. Save summary
# ------------------------------------------------------------

summary = {
    "experiment": (
        "B1 Gemma 4 E4B epoch 2"
    ),

    "model_id": MODEL_ID,

    "adapter": {
        "directory": str(
            ADAPTER_DIR
        ),

        "sha256": adapter_sha,
    },

    "prompt": {
        "name": "BASE",
        "thinking": False,
    },

    "organizer_validation": {
        "path": str(
            validation_path
        ),

        "sha256": (
            EXPECTED_VALIDATION_SHA256
        ),

        "rows": 300,

        "historical_status": (
            "adaptive; previously inspected during A-series"
        ),

        "accuracy": float(
            accuracy
        ),

        "macro_f1": (
            macro_f1
        ),

        "invalid": int(
            invalid
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class_f1": (
            per_class_f1
        ),
    },

    "references": {
        "A1": {
            "accuracy": A1_ACC,
            "macro_f1": A1_F1,
        },

        "A2": {
            "accuracy": A2_ACC,
            "macro_f1": A2_F1,
        },

        "A3": {
            "accuracy": A3_ACC,
            "macro_f1": A3_F1,
        },

        "B1_internal_dev": {
            "accuracy": 0.8857,
            "macro_f1": 0.8880,
        },

        "B1_external_stress": {
            "accuracy": 0.8000,
            "macro_f1": 0.8038,
        },
    },

    "training_occurred": False,
}


OUT_JSON.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Q. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B1.3 — FINAL ORGANIZER-VALIDATION RESULT")
print("=" * 78)


print(
    "Checkpoint:",
    "B1 epoch 2",
)

print(
    "Adapter SHA-256:",
    adapter_sha,
)


print("\nB1")

print(
    "  Accuracy:",
    f"{accuracy:.4f}",
)

print(
    "  Macro-F1:",
    f"{macro_f1:.4f}",
)

print(
    "  Invalid:",
    invalid,
)


print("\nPer-class F1")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{per_class_f1[label]:.4f}"
    )


print("\nDeltas")

print(
    "  vs A1 Macro-F1:",
    f"{macro_f1 - A1_F1:+.4f}",
)

print(
    "  vs A3 Macro-F1:",
    f"{macro_f1 - A3_F1:+.4f}",
)


print("\nSaved")

print(
    " ",
    OUT_CSV,
)

print(
    " ",
    OUT_JSON,
)


print("\nHistorical note:")
print(
    "  Organizer validation is adaptive, "
    "not a pristine final test."
)

print("\nNO training occurred.")

In [ ]:
# ============================================================
# B2 — GEMMA 4 E4B + FROZEN SYNTHETIC AUGMENTATION
#
# CONTROLLED AUGMENTATION EXPERIMENT
#
# B1:
#   795 real fit
#
# B2:
#   795 real fit
# + 180 frozen synthetic
# = 975 training examples
#
# Internal dev:
#   same untouched 140 REAL examples
#
# The ONLY primary experimental change vs B1 is:
#   + frozen A2 synthetic augmentation
#
# NO organizer validation.
# NO external-stress model selection.
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import os
import random
import re
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ------------------------------------------------------------
# A. Experiment configuration
# ------------------------------------------------------------

RUN_NAME = (
    "b2_gemma4_e4b_real795_plus_synth180"
)

WORK_ROOT = Path(
    "/kaggle/working"
)

RUN_DIR = (
    WORK_ROOT
    / RUN_NAME
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_ID = (
    "google/gemma-4-E4B-it"
)


# Frozen augmented training set created during A2.
TRAIN_PATH = (
    WORK_ROOT
    / "train_fit_v2_plus_a2_v1.jsonl"
)

EXPECTED_TRAIN_SHA256 = (
    "6dae1152d9fc310f564ee2841b44b39a"
    "ee1a3dbee559bb040a3cfc330806602f"
)


DEV_PATH = (
    WORK_ROOT
    / "internal_dev_v2.jsonl"
)

EXPECTED_DEV_SHA256 = (
    "aaca3e5e0aa0ba128c91091c7c0499c"
    "c45e4d69c850a98a11c7d5a610aa9a433"
)


# Also verify original frozen synthetic file.
SYNTH_PATH = (
    WORK_ROOT
    / "synthetic_a2_v1_train.jsonl"
)

EXPECTED_SYNTH_SHA256 = (
    "9b0aad0c39b119e47a91e8c901c5372e"
    "0b5e70263606988cfb6f6eee6ba75f52"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


SEED = 42

MAX_LENGTH = 256

EPOCHS = 2

PHYSICAL_BATCH_SIZE = 1

GRAD_ACCUM_STEPS = 16

LEARNING_RATE = 2e-4

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.05

MAX_GRAD_NORM = 1.0

MAX_NEW_TOKENS = 16


LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


# ------------------------------------------------------------
# B. Exact B1/B2 BASE prompt
# ------------------------------------------------------------

PROMPT_BASE = (
    "Classify the claim using only the supplied evidence.\n\n"

    "SUPPORTS: the evidence establishes the claim.\n"

    "REFUTES: the evidence contradicts the claim.\n"

    "NOT_ENOUGH_INFO: the evidence neither establishes nor "
    "contradicts the specific claim.\n\n"

    "End your response exactly as:\n"
    "FINAL: SUPPORTS\n"
    "or\n"
    "FINAL: REFUTES\n"
    "or\n"
    "FINAL: NOT_ENOUGH_INFO"
)


# ------------------------------------------------------------
# C. Helpers
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line_number, line in enumerate(
            f,
            1,
        ):

            if not line.strip():
                continue

            try:

                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{path.name} line "
                    f"{line_number}: {exc}"
                )

    return rows


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ------------------------------------------------------------
# D. Verify frozen datasets
# ------------------------------------------------------------

assert TRAIN_PATH.exists()
assert DEV_PATH.exists()
assert SYNTH_PATH.exists()


assert (
    sha256_file(TRAIN_PATH)
    == EXPECTED_TRAIN_SHA256
)

assert (
    sha256_file(DEV_PATH)
    == EXPECTED_DEV_SHA256
)

assert (
    sha256_file(SYNTH_PATH)
    == EXPECTED_SYNTH_SHA256
)


train_rows = read_jsonl(
    TRAIN_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

synth_rows = read_jsonl(
    SYNTH_PATH
)


assert len(train_rows) == 975
assert len(dev_rows) == 140
assert len(synth_rows) == 180


assert Counter(
    row["label"]
    for row in train_rows
) == Counter({
    "SUPPORTS": 355,
    "REFUTES": 326,
    "NOT_ENOUGH_INFO": 294,
})


assert Counter(
    row["label"]
    for row in dev_rows
) == Counter({
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
})


assert Counter(
    row["label"]
    for row in synth_rows
) == Counter({
    "SUPPORTS": 60,
    "REFUTES": 60,
    "NOT_ENOUGH_INFO": 60,
})


print("=" * 78)
print("B2 — EXPERIMENT DEFINITION")
print("=" * 78)


print(
    "Model:",
    MODEL_ID,
)

print(
    "Training rows:",
    len(train_rows),
)

print(
    "  Real fit:",
    795,
)

print(
    "  Frozen synthetic:",
    len(synth_rows),
)

print(
    "Internal dev:",
    len(dev_rows),
)

print(
    "Organizer validation used:",
    False,
)

print(
    "Thinking:",
    False,
)

print(
    "Primary change vs B1:",
    "+180 frozen synthetic examples",
)


# ------------------------------------------------------------
# E. Authentication
# ------------------------------------------------------------

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ------------------------------------------------------------
# F. Clean runtime guard
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


torch.cuda.set_device(0)


start_gpu_gb = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print("\n" + "=" * 78)
print("B2 — CLEAN GPU STATE")
print("=" * 78)


print(
    "GPU0 allocated:",
    f"{start_gpu_gb:.3f} GB",
)


if torch.cuda.device_count() > 1:

    print(
        "GPU1 allocated:",
        f"{torch.cuda.memory_allocated(1) / 1024**3:.3f} GB",
    )


if start_gpu_gb > 0.5:

    raise RuntimeError(
        "GPU0 is not clean. Restart the Kaggle "
        "kernel and run B2 directly."
    )


# ------------------------------------------------------------
# G. Processor + fresh Gemma 4
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

tokenizer = (
    processor.tokenizer
)


quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


print("\n" + "=" * 78)
print("B2 — FRESH MODEL LOAD")
print("=" * 78)


load_start = (
    time.perf_counter()
)


model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
        quantization_config=(
            quant_config
        ),
        device_map={"": 0},
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)


torch.cuda.synchronize(0)


print(
    "Model class:",
    type(model).__name__,
)

print(
    "4-bit:",
    getattr(
        model,
        "is_loaded_in_4bit",
        False,
    ),
)

print(
    "Load seconds:",
    round(
        time.perf_counter()
        - load_start,
        1,
    ),
)

print(
    "GPU0 allocated:",
    f"{torch.cuda.memory_allocated(0) / 1024**3:.3f} GB",
)


assert (
    type(model).__name__
    == "Gemma4ForConditionalGeneration"
)

assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
)


# ------------------------------------------------------------
# H. Freeze entire base WITHOUT dtype conversion
# ------------------------------------------------------------

for parameter in model.parameters():

    parameter.requires_grad = False


assert sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
) == 0


# ------------------------------------------------------------
# I. Language-only LoRA
# ------------------------------------------------------------

TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\.(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\.(?:gate_proj|up_proj|down_proj)"
    r")"
)


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_REGEX,
)


set_seed(SEED)


model = get_peft_model(
    model,
    lora_config,
)


# ------------------------------------------------------------
# J. LoRA verification
# ------------------------------------------------------------

lora_module_names = []


for name, module in (
    model.named_modules()
):

    if (
        hasattr(
            module,
            "lora_A",
        )
        and len(
            module.lora_A
        ) > 0
    ):

        lora_module_names.append(
            name
        )


language_lora = [
    name
    for name in lora_module_names
    if "language_model.layers"
    in name
]

vision_lora = [
    name
    for name in lora_module_names
    if "vision" in name.lower()
]

audio_lora = [
    name
    for name in lora_module_names
    if "audio" in name.lower()
]


assert len(language_lora) == 258
assert len(vision_lora) == 0
assert len(audio_lora) == 0


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


assert (
    trainable_params
    == 17_440_768
)


print(
    "Language LoRA modules:",
    len(language_lora),
)

print(
    "Vision LoRA modules:",
    len(vision_lora),
)

print(
    "Audio LoRA modules:",
    len(audio_lora),
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)


# ------------------------------------------------------------
# K. Ensure giant embeddings stay frozen FP16
# ------------------------------------------------------------

critical_embeddings = []


for name, parameter in (
    model.named_parameters()
):

    if (
        "embed_tokens_per_layer.weight"
        in name
        or name.endswith(
            "language_model.embed_tokens.weight"
        )
    ):

        critical_embeddings.append({
            "name": name,
            "dtype": str(
                parameter.dtype
            ),
            "requires_grad": bool(
                parameter.requires_grad
            ),
        })


assert len(
    critical_embeddings
) >= 2


for item in critical_embeddings:

    assert (
        item["dtype"]
        == "torch.float16"
    )

    assert (
        item["requires_grad"]
        is False
    )


print(
    "Critical FP16 embeddings frozen:",
    True,
)


# ------------------------------------------------------------
# L. Gradient checkpointing
# ------------------------------------------------------------

model.config.use_cache = False


model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    }
)


# ------------------------------------------------------------
# M. Training-message helpers
# ------------------------------------------------------------

def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"
        for i, passage in enumerate(
            row["evidence"],
            1,
        )
    )


    return (
        PROMPT_BASE
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        }
    ]


def full_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        },

        {
            "role": "assistant",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "FINAL: "
                        + row["label"]
                    ),
                }
            ],
        },
    ]


def get_prompt_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            user_messages(row),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


def get_full_ids(row):

    rendered = (
        processor
        .apply_chat_template(
            full_messages(row),
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    )


    return tokenizer(
        rendered,
        add_special_tokens=False,
    )["input_ids"]


# ------------------------------------------------------------
# N. Pre-tokenize all 975 training examples
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B2 — TRAINING TOKENIZATION")
print("=" * 78)


train_examples = []


for row in train_rows:

    prompt_ids = get_prompt_ids(
        row
    )

    full_ids = get_full_ids(
        row
    )


    assert (
        full_ids[
            :len(prompt_ids)
        ]
        == prompt_ids
    )


    if len(full_ids) > MAX_LENGTH:

        raise RuntimeError(
            f"Sequence exceeds max_length: "
            f"id={row['id']} "
            f"length={len(full_ids)}"
        )


    labels = (
        [-100] * len(
            prompt_ids
        )
        + full_ids[
            len(prompt_ids):
        ]
    )


    train_examples.append({
        "id": row["id"],

        "gold": row["label"],

        "input_ids": (
            full_ids
        ),

        "labels": labels,
    })


max_train_length = max(
    len(
        example[
            "input_ids"
        ]
    )
    for example in train_examples
)


print(
    "Training examples:",
    len(train_examples),
)

print(
    "Longest sequence:",
    max_train_length,
)

print(
    "max_length:",
    MAX_LENGTH,
)

print(
    "Margin:",
    MAX_LENGTH
    - max_train_length,
)


# ------------------------------------------------------------
# O. Deterministic dev evaluator
# ------------------------------------------------------------

FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.IGNORECASE,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.IGNORECASE,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(text)
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    fallback = list(
        LABEL_RE.finditer(text)
    )


    if fallback:

        return (
            fallback[-1]
            .group(1)
            .upper()
        )


    return None


def evaluate_internal_dev(tag):

    model.gradient_checkpointing_disable()

    model.config.use_cache = True

    model.eval()


    predictions = []
    records = []


    start = time.perf_counter()


    try:

        for index, row in enumerate(
            dev_rows,
            1,
        ):

            inputs = (
                processor
                .apply_chat_template(
                    user_messages(row),
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            )


            inputs = {
                key: (
                    value.to("cuda:0")
                    if torch.is_tensor(
                        value
                    )
                    else value
                )

                for key, value
                in inputs.items()
            }


            input_len = (
                inputs["input_ids"]
                .shape[-1]
            )


            with torch.inference_mode():

                generated = (
                    model.generate(
                        **inputs,
                        do_sample=False,
                        num_beams=1,
                        max_new_tokens=(
                            MAX_NEW_TOKENS
                        ),
                        pad_token_id=(
                            tokenizer.pad_token_id
                        ),
                    )
                )


            output_tokens = generated[
                0,
                input_len:
            ]


            raw = processor.decode(
                output_tokens,
                skip_special_tokens=False,
            )


            prediction = (
                parse_prediction(
                    raw
                )
            )


            predictions.append(
                prediction
            )


            records.append({
                "id": row["id"],
                "gold": row["label"],
                "prediction": (
                    prediction
                ),
                "raw_output": raw,
            })


            del inputs
            del generated
            del output_tokens

            torch.cuda.empty_cache()


    finally:

        model.config.use_cache = False

        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False,
            }
        )


    gold = [
        row["label"]
        for row in dev_rows
    ]


    metric_predictions = [
        prediction
        if prediction is not None
        else "__INVALID__"

        for prediction in predictions
    ]


    accuracy = accuracy_score(
        gold,
        metric_predictions,
    )


    (
        precision,
        recall,
        f1,
        support,
    ) = precision_recall_fscore_support(
        gold,
        metric_predictions,
        labels=LABELS,
        zero_division=0,
    )


    macro_f1 = float(
        np.mean(f1)
    )


    matrix = confusion_matrix(
        gold,
        metric_predictions,
        labels=LABELS,
    )


    invalid = sum(
        prediction is None
        for prediction
        in predictions
    )


    result = {
        "tag": tag,

        "accuracy": float(
            accuracy
        ),

        "macro_f1": (
            macro_f1
        ),

        "invalid": int(
            invalid
        ),

        "confusion_matrix": (
            matrix.tolist()
        ),

        "per_class": {
            label: {
                "f1": float(
                    f1[i]
                ),

                "precision": float(
                    precision[i]
                ),

                "recall": float(
                    recall[i]
                ),

                "support": int(
                    support[i]
                ),
            }

            for i, label in enumerate(
                LABELS
            )
        },

        "seconds": float(
            time.perf_counter()
            - start
        ),
    }


    print("\n" + "-" * 78)
    print(tag)
    print("-" * 78)


    print(
        "Accuracy:",
        f"{accuracy:.4f}",
    )

    print(
        "Macro-F1:",
        f"{macro_f1:.4f}",
    )

    print(
        "Invalid:",
        invalid,
    )


    print(
        "Confusion matrix:"
    )

    print(
        "                "
        "SUPPORTS  REFUTES  NOT_ENOUGH_INFO"
    )


    for label, row_values in zip(
        LABELS,
        matrix.tolist(),
    ):

        print(
            f"{label:15}",
            row_values,
        )


    print(
        "Per-class F1:"
    )


    for label in LABELS:

        print(
            f"  {label:15} "
            f"{result['per_class'][label]['f1']:.4f}"
        )


    return (
        result,
        records,
    )


# ------------------------------------------------------------
# P. Pretrain dev
#
# Should reproduce raw BASE behavior.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B2 — PRETRAIN INTERNAL DEV")
print("=" * 78)


(
    pretrain_dev,
    pretrain_records,
) = evaluate_internal_dev(
    "INTERNAL DEV — B2 PRETRAIN"
)


# ------------------------------------------------------------
# Q. Optimizer / scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        p
        for p in model.parameters()
        if p.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


steps_per_epoch = math.ceil(
    len(train_examples)
    / GRAD_ACCUM_STEPS
)


total_steps = (
    steps_per_epoch
    * EPOCHS
)


warmup_steps = int(
    round(
        total_steps
        * WARMUP_RATIO
    )
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=(
            warmup_steps
        ),
        num_training_steps=(
            total_steps
        ),
    )
)


print("\n" + "=" * 78)
print("B2 — OPTIMIZATION")
print("=" * 78)


print(
    "Physical batch:",
    PHYSICAL_BATCH_SIZE,
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS,
)

print(
    "Effective batch:",
    PHYSICAL_BATCH_SIZE
    * GRAD_ACCUM_STEPS,
)

print(
    "Steps / epoch:",
    steps_per_epoch,
)

print(
    "Total steps:",
    total_steps,
)

print(
    "Warmup steps:",
    warmup_steps,
)

print(
    "Learning rate:",
    LEARNING_RATE,
)


# ------------------------------------------------------------
# R. Deterministic epoch orders
# ------------------------------------------------------------

epoch_orders = []


for epoch_index in range(
    EPOCHS
):

    rng = random.Random(
        SEED
        + epoch_index
    )


    indices = list(
        range(
            len(
                train_examples
            )
        )
    )


    rng.shuffle(indices)


    epoch_orders.append(
        indices
    )


# ------------------------------------------------------------
# S. Tensor builder
# ------------------------------------------------------------

def make_training_tensors(
    example,
):

    input_ids = torch.tensor(
        [
            example["input_ids"]
        ],
        dtype=torch.long,
        device="cuda:0",
    )


    attention_mask = (
        torch.ones_like(
            input_ids
        )
    )


    labels = torch.tensor(
        [
            example["labels"]
        ],
        dtype=torch.long,
        device="cuda:0",
    )


    return (
        input_ids,
        attention_mask,
        labels,
    )


# ------------------------------------------------------------
# T. Training
# ------------------------------------------------------------

training_history = []

dev_history = [
    pretrain_dev
]

global_step = 0


for epoch_index in range(
    EPOCHS
):

    epoch_number = (
        epoch_index + 1
    )


    print("\n" + "=" * 78)
    print(
        f"B2 TRAINING EPOCH "
        f"{epoch_number}/{EPOCHS}"
    )
    print("=" * 78)


    model.train()

    model.config.use_cache = False


    optimizer.zero_grad(
        set_to_none=True
    )


    epoch_start = (
        time.perf_counter()
    )


    loss_sum = 0.0

    examples_seen = 0


    order = epoch_orders[
        epoch_index
    ]


    for position, example_index in enumerate(
        order
    ):

        example = (
            train_examples[
                example_index
            ]
        )


        window_start = (
            position
            // GRAD_ACCUM_STEPS
        ) * GRAD_ACCUM_STEPS


        window_end = min(
            window_start
            + GRAD_ACCUM_STEPS,

            len(order),
        )


        actual_window_size = (
            window_end
            - window_start
        )


        (
            input_ids,
            attention_mask,
            labels,
        ) = make_training_tensors(
            example
        )


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )


        raw_loss = outputs.loss


        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                "Non-finite loss at "
                f"epoch={epoch_number}, "
                f"id={example['id']}"
            )


        loss_sum += float(
            raw_loss
            .detach()
            .cpu()
        )


        examples_seen += 1


        scaled_loss = (
            raw_loss
            / actual_window_size
        )


        scaled_loss.backward()


        end_of_window = (
            position + 1
            == window_end
        )


        if end_of_window:

            grad_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    [
                        p
                        for p
                        in model.parameters()
                        if (
                            p.requires_grad
                            and p.grad
                            is not None
                        )
                    ],

                    MAX_GRAD_NORM,
                )
            )


            if not torch.isfinite(
                grad_norm
            ):

                raise RuntimeError(
                    "Non-finite gradient norm."
                )


            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )


            global_step += 1


            step_in_epoch = (
                global_step
                - epoch_index
                * steps_per_epoch
            )


            if (
                step_in_epoch == 1
                or step_in_epoch % 10 == 0
                or step_in_epoch
                == steps_per_epoch
            ):

                print(
                    f"epoch={epoch_number} | "
                    f"step={global_step:3d}/"
                    f"{total_steps} | "
                    f"examples="
                    f"{examples_seen:3d}/"
                    f"{len(order)} | "
                    f"epoch_loss="
                    f"{loss_sum / examples_seen:.4f} | "
                    f"lr="
                    f"{optimizer.param_groups[0]['lr']:.7f} | "
                    f"grad_norm="
                    f"{float(grad_norm):.4f}"
                )


        del input_ids
        del attention_mask
        del labels
        del outputs
        del raw_loss
        del scaled_loss


    torch.cuda.synchronize(0)


    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )


    mean_loss = (
        loss_sum
        / examples_seen
    )


    print(
        "\nEpoch training complete"
    )

    print(
        "  Mean example loss:",
        f"{mean_loss:.4f}",
    )

    print(
        "  Seconds:",
        round(
            epoch_seconds,
            1,
        ),
    )


    # --------------------------------------------------------
    # Save checkpoint before evaluation
    # --------------------------------------------------------

    adapter_dir = (
        RUN_DIR
        / f"epoch_{epoch_number}_adapter"
    )


    model.save_pretrained(
        adapter_dir
    )

    processor.save_pretrained(
        adapter_dir
    )


    adapter_file = (
        adapter_dir
        / "adapter_model.safetensors"
    )


    assert adapter_file.exists()


    print(
        "  Adapter:",
        adapter_dir,
    )

    print(
        "  Adapter size:",
        round(
            adapter_file.stat().st_size
            / 1024**2,
            2,
        ),
        "MB",
    )


    # --------------------------------------------------------
    # Internal dev only
    # --------------------------------------------------------

    (
        dev_result,
        dev_records,
    ) = evaluate_internal_dev(
        f"INTERNAL DEV — B2 EPOCH {epoch_number}"
    )


    dev_history.append(
        dev_result
    )


    pd.DataFrame(
        dev_records
    ).to_csv(
        RUN_DIR
        / (
            f"internal_dev_epoch_"
            f"{epoch_number}.csv"
        ),
        index=False,
    )


    training_history.append({
        "epoch": epoch_number,

        "mean_example_loss": (
            mean_loss
        ),

        "seconds": (
            epoch_seconds
        ),

        "adapter_dir": str(
            adapter_dir
        ),

        "adapter_sha256": (
            sha256_file(
                adapter_file
            )
        ),

        "dev_accuracy": (
            dev_result[
                "accuracy"
            ]
        ),

        "dev_macro_f1": (
            dev_result[
                "macro_f1"
            ]
        ),
    })


# ------------------------------------------------------------
# U. Select epoch ONLY by internal dev
# ------------------------------------------------------------

best_result = max(
    dev_history[1:],

    key=lambda result: (
        result[
            "macro_f1"
        ],

        result[
            "accuracy"
        ],
    ),
)


best_epoch = int(
    re.search(
        r"EPOCH\s+(\d+)",
        best_result[
            "tag"
        ],
    ).group(1)
)


best_adapter_dir = (
    RUN_DIR
    / f"epoch_{best_epoch}_adapter"
)


# ------------------------------------------------------------
# V. Save run configuration
# ------------------------------------------------------------

run_config = {
    "experiment": "B2",

    "model_id": MODEL_ID,

    "principal_change_vs_B1": (
        "+180 frozen synthetic examples"
    ),

    "training_data": {
        "rows": 975,

        "real_fit_rows": 795,

        "synthetic_rows": 180,

        "augmented_sha256": (
            EXPECTED_TRAIN_SHA256
        ),

        "synthetic_sha256": (
            EXPECTED_SYNTH_SHA256
        ),
    },

    "internal_dev": {
        "rows": 140,

        "sha256": (
            EXPECTED_DEV_SHA256
        ),
    },

    "prompt": {
        "name": "BASE",

        "enable_thinking": False,

        "text": PROMPT_BASE,
    },

    "max_length": MAX_LENGTH,

    "lora": {
        "r": LORA_R,

        "alpha": LORA_ALPHA,

        "dropout": (
            LORA_DROPOUT
        ),

        "language_modules": 258,

        "trainable_parameters": (
            trainable_params
        ),
    },

    "optimization": {
        "epochs": EPOCHS,

        "physical_batch_size": (
            PHYSICAL_BATCH_SIZE
        ),

        "gradient_accumulation": (
            GRAD_ACCUM_STEPS
        ),

        "effective_batch_size": (
            PHYSICAL_BATCH_SIZE
            * GRAD_ACCUM_STEPS
        ),

        "learning_rate": (
            LEARNING_RATE
        ),

        "weight_decay": (
            WEIGHT_DECAY
        ),

        "warmup_ratio": (
            WARMUP_RATIO
        ),

        "warmup_steps": (
            warmup_steps
        ),

        "scheduler": "cosine",

        "seed": SEED,

        "grad_scaler": False,
    },

    "epoch_selection": (
        "internal dev only"
    ),

    "organizer_validation_used": False,
}


(
    RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# W. Save results
# ------------------------------------------------------------

results_payload = {
    "pretrain_internal_dev": (
        pretrain_dev
    ),

    "training_history": (
        training_history
    ),

    "internal_dev_history": (
        dev_history
    ),

    "best_epoch": (
        best_epoch
    ),

    "best_adapter_dir": str(
        best_adapter_dir
    ),

    "best_internal_dev": (
        best_result
    ),

    "references": {
        "B1": {
            "internal_dev_accuracy": (
                0.8857
            ),

            "internal_dev_macro_f1": (
                0.8880
            ),

            "organizer_validation_accuracy": (
                0.8800
            ),

            "organizer_validation_macro_f1": (
                0.8800
            ),

            "external_stress_accuracy": (
                0.8000
            ),

            "external_stress_macro_f1": (
                0.8038
            ),
        },

        "A1": {
            "internal_dev_macro_f1": (
                0.8249
            ),

            "organizer_validation_macro_f1": (
                0.8207
            ),
        },
    },

    "organizer_validation_used": (
        False
    ),
}


(
    RUN_DIR
    / "results.json"
).write_text(
    json.dumps(
        results_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# X. Final report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("B2 — RESULT")
print("=" * 78)


print(
    "Primary variable:",
    "+180 frozen synthetic examples",
)


print("\nTraining data")

print(
    "  Real:",
    795,
)

print(
    "  Synthetic:",
    180,
)

print(
    "  Total:",
    975,
)

print(
    "  Internal dev:",
    140,
)

print(
    "  Organizer validation used:",
    False,
)


print("\nB1 reference")

print(
    "  Internal dev | "
    "Accuracy=0.8857 | "
    "Macro-F1=0.8880"
)

print(
    "  Organizer val | "
    "Accuracy=0.8800 | "
    "Macro-F1=0.8800"
)

print(
    "  External stress | "
    "Accuracy=0.8000 | "
    "Macro-F1=0.8038"
)


print("\nB2 checkpoints")

print(
    f"  PRETRAIN | "
    f"Accuracy="
    f"{pretrain_dev['accuracy']:.4f} | "
    f"Macro-F1="
    f"{pretrain_dev['macro_f1']:.4f}"
)


for item in training_history:

    print(
        f"  EPOCH {item['epoch']} | "
        f"loss="
        f"{item['mean_example_loss']:.4f} | "
        f"Accuracy="
        f"{item['dev_accuracy']:.4f} | "
        f"Macro-F1="
        f"{item['dev_macro_f1']:.4f} | "
        f"{item['seconds']:.1f}s"
    )


print(
    "\nBest B2 checkpoint "
    "by INTERNAL DEV ONLY"
)

print(
    "  Epoch:",
    best_epoch,
)

print(
    "  Adapter:",
    best_adapter_dir,
)

print(
    "  Accuracy:",
    f"{best_result['accuracy']:.4f}",
)

print(
    "  Macro-F1:",
    f"{best_result['macro_f1']:.4f}",
)


print("\nB2 minus B1 internal dev")

print(
    "  Accuracy delta:",
    f"{best_result['accuracy'] - 0.8857:+.4f}",
)

print(
    "  Macro-F1 delta:",
    f"{best_result['macro_f1'] - 0.8880:+.4f}",
)


print("\nPer-class F1 — best B2")

for label in LABELS:

    print(
        f"  {label:15} "
        f"{best_result['per_class'][label]['f1']:.4f}"
    )


print("\nArtifacts")

print(
    " ",
    RUN_DIR,
)

print(
    " ",
    RUN_DIR
    / "run_config.json",
)

print(
    " ",
    RUN_DIR
    / "results.json",
)


print(
    "\nNO organizer-validation "
    "inference occurred."
)

In [ ]:
# ============================================================
# CELL 1 — D1 CHAMPION RECOVERY + CONSOLIDATED 4-SET AUDIT
#
# PURPOSE
# -------
# After a Kaggle runtime reset:
#
# 1. Recover the saved D1 champion package.
# 2. Verify exact D1 epoch-2 adapter SHA.
# 3. Locate original organizer train/validation files.
# 4. Reconstruct semantically-clean V3 (935 rows) if the
#    old /kaggle/working files disappeared.
# 5. Recover exact D1 predictions for:
#       - V3 internal dev 140
#       - external stress 30
#       - organizer validation 300
#       - synthetic diagnostic 180
# 6. Build ONE consolidated failure taxonomy.
# 7. Compare training-family coverage vs D1 failures.
#
# NO MODEL LOAD
# NO INFERENCE
# NO TRAINING
# NO GPU REQUIRED
# ============================================================

import re
import json
import csv
import hashlib
import zipfile
import shutil
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd


# ============================================================
# 0. CONSTANTS
# ============================================================

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

RECOVERY.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_RAW_TRAIN_SHA = (
    "26ea9a6998815d0f99f45aff2206f435"
    "781cc71bd92638101d7ea08c2e175d3c"
)

EXPECTED_ORGANIZER_SHA = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)

EXPECTED_D1_ADAPTER_SHA = (
    "2c64a87e227e0638be9dc73e021f877f"
    "2536876ccf94cc116e04d60075164f8f"
)

EXPECTED_V3_COUNTS = {
    "SUPPORTS": 348,
    "REFUTES": 311,
    "NOT_ENOUGH_INFO": 276,
}

EXPECTED_FIT_COUNTS = {
    "SUPPORTS": 296,
    "REFUTES": 264,
    "NOT_ENOUGH_INFO": 235,
}

EXPECTED_DEV_COUNTS = {
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
}


EXPECTED_METRICS = {
    "v3_dev": (134, 140),
    "external_30": (27, 30),
    "organizer_300": (264, 300),
    "synthetic_180": (147, 180),
}


CONFLICT_IDS = {
    "gfcc_v5_tr_0911",
    "gfcc_v5_tr_0102",
}


# ============================================================
# 1. 23 SEMANTIC CORRECTIONS
# ============================================================

SEMANTIC_CORRECTIONS = {
    "gfcc_v5_tr_0092": "REFUTES",
    "gfcc_v5_tr_0015": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0520": "SUPPORTS",
    "gfcc_v5_tr_0061": "SUPPORTS",
    "gfcc_v5_tr_0106": "SUPPORTS",
    "gfcc_v5_tr_0368": "REFUTES",
    "gfcc_v5_tr_0211": "SUPPORTS",
    "gfcc_v5_tr_0797": "SUPPORTS",
    "gfcc_v5_tr_0270": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0892": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0074": "REFUTES",
    "gfcc_v5_tr_0467": "REFUTES",
    "gfcc_v5_tr_0417": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0637": "SUPPORTS",
    "gfcc_v5_tr_0333": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0556": "SUPPORTS",
    "gfcc_v5_tr_0472": "SUPPORTS",
    "gfcc_v5_tr_0360": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0464": "REFUTES",
    "gfcc_v5_tr_0458": "REFUTES",
    "gfcc_v5_tr_0246": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0822": "NOT_ENOUGH_INFO",
    "gfcc_v5_tr_0644": "REFUTES",
}

assert len(SEMANTIC_CORRECTIONS) == 23


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def normalize_text(value):

    if value is None:
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


def normalize_label(value):

    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip().upper()

    if not text:
        return None

    compact = re.sub(
        r"[^A-Z]",
        "",
        text,
    )

    # SUPPORT variants
    if "SUPPORT" in compact:
        return "SUPPORTS"

    # REFUTE / contradiction variants
    if (
        "REFUT" in compact
        or "CONTRADICT" in compact
    ):
        return "REFUTES"

    # NEI variants
    if compact in {
        "NEI",
        "NOTENOUGHINFO",
        "NOTENOUGHINFORMATION",
        "INSUFFICIENTINFO",
        "INSUFFICIENTINFORMATION",
        "UNKNOWN",
        "UNDETERMINED",
    }:
        return "NOT_ENOUGH_INFO"

    if (
        "NOT" in compact
        and "ENOUGH" in compact
        and (
            "INFO" in compact
            or "INFORMATION" in compact
        )
    ):
        return "NOT_ENOUGH_INFO"

    return None


def normalize_evidence(value):

    if isinstance(value, str):
        passages = [value]

    elif isinstance(value, list):
        passages = value

    else:
        passages = list(value)

    cleaned = []
    seen = set()

    for passage in passages:

        passage = normalize_text(
            passage
        )

        # Remove blank passages.
        if not passage:
            continue

        # Remove duplicate normalized passages.
        if passage in seen:
            continue

        seen.add(passage)
        cleaned.append(passage)

    return cleaned


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():
                rows.append(
                    json.loads(line)
                )

    return rows


def write_jsonl(path, rows):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def find_by_basename(root, basename):

    if not root.exists():
        return []

    return list(
        root.rglob(
            basename
        )
    )


def parse_evidence_cell(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value)

    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return parsed

        return [str(parsed)]

    except Exception:
        return [text]


# ============================================================
# 3. LOCATE / EXTRACT D1 CHAMPION PACKAGE
# ============================================================

print("=" * 78)
print("STEP 1 — LOCATE D1 CHAMPION PACKAGE")
print("=" * 78)


manifest_candidates = (
    find_by_basename(
        INPUT,
        "CHAMPION_MANIFEST.json",
    )
    + find_by_basename(
        WORK,
        "CHAMPION_MANIFEST.json",
    )
)


champion_root = None


for candidate in manifest_candidates:

    try:
        data = json.loads(
            candidate.read_text(
                encoding="utf-8"
            )
        )

        if (
            data.get("artifact_name")
            == "D1_GEMMA4_12B_CHAMPION"
        ):
            champion_root = (
                candidate.parent
            )
            break

    except Exception:
        pass


if champion_root is None:

    zip_candidates = (
        find_by_basename(
            INPUT,
            "D1_GEMMA4_12B_CHAMPION.zip",
        )
        + find_by_basename(
            WORK,
            "D1_GEMMA4_12B_CHAMPION.zip",
        )
    )

    if not zip_candidates:

        raise RuntimeError(
            "\nD1_GEMMA4_12B_CHAMPION.zip was not found.\n\n"
            "Add your downloaded D1_GEMMA4_12B_CHAMPION.zip "
            "to this Kaggle notebook as an INPUT, then rerun "
            "this exact cell."
        )

    zip_path = zip_candidates[0]

    print(
        "Champion ZIP:",
        zip_path,
    )

    extract_root = (
        RECOVERY
        / "champion_extracted"
    )

    if extract_root.exists():
        shutil.rmtree(
            extract_root
        )

    extract_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as z:

        z.extractall(
            extract_root
        )

    extracted_manifests = (
        find_by_basename(
            extract_root,
            "CHAMPION_MANIFEST.json",
        )
    )

    assert extracted_manifests, (
        "ZIP extracted, but CHAMPION_MANIFEST.json "
        "was not found."
    )

    champion_root = (
        extracted_manifests[0]
        .parent
    )


print(
    "Champion root:",
    champion_root,
)


# ============================================================
# 4. VERIFY EXACT D1 ADAPTER
# ============================================================

adapter_candidates = (
    find_by_basename(
        champion_root,
        "adapter_model.safetensors",
    )
)


verified_adapter = None


for candidate in adapter_candidates:

    digest = sha256_file(
        candidate
    )

    if digest == EXPECTED_D1_ADAPTER_SHA:

        verified_adapter = candidate
        break


assert verified_adapter is not None, (
    "Exact D1 champion adapter SHA was not found "
    "inside the package."
)


print(
    "Verified adapter:",
    verified_adapter,
)

print(
    "Adapter SHA:",
    sha256_file(
        verified_adapter
    ),
)


# ============================================================
# 5. LOCATE RAW ORGANIZER TRAIN + VALIDATION BY SHA
# ============================================================

print("\n" + "=" * 78)
print("STEP 2 — LOCATE ORIGINAL DATASETS")
print("=" * 78)


known_sha_targets = {
    EXPECTED_RAW_TRAIN_SHA: None,
    EXPECTED_ORGANIZER_SHA: None,
}


jsonl_candidates = []


for root in [
    INPUT,
    WORK,
]:

    if not root.exists():
        continue

    for path in root.rglob(
        "*.jsonl"
    ):

        try:

            # Avoid hashing unexpectedly huge files.
            if (
                path.stat().st_size
                <= 100 * 1024 * 1024
            ):
                jsonl_candidates.append(
                    path
                )

        except Exception:
            pass


for path in jsonl_candidates:

    try:
        digest = sha256_file(
            path
        )

    except Exception:
        continue

    if digest in known_sha_targets:

        known_sha_targets[
            digest
        ] = path


RAW_TRAIN_PATH = (
    known_sha_targets[
        EXPECTED_RAW_TRAIN_SHA
    ]
)

ORGANIZER_PATH = (
    known_sha_targets[
        EXPECTED_ORGANIZER_SHA
    ]
)


assert RAW_TRAIN_PATH is not None, (
    "Could not locate original 1000-row train JSONL "
    "with the expected SHA."
)

assert ORGANIZER_PATH is not None, (
    "Could not locate organizer validation JSONL "
    "with the expected SHA."
)


print(
    "Raw train:",
    RAW_TRAIN_PATH,
)

print(
    "Raw train SHA:",
    sha256_file(
        RAW_TRAIN_PATH
    ),
)


print(
    "\nOrganizer validation:",
    ORGANIZER_PATH,
)

print(
    "Organizer SHA:",
    sha256_file(
        ORGANIZER_PATH
    ),
)


# ============================================================
# 6. LOCATE SAVED D1 PREDICTIONS
# ============================================================

print("\n" + "=" * 78)
print("STEP 3 — RECOVER D1 PREDICTION FILES")
print("=" * 78)


wanted_prediction_files = {
    "v3_dev": (
        "internal_dev_epoch_2.csv"
    ),

    "external_30": (
        "d1_12b_external_30_predictions.csv"
    ),

    "organizer_300": (
        "d1_12b_organizer_300_predictions.csv"
    ),

    "synthetic_180": (
        "d1_12b_synthetic_180_predictions.csv"
    ),
}


prediction_paths = {}


for dataset_name, basename in (
    wanted_prediction_files.items()
):

    candidates = (
        find_by_basename(
            champion_root,
            basename,
        )
    )

    if not candidates:

        raise RuntimeError(
            f"Missing saved D1 prediction file: {basename}\n"
            "The champion ZIP does not contain all broad "
            "evaluation artifacts."
        )

    prediction_paths[
        dataset_name
    ] = candidates[0]

    print(
        f"{dataset_name:16}: "
        f"{candidates[0]}"
    )


# ============================================================
# 7. RECONSTRUCT V3 CLEAN 935
# ============================================================

print("\n" + "=" * 78)
print("STEP 4 — RECONSTRUCT SEMANTIC-CLEAN V3")
print("=" * 78)


raw_rows = read_jsonl(
    RAW_TRAIN_PATH
)


assert len(raw_rows) == 1000


clean_rows = []


for row in raw_rows:

    example_id = str(
        row["id"]
    )

    label = normalize_label(
        row.get(
            "label"
        )
    )

    # Original cleaning removed 10 blank labels.
    if label is None:
        continue

    if example_id in CONFLICT_IDS:
        continue

    cleaned = dict(
        row
    )

    cleaned["id"] = (
        example_id
    )

    cleaned["claim"] = (
        normalize_text(
            row["claim"]
        )
    )

    cleaned["evidence"] = (
        normalize_evidence(
            row["evidence"]
        )
    )

    cleaned["label"] = (
        label
    )

    clean_rows.append(
        cleaned
    )


# Same-label exact normalized duplicates:
# keep earliest row.

deduped = []

seen_keys = set()


for row in clean_rows:

    key = (
        row["claim"],
        tuple(
            row["evidence"]
        ),
        row["label"],
    )

    if key in seen_keys:
        continue

    seen_keys.add(
        key
    )

    deduped.append(
        row
    )


assert len(deduped) == 935, (
    f"Expected 935 post-clean rows; "
    f"got {len(deduped)}."
)


# Apply 23 semantic corrections.

seen_corrections = set()


for row in deduped:

    example_id = (
        str(
            row["id"]
        )
    )

    if example_id in SEMANTIC_CORRECTIONS:

        row["label"] = (
            SEMANTIC_CORRECTIONS[
                example_id
            ]
        )

        seen_corrections.add(
            example_id
        )


assert (
    seen_corrections
    == set(
        SEMANTIC_CORRECTIONS
    )
)


v3_counts = Counter(
    row["label"]
    for row in deduped
)


assert dict(
    v3_counts
) == EXPECTED_V3_COUNTS, (
    v3_counts
)


print(
    "V3 clean rows:",
    len(deduped),
)

print(
    "V3 labels:",
    v3_counts,
)

print(
    "Semantic corrections:",
    len(
        seen_corrections
    ),
)


# ============================================================
# 8. RECONSTRUCT EXACT D1 FIT/DEV MEMBERSHIP
#
# Dev IDs are recovered from D1's saved epoch-2 predictions,
# so we do NOT need to guess the old split RNG implementation.
# ============================================================

dev_pred_df = pd.read_csv(
    prediction_paths[
        "v3_dev"
    ]
)


assert len(
    dev_pred_df
) == 140


dev_ids = set(
    dev_pred_df[
        "id"
    ]
    .astype(str)
)


v3_dev_rows = [
    row
    for row in deduped
    if str(
        row["id"]
    )
    in dev_ids
]


v3_fit_rows = [
    row
    for row in deduped
    if str(
        row["id"]
    )
    not in dev_ids
]


assert len(
    v3_dev_rows
) == 140

assert len(
    v3_fit_rows
) == 795


fit_counts = Counter(
    row["label"]
    for row in v3_fit_rows
)

dev_counts = Counter(
    row["label"]
    for row in v3_dev_rows
)


assert dict(
    fit_counts
) == EXPECTED_FIT_COUNTS

assert dict(
    dev_counts
) == EXPECTED_DEV_COUNTS


RECOVERED_CLEAN = (
    RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

RECOVERED_FIT = (
    RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

RECOVERED_DEV = (
    RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)


write_jsonl(
    RECOVERED_CLEAN,
    deduped,
)

write_jsonl(
    RECOVERED_FIT,
    v3_fit_rows,
)

write_jsonl(
    RECOVERED_DEV,
    v3_dev_rows,
)


print(
    "\nRecovered fit:",
    len(v3_fit_rows),
    fit_counts,
)

print(
    "Recovered dev:",
    len(v3_dev_rows),
    dev_counts,
)


# ============================================================
# 9. FAMILY TAXONOMY
#
# This taxonomy is intentionally based on GENERAL reasoning
# primitives rather than individual validation IDs.
# ============================================================

def classify_family(
    claim,
    example_id="",
):

    c = normalize_text(
        claim
    ).lower()

    eid = str(
        example_id
    ).lower()


    # Revenue percentage arithmetic
    if (
        "revenue" in c
        and (
            "%" in c
            or "percent" in c
            or "growth" in c
            or "increase" in c
            or "rose" in c
        )
    ):
        return "revenue_percentage"


    # Explicit synthetic revenue IDs
    if "revenue" in eid:
        return "revenue_percentage"


    # Two-component addition
    if (
        "combined" in c
        and (
            "north" in c
            or "south" in c
        )
    ):
        return "two_component_sum"

    if (
        "jointly made" in c
        or "two named sites" in c
    ):
        return "two_component_sum"

    if "two_comp" in eid:
        return "two_component_sum"


    # Project + location + year conjunction
    if (
        "carried out project" in c
        or (
            "project " in c
            and " in 202" in c
            and (
                "laboratory" in c
                or "project" in c
            )
        )
    ):
        return "project_location_mapping"


    # Closed-world manifest omission
    if (
        "left out" in c
        and (
            "payload" in c
            or "manifest" in c
        )
    ):
        return "complete_manifest"


    # Numeric bound / storage range
    if (
        "storage" in c
        and (
            "minimum" in c
            or "maximum" in c
            or "at least" in c
            or "at most" in c
        )
    ):
        return "range_or_bound"


    # Paired depot comparison
    if (
        "depot" in c
        and (
            "busier" in c
            or "more parcels" in c
        )
    ):
        return "depot_comparison"


    # Temporal office-holder transition
    if (
        "office-holder" in c
        or (
            "director" in c
            and re.search(
                r"\b20\d{2}\b",
                c,
            )
        )
    ):
        return "temporal_transition"


    # General duration threshold
    if (
        any(
            unit in c
            for unit in [
                " year",
                " month",
                " day",
                " week",
            ]
        )
        and any(
            phrase in c
            for phrase in [
                "more than",
                "less than",
                "at least",
                "at most",
                "within",
            ]
        )
    ):
        return "duration_threshold"


    # General % / fraction threshold outside revenue
    if (
        "%" in c
        or "percent" in c
    ):
        return "percentage_threshold"


    # Full-year service
    if (
        "full-year service" in c
        or "operating in every month" in c
    ):
        return "full_year_service"


    # Chronological order
    if (
        "came first" in c
        or "ahead of" in c
        or "before" in c
        and "opening" in c
    ):
        return "chronological_order"


    return "other"


# ============================================================
# 10. BUILD CANONICAL DATA LOOKUPS
# ============================================================

v3_dev_by_id = {
    str(row["id"]): row
    for row in v3_dev_rows
}


organizer_rows = read_jsonl(
    ORGANIZER_PATH
)


assert len(
    organizer_rows
) == 300


organizer_by_id = {
    str(row["id"]): row
    for row in organizer_rows
}


# ============================================================
# 11. STANDARDIZE PREDICTION DATASETS
# ============================================================

all_eval_records = []


def add_eval_from_predictions(
    dataset_name,
    prediction_path,
    source_rows_by_id=None,
):

    df = pd.read_csv(
        prediction_path
    )


    required = {
        "id",
        "gold",
        "prediction",
    }

    assert required.issubset(
        set(df.columns)
    ), (
        dataset_name,
        df.columns.tolist(),
    )


    for raw in df.to_dict(
        orient="records"
    ):

        example_id = str(
            raw["id"]
        )

        gold = normalize_label(
            raw["gold"]
        )

        prediction = normalize_label(
            raw["prediction"]
        )


        if source_rows_by_id is not None:

            source = (
                source_rows_by_id[
                    example_id
                ]
            )

            claim = (
                source["claim"]
            )

            evidence = (
                source["evidence"]
            )

        else:

            claim = (
                raw.get(
                    "claim",
                    "",
                )
            )

            evidence = parse_evidence_cell(
                raw.get(
                    "evidence",
                    "[]",
                )
            )


        family = classify_family(
            claim,
            example_id,
        )


        all_eval_records.append({
            "dataset": (
                dataset_name
            ),

            "id": (
                example_id
            ),

            "family": (
                family
            ),

            "gold": (
                gold
            ),

            "prediction": (
                prediction
            ),

            "correct": (
                gold == prediction
            ),

            "claim": (
                claim
            ),

            "evidence": json.dumps(
                evidence,
                ensure_ascii=False,
            ),
        })


add_eval_from_predictions(
    "v3_dev",
    prediction_paths[
        "v3_dev"
    ],
    source_rows_by_id=(
        v3_dev_by_id
    ),
)


add_eval_from_predictions(
    "external_30",
    prediction_paths[
        "external_30"
    ],
)


add_eval_from_predictions(
    "organizer_300",
    prediction_paths[
        "organizer_300"
    ],
    source_rows_by_id=(
        organizer_by_id
    ),
)


add_eval_from_predictions(
    "synthetic_180",
    prediction_paths[
        "synthetic_180"
    ],
)


audit_df = pd.DataFrame(
    all_eval_records
)


# ============================================================
# 12. VERIFY RECOVERED D1 RESULTS
# ============================================================

print("\n" + "=" * 78)
print("STEP 5 — VERIFY RECOVERED D1 RESULTS")
print("=" * 78)


for dataset_name, (
    expected_correct,
    expected_total,
) in EXPECTED_METRICS.items():

    subset = audit_df[
        audit_df[
            "dataset"
        ]
        == dataset_name
    ]

    actual_total = len(
        subset
    )

    actual_correct = int(
        subset[
            "correct"
        ].sum()
    )


    assert actual_total == expected_total, (
        dataset_name,
        actual_total,
    )

    assert actual_correct == expected_correct, (
        dataset_name,
        actual_correct,
    )


    print(
        f"{dataset_name:16} | "
        f"{actual_correct:3d}/{actual_total:3d} | "
        f"errors={actual_total-actual_correct:3d}"
    )


# ============================================================
# 13. TRAINING FAMILY COVERAGE
# ============================================================

fit_family_counts = Counter(
    classify_family(
        row["claim"],
        row["id"],
    )
    for row in v3_fit_rows
)


# ============================================================
# 14. CROSS-DATASET FAMILY PERFORMANCE
# ============================================================

print("\n" + "=" * 78)
print("STEP 6 — COLLECTIVE D1 FAILURE MAP")
print("=" * 78)


family_rows = []


for family in sorted(
    audit_df[
        "family"
    ].unique()
):

    subset = audit_df[
        audit_df[
            "family"
        ]
        == family
    ]

    n = len(
        subset
    )

    errors = int(
        (
            ~subset[
                "correct"
            ]
        ).sum()
    )

    family_rows.append({
        "family": family,
        "fit795_examples": (
            fit_family_counts[
                family
            ]
        ),
        "eval_examples": n,
        "errors": errors,
        "error_rate": (
            errors / n
            if n
            else 0.0
        ),
        "datasets_with_errors": (
            ", ".join(
                sorted(
                    subset.loc[
                        ~subset[
                            "correct"
                        ],
                        "dataset",
                    ].unique()
                )
            )
        ),
    })


family_summary_df = (
    pd.DataFrame(
        family_rows
    )
    .sort_values(
        [
            "errors",
            "error_rate",
        ],
        ascending=False,
    )
)


print(
    f"{'Family':28} | "
    f"{'Fit':>4} | "
    f"{'Eval':>4} | "
    f"{'Err':>3} | "
    f"{'Rate':>7} | "
    f"Error datasets"
)

print(
    "-" * 100
)


for _, row in (
    family_summary_df
    .iterrows()
):

    print(
        f"{row['family']:28} | "
        f"{int(row['fit795_examples']):4d} | "
        f"{int(row['eval_examples']):4d} | "
        f"{int(row['errors']):3d} | "
        f"{row['error_rate']:7.1%} | "
        f"{row['datasets_with_errors']}"
    )


# ============================================================
# 15. ERROR COUNTS BY DATASET × FAMILY
# ============================================================

errors_df = audit_df[
    ~audit_df[
        "correct"
    ]
].copy()


print("\n" + "=" * 78)
print("STEP 7 — ERRORS BY DATASET × FAMILY")
print("=" * 78)


error_pivot = pd.crosstab(
    errors_df[
        "family"
    ],
    errors_df[
        "dataset"
    ],
)


dataset_order = [
    "v3_dev",
    "external_30",
    "organizer_300",
    "synthetic_180",
]


for name in dataset_order:

    if name not in error_pivot.columns:
        error_pivot[
            name
        ] = 0


error_pivot = (
    error_pivot[
        dataset_order
    ]
    .assign(
        TOTAL=lambda x: (
            x.sum(
                axis=1
            )
        )
    )
    .sort_values(
        "TOTAL",
        ascending=False,
    )
)


print(
    error_pivot.to_string()
)


# ============================================================
# 16. CONFUSION TYPES
# ============================================================

print("\n" + "=" * 78)
print("STEP 8 — COLLECTIVE CONFUSION TYPES")
print("=" * 78)


confusions = Counter(
    f"{row.gold} -> {row.prediction}"

    for row in errors_df.itertuples()
)


for confusion, count in (
    confusions.most_common()
):

    print(
        f"{confusion:40}: "
        f"{count}"
    )


# ============================================================
# 17. MACRO REASONING GROUPS
# ============================================================

NUMERIC_RELATIONAL = {
    "revenue_percentage",
    "two_component_sum",
    "range_or_bound",
    "depot_comparison",
    "temporal_transition",
    "duration_threshold",
    "percentage_threshold",
}

STRUCTURED_LOOKUP = {
    "project_location_mapping",
    "complete_manifest",
}


def macro_group(family):

    if family in NUMERIC_RELATIONAL:
        return "NUMERIC_OR_RELATIONAL"

    if family in STRUCTURED_LOOKUP:
        return "STRUCTURED_LOOKUP_OR_CLOSED_WORLD"

    return "OTHER"


errors_df[
    "macro_group"
] = (
    errors_df[
        "family"
    ]
    .map(
        macro_group
    )
)


macro_counts = Counter(
    errors_df[
        "macro_group"
    ]
)


print("\n" + "=" * 78)
print("STEP 9 — MACRO FAILURE SIGNAL")
print("=" * 78)


total_errors = len(
    errors_df
)


print(
    "Total known D1 errors:",
    total_errors,
)


for group, count in (
    macro_counts.most_common()
):

    print(
        f"{group:36} | "
        f"{count:3d}/{total_errors} "
        f"({count/total_errors:.1%})"
    )


# ============================================================
# 18. TOP REPRESENTATIVE ERRORS
# ============================================================

print("\n" + "=" * 78)
print("STEP 10 — REPRESENTATIVE ERRORS BY FAMILY")
print("=" * 78)


for family in (
    family_summary_df[
        "family"
    ].tolist()
):

    family_errors = errors_df[
        errors_df[
            "family"
        ]
        == family
    ]


    if len(
        family_errors
    ) == 0:
        continue


    print("\n" + "-" * 78)

    print(
        f"{family} | "
        f"errors={len(family_errors)}"
    )

    print("-" * 78)


    for _, row in (
        family_errors
        .head(3)
        .iterrows()
    ):

        print(
            f"[{row['dataset']}] "
            f"{row['gold']} -> "
            f"{row['prediction']}"
        )

        print(
            "ID:",
            row["id"],
        )

        print(
            "Claim:",
            row["claim"],
        )

        print()


# ============================================================
# 19. SAVE AUDIT ARTIFACTS
# ============================================================

AUDIT_CSV = (
    RECOVERY
    / "D1_CONSOLIDATED_4SET_AUDIT.csv"
)

ERRORS_CSV = (
    RECOVERY
    / "D1_CONSOLIDATED_4SET_ERRORS.csv"
)

FAMILY_CSV = (
    RECOVERY
    / "D1_CONSOLIDATED_FAMILY_SUMMARY.csv"
)

SUMMARY_JSON = (
    RECOVERY
    / "D1_CONSOLIDATED_4SET_AUDIT_SUMMARY.json"
)


audit_df.to_csv(
    AUDIT_CSV,
    index=False,
)

errors_df.to_csv(
    ERRORS_CSV,
    index=False,
)

family_summary_df.to_csv(
    FAMILY_CSV,
    index=False,
)


summary = {
    "champion": {
        "model": (
            "google/gemma-4-12B-it"
        ),

        "adapter_sha256": (
            EXPECTED_D1_ADAPTER_SHA
        ),

        "recipe": (
            "D1 epoch2 BASE greedy "
            "r8 alpha16 dropout0.05 all-language-projections"
        ),
    },

    "datasets": {
        dataset_name: {
            "rows": int(
                len(
                    audit_df[
                        audit_df["dataset"]
                        == dataset_name
                    ]
                )
            ),

            "errors": int(
                (
                    ~audit_df.loc[
                        audit_df["dataset"]
                        == dataset_name,
                        "correct",
                    ]
                ).sum()
            ),
        }
        for dataset_name in dataset_order
    },

    "total_known_errors": int(
        total_errors
    ),

    "macro_failure_counts": dict(
        macro_counts
    ),

    "confusions": dict(
        confusions
    ),

    "family_summary": (
        family_summary_df
        .to_dict(
            orient="records"
        )
    ),

    "recovered_files": {
        "v3_clean": str(
            RECOVERED_CLEAN
        ),

        "v3_fit": str(
            RECOVERED_FIT
        ),

        "v3_dev": str(
            RECOVERED_DEV
        ),

        "d1_adapter": str(
            verified_adapter
        ),
    },
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 78)
print("CELL 1 COMPLETE")
print("=" * 78)


print(
    "Recovered V3 fit:",
    RECOVERED_FIT,
)

print(
    "Recovered V3 dev:",
    RECOVERED_DEV,
)

print(
    "Consolidated audit:",
    AUDIT_CSV,
)

print(
    "Errors only:",
    ERRORS_CSV,
)

print(
    "Family summary:",
    FAMILY_CSV,
)

print(
    "Summary:",
    SUMMARY_JSON,
)


print(
    "\nNO MODEL was loaded."
)

print(
    "NO inference occurred."
)

print(
    "NO training occurred."
)

print(
    "\nNEXT: send me this cell's output. "
    "I will use the measured family table to design "
    "the NEW D4 training curriculum + untouched holdout."
)

In [ ]:
# ============================================================
# CELL 2 — BUILD D4 CONTRASTIVE CURRICULUM + FROZEN HOLDOUT
#
# INPUT:
#   Outputs of Cell 1.
#
# OUTPUT:
#
#   TRAIN CURRICULUM:
#     150 examples = 50 SUPPORTS / 50 REFUTES / 50 NEI
#
#   FROZEN HOLDOUT:
#      75 examples = 25 SUPPORTS / 25 REFUTES / 25 NEI
#
# DESIGN:
#   Every scenario forms a contrastive trio:
#
#       same evidence
#          ├── SUPPORTS claim
#          ├── REFUTES claim
#          └── NOT_ENOUGH_INFO claim
#
# TRAIN and HOLDOUT use:
#   - different entities
#   - different IDs
#   - different scenario instances
#   - different wording/templates
#   - different numbers
#
# PRIORITIES ARE BASED ON D1'S COLLECTIVE FAILURE MAP,
# NOT ON INDIVIDUAL VALIDATION ANSWERS.
#
# NO MODEL
# NO GPU
# NO TRAINING
# NO INFERENCE
#
# IMPORTANT:
#   The frozen holdout produced here must NEVER be added
#   to D4 training.
# ============================================================

import re
import json
import math
import random
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date

import pandas as pd


# ============================================================
# 0. PATHS / CONFIG
# ============================================================

WORK = Path("/kaggle/working")

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

CLEAN_PATH = (
    RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

FIT_PATH = (
    RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

DEV_PATH = (
    RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)

EVAL_AUDIT_PATH = (
    RECOVERY
    / "D1_CONSOLIDATED_4SET_AUDIT.csv"
)


OUT_DIR = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_OUT = (
    OUT_DIR
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT_OUT = (
    OUT_DIR
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)

TRAIN_META_OUT = (
    OUT_DIR
    / "d4_contrastive_train_v1_metadata.csv"
)

HOLDOUT_META_OUT = (
    OUT_DIR
    / "D4_FROZEN_HOLDOUT_V1_metadata.csv"
)

MANIFEST_OUT = (
    OUT_DIR
    / "d4_curriculum_v1_manifest.json"
)


SEED = 20260827


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


# ============================================================
# 1. FAILURE-INFORMED ALLOCATION
#
# Number here = number of contrastive scenarios.
# Each scenario produces exactly 3 examples.
#
# TRAIN:
# 50 scenarios × 3 = 150 examples.
#
# HOLDOUT:
# 25 scenarios × 3 = 75 examples.
# ============================================================

TRAIN_SCENARIOS = {
    "revenue_percentage": 15,
    "two_component_sum": 10,
    "project_location_mapping": 6,
    "range_or_bound": 4,
    "complete_manifest": 4,
    "percentage_threshold": 3,
    "duration_threshold": 2,
    "depot_comparison": 2,
    "temporal_transition": 2,

    # Small transfer bucket:
    # similar reasoning primitives but wording/operation not
    # directly copied from known benchmark families.
    "derived_comparison_transfer": 2,
}


HOLDOUT_SCENARIOS = {
    "revenue_percentage": 5,
    "two_component_sum": 4,
    "project_location_mapping": 3,
    "range_or_bound": 3,
    "complete_manifest": 2,
    "percentage_threshold": 2,
    "duration_threshold": 1,
    "depot_comparison": 1,
    "temporal_transition": 2,
    "derived_comparison_transfer": 2,
}


assert sum(
    TRAIN_SCENARIOS.values()
) == 50

assert sum(
    HOLDOUT_SCENARIOS.values()
) == 25


# ============================================================
# 2. HELPERS
# ============================================================

def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def write_jsonl(
    path,
    rows,
):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


def norm_text(value):

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    )

    return (
        value
        .strip()
        .lower()
    )


def parse_evidence(value):

    if isinstance(
        value,
        list,
    ):

        return [
            str(x)
            for x in value
        ]


    if value is None:

        return []


    try:

        if pd.isna(value):
            return []

    except Exception:

        pass


    try:

        parsed = json.loads(
            str(value)
        )

        if isinstance(
            parsed,
            list,
        ):

            return [
                str(x)
                for x in parsed
            ]

    except Exception:

        pass


    return [
        str(value)
    ]


def fingerprint(
    claim,
    evidence,
):

    # Sort evidence so passage ordering alone cannot evade
    # overlap detection.

    evidence_norm = sorted(
        norm_text(x)
        for x in evidence
    )

    material = (
        norm_text(
            claim
        )
        + " || "
        + " || ".join(
            evidence_norm
        )
    )

    return hashlib.sha256(
        material.encode(
            "utf-8"
        )
    ).hexdigest()


def scenario_rng(
    split,
    family,
    index,
):

    material = (
        f"{SEED}|"
        f"{split}|"
        f"{family}|"
        f"{index}"
    )

    digest = hashlib.sha256(
        material.encode(
            "utf-8"
        )
    ).hexdigest()


    integer_seed = int(
        digest[:16],
        16,
    )


    return random.Random(
        integer_seed
    )


def shuffled(
    values,
    rng,
):

    values = list(
        values
    )

    rng.shuffle(
        values
    )

    return values


def next_quarter(
    quarter,
    year,
):

    if quarter < 4:

        return (
            quarter + 1,
            year,
        )

    return (
        1,
        year + 1,
    )


# ============================================================
# 3. VERIFY CELL-1 INPUTS
# ============================================================

for path in [
    CLEAN_PATH,
    FIT_PATH,
    DEV_PATH,
    EVAL_AUDIT_PATH,
]:

    assert path.exists(), path


clean_rows = read_jsonl(
    CLEAN_PATH
)

fit_rows = read_jsonl(
    FIT_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

eval_df = pd.read_csv(
    EVAL_AUDIT_PATH
)


assert len(clean_rows) == 935
assert len(fit_rows) == 795
assert len(dev_rows) == 140
assert len(eval_df) == 650


print("=" * 78)
print("CELL 2 — D4 CONTRASTIVE CURRICULUM")
print("=" * 78)

print(
    "Recovered clean:",
    len(clean_rows),
)

print(
    "Recovered fit:",
    len(fit_rows),
)

print(
    "Recovered dev:",
    len(dev_rows),
)

print(
    "Existing diagnostic examples:",
    len(eval_df),
)


# ============================================================
# 4. EXISTING CORPUS — USED ONLY FOR LEAKAGE CHECKING
# ============================================================

existing_claims = set()

existing_fingerprints = set()

existing_text_parts = []


for row in clean_rows:

    claim = row[
        "claim"
    ]

    evidence = row[
        "evidence"
    ]

    existing_claims.add(
        norm_text(
            claim
        )
    )

    existing_fingerprints.add(
        fingerprint(
            claim,
            evidence,
        )
    )

    existing_text_parts.append(
        claim
    )

    existing_text_parts.extend(
        evidence
    )


for row in eval_df.to_dict(
    orient="records"
):

    claim = str(
        row.get(
            "claim",
            "",
        )
    )

    evidence = parse_evidence(
        row.get(
            "evidence",
            "[]",
        )
    )

    existing_claims.add(
        norm_text(
            claim
        )
    )

    existing_fingerprints.add(
        fingerprint(
            claim,
            evidence,
        )
    )

    existing_text_parts.append(
        claim
    )

    existing_text_parts.extend(
        evidence
    )


existing_text = norm_text(
    "\n".join(
        existing_text_parts
    )
)


# ============================================================
# 5. ENTITY / WORD BANKS
#
# TRAIN and HOLDOUT intentionally use different banks.
# ============================================================

TRAIN_ROOTS = [
    "Caldera",
    "Meridian",
    "Solstice",
    "Brightmere",
    "Stonehaven",
    "Vesper",
    "Clearbrook",
    "Northcove",
    "Lakewind",
    "Marigold",
    "Silverpine",
    "Redcliff",
    "Alderwick",
    "Bluehaven",
    "Suncrest",
    "Hawthorne",
]


HOLDOUT_ROOTS = [
    "Quarrygate",
    "Evergreen",
    "Sableford",
    "Rivermark",
    "Goldhaven",
    "Pinecrest",
    "Meadowgate",
    "Ironwood",
    "Starfall",
    "Cloudmere",
    "Westhaven",
    "Eastbrook",
]


TRAIN_PROJECTS = [
    "Project Emberline",
    "Project Glasswing",
    "Project Tidemark",
    "Project Sandpiper",
    "Project Blue Lantern",
    "Project Silver Current",
    "Project Cedar Arc",
]


HOLDOUT_PROJECTS = [
    "Project Moonridge",
    "Project Frost Relay",
    "Project Copper Finch",
    "Project Delta Loom",
    "Project Sunward",
]


TRAIN_LOCATIONS = [
    "Northern Basin",
    "Amber Coast",
    "Highland Plain",
    "Eastern Marsh",
    "Granite Valley",
    "Western Reach",
    "River Plateau",
]


HOLDOUT_LOCATIONS = [
    "Cobalt Ridge",
    "Southern Steppe",
    "Mist Valley",
    "Lake Frontier",
    "Redstone Plain",
]


TRAIN_PEOPLE = [
    "Elena Marr",
    "Jonas Vale",
    "Mira Hoss",
    "Tariq Rowan",
    "Leonie Park",
    "Samir Holt",
    "Nadia Fenn",
    "Iris Cole",
]


HOLDOUT_PEOPLE = [
    "Avery Sloan",
    "Rhea Mercer",
    "Devon Pike",
    "Malik Arden",
    "Sofia Trent",
    "Noah Venn",
]


TRAIN_DISTRACTORS = [
    "A routine archival checksum was completed after publication.",
    "The catalog identifier reflects export order rather than magnitude.",
    "Independent clerks verified the transcription format.",
    "A duplicate copy is stored in the records annex.",
]


HOLDOUT_DISTRACTORS = [
    "The register was migrated to a new database after certification.",
    "A document-control stamp records the filing sequence only.",
    "The digital copy preserves the original pagination.",
    "A secondary archive retains the signed record.",
]


def entity_name(
    split,
    family,
    index,
):

    if split == "train":

        root = TRAIN_ROOTS[
            index
            % len(
                TRAIN_ROOTS
            )
        ]

        number = (
            7000
            + list(
                TRAIN_SCENARIOS
            ).index(
                family
            )
            * 100
            + index
        )

    else:

        root = HOLDOUT_ROOTS[
            index
            % len(
                HOLDOUT_ROOTS
            )
        ]

        number = (
            9000
            + list(
                HOLDOUT_SCENARIOS
            ).index(
                family
            )
            * 100
            + index
        )


    suffixes = {
        "revenue_percentage": (
            "Ledger Cooperative"
        ),

        "two_component_sum": (
            "Production Archive"
        ),

        "project_location_mapping": (
            "Civic Observatory"
        ),

        "range_or_bound": (
            "Water Authority"
        ),

        "complete_manifest": (
            "Exploration Centre"
        ),

        "percentage_threshold": (
            "Regional College"
        ),

        "duration_threshold": (
            "Orbital Mission"
        ),

        "depot_comparison": (
            "Records Office"
        ),

        "temporal_transition": (
            "Policy Institute"
        ),

        "derived_comparison_transfer": (
            "Operations Centre"
        ),
    }


    return (
        f"{root} "
        f"{suffixes[family]} "
        f"{number}"
    )


def distractor(
    split,
    index,
):

    bank = (
        TRAIN_DISTRACTORS
        if split == "train"
        else HOLDOUT_DISTRACTORS
    )

    return bank[
        index
        % len(
            bank
        )
    ]


# ============================================================
# 6. SCENARIO GENERATORS
#
# Each function returns:
#
# {
#   evidence: [...]
#   claims: {
#       SUPPORTS: "...",
#       REFUTES: "...",
#       NOT_ENOUGH_INFO: "..."
#   }
#   entity: ...
#   template_id: ...
#   reasoning: ...
# }
# ============================================================

def generate_revenue(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "revenue_percentage",
        index,
    )

    org = entity_name(
        split,
        "revenue_percentage",
        index,
    )


    percentages = [
        10,
        15,
        20,
        25,
        30,
        40,
    ]

    pct = percentages[
        index
        % len(
            percentages
        )
    ]


    # Multiple of 20 makes all chosen percentages integral.

    base = (
        240
        + 20
        * (
            (
                index * 7
                + (
                    2
                    if split == "holdout"
                    else 0
                )
            )
            % 25
        )
    )


    new = int(
        base
        * (
            100 + pct
        )
        / 100
    )


    wrong_pct = (
        pct + 5
        if pct != 40
        else 35
    )


    y0 = (
        2020
        + (
            index
            % 4
        )
    )

    y1 = (
        y0 + 1
    )

    y2 = (
        y1 + 1
    )


    style = index % 3


    if split == "train":

        if style == 0:

            evidence = [
                (
                    f"Certified accounts list "
                    f"{org}'s revenue as "
                    f"{base} million credits "
                    f"in {y0}."
                ),

                (
                    f"The audited figure for "
                    f"{y1} is {new} million "
                    f"credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"{org}'s audited revenue "
                f"increased by exactly {pct}% "
                f"from {y0} to {y1}."
            )

            r_claim = (
                f"{org}'s audited revenue "
                f"increased by exactly "
                f"{wrong_pct}% from {y0} "
                f"to {y1}."
            )

            n_claim = (
                f"{org}'s audited revenue "
                f"increased by exactly {pct}% "
                f"from {y1} to {y2}."
            )


        elif style == 1:

            evidence = [
                (
                    f"In {y0}, the verified "
                    f"revenue ledger for {org} "
                    f"records {base} million "
                    f"credits."
                ),

                (
                    f"In {y1}, the same ledger "
                    f"records {new} million "
                    f"credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"The year-on-year rise in "
                f"verified revenue at {org} "
                f"from {y0} to {y1} was "
                f"precisely {pct}%."
            )

            r_claim = (
                f"The year-on-year rise in "
                f"verified revenue at {org} "
                f"from {y0} to {y1} was "
                f"precisely {wrong_pct}%."
            )

            n_claim = (
                f"The year-on-year rise in "
                f"verified revenue at {org} "
                f"from {y1} to {y2} was "
                f"precisely {pct}%."
            )


        else:

            evidence = [
                (
                    f"{org} reported audited "
                    f"revenue of {new} million "
                    f"credits for {y1}."
                ),

                (
                    f"Its audited revenue for "
                    f"{y0} was {base} million "
                    f"credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"Using the audited totals, "
                f"revenue at {org} was "
                f"exactly {pct}% higher in "
                f"{y1} than in {y0}."
            )

            r_claim = (
                f"Using the audited totals, "
                f"revenue at {org} was "
                f"exactly {wrong_pct}% higher "
                f"in {y1} than in {y0}."
            )

            n_claim = (
                f"Using the audited totals, "
                f"revenue at {org} was "
                f"exactly {pct}% higher in "
                f"{y2} than in {y1}."
            )


    else:

        # HOLDOUT wording is deliberately different.

        if style == 0:

            evidence = [
                (
                    f"The signed financial "
                    f"review fixes {org}'s "
                    f"{y0} revenue at "
                    f"{base} million credits."
                ),

                (
                    f"For {y1}, the signed-off "
                    f"total is {new} million "
                    f"credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"Comparing the signed-off "
                f"totals for {y0} and {y1}, "
                f"{org} recorded exactly "
                f"{pct}% revenue growth."
            )

            r_claim = (
                f"Comparing the signed-off "
                f"totals for {y0} and {y1}, "
                f"{org} recorded exactly "
                f"{wrong_pct}% revenue growth."
            )

            n_claim = (
                f"Comparing the signed-off "
                f"totals for {y1} and {y2}, "
                f"{org} recorded exactly "
                f"{pct}% revenue growth."
            )


        elif style == 1:

            evidence = [
                (
                    f"Revenue certified for "
                    f"{org} in {y1}: "
                    f"{new} million credits."
                ),

                (
                    f"Revenue certified for "
                    f"{org} in {y0}: "
                    f"{base} million credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"The certified revenue figures "
                f"show a {pct}% increase at "
                f"{org} between {y0} and {y1}."
            )

            r_claim = (
                f"The certified revenue figures "
                f"show a {wrong_pct}% increase "
                f"at {org} between {y0} "
                f"and {y1}."
            )

            n_claim = (
                f"The certified revenue figures "
                f"show a {pct}% increase at "
                f"{org} between {y1} and {y2}."
            )


        else:

            evidence = [
                (
                    f"The finance register gives "
                    f"{base} million credits as "
                    f"{org}'s revenue for {y0}."
                ),

                (
                    f"The corresponding {y1} "
                    f"entry is {new} million "
                    f"credits."
                ),

                distractor(
                    split,
                    index,
                ),
            ]


            s_claim = (
                f"{org}'s later revenue total "
                f"was exactly {pct}% above its "
                f"earlier total from {y0}."
            )

            r_claim = (
                f"{org}'s later revenue total "
                f"was exactly {wrong_pct}% "
                f"above its earlier total "
                f"from {y0}."
            )

            n_claim = (
                f"{org}'s revenue in {y2} "
                f"was exactly {pct}% above "
                f"its {y1} total."
            )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"revenue_{split}_v{style}"
        ),

        "reasoning": (
            f"percent_change="
            f"({new}-{base})/{base}*100="
            f"{pct}%"
        ),
    }


def generate_sum(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "two_component_sum",
        index,
    )

    org = entity_name(
        split,
        "two_component_sum",
        index,
    )


    quarter = (
        index % 4
    ) + 1

    year = (
        2023
        + (
            index % 3
        )
    )


    north = (
        82
        + (
            index * 13
        )
        % 111
    )

    south = (
        74
        + (
            index * 17
        )
        % 109
    )

    total = (
        north + south
    )


    wrong_total = (
        total
        + (
            7
            if index % 2 == 0
            else -9
        )
    )


    nq, ny = next_quarter(
        quarter,
        year,
    )

    next_north = (
        north + 12
    )


    evidence = [
        (
            f"{org}'s North site recorded "
            f"{north} units in Q{quarter} "
            f"{year}."
        ),

        (
            f"The South site recorded "
            f"{south} units in the same "
            f"quarter."
        ),

        (
            f"For Q{nq} {ny}, only the North "
            f"site figure is available: "
            f"{next_north} units."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        style = index % 2

        if style == 0:

            s_claim = (
                f"In Q{quarter} {year}, "
                f"{org}'s North and South sites "
                f"produced {total} units "
                f"altogether."
            )

            r_claim = (
                f"In Q{quarter} {year}, "
                f"{org}'s North and South sites "
                f"produced {wrong_total} units "
                f"altogether."
            )

            n_claim = (
                f"In Q{nq} {ny}, "
                f"{org}'s North and South sites "
                f"produced {next_north + south} "
                f"units altogether."
            )

        else:

            s_claim = (
                f"Adding the two site totals "
                f"for quarter {quarter} of "
                f"{year} gives {total} units "
                f"at {org}."
            )

            r_claim = (
                f"Adding the two site totals "
                f"for quarter {quarter} of "
                f"{year} gives {wrong_total} "
                f"units at {org}."
            )

            n_claim = (
                f"Adding the two site totals "
                f"for quarter {nq} of {ny} "
                f"gives {next_north + south} "
                f"units at {org}."
            )

    else:

        style = index % 2

        if style == 0:

            s_claim = (
                f"The combined output of both "
                f"recorded facilities at {org} "
                f"during Q{quarter} {year} "
                f"was {total} units."
            )

            r_claim = (
                f"The combined output of both "
                f"recorded facilities at {org} "
                f"during Q{quarter} {year} "
                f"was {wrong_total} units."
            )

            n_claim = (
                f"The combined output of both "
                f"recorded facilities at {org} "
                f"during Q{nq} {ny} was "
                f"{next_north + south} units."
            )

        else:

            s_claim = (
                f"Together, the North and "
                f"South entries for "
                f"Q{quarter} {year} at {org} "
                f"sum to {total} units."
            )

            r_claim = (
                f"Together, the North and "
                f"South entries for "
                f"Q{quarter} {year} at {org} "
                f"sum to {wrong_total} units."
            )

            n_claim = (
                f"Together, the North and "
                f"South entries for "
                f"Q{nq} {ny} at {org} sum "
                f"to {next_north + south} units."
            )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"sum_{split}_v{style}"
        ),

        "reasoning": (
            f"{north}+{south}={total}"
        ),
    }


def generate_project(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "project_location_mapping",
        index,
    )

    org = entity_name(
        split,
        "project_location_mapping",
        index,
    )


    projects = (
        TRAIN_PROJECTS
        if split == "train"
        else HOLDOUT_PROJECTS
    )

    locations = (
        TRAIN_LOCATIONS
        if split == "train"
        else HOLDOUT_LOCATIONS
    )


    project = projects[
        index
        % len(
            projects
        )
    ]

    other_project = projects[
        (
            index + 1
        )
        % len(
            projects
        )
    ]


    location = locations[
        index
        % len(
            locations
        )
    ]

    wrong_location = locations[
        (
            index + 2
        )
        % len(
            locations
        )
    ]


    year = (
        2024
        + (
            index % 2
        )
    )


    evidence = [
        (
            f"The certified {year} project "
            f"register for {org} assigns "
            f"{project} exactly one operating "
            f"region: {location}."
        ),

        (
            f"{other_project} is separately "
            f"listed for {wrong_location} "
            f"in {year}."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        s_claim = (
            f"In {year}, {org} carried out "
            f"{project} in {location}."
        )

        r_claim = (
            f"In {year}, {org} carried out "
            f"{project} in {wrong_location}."
        )

        n_claim = (
            f"In {year + 1}, {org} carried "
            f"out {project} in {location}."
        )

    else:

        s_claim = (
            f"The operating region recorded "
            f"for {org}'s {project} in "
            f"{year} was {location}."
        )

        r_claim = (
            f"The operating region recorded "
            f"for {org}'s {project} in "
            f"{year} was {wrong_location}."
        )

        n_claim = (
            f"The operating region recorded "
            f"for {org}'s {project} in "
            f"{year + 1} was {location}."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"project_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"{project}|{year}|"
            f"location={location}"
        ),
    }


def generate_range(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "range_or_bound",
        index,
    )

    org = entity_name(
        split,
        "range_or_bound",
        index,
    )


    reading = (
        525
        + 25
        * (
            (
                index * 3
                + (
                    1
                    if split == "holdout"
                    else 0
                )
            )
            % 18
        )
    )


    day = (
        10
        + (
            index % 8
        )
    )

    month = (
        "August"
        if index % 2 == 0
        else "September"
    )

    year = (
        2025
    )

    date_text = (
        f"{day} {month} {year}"
    )

    missing_date = (
        f"{day + 1} {month} {year}"
    )


    mode = index % 3


    if mode == 0:

        good_bound = (
            reading - 25
        )

        bad_bound = (
            reading + 25
        )

        if split == "train":

            s_claim = (
                f"On {date_text}, storage at "
                f"{org} was at least "
                f"{good_bound} megalitres."
            )

            r_claim = (
                f"On {date_text}, storage at "
                f"{org} was at least "
                f"{bad_bound} megalitres."
            )

            n_claim = (
                f"On {missing_date}, storage "
                f"at {org} was at least "
                f"{good_bound} megalitres."
            )

        else:

            s_claim = (
                f"The {date_text} reading at "
                f"{org} did not fall below "
                f"{good_bound} megalitres."
            )

            r_claim = (
                f"The {date_text} reading at "
                f"{org} did not fall below "
                f"{bad_bound} megalitres."
            )

            n_claim = (
                f"The {missing_date} reading "
                f"at {org} did not fall below "
                f"{good_bound} megalitres."
            )


    elif mode == 1:

        good_bound = (
            reading + 25
        )

        bad_bound = (
            reading - 25
        )

        if split == "train":

            s_claim = (
                f"On {date_text}, storage at "
                f"{org} was no more than "
                f"{good_bound} megalitres."
            )

            r_claim = (
                f"On {date_text}, storage at "
                f"{org} was no more than "
                f"{bad_bound} megalitres."
            )

            n_claim = (
                f"On {missing_date}, storage "
                f"at {org} was no more than "
                f"{good_bound} megalitres."
            )

        else:

            s_claim = (
                f"The certified {date_text} "
                f"storage figure at {org} did "
                f"not exceed {good_bound} "
                f"megalitres."
            )

            r_claim = (
                f"The certified {date_text} "
                f"storage figure at {org} did "
                f"not exceed {bad_bound} "
                f"megalitres."
            )

            n_claim = (
                f"The certified {missing_date} "
                f"storage figure at {org} did "
                f"not exceed {good_bound} "
                f"megalitres."
            )


    else:

        good_bound = (
            reading - 25
        )

        bad_bound = (
            reading
        )

        if split == "train":

            s_claim = (
                f"Storage at {org} exceeded "
                f"{good_bound} megalitres on "
                f"{date_text}."
            )

            r_claim = (
                f"Storage at {org} exceeded "
                f"{bad_bound} megalitres on "
                f"{date_text}."
            )

            n_claim = (
                f"Storage at {org} exceeded "
                f"{good_bound} megalitres on "
                f"{missing_date}."
            )

        else:

            s_claim = (
                f"The {date_text} gauge value "
                f"for {org} was greater than "
                f"{good_bound} megalitres."
            )

            r_claim = (
                f"The {date_text} gauge value "
                f"for {org} was greater than "
                f"{bad_bound} megalitres."
            )

            n_claim = (
                f"The {missing_date} gauge "
                f"value for {org} was greater "
                f"than {good_bound} megalitres."
            )


    evidence = [
        (
            f"The certified gauge reading at "
            f"{org} on {date_text} was exactly "
            f"{reading} megalitres."
        ),

        (
            "The instrument had completed its "
            "scheduled calibration before "
            "this reading."
        ),

        distractor(
            split,
            index,
        ),
    ]


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"range_{split}_v{mode}"
        ),

        "reasoning": (
            f"exact_reading={reading};"
            f"mode={mode}"
        ),
    }


def generate_manifest(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "complete_manifest",
        index,
    )

    org = entity_name(
        split,
        "complete_manifest",
        index,
    )


    items = [
        "thermal camera",
        "radar scanner",
        "radio beacon",
        "spectral sensor",
        "navigation transponder",
        "sample drill",
        "laser altimeter",
    ]


    a = items[
        index
        % len(
            items
        )
    ]

    b = items[
        (
            index + 1
        )
        % len(
            items
        )
    ]

    c = items[
        (
            index + 2
        )
        % len(
            items
        )
    ]

    omitted = items[
        (
            index + 4
        )
        % len(
            items
        )
    ]


    year = (
        2024
        + (
            index % 2
        )
    )


    evidence = [
        (
            f"The complete and exhaustive "
            f"{year} mission manifest for "
            f"{org} lists: {a}, {b}, and {c}."
        ),

        (
            "The register explicitly states "
            "that no additional payload items "
            "were carried."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        s_claim = (
            f"{org}'s {year} mission left "
            f"out the {omitted}."
        )

        r_claim = (
            f"{org}'s {year} mission left "
            f"out the {a}."
        )

        n_claim = (
            f"{org}'s {year - 1} mission "
            f"left out the {omitted}."
        )

    else:

        s_claim = (
            f"The {omitted} was absent from "
            f"{org}'s complete {year} payload."
        )

        r_claim = (
            f"The {a} was absent from "
            f"{org}'s complete {year} payload."
        )

        n_claim = (
            f"The {omitted} was absent from "
            f"{org}'s complete {year - 1} "
            f"payload."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"manifest_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"complete_manifest="
            f"{a}|{b}|{c};"
            f"omitted={omitted}"
        ),
    }


def generate_percentage_threshold(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "percentage_threshold",
        index,
    )

    org = entity_name(
        split,
        "percentage_threshold",
        index,
    )


    actual_pct_choices = [
        55,
        60,
        65,
        70,
        75,
        80,
    ]


    actual_pct = actual_pct_choices[
        index
        % len(
            actual_pct_choices
        )
    ]


    total = 200

    selected = (
        actual_pct * 2
    )


    lower = (
        actual_pct - 5
    )

    upper = (
        actual_pct + 5
    )


    year = (
        2019
        + (
            index % 5
        )
    )


    evidence = [
        (
            f"{org} recorded {total} graduates "
            f"in {year}."
        ),

        (
            f"Of those graduates, {selected} "
            f"entered a four-year college."
        ),

        (
            f"The following year's graduating "
            f"class contained {total + 20} "
            f"students, but destination data "
            f"were not yet published."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        s_claim = (
            f"More than {lower}% of {org}'s "
            f"{year} graduates entered a "
            f"four-year college."
        )

        r_claim = (
            f"More than {upper}% of {org}'s "
            f"{year} graduates entered a "
            f"four-year college."
        )

        n_claim = (
            f"More than {lower}% of {org}'s "
            f"{year + 1} graduates entered a "
            f"four-year college."
        )

    else:

        s_claim = (
            f"The share of {org}'s {year} "
            f"graduates entering four-year "
            f"college exceeded {lower}%."
        )

        r_claim = (
            f"The share of {org}'s {year} "
            f"graduates entering four-year "
            f"college exceeded {upper}%."
        )

        n_claim = (
            f"The share of {org}'s {year + 1} "
            f"graduates entering four-year "
            f"college exceeded {lower}%."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"pct_threshold_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"{selected}/{total}="
            f"{actual_pct}%"
        ),
    }


def generate_duration(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "duration_threshold",
        index,
    )

    org = entity_name(
        split,
        "duration_threshold",
        index,
    )


    start_year = (
        2014
        + (
            index % 5
        )
    )


    full_years = (
        3
        + (
            index % 3
        )
    )


    launch = (
        f"12 February {start_year}"
    )

    arrival = (
        f"20 September "
        f"{start_year + full_years}"
    )


    evidence = [
        (
            f"{org} launched on {launch}."
        ),

        (
            f"It began orbital operations on "
            f"{arrival}."
        ),

        (
            "The mission log does not provide "
            "a date for its final calibration."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        s_claim = (
            f"{org} began orbital operations "
            f"more than {full_years} years "
            f"after launch."
        )

        r_claim = (
            f"{org} began orbital operations "
            f"more than {full_years + 1} years "
            f"after launch."
        )

        n_claim = (
            f"{org} began orbital operations "
            f"more than two years after its "
            f"final calibration."
        )

    else:

        s_claim = (
            f"The interval from launch to "
            f"orbital operations for {org} "
            f"exceeded {full_years} years."
        )

        r_claim = (
            f"The interval from launch to "
            f"orbital operations for {org} "
            f"exceeded {full_years + 1} years."
        )

        n_claim = (
            f"The interval from final "
            f"calibration to orbital operations "
            f"for {org} exceeded two years."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"duration_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"launch={launch};"
            f"arrival={arrival};"
            f"greater_than={full_years}y"
        ),
    }


def generate_depot(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "depot_comparison",
        index,
    )

    org = entity_name(
        split,
        "depot_comparison",
        index,
    )


    year = (
        2023
        + (
            index % 3
        )
    )


    east = (
        430
        + (
            index * 31
        )
        % 160
    )

    west = (
        440
        + (
            index * 47
        )
        % 160
    )


    if east == west:

        west += 11


    next_east = (
        east + 17
    )


    evidence = [
        (
            f"The {year} East-depot ledger "
            f"for {org} records {east} parcels."
        ),

        (
            f"The {year} West-depot ledger "
            f"records {west} parcels."
        ),

        (
            f"For {year + 1}, only the East "
            f"depot total is available: "
            f"{next_east} parcels."
        ),

        distractor(
            split,
            index,
        ),
    ]


    east_higher = (
        east > west
    )


    if split == "train":

        if east_higher:

            s_claim = (
                f"In {year}, the East depot "
                f"at {org} handled more parcels "
                f"than the West depot."
            )

            r_claim = (
                f"In {year}, the West depot "
                f"at {org} handled more parcels "
                f"than the East depot."
            )

        else:

            s_claim = (
                f"In {year}, the West depot "
                f"at {org} handled more parcels "
                f"than the East depot."
            )

            r_claim = (
                f"In {year}, the East depot "
                f"at {org} handled more parcels "
                f"than the West depot."
            )


        n_claim = (
            f"In {year + 1}, the East depot "
            f"at {org} handled more parcels "
            f"than the West depot."
        )


    else:

        if east_higher:

            s_claim = (
                f"The busier depot at {org} "
                f"in {year} was East rather "
                f"than West."
            )

            r_claim = (
                f"The busier depot at {org} "
                f"in {year} was West rather "
                f"than East."
            )

        else:

            s_claim = (
                f"The busier depot at {org} "
                f"in {year} was West rather "
                f"than East."
            )

            r_claim = (
                f"The busier depot at {org} "
                f"in {year} was East rather "
                f"than West."
            )


        n_claim = (
            f"The busier depot at {org} "
            f"in {year + 1} was East rather "
            f"than West."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"depot_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"east={east};west={west}"
        ),
    }


def generate_temporal(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "temporal_transition",
        index,
    )

    org = entity_name(
        split,
        "temporal_transition",
        index,
    )


    people = (
        TRAIN_PEOPLE
        if split == "train"
        else HOLDOUT_PEOPLE
    )


    old_person = people[
        (
            index * 2
        )
        % len(
            people
        )
    ]

    new_person = people[
        (
            index * 2
            + 1
        )
        % len(
            people
        )
    ]


    month = (
        "May"
        if index % 2 == 0
        else "March"
    )


    year = 2025


    evidence = [
        (
            f"{old_person}'s term as director "
            f"of {org} ended when "
            f"{new_person}'s appointment "
            f"took effect."
        ),

        (
            f"The transition resolution places "
            f"that effective date somewhere "
            f"from 21 to 23 {month} {year}, "
            f"inclusive."
        ),

        distractor(
            split,
            index,
        ),
    ]


    if split == "train":

        s_claim = (
            f"{old_person} was director of "
            f"{org} on 18 {month} {year}."
        )

        r_claim = (
            f"{new_person} was director of "
            f"{org} on 18 {month} {year}."
        )

        n_claim = (
            f"{old_person} was director of "
            f"{org} on 22 {month} {year}."
        )

    else:

        s_claim = (
            f"On 18 {month} {year}, the "
            f"office-holder directing {org} "
            f"was {old_person}."
        )

        r_claim = (
            f"On 18 {month} {year}, the "
            f"office-holder directing {org} "
            f"was {new_person}."
        )

        n_claim = (
            f"On 22 {month} {year}, the "
            f"office-holder directing {org} "
            f"was {old_person}."
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"temporal_{split}_v"
            f"{index % 2}"
        ),

        "reasoning": (
            f"transition_window="
            f"21-23 {month} {year}"
        ),
    }


def generate_transfer(
    split,
    index,
):

    rng = scenario_rng(
        split,
        "derived_comparison_transfer",
        index,
    )

    org = entity_name(
        split,
        "derived_comparison_transfer",
        index,
    )


    year = (
        2024
        + (
            index % 2
        )
    )


    if index % 2 == 0:

        # Difference reasoning.

        b = (
            90
            + 10
            * index
        )

        difference = (
            35
            + 5
            * (
                index % 3
            )
        )

        a = (
            b + difference
        )


        evidence = [
            (
                f"Unit A at {org} recorded "
                f"{a} completed cases in "
                f"{year}."
            ),

            (
                f"Unit B recorded {b} completed "
                f"cases in the same year."
            ),

            (
                f"For {year + 1}, only Unit A's "
                f"figure has been released."
            ),

            distractor(
                split,
                index,
            ),
        ]


        if split == "train":

            s_claim = (
                f"In {year}, Unit A at {org} "
                f"completed exactly {difference} "
                f"more cases than Unit B."
            )

            r_claim = (
                f"In {year}, Unit A at {org} "
                f"completed exactly "
                f"{difference + 10} more cases "
                f"than Unit B."
            )

            n_claim = (
                f"In {year + 1}, Unit A at "
                f"{org} completed exactly "
                f"{difference} more cases than "
                f"Unit B."
            )

        else:

            s_claim = (
                f"The {year} gap between "
                f"Unit A and Unit B at {org} "
                f"was {difference} completed "
                f"cases."
            )

            r_claim = (
                f"The {year} gap between "
                f"Unit A and Unit B at {org} "
                f"was {difference + 10} "
                f"completed cases."
            )

            n_claim = (
                f"The {year + 1} gap between "
                f"Unit A and Unit B at {org} "
                f"was {difference} completed "
                f"cases."
            )


        reasoning = (
            f"difference={a}-{b}="
            f"{difference}"
        )


    else:

        # Ratio reasoning.

        b = (
            40
            + 5
            * index
        )

        ratio = (
            2
            + (
                index % 2
            )
        )

        a = (
            b * ratio
        )


        evidence = [
            (
                f"Section A at {org} processed "
                f"{a} records in {year}."
            ),

            (
                f"Section B processed {b} "
                f"records in {year}."
            ),

            (
                f"For {year + 1}, the published "
                f"table contains only Section "
                f"A's value."
            ),

            distractor(
                split,
                index,
            ),
        ]


        if split == "train":

            s_claim = (
                f"In {year}, Section A at "
                f"{org} processed exactly "
                f"{ratio} times as many records "
                f"as Section B."
            )

            r_claim = (
                f"In {year}, Section A at "
                f"{org} processed exactly "
                f"{ratio + 1} times as many "
                f"records as Section B."
            )

            n_claim = (
                f"In {year + 1}, Section A at "
                f"{org} processed exactly "
                f"{ratio} times as many records "
                f"as Section B."
            )

        else:

            s_claim = (
                f"Section A's {year} total at "
                f"{org} was precisely "
                f"{ratio}× Section B's total."
            )

            r_claim = (
                f"Section A's {year} total at "
                f"{org} was precisely "
                f"{ratio + 1}× Section B's "
                f"total."
            )

            n_claim = (
                f"Section A's {year + 1} total "
                f"at {org} was precisely "
                f"{ratio}× Section B's total."
            )


        reasoning = (
            f"ratio={a}/{b}="
            f"{ratio}"
        )


    return {
        "entity": org,

        "evidence": shuffled(
            evidence,
            rng,
        ),

        "claims": {
            "SUPPORTS": s_claim,
            "REFUTES": r_claim,
            "NOT_ENOUGH_INFO": n_claim,
        },

        "template_id": (
            f"transfer_{split}_"
            f"{'difference' if index % 2 == 0 else 'ratio'}"
        ),

        "reasoning": reasoning,
    }


GENERATORS = {
    "revenue_percentage": (
        generate_revenue
    ),

    "two_component_sum": (
        generate_sum
    ),

    "project_location_mapping": (
        generate_project
    ),

    "range_or_bound": (
        generate_range
    ),

    "complete_manifest": (
        generate_manifest
    ),

    "percentage_threshold": (
        generate_percentage_threshold
    ),

    "duration_threshold": (
        generate_duration
    ),

    "depot_comparison": (
        generate_depot
    ),

    "temporal_transition": (
        generate_temporal
    ),

    "derived_comparison_transfer": (
        generate_transfer
    ),
}


# ============================================================
# 7. BUILD A SPLIT
# ============================================================

def build_split(
    split,
    allocation,
):

    rows = []

    metadata = []


    label_codes = {
        "SUPPORTS": "S",
        "REFUTES": "R",
        "NOT_ENOUGH_INFO": "N",
    }


    for family, count in (
        allocation.items()
    ):

        generator = (
            GENERATORS[
                family
            ]
        )


        for scenario_index in range(
            count
        ):

            scenario = generator(
                split,
                scenario_index,
            )


            scenario_id = (
                f"d4v1_{split}_"
                f"{family}_"
                f"{scenario_index:03d}"
            )


            evidence = (
                scenario[
                    "evidence"
                ]
            )


            for label in LABELS:

                example_id = (
                    f"{scenario_id}_"
                    f"{label_codes[label]}"
                )


                claim = (
                    scenario[
                        "claims"
                    ][
                        label
                    ]
                )


                rows.append({
                    "id": (
                        example_id
                    ),

                    "claim": (
                        claim
                    ),

                    "evidence": (
                        evidence
                    ),

                    "label": (
                        label
                    ),
                })


                metadata.append({
                    "id": (
                        example_id
                    ),

                    "split": (
                        split
                    ),

                    "scenario_id": (
                        scenario_id
                    ),

                    "family": (
                        family
                    ),

                    "template_id": (
                        scenario[
                            "template_id"
                        ]
                    ),

                    "entity": (
                        scenario[
                            "entity"
                        ]
                    ),

                    "label": (
                        label
                    ),

                    "reasoning": (
                        scenario[
                            "reasoning"
                        ]
                    ),

                    "claim": (
                        claim
                    ),

                    "evidence": json.dumps(
                        evidence,
                        ensure_ascii=False,
                    ),
                })


    return (
        rows,
        metadata,
    )


train_rows, train_meta = (
    build_split(
        "train",
        TRAIN_SCENARIOS,
    )
)


holdout_rows, holdout_meta = (
    build_split(
        "holdout",
        HOLDOUT_SCENARIOS,
    )
)


assert len(
    train_rows
) == 150

assert len(
    holdout_rows
) == 75


# ============================================================
# 8. STRUCTURAL VALIDATION
# ============================================================

print("\n" + "=" * 78)
print("STRUCTURAL VALIDATION")
print("=" * 78)


def validate_triplets(
    metadata,
    expected_scenarios,
):

    by_scenario = defaultdict(
        list
    )


    for row in metadata:

        by_scenario[
            row[
                "scenario_id"
            ]
        ].append(
            row
        )


    assert len(
        by_scenario
    ) == expected_scenarios


    for scenario_id, rows in (
        by_scenario.items()
    ):

        assert len(
            rows
        ) == 3


        labels = {
            row["label"]
            for row in rows
        }


        assert labels == set(
            LABELS
        )


        # Same evidence across the trio.

        evidence_values = {
            row["evidence"]
            for row in rows
        }


        assert len(
            evidence_values
        ) == 1


validate_triplets(
    train_meta,
    50,
)

validate_triplets(
    holdout_meta,
    25,
)


train_label_counts = Counter(
    row["label"]
    for row in train_rows
)

holdout_label_counts = Counter(
    row["label"]
    for row in holdout_rows
)


assert train_label_counts == Counter({
    "SUPPORTS": 50,
    "REFUTES": 50,
    "NOT_ENOUGH_INFO": 50,
})


assert holdout_label_counts == Counter({
    "SUPPORTS": 25,
    "REFUTES": 25,
    "NOT_ENOUGH_INFO": 25,
})


print(
    "Train rows:",
    len(train_rows),
)

print(
    "Train labels:",
    train_label_counts,
)

print(
    "Holdout rows:",
    len(holdout_rows),
)

print(
    "Holdout labels:",
    holdout_label_counts,
)


# ============================================================
# 9. FAMILY BALANCE
# ============================================================

train_family_counts = Counter(
    row["family"]
    for row in train_meta
)

holdout_family_counts = Counter(
    row["family"]
    for row in holdout_meta
)


print("\n" + "=" * 78)
print("FAMILY DISTRIBUTION")
print("=" * 78)


print(
    f"{'Family':30} | "
    f"{'Train':>5} | "
    f"{'Holdout':>7}"
)

print(
    "-" * 52
)


for family in (
    TRAIN_SCENARIOS
):

    print(
        f"{family:30} | "
        f"{train_family_counts[family]:5d} | "
        f"{holdout_family_counts[family]:7d}"
    )


# ============================================================
# 10. LEAKAGE / OVERLAP CHECKS
# ============================================================

print("\n" + "=" * 78)
print("LEAKAGE / OVERLAP CHECKS")
print("=" * 78)


train_ids = {
    row["id"]
    for row in train_rows
}

holdout_ids = {
    row["id"]
    for row in holdout_rows
}


assert len(
    train_ids
) == len(
    train_rows
)

assert len(
    holdout_ids
) == len(
    holdout_rows
)

assert train_ids.isdisjoint(
    holdout_ids
)


train_entities = {
    row["entity"]
    for row in train_meta
}

holdout_entities = {
    row["entity"]
    for row in holdout_meta
}


assert train_entities.isdisjoint(
    holdout_entities
)


train_templates = {
    row["template_id"]
    for row in train_meta
}

holdout_templates = {
    row["template_id"]
    for row in holdout_meta
}


assert train_templates.isdisjoint(
    holdout_templates
)


train_claims = {
    norm_text(
        row["claim"]
    )
    for row in train_rows
}

holdout_claims = {
    norm_text(
        row["claim"]
    )
    for row in holdout_rows
}


assert train_claims.isdisjoint(
    holdout_claims
)


# No exact claim match to any known real/diagnostic example.

train_existing_claim_overlap = (
    train_claims
    & existing_claims
)

holdout_existing_claim_overlap = (
    holdout_claims
    & existing_claims
)


assert not train_existing_claim_overlap
assert not holdout_existing_claim_overlap


train_fps = {
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in train_rows
}

holdout_fps = {
    fingerprint(
        row["claim"],
        row["evidence"],
    )
    for row in holdout_rows
}


assert len(
    train_fps
) == len(
    train_rows
)

assert len(
    holdout_fps
) == len(
    holdout_rows
)


assert train_fps.isdisjoint(
    holdout_fps
)

assert not (
    train_fps
    & existing_fingerprints
)

assert not (
    holdout_fps
    & existing_fingerprints
)


# Synthetic entities must not already occur in known corpus.

entity_leaks = []


for entity in sorted(
    train_entities
    | holdout_entities
):

    if norm_text(
        entity
    ) in existing_text:

        entity_leaks.append(
            entity
        )


assert not entity_leaks, (
    entity_leaks
)


print(
    "Train/Holdout ID overlap:",
    0,
)

print(
    "Train/Holdout entity overlap:",
    0,
)

print(
    "Train/Holdout template overlap:",
    0,
)

print(
    "Train/Holdout exact claim overlap:",
    0,
)

print(
    "Synthetic vs existing exact claim overlap:",
    0,
)

print(
    "Synthetic vs existing claim+evidence overlap:",
    0,
)

print(
    "Synthetic entity leakage into existing corpus:",
    0,
)


# ============================================================
# 11. WRITE FILES
# ============================================================

write_jsonl(
    TRAIN_OUT,
    train_rows,
)

write_jsonl(
    HOLDOUT_OUT,
    holdout_rows,
)


pd.DataFrame(
    train_meta
).to_csv(
    TRAIN_META_OUT,
    index=False,
)


pd.DataFrame(
    holdout_meta
).to_csv(
    HOLDOUT_META_OUT,
    index=False,
)


train_sha = sha256_file(
    TRAIN_OUT
)

holdout_sha = sha256_file(
    HOLDOUT_OUT
)


# ============================================================
# 12. FREEZE MANIFEST
# ============================================================

manifest = {
    "name": (
        "D4_CONTRASTIVE_CURRICULUM_V1"
    ),

    "seed": (
        SEED
    ),

    "purpose": (
        "Failure-informed but validation-example-independent "
        "contrastive augmentation for D1."
    ),

    "design": {
        "contrastive_trios": True,

        "same_evidence_three_labels": True,

        "train_holdout_entity_disjoint": True,

        "train_holdout_template_disjoint": True,

        "no_exact_overlap_with_existing_data": True,

        "holdout_must_never_be_used_for_training": True,
    },

    "train": {
        "path": str(
            TRAIN_OUT
        ),

        "sha256": (
            train_sha
        ),

        "rows": (
            len(
                train_rows
            )
        ),

        "labels": dict(
            train_label_counts
        ),

        "scenario_allocation": (
            TRAIN_SCENARIOS
        ),
    },

    "frozen_holdout": {
        "path": str(
            HOLDOUT_OUT
        ),

        "sha256": (
            holdout_sha
        ),

        "rows": (
            len(
                holdout_rows
            )
        ),

        "labels": dict(
            holdout_label_counts
        ),

        "scenario_allocation": (
            HOLDOUT_SCENARIOS
        ),

        "status": (
            "FROZEN_BEFORE_D4_TRAINING"
        ),
    },

    "known_D1_failure_signal_used_only_for_family_weighting": {
        "revenue_percentage_errors": 33,
        "two_component_sum_errors": 19,
        "project_location_mapping_errors": 10,
        "range_or_bound_errors": 6,
        "complete_manifest_errors": 4,
        "percentage_threshold_errors": 2,
        "depot_comparison_errors": 2,
        "duration_threshold_errors": 1,
        "temporal_transition_errors": 1,
    },

    "important_rule": (
        "No known validation/dev/external example was copied "
        "or edited into either synthetic split."
    ),
}


with open(
    MANIFEST_OUT,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. COMPACT HUMAN REVIEW
# ============================================================

print("\n" + "=" * 78)
print("ONE TRAIN SCENARIO PER FAMILY")
print("=" * 78)


train_meta_df = pd.DataFrame(
    train_meta
)


for family in (
    TRAIN_SCENARIOS
):

    subset = train_meta_df[
        train_meta_df[
            "family"
        ]
        == family
    ]


    scenario_id = (
        subset.iloc[0][
            "scenario_id"
        ]
    )


    trio = subset[
        subset[
            "scenario_id"
        ]
        == scenario_id
    ]


    print("\n" + "-" * 78)

    print(
        family
    )

    print("-" * 78)


    evidence = json.loads(
        trio.iloc[0][
            "evidence"
        ]
    )


    print(
        "Evidence:"
    )


    for i, passage in enumerate(
        evidence,
        1,
    ):

        print(
            f"  [{i}] {passage}"
        )


    print(
        "\nClaims:"
    )


    for label in LABELS:

        row = trio[
            trio[
                "label"
            ]
            == label
        ].iloc[0]


        print(
            f"  {label:16} "
            f"{row['claim']}"
        )


# ============================================================
# 14. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 78)
print("CELL 2 COMPLETE")
print("=" * 78)


print(
    "TRAIN curriculum:",
    TRAIN_OUT,
)

print(
    "TRAIN SHA256:",
    train_sha,
)

print(
    "TRAIN rows:",
    len(train_rows),
)


print(
    "\nFROZEN holdout:",
    HOLDOUT_OUT,
)

print(
    "FROZEN HOLDOUT SHA256:",
    holdout_sha,
)

print(
    "FROZEN HOLDOUT rows:",
    len(holdout_rows),
)


print(
    "\nManifest:",
    MANIFEST_OUT,
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT train on "
    "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)


print(
    "\nNO MODEL was loaded."
)

print(
    "NO inference occurred."
)

print(
    "NO training occurred."
)

print(
    "\nNEXT: send me the complete Cell-2 output. "
    "Cell 3 will independently audit the synthetic semantics "
    "and freeze the datasets before any D4 model training."
)

In [ ]:
# ============================================================
# CELL 3 — INDEPENDENT SEMANTIC AUDIT + FREEZE D4 DATA
#
# PURPOSE
# -------
# Independently recompute the correct label from the actual
# CLAIM + EVIDENCE for every D4 synthetic example.
#
# We DO NOT trust the generated label during semantic checking.
#
# Audited:
#   Training curriculum: 150
#   Frozen holdout:       75
#   Total:               225
#
# REQUIREMENT BEFORE D4 TRAINING:
#   225/225 semantic agreement
#   0 parser failures
#   0 structural failures
#
# ALSO:
#   - verify hashes from Cell 2
#   - verify contrastive trios
#   - verify train/holdout separation again
#   - create final audited manifest
#   - create an audited backup ZIP
#
# NO MODEL LOAD
# NO GPU
# NO INFERENCE
# NO TRAINING
# ============================================================

import re
import json
import math
import shutil
import hashlib
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from fractions import Fraction

import pandas as pd


# ============================================================
# 0. PATHS + EXPECTED HASHES FROM CELL 2
# ============================================================

WORK = Path("/kaggle/working")

D4_DIR = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

TRAIN_PATH = (
    D4_DIR
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT_PATH = (
    D4_DIR
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)

TRAIN_META_PATH = (
    D4_DIR
    / "d4_contrastive_train_v1_metadata.csv"
)

HOLDOUT_META_PATH = (
    D4_DIR
    / "D4_FROZEN_HOLDOUT_V1_metadata.csv"
)

ORIGINAL_MANIFEST = (
    D4_DIR
    / "d4_curriculum_v1_manifest.json"
)


EXPECTED_TRAIN_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)


AUDIT_DIR = (
    D4_DIR
    / "semantic_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


AUDIT_CSV = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_SEMANTIC_AUDIT.csv"
)

AUDIT_ERRORS_CSV = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_AUDIT_FAILURES.csv"
)

AUDIT_SUMMARY = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_AUDIT_SUMMARY.json"
)

FINAL_MANIFEST = (
    D4_DIR
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


BACKUP_DIR = (
    WORK
    / "D4_CURRICULUM_V1_AUDITED"
)

ZIP_BASE = (
    WORK
    / "D4_CURRICULUM_V1_AUDITED"
)


LABELS = {
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
}


FAMILIES = [
    "revenue_percentage",
    "two_component_sum",
    "project_location_mapping",
    "range_or_bound",
    "complete_manifest",
    "percentage_threshold",
    "duration_threshold",
    "depot_comparison",
    "temporal_transition",
    "derived_comparison_transfer",
]


MONTHS = (
    "January|February|March|April|May|June|"
    "July|August|September|October|November|December"
)


# ============================================================
# 1. BASIC HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def norm(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def infer_family(example_id):

    for family in FAMILIES:

        token = (
            "_"
            + family
            + "_"
        )

        if token in example_id:

            return family

    raise ValueError(
        f"Cannot infer family from ID: "
        f"{example_id}"
    )


def result_from_truth(
    truth,
):

    return (
        "SUPPORTS"
        if truth
        else "REFUTES"
    )


# ============================================================
# 2. VERIFY CELL-2 ARTIFACTS
# ============================================================

for path in [
    TRAIN_PATH,
    HOLDOUT_PATH,
    TRAIN_META_PATH,
    HOLDOUT_META_PATH,
    ORIGINAL_MANIFEST,
]:

    assert path.exists(), path


train_sha = sha256_file(
    TRAIN_PATH
)

holdout_sha = sha256_file(
    HOLDOUT_PATH
)


assert train_sha == EXPECTED_TRAIN_SHA, (
    "TRAIN dataset changed after Cell 2."
)

assert holdout_sha == EXPECTED_HOLDOUT_SHA, (
    "FROZEN HOLDOUT changed after Cell 2."
)


train_rows = read_jsonl(
    TRAIN_PATH
)

holdout_rows = read_jsonl(
    HOLDOUT_PATH
)


assert len(train_rows) == 150
assert len(holdout_rows) == 75


print("=" * 78)
print("CELL 3 — D4 INDEPENDENT SEMANTIC AUDIT")
print("=" * 78)


print(
    "Train SHA:",
    train_sha,
)

print(
    "Train rows:",
    len(train_rows),
)


print(
    "\nFrozen holdout SHA:",
    holdout_sha,
)

print(
    "Frozen holdout rows:",
    len(holdout_rows),
)


# ============================================================
# 3. REVENUE AUDITOR
# ============================================================

def audit_revenue(
    claim,
    evidence,
):

    year_to_value = {}


    for passage in evidence:

        years = re.findall(
            r"\b(20\d{2})\b",
            passage,
        )

        amounts = re.findall(
            r"\b(\d+)\s+million credits\b",
            passage,
            flags=re.I,
        )


        if (
            len(years) == 1
            and len(amounts) == 1
        ):

            year_to_value[
                int(years[0])
            ] = int(
                amounts[0]
            )


    if len(year_to_value) != 2:

        raise ValueError(
            f"Revenue parser expected "
            f"2 year/value pairs; got "
            f"{year_to_value}"
        )


    pct_match = re.search(
        r"(\d+)%",
        claim,
    )

    if not pct_match:

        raise ValueError(
            "Revenue claim lacks percentage."
        )


    claimed_pct = int(
        pct_match.group(1)
    )


    claim_years = [
        int(x)

        for x in re.findall(
            r"\b(20\d{2})\b",
            claim,
        )
    ]


    known_years = sorted(
        year_to_value
    )


    # --------------------------------------------------------
    # Usually claim names both comparison years.
    # --------------------------------------------------------

    if len(claim_years) >= 2:

        y0 = claim_years[-2]
        y1 = claim_years[-1]


        if (
            y0 not in year_to_value
            or y1 not in year_to_value
        ):

            return (
                "NOT_ENOUGH_INFO",
                (
                    f"Revenue values unavailable "
                    f"for {y0}->{y1}"
                ),
            )


    # --------------------------------------------------------
    # Holdout paraphrase:
    # "later total ... earlier total from YEAR"
    #
    # Only the earlier year is explicitly named.
    # --------------------------------------------------------

    elif len(claim_years) == 1:

        stated_year = (
            claim_years[0]
        )


        if (
            "later revenue total" in claim.lower()
            and stated_year == known_years[0]
        ):

            y0 = known_years[0]
            y1 = known_years[1]


        else:

            return (
                "NOT_ENOUGH_INFO",
                (
                    f"Only claim year "
                    f"{stated_year} available; "
                    f"comparison not established."
                ),
            )


    else:

        raise ValueError(
            "Revenue claim has no year."
        )


    old = year_to_value[
        y0
    ]

    new = year_to_value[
        y1
    ]


    actual_pct = (
        Fraction(
            new - old,
            old,
        )
        * 100
    )


    truth = (
        actual_pct
        == claimed_pct
    )


    return (
        result_from_truth(
            truth
        ),

        (
            f"({new}-{old})/{old}*100"
            f"={float(actual_pct):g}%; "
            f"claim={claimed_pct}%"
        ),
    )


# ============================================================
# 4. TWO-COMPONENT SUM AUDITOR
# ============================================================

def audit_sum(
    claim,
    evidence,
):

    north_value = None
    south_value = None

    known_q = None
    known_year = None


    for passage in evidence:

        m = re.search(
            r"North site recorded "
            r"(\d+) units in Q(\d) "
            r"(20\d{2})",
            passage,
            flags=re.I,
        )

        if m:

            north_value = int(
                m.group(1)
            )

            known_q = int(
                m.group(2)
            )

            known_year = int(
                m.group(3)
            )


        m = re.search(
            r"South site recorded "
            r"(\d+) units in the same "
            r"quarter",
            passage,
            flags=re.I,
        )

        if m:

            south_value = int(
                m.group(1)
            )


    if None in [
        north_value,
        south_value,
        known_q,
        known_year,
    ]:

        raise ValueError(
            "Failed to parse primary sum evidence."
        )


    q_match = re.search(
        r"\bQ(\d)\s+(20\d{2})\b",
        claim,
        flags=re.I,
    )


    if q_match:

        claim_q = int(
            q_match.group(1)
        )

        claim_year = int(
            q_match.group(2)
        )


    else:

        q_match = re.search(
            r"quarter\s+(\d)\s+of\s+(20\d{2})",
            claim,
            flags=re.I,
        )

        if not q_match:

            raise ValueError(
                f"Could not parse claim quarter: "
                f"{claim}"
            )

        claim_q = int(
            q_match.group(1)
        )

        claim_year = int(
            q_match.group(2)
        )


    total_match = re.search(
        r"\b(\d+)\s+units\b",
        claim,
        flags=re.I,
    )


    if not total_match:

        raise ValueError(
            "Could not parse claimed total."
        )


    claimed_total = int(
        total_match.group(1)
    )


    if (
        claim_q != known_q
        or claim_year != known_year
    ):

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Both site totals only known "
                f"for Q{known_q} {known_year}."
            ),
        )


    actual = (
        north_value
        + south_value
    )


    return (
        result_from_truth(
            actual
            == claimed_total
        ),

        (
            f"{north_value}+{south_value}"
            f"={actual}; "
            f"claim={claimed_total}"
        ),
    )


# ============================================================
# 5. PROJECT / LOCATION AUDITOR
# ============================================================

def audit_project(
    claim,
    evidence,
):

    mapped_project = None
    mapped_location = None
    mapped_year = None

    other_location = None


    for passage in evidence:

        m = re.search(
            r"certified\s+(20\d{2})\s+"
            r"project register .*?"
            r"assigns\s+"
            r"(Project .+?)\s+"
            r"exactly one operating region:\s+"
            r"(.+?)\.",
            passage,
            flags=re.I,
        )


        if m:

            mapped_year = int(
                m.group(1)
            )

            mapped_project = (
                m.group(2)
                .strip()
            )

            mapped_location = (
                m.group(3)
                .strip()
            )


        m2 = re.search(
            r"is separately listed for "
            r"(.+?) in (20\d{2})",
            passage,
            flags=re.I,
        )


        if m2:

            other_location = (
                m2.group(1)
                .strip()
            )


    if None in [
        mapped_project,
        mapped_location,
        mapped_year,
    ]:

        raise ValueError(
            "Could not parse project mapping."
        )


    claim_year_match = re.search(
        r"\b(20\d{2})\b",
        claim,
    )

    if not claim_year_match:

        raise ValueError(
            "Project claim has no year."
        )


    claim_year = int(
        claim_year_match.group(1)
    )


    if claim_year != mapped_year:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Mapping known for "
                f"{mapped_year}, not "
                f"{claim_year}."
            ),
        )


    if mapped_project.lower() not in (
        claim.lower()
    ):

        return (
            "NOT_ENOUGH_INFO",
            "Claimed project not mapped.",
        )


    if mapped_location.lower() in (
        claim.lower()
    ):

        return (
            "SUPPORTS",
            (
                f"{mapped_project} -> "
                f"{mapped_location}"
            ),
        )


    if (
        other_location is not None
        and other_location.lower()
        in claim.lower()
    ):

        return (
            "REFUTES",
            (
                f"{mapped_project} maps to "
                f"{mapped_location}, not "
                f"{other_location}."
            ),
        )


    raise ValueError(
        "Could not identify claimed location."
    )


# ============================================================
# 6. RANGE / BOUND AUDITOR
# ============================================================

def audit_range(
    claim,
    evidence,
):

    reading = None
    evidence_date = None


    for passage in evidence:

        m = re.search(
            rf"on\s+"
            rf"(\d{{1,2}}\s+(?:{MONTHS})\s+20\d{{2}})"
            rf"\s+was exactly\s+"
            rf"(\d+)\s+megalitres",
            passage,
            flags=re.I,
        )


        if m:

            evidence_date = (
                m.group(1)
                .lower()
            )

            reading = int(
                m.group(2)
            )


    if reading is None:

        raise ValueError(
            "Could not parse range evidence."
        )


    date_match = re.search(
        rf"\b(\d{{1,2}}\s+(?:{MONTHS})"
        rf"\s+20\d{{2}})\b",
        claim,
        flags=re.I,
    )


    bound_match = re.search(
        r"\b(\d+)\s+megalitres\b",
        claim,
        flags=re.I,
    )


    if (
        not date_match
        or not bound_match
    ):

        raise ValueError(
            "Could not parse range claim."
        )


    claim_date = (
        date_match.group(1)
        .lower()
    )

    bound = int(
        bound_match.group(1)
    )


    if claim_date != evidence_date:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"No exact reading for "
                f"{claim_date}."
            ),
        )


    c = claim.lower()


    if (
        "at least" in c
        or "did not fall below" in c
    ):

        truth = (
            reading >= bound
        )

        op = ">="


    elif (
        "no more than" in c
        or "did not exceed" in c
    ):

        truth = (
            reading <= bound
        )

        op = "<="


    elif (
        "exceeded" in c
        or "greater than" in c
    ):

        truth = (
            reading > bound
        )

        op = ">"


    else:

        raise ValueError(
            f"Unknown bound operator: "
            f"{claim}"
        )


    return (
        result_from_truth(
            truth
        ),

        (
            f"{reading} {op} {bound} "
            f"is {truth}"
        ),
    )


# ============================================================
# 7. COMPLETE MANIFEST AUDITOR
# ============================================================

def audit_manifest(
    claim,
    evidence,
):

    manifest_year = None
    items = None


    for passage in evidence:

        m = re.search(
            r"complete and exhaustive "
            r"(20\d{2}) mission manifest .*?"
            r"lists:\s*(.+?)\.",
            passage,
            flags=re.I,
        )


        if m:

            manifest_year = int(
                m.group(1)
            )


            raw_items = (
                m.group(2)
                .replace(
                    ", and ",
                    ", ",
                )
                .replace(
                    " and ",
                    ", ",
                )
            )


            items = {
                x.strip().lower()

                for x in raw_items.split(
                    ","
                )

                if x.strip()
            }


    if (
        manifest_year is None
        or items is None
    ):

        raise ValueError(
            "Manifest evidence parser failed."
        )


    year_match = re.search(
        r"\b(20\d{2})\b",
        claim,
    )


    if not year_match:

        raise ValueError(
            "Manifest claim has no year."
        )


    claim_year = int(
        year_match.group(1)
    )


    if claim_year != manifest_year:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Complete manifest only "
                f"available for {manifest_year}."
            ),
        )


    m = re.search(
        r"left out the (.+?)\.",
        claim,
        flags=re.I,
    )


    if m:

        item = (
            m.group(1)
            .strip()
            .lower()
        )


    else:

        m = re.search(
            r"The (.+?) was absent from",
            claim,
            flags=re.I,
        )


        if not m:

            raise ValueError(
                "Could not parse omitted item."
            )


        item = (
            m.group(1)
            .strip()
            .lower()
        )


    if item in items:

        return (
            "REFUTES",
            (
                f"{item} is explicitly present "
                f"in exhaustive manifest."
            ),
        )


    return (
        "SUPPORTS",
        (
            f"{item} absent from explicitly "
            f"exhaustive manifest."
        ),
    )


# ============================================================
# 8. PERCENTAGE THRESHOLD AUDITOR
# ============================================================

def audit_percentage_threshold(
    claim,
    evidence,
):

    total = None
    selected = None
    known_year = None


    for passage in evidence:

        m = re.search(
            r"recorded\s+(\d+)\s+graduates "
            r"in\s+(20\d{2})",
            passage,
            flags=re.I,
        )


        if m:

            total = int(
                m.group(1)
            )

            known_year = int(
                m.group(2)
            )


        m = re.search(
            r"Of those graduates,\s+"
            r"(\d+)\s+entered",
            passage,
            flags=re.I,
        )


        if m:

            selected = int(
                m.group(1)
            )


    if None in [
        total,
        selected,
        known_year,
    ]:

        raise ValueError(
            "Percentage threshold evidence "
            "parser failed."
        )


    year_match = re.search(
        r"\b(20\d{2})\b",
        claim,
    )

    pct_match = re.search(
        r"(\d+)%",
        claim,
    )


    if (
        not year_match
        or not pct_match
    ):

        raise ValueError(
            "Percentage threshold claim "
            "parser failed."
        )


    claim_year = int(
        year_match.group(1)
    )

    threshold = int(
        pct_match.group(1)
    )


    if claim_year != known_year:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Destination count unavailable "
                f"for {claim_year}."
            ),
        )


    actual_pct = (
        Fraction(
            selected,
            total,
        )
        * 100
    )


    truth = (
        actual_pct
        > threshold
    )


    return (
        result_from_truth(
            truth
        ),

        (
            f"{selected}/{total}*100"
            f"={float(actual_pct):g}%; "
            f">{threshold}% is {truth}"
        ),
    )


# ============================================================
# 9. DURATION THRESHOLD AUDITOR
# ============================================================

def parse_date_text(text):

    return datetime.strptime(
        text,
        "%d %B %Y",
    )


def audit_duration(
    claim,
    evidence,
):

    launch_text = None
    arrival_text = None


    for passage in evidence:

        m = re.search(
            rf"launched on "
            rf"(\d{{1,2}}\s+(?:{MONTHS})"
            rf"\s+20\d{{2}})",
            passage,
            flags=re.I,
        )


        if m:

            launch_text = (
                m.group(1)
            )


        m = re.search(
            rf"began orbital operations on "
            rf"(\d{{1,2}}\s+(?:{MONTHS})"
            rf"\s+20\d{{2}})",
            passage,
            flags=re.I,
        )


        if m:

            arrival_text = (
                m.group(1)
            )


    if (
        launch_text is None
        or arrival_text is None
    ):

        raise ValueError(
            "Duration evidence parser failed."
        )


    if "final calibration" in (
        claim.lower()
    ):

        return (
            "NOT_ENOUGH_INFO",
            (
                "Final calibration date "
                "explicitly unavailable."
            ),
        )


    m = re.search(
        r"(?:more than|exceeded)\s+"
        r"(\d+)\s+years",
        claim,
        flags=re.I,
    )


    if not m:

        raise ValueError(
            "Duration threshold missing."
        )


    threshold_years = int(
        m.group(1)
    )


    launch = parse_date_text(
        launch_text
    )

    arrival = parse_date_text(
        arrival_text
    )


    anniversary = launch.replace(
        year=(
            launch.year
            + threshold_years
        )
    )


    truth = (
        arrival > anniversary
    )


    return (
        result_from_truth(
            truth
        ),

        (
            f"arrival={arrival.date()}, "
            f"{threshold_years}y anniversary="
            f"{anniversary.date()}, "
            f"greater={truth}"
        ),
    )


# ============================================================
# 10. DEPOT COMPARISON AUDITOR
# ============================================================

def audit_depot(
    claim,
    evidence,
):

    east = None
    west = None
    known_year = None


    for passage in evidence:

        m = re.search(
            r"The\s+(20\d{2})\s+East-depot "
            r"ledger .*? records\s+"
            r"(\d+)\s+parcels",
            passage,
            flags=re.I,
        )


        if m:

            known_year = int(
                m.group(1)
            )

            east = int(
                m.group(2)
            )


        m = re.search(
            r"The\s+(20\d{2})\s+West-depot "
            r"ledger records\s+"
            r"(\d+)\s+parcels",
            passage,
            flags=re.I,
        )


        if m:

            west_year = int(
                m.group(1)
            )

            west = int(
                m.group(2)
            )


            if known_year is not None:

                assert (
                    known_year
                    == west_year
                )

            known_year = (
                west_year
            )


    if None in [
        east,
        west,
        known_year,
    ]:

        raise ValueError(
            "Depot evidence parser failed."
        )


    year_match = re.search(
        r"\b(20\d{2})\b",
        claim,
    )


    if not year_match:

        raise ValueError(
            "Depot claim has no year."
        )


    claim_year = int(
        year_match.group(1)
    )


    if claim_year != known_year:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Both depot values only "
                f"known for {known_year}."
            ),
        )


    c = claim.lower()


    m = re.search(
        r"the\s+(east|west)\s+depot .*?"
        r"handled more parcels than the "
        r"(east|west)\s+depot",
        c,
    )


    if m:

        claimed_winner = (
            m.group(1)
            .upper()
        )


    else:

        m = re.search(
            r"was\s+(east|west)\s+rather "
            r"than\s+(east|west)",
            c,
        )


        if not m:

            raise ValueError(
                "Could not parse claimed "
                "busier depot."
            )


        claimed_winner = (
            m.group(1)
            .upper()
        )


    actual_winner = (
        "EAST"
        if east > west
        else "WEST"
    )


    return (
        result_from_truth(
            claimed_winner
            == actual_winner
        ),

        (
            f"East={east}; West={west}; "
            f"actual={actual_winner}; "
            f"claim={claimed_winner}"
        ),
    )


# ============================================================
# 11. TEMPORAL TRANSITION AUDITOR
# ============================================================

def audit_temporal(
    claim,
    evidence,
):

    old_person = None
    new_person = None

    start_day = None
    end_day = None
    month = None
    year = None


    for passage in evidence:

        m = re.search(
            r"(.+?)'s term as director .*?"
            r"ended when (.+?)'s appointment "
            r"took effect",
            passage,
            flags=re.I,
        )


        if m:

            old_person = (
                m.group(1)
                .strip()
            )

            new_person = (
                m.group(2)
                .strip()
            )


        m = re.search(
            rf"from\s+(\d+)\s+to\s+(\d+)\s+"
            rf"({MONTHS})\s+(20\d{{2}}),"
            rf"\s+inclusive",
            passage,
            flags=re.I,
        )


        if m:

            start_day = int(
                m.group(1)
            )

            end_day = int(
                m.group(2)
            )

            month = (
                m.group(3)
            )

            year = int(
                m.group(4)
            )


    if None in [
        old_person,
        new_person,
        start_day,
        end_day,
        month,
        year,
    ]:

        raise ValueError(
            "Temporal evidence parser failed."
        )


    m = re.search(
        rf"\b(\d{{1,2}})\s+"
        rf"({MONTHS})\s+"
        rf"(20\d{{2}})\b",
        claim,
        flags=re.I,
    )


    if not m:

        raise ValueError(
            "Temporal claim date parser failed."
        )


    day = int(
        m.group(1)
    )

    claim_month = (
        m.group(2)
    )

    claim_year = int(
        m.group(3)
    )


    if (
        claim_month.lower()
        != month.lower()
        or claim_year != year
    ):

        return (
            "NOT_ENOUGH_INFO",
            "Claim date outside known transition record.",
        )


    # Determine which person the claim asserts.

    old_in_claim = (
        old_person.lower()
        in claim.lower()
    )

    new_in_claim = (
        new_person.lower()
        in claim.lower()
    )


    if (
        old_in_claim
        == new_in_claim
    ):

        raise ValueError(
            "Could not uniquely determine "
            "claimed office-holder."
        )


    claimed_person = (
        old_person
        if old_in_claim
        else new_person
    )


    # Inside uncertain transition window:
    # exact office-holder cannot be established.

    if (
        start_day
        <= day
        <= end_day
    ):

        return (
            "NOT_ENOUGH_INFO",
            (
                f"{day} lies inside uncertain "
                f"{start_day}-{end_day} "
                f"transition interval."
            ),
        )


    if day < start_day:

        actual_person = (
            old_person
        )

    else:

        actual_person = (
            new_person
        )


    return (
        result_from_truth(
            claimed_person
            == actual_person
        ),

        (
            f"actual={actual_person}; "
            f"claim={claimed_person}"
        ),
    )


# ============================================================
# 12. TRANSFER ARITHMETIC AUDITOR
# ============================================================

def audit_transfer(
    claim,
    evidence,
):

    # --------------------------------------------------------
    # DIFFERENCE variant
    # --------------------------------------------------------

    unit_a = None
    unit_b = None
    unit_year = None


    for passage in evidence:

        m = re.search(
            r"Unit A .*? recorded "
            r"(\d+) completed cases in "
            r"(20\d{2})",
            passage,
            flags=re.I,
        )


        if m:

            unit_a = int(
                m.group(1)
            )

            unit_year = int(
                m.group(2)
            )


        m = re.search(
            r"Unit B recorded "
            r"(\d+) completed cases",
            passage,
            flags=re.I,
        )


        if m:

            unit_b = int(
                m.group(1)
            )


    if (
        unit_a is not None
        and unit_b is not None
    ):

        year_match = re.search(
            r"\b(20\d{2})\b",
            claim,
        )


        if not year_match:

            raise ValueError(
                "Transfer difference claim "
                "has no year."
            )


        claim_year = int(
            year_match.group(1)
        )


        if claim_year != unit_year:

            return (
                "NOT_ENOUGH_INFO",
                (
                    f"Both units only known "
                    f"for {unit_year}."
                ),
            )


        m = re.search(
            r"(\d+)\s+more cases",
            claim,
            flags=re.I,
        )


        if not m:

            m = re.search(
                r"gap .*? was "
                r"(\d+)\s+completed cases",
                claim,
                flags=re.I,
            )


        if not m:

            raise ValueError(
                "Difference amount parser failed."
            )


        claimed_diff = int(
            m.group(1)
        )


        actual_diff = (
            unit_a
            - unit_b
        )


        return (
            result_from_truth(
                actual_diff
                == claimed_diff
            ),

            (
                f"{unit_a}-{unit_b}"
                f"={actual_diff}; "
                f"claim={claimed_diff}"
            ),
        )


    # --------------------------------------------------------
    # RATIO variant
    # --------------------------------------------------------

    section_a = None
    section_b = None
    section_year = None


    for passage in evidence:

        m = re.search(
            r"Section A .*? processed "
            r"(\d+) records in "
            r"(20\d{2})",
            passage,
            flags=re.I,
        )


        if m:

            section_a = int(
                m.group(1)
            )

            section_year = int(
                m.group(2)
            )


        m = re.search(
            r"Section B processed "
            r"(\d+) records in "
            r"(20\d{2})",
            passage,
            flags=re.I,
        )


        if m:

            section_b = int(
                m.group(1)
            )


    if None in [
        section_a,
        section_b,
        section_year,
    ]:

        raise ValueError(
            "Transfer ratio evidence "
            "parser failed."
        )


    year_match = re.search(
        r"\b(20\d{2})\b",
        claim,
    )


    if not year_match:

        raise ValueError(
            "Transfer ratio claim "
            "has no year."
        )


    claim_year = int(
        year_match.group(1)
    )


    if claim_year != section_year:

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Both sections only known "
                f"for {section_year}."
            ),
        )


    m = re.search(
        r"exactly\s+(\d+)\s+times",
        claim,
        flags=re.I,
    )


    if not m:

        m = re.search(
            r"precisely\s+(\d+)×",
            claim,
            flags=re.I,
        )


    if not m:

        raise ValueError(
            "Ratio multiplier parser failed."
        )


    claimed_ratio = int(
        m.group(1)
    )


    actual_ratio = Fraction(
        section_a,
        section_b,
    )


    return (
        result_from_truth(
            actual_ratio
            == claimed_ratio
        ),

        (
            f"{section_a}/{section_b}"
            f"={float(actual_ratio):g}; "
            f"claim={claimed_ratio}"
        ),
    )


# ============================================================
# 13. ROUTER
# ============================================================

AUDITORS = {
    "revenue_percentage": (
        audit_revenue
    ),

    "two_component_sum": (
        audit_sum
    ),

    "project_location_mapping": (
        audit_project
    ),

    "range_or_bound": (
        audit_range
    ),

    "complete_manifest": (
        audit_manifest
    ),

    "percentage_threshold": (
        audit_percentage_threshold
    ),

    "duration_threshold": (
        audit_duration
    ),

    "depot_comparison": (
        audit_depot
    ),

    "temporal_transition": (
        audit_temporal
    ),

    "derived_comparison_transfer": (
        audit_transfer
    ),
}


# ============================================================
# 14. AUDIT ALL 225 EXAMPLES
# ============================================================

audit_records = []


def audit_split(
    split_name,
    rows,
):

    print(
        f"\nAuditing {split_name}: "
        f"{len(rows)} examples"
    )


    for index, row in enumerate(
        rows,
        1,
    ):

        example_id = str(
            row["id"]
        )

        claim = row[
            "claim"
        ]

        evidence = row[
            "evidence"
        ]

        assigned_label = row[
            "label"
        ]


        family = infer_family(
            example_id
        )


        parser_failed = False

        error_text = None

        derived_label = None

        reasoning = None


        try:

            derived_label, reasoning = (
                AUDITORS[
                    family
                ](
                    claim,
                    evidence,
                )
            )


            if derived_label not in LABELS:

                raise ValueError(
                    f"Invalid derived label: "
                    f"{derived_label}"
                )


        except Exception as e:

            parser_failed = True

            error_text = (
                f"{type(e).__name__}: "
                f"{e}"
            )


        agreement = (
            (
                not parser_failed
            )
            and (
                derived_label
                == assigned_label
            )
        )


        audit_records.append({
            "split": (
                split_name
            ),

            "id": (
                example_id
            ),

            "family": (
                family
            ),

            "assigned_label": (
                assigned_label
            ),

            "derived_label": (
                derived_label
            ),

            "agreement": (
                agreement
            ),

            "parser_failed": (
                parser_failed
            ),

            "reasoning_check": (
                reasoning
            ),

            "parser_error": (
                error_text
            ),

            "claim": (
                claim
            ),

            "evidence": json.dumps(
                evidence,
                ensure_ascii=False,
            ),
        })


        if (
            index % 30 == 0
            or index == len(rows)
        ):

            print(
                f"  checked "
                f"{index}/{len(rows)}"
            )


audit_split(
    "train",
    train_rows,
)


audit_split(
    "frozen_holdout",
    holdout_rows,
)


audit_df = pd.DataFrame(
    audit_records
)


assert len(
    audit_df
) == 225


# ============================================================
# 15. AUDIT RESULTS
# ============================================================

parser_failures = audit_df[
    audit_df[
        "parser_failed"
    ]
]


disagreements = audit_df[
    ~audit_df[
        "agreement"
    ]
]


agreement_count = int(
    audit_df[
        "agreement"
    ].sum()
)


print("\n" + "=" * 78)
print("SEMANTIC AUDIT RESULT")
print("=" * 78)


print(
    "Total examples:",
    len(audit_df),
)

print(
    "Semantic agreements:",
    f"{agreement_count}/225",
)

print(
    "Parser failures:",
    len(parser_failures),
)

print(
    "Label disagreements:",
    len(disagreements),
)


# ============================================================
# 16. RESULTS BY SPLIT
# ============================================================

print("\n" + "=" * 78)
print("AUDIT BY SPLIT")
print("=" * 78)


for split_name in [
    "train",
    "frozen_holdout",
]:

    subset = audit_df[
        audit_df[
            "split"
        ]
        == split_name
    ]


    correct = int(
        subset[
            "agreement"
        ].sum()
    )


    print(
        f"{split_name:16} | "
        f"{correct:3d}/{len(subset):3d} | "
        f"parser_fail="
        f"{int(subset['parser_failed'].sum()):2d}"
    )


# ============================================================
# 17. RESULTS BY FAMILY
# ============================================================

print("\n" + "=" * 78)
print("AUDIT BY FAMILY")
print("=" * 78)


print(
    f"{'Family':30} | "
    f"{'N':>3} | "
    f"{'Agree':>5} | "
    f"{'Fail':>4}"
)

print(
    "-" * 52
)


family_summary = {}


for family in FAMILIES:

    subset = audit_df[
        audit_df[
            "family"
        ]
        == family
    ]


    n = len(
        subset
    )

    agree = int(
        subset[
            "agreement"
        ].sum()
    )

    fails = int(
        subset[
            "parser_failed"
        ].sum()
    )


    family_summary[
        family
    ] = {
        "n": (
            n
        ),

        "agreement": (
            agree
        ),

        "parser_failures": (
            fails
        ),
    }


    print(
        f"{family:30} | "
        f"{n:3d} | "
        f"{agree:5d} | "
        f"{fails:4d}"
    )


# ============================================================
# 18. CONTRASTIVE TRIO INVARIANT
# ============================================================

print("\n" + "=" * 78)
print("CONTRASTIVE TRIO AUDIT")
print("=" * 78)


def scenario_id_from_example(
    example_id,
):

    return re.sub(
        r"_(S|R|N)$",
        "",
        example_id,
    )


scenario_groups = defaultdict(
    list
)


for row in (
    train_rows
    + holdout_rows
):

    scenario_groups[
        scenario_id_from_example(
            row["id"]
        )
    ].append(
        row
    )


assert len(
    scenario_groups
) == 75


trio_failures = []


for scenario_id, rows in (
    scenario_groups.items()
):

    labels = {
        row["label"]
        for row in rows
    }


    evidence_versions = {
        json.dumps(
            row["evidence"],
            ensure_ascii=False,
            sort_keys=True,
        )
        for row in rows
    }


    if (
        len(rows) != 3
        or labels != LABELS
        or len(
            evidence_versions
        ) != 1
    ):

        trio_failures.append(
            scenario_id
        )


print(
    "Total scenarios:",
    len(
        scenario_groups
    ),
)

print(
    "Valid contrastive trios:",
    (
        len(
            scenario_groups
        )
        - len(
            trio_failures
        )
    ),
)

print(
    "Trio failures:",
    len(
        trio_failures
    ),
)


# ============================================================
# 19. TRAIN / HOLDOUT SEPARATION RECHECK
# ============================================================

print("\n" + "=" * 78)
print("TRAIN / HOLDOUT SEPARATION RECHECK")
print("=" * 78)


train_ids = {
    row["id"]
    for row in train_rows
}

holdout_ids = {
    row["id"]
    for row in holdout_rows
}


assert train_ids.isdisjoint(
    holdout_ids
)


train_claims = {
    norm(
        row["claim"]
    ).lower()

    for row in train_rows
}

holdout_claims = {
    norm(
        row["claim"]
    ).lower()

    for row in holdout_rows
}


assert train_claims.isdisjoint(
    holdout_claims
)


train_meta = pd.read_csv(
    TRAIN_META_PATH
)

holdout_meta = pd.read_csv(
    HOLDOUT_META_PATH
)


train_entities = set(
    train_meta[
        "entity"
    ].astype(str)
)

holdout_entities = set(
    holdout_meta[
        "entity"
    ].astype(str)
)


train_templates = set(
    train_meta[
        "template_id"
    ].astype(str)
)

holdout_templates = set(
    holdout_meta[
        "template_id"
    ].astype(str)
)


assert train_entities.isdisjoint(
    holdout_entities
)

assert train_templates.isdisjoint(
    holdout_templates
)


print(
    "ID overlap:",
    len(
        train_ids
        & holdout_ids
    ),
)

print(
    "Exact claim overlap:",
    len(
        train_claims
        & holdout_claims
    ),
)

print(
    "Entity overlap:",
    len(
        train_entities
        & holdout_entities
    ),
)

print(
    "Template-ID overlap:",
    len(
        train_templates
        & holdout_templates
    ),
)


# ============================================================
# 20. SAVE AUDIT CSV
# ============================================================

audit_df.to_csv(
    AUDIT_CSV,
    index=False,
)


disagreements.to_csv(
    AUDIT_ERRORS_CSV,
    index=False,
)


# ============================================================
# 21. HARD GATE
#
# DO NOT FREEZE / TRAIN if anything fails.
# ============================================================

passed = (
    len(
        parser_failures
    ) == 0

    and len(
        disagreements
    ) == 0

    and len(
        trio_failures
    ) == 0

    and agreement_count
    == 225
)


if not passed:

    print("\n" + "!" * 78)

    print(
        "D4 DATASET AUDIT FAILED"
    )

    print("!" * 78)


    if len(
        disagreements
    ):

        print(
            "\nFirst failures:"
        )

        print(
            disagreements[
                [
                    "split",
                    "id",
                    "family",
                    "assigned_label",
                    "derived_label",
                    "parser_error",
                    "claim",
                ]
            ]
            .head(20)
            .to_string(
                index=False
            )
        )


    raise RuntimeError(
        "D4 synthetic data did NOT pass "
        "the independent semantic audit. "
        "DO NOT TRAIN D4."
    )


# ============================================================
# 22. WRITE AUDIT SUMMARY
# ============================================================

summary = {
    "audit_name": (
        "D4_SYNTHETIC_225_INDEPENDENT_SEMANTIC_AUDIT"
    ),

    "status": (
        "PASSED"
    ),

    "train": {
        "path": (
            str(
                TRAIN_PATH
            )
        ),

        "sha256": (
            train_sha
        ),

        "rows": (
            len(
                train_rows
            )
        ),

        "semantic_agreement": (
            int(
                audit_df.loc[
                    audit_df["split"]
                    == "train",
                    "agreement",
                ].sum()
            )
        ),
    },

    "frozen_holdout": {
        "path": (
            str(
                HOLDOUT_PATH
            )
        ),

        "sha256": (
            holdout_sha
        ),

        "rows": (
            len(
                holdout_rows
            )
        ),

        "semantic_agreement": (
            int(
                audit_df.loc[
                    audit_df["split"]
                    == "frozen_holdout",
                    "agreement",
                ].sum()
            )
        ),
    },

    "total_examples": (
        225
    ),

    "semantic_agreements": (
        agreement_count
    ),

    "parser_failures": (
        len(
            parser_failures
        )
    ),

    "label_disagreements": (
        len(
            disagreements
        )
    ),

    "contrastive_scenarios": (
        len(
            scenario_groups
        )
    ),

    "contrastive_trio_failures": (
        len(
            trio_failures
        )
    ),

    "family_summary": (
        family_summary
    ),

    "training_rule": (
        "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl "
        "must never be included in optimization."
    ),
}


with open(
    AUDIT_SUMMARY,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 23. FINAL FROZEN MANIFEST
# ============================================================

frozen_manifest = {
    "artifact": (
        "D4_CONTRASTIVE_CURRICULUM_V1_AUDITED"
    ),

    "status": (
        "SEMANTICALLY_AUDITED_AND_FROZEN"
    ),

    "train": {
        "filename": (
            TRAIN_PATH.name
        ),

        "sha256": (
            train_sha
        ),

        "rows": 150,

        "use_for_training": True,
    },

    "frozen_holdout": {
        "filename": (
            HOLDOUT_PATH.name
        ),

        "sha256": (
            holdout_sha
        ),

        "rows": 75,

        "use_for_training": False,

        "status": (
            "FROZEN_BEFORE_FIRST_D4_TRAINING_RUN"
        ),
    },

    "semantic_audit": {
        "examples_checked": 225,

        "agreements": 225,

        "parser_failures": 0,

        "label_disagreements": 0,

        "contrastive_trio_failures": 0,

        "audit_csv": (
            str(
                AUDIT_CSV
            )
        ),
    },

    "important_experimental_rule": (
        "Do not alter the frozen holdout, "
        "do not train on it, and do not use "
        "its individual errors to generate "
        "additional D4 training examples "
        "before the D4 recipe is evaluated."
    ),
}


with open(
    FINAL_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        frozen_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 24. CREATE AUDITED BACKUP PACKAGE
# ============================================================

if BACKUP_DIR.exists():

    shutil.rmtree(
        BACKUP_DIR
    )


BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


copy_files = [
    TRAIN_PATH,
    HOLDOUT_PATH,
    TRAIN_META_PATH,
    HOLDOUT_META_PATH,
    ORIGINAL_MANIFEST,
    FINAL_MANIFEST,
    AUDIT_CSV,
    AUDIT_SUMMARY,
]


for source in copy_files:

    shutil.copy2(
        source,
        BACKUP_DIR
        / source.name,
    )


README = (
    BACKUP_DIR
    / "README_DO_NOT_CONTAMINATE_HOLDOUT.txt"
)


README.write_text(
    """D4 CONTRASTIVE CURRICULUM V1 — AUDITED

TRAIN:
d4_contrastive_train_v1.jsonl
150 examples
Use for D4 augmentation.

FROZEN HOLDOUT:
D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl
75 examples
NEVER use for training.

The holdout was created and frozen BEFORE the first D4
training experiment.

Independent deterministic semantic audit:
225/225 agreement
0 parser failures
0 label disagreements

Do not generate additional examples from individual holdout
errors before evaluating the planned D4 experiment.
""",
    encoding="utf-8",
)


zip_existing = Path(
    str(
        ZIP_BASE
    )
    + ".zip"
)


if zip_existing.exists():

    zip_existing.unlink()


zip_path = shutil.make_archive(
    base_name=str(
        ZIP_BASE
    ),

    format="zip",

    root_dir=str(
        BACKUP_DIR.parent
    ),

    base_dir=(
        BACKUP_DIR.name
    ),
)


zip_path = Path(
    zip_path
)


zip_sha = sha256_file(
    zip_path
)

zip_mb = (
    zip_path.stat().st_size
    / 1024**2
)


# Make the holdout read-only as an extra reminder.
# It remains readable for later evaluation.

try:

    HOLDOUT_PATH.chmod(
        0o444
    )

except Exception:

    pass


# ============================================================
# 25. FINAL REPORT
# ============================================================

print("\n" + "=" * 78)
print("D4 DATASET — AUDIT PASSED")
print("=" * 78)


print(
    "Semantic agreement:",
    "225/225",
)

print(
    "Parser failures:",
    0,
)

print(
    "Label disagreements:",
    0,
)

print(
    "Contrastive trio failures:",
    0,
)


print(
    "\nTRAIN:"
)

print(
    TRAIN_PATH,
)

print(
    "SHA256:",
    train_sha,
)

print(
    "Rows:",
    150,
)


print(
    "\nFROZEN HOLDOUT — DO NOT TRAIN:"
)

print(
    HOLDOUT_PATH,
)

print(
    "SHA256:",
    holdout_sha,
)

print(
    "Rows:",
    75,
)


print(
    "\nAudit CSV:",
    AUDIT_CSV,
)

print(
    "Audit summary:",
    AUDIT_SUMMARY,
)

print(
    "Final manifest:",
    FINAL_MANIFEST,
)


print(
    "\nAudited backup ZIP:",
    zip_path,
)

print(
    "ZIP size:",
    f"{zip_mb:.3f} MB",
)

print(
    "ZIP SHA256:",
    zip_sha,
)


print("\n" + "=" * 78)
print("CELL 3 COMPLETE")
print("=" * 78)


print(
    "\nNO MODEL was loaded."
)

print(
    "NO GPU was used."
)

print(
    "NO inference occurred."
)

print(
    "NO training occurred."
)


print(
    "\nNEXT: send me the complete Cell-3 output."
)

print(
    "If and only if it reports 225/225 agreement "
    "with 0 parser failures, I will give you the "
    "D4 training cell."
)

In [ ]:
# ============================================================
# CELL 3B — FIX REVENUE YEAR-DIRECTION AUDIT + FINAL FREEZE
#
# WHY THIS CELL EXISTS
# --------------------
# Cell 3 found 5 disagreements, ALL in one revenue wording:
#
#   "... X% higher in 2023 than in 2022."
#
# The dataset label is correct.
# The auditor incorrectly interpreted the textual year order
# as old_year -> new_year.
#
# Correct semantics:
#
#   "higher in 2023 than in 2022"
#       old = 2022
#       new = 2023
#
# This cell:
#
#   1. preserves the old buggy audit CSV;
#   2. independently re-audits ALL 60 revenue examples;
#   3. patches their audit results;
#   4. requires final 225/225 agreement;
#   5. rechecks train/holdout structural separation;
#   6. creates the final frozen manifest;
#   7. creates D4_CURRICULUM_V1_AUDITED.zip.
#
# NO MODEL
# NO GPU
# NO TRAINING
# NO INFERENCE
# ============================================================

import re
import json
import shutil
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction

import pandas as pd


# ============================================================
# 0. PATHS / EXPECTED HASHES
# ============================================================

WORK = Path("/kaggle/working")

D4_DIR = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

TRAIN_PATH = (
    D4_DIR
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT_PATH = (
    D4_DIR
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)

TRAIN_META_PATH = (
    D4_DIR
    / "d4_contrastive_train_v1_metadata.csv"
)

HOLDOUT_META_PATH = (
    D4_DIR
    / "D4_FROZEN_HOLDOUT_V1_metadata.csv"
)

ORIGINAL_MANIFEST = (
    D4_DIR
    / "d4_curriculum_v1_manifest.json"
)


AUDIT_DIR = (
    D4_DIR
    / "semantic_audit"
)

OLD_AUDIT_CSV = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_SEMANTIC_AUDIT.csv"
)

BUGGY_AUDIT_BACKUP = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_SEMANTIC_AUDIT_"
      "BUGGY_REVENUE_DIRECTION_BACKUP.csv"
)

FINAL_AUDIT_CSV = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_SEMANTIC_AUDIT.csv"
)

FINAL_FAILURES_CSV = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_AUDIT_FAILURES.csv"
)

AUDIT_SUMMARY = (
    AUDIT_DIR
    / "D4_SYNTHETIC_225_AUDIT_SUMMARY.json"
)

FINAL_MANIFEST = (
    D4_DIR
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


BACKUP_DIR = (
    WORK
    / "D4_CURRICULUM_V1_AUDITED"
)

ZIP_BASE = (
    WORK
    / "D4_CURRICULUM_V1_AUDITED"
)


EXPECTED_TRAIN_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)


LABELS = {
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
}


# ============================================================
# 1. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def norm(text):

    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip().lower()


def result_from_truth(truth):

    return (
        "SUPPORTS"
        if truth
        else "REFUTES"
    )


# ============================================================
# 2. VERIFY CELL-2 DATA HAS NOT CHANGED
# ============================================================

for path in [
    TRAIN_PATH,
    HOLDOUT_PATH,
    TRAIN_META_PATH,
    HOLDOUT_META_PATH,
    ORIGINAL_MANIFEST,
    OLD_AUDIT_CSV,
]:

    assert path.exists(), path


train_sha = sha256_file(
    TRAIN_PATH
)

holdout_sha = sha256_file(
    HOLDOUT_PATH
)


assert train_sha == EXPECTED_TRAIN_SHA
assert holdout_sha == EXPECTED_HOLDOUT_SHA


train_rows = read_jsonl(
    TRAIN_PATH
)

holdout_rows = read_jsonl(
    HOLDOUT_PATH
)


assert len(train_rows) == 150
assert len(holdout_rows) == 75


print("=" * 78)
print("CELL 3B — REVENUE AUDIT CORRECTION")
print("=" * 78)

print(
    "Train SHA:",
    train_sha,
)

print(
    "Holdout SHA:",
    holdout_sha,
)

print(
    "Train rows:",
    len(train_rows),
)

print(
    "Holdout rows:",
    len(holdout_rows),
)


# ============================================================
# 3. PRESERVE ORIGINAL FAILED AUDIT
# ============================================================

if not BUGGY_AUDIT_BACKUP.exists():

    shutil.copy2(
        OLD_AUDIT_CSV,
        BUGGY_AUDIT_BACKUP,
    )


print(
    "\nBuggy audit preserved at:",
    BUGGY_AUDIT_BACKUP,
)


old_audit_df = pd.read_csv(
    OLD_AUDIT_CSV
)


assert len(old_audit_df) == 225


print(
    "Original semantic agreements:",
    int(
        old_audit_df[
            "agreement"
        ].sum()
    ),
    "/225",
)


# ============================================================
# 4. CORRECTED, PHRASE-AWARE REVENUE AUDITOR
#
# IMPORTANT:
#
# It does NOT infer direction merely from the order in which
# years happen to appear in the sentence.
#
# Explicitly supported semantic patterns:
#
#   from 2022 to 2023
#       old=2022, new=2023
#
#   between 2022 and 2023
#       old=2022, new=2023
#
#   higher in 2023 than in 2022
#       old=2022, new=2023
#
#   comparing totals for 2022 and 2023
#       old=2022, new=2023
#
#   later total ... earlier total from 2022
#       old=2022, new=other known later year
# ============================================================

def audit_revenue_corrected(
    claim,
    evidence,
):

    # --------------------------------------------------------
    # Extract exact year -> revenue mapping from evidence.
    # --------------------------------------------------------

    year_to_value = {}


    for passage in evidence:

        years = re.findall(
            r"\b(20\d{2})\b",
            passage,
        )

        amounts = re.findall(
            r"\b(\d+)\s+million credits\b",
            passage,
            flags=re.I,
        )


        if (
            len(years) == 1
            and len(amounts) == 1
        ):

            year_to_value[
                int(years[0])
            ] = int(
                amounts[0]
            )


    if len(year_to_value) != 2:

        raise ValueError(
            f"Expected exactly two revenue "
            f"year/value pairs; got "
            f"{year_to_value}"
        )


    known_years = sorted(
        year_to_value
    )


    # --------------------------------------------------------
    # Extract claimed percentage.
    # --------------------------------------------------------

    pct_match = re.search(
        r"(\d+)%",
        claim,
    )


    if not pct_match:

        raise ValueError(
            "Revenue claim has no percentage."
        )


    claimed_pct = int(
        pct_match.group(1)
    )


    c = claim.lower()


    y0 = None
    y1 = None
    semantic_pattern = None


    # --------------------------------------------------------
    # PATTERN 1:
    # "from YEAR0 to YEAR1"
    # --------------------------------------------------------

    m = re.search(
        r"from\s+(20\d{2})\s+to\s+(20\d{2})",
        c,
    )


    if m:

        y0 = int(
            m.group(1)
        )

        y1 = int(
            m.group(2)
        )

        semantic_pattern = (
            "from_old_to_new"
        )


    # --------------------------------------------------------
    # PATTERN 2:
    # "between YEAR0 and YEAR1"
    # --------------------------------------------------------

    if y0 is None:

        m = re.search(
            r"between\s+(20\d{2})\s+and\s+(20\d{2})",
            c,
        )


        if m:

            first = int(
                m.group(1)
            )

            second = int(
                m.group(2)
            )


            # For "growth between", use chronological
            # earlier -> later.

            y0 = min(
                first,
                second,
            )

            y1 = max(
                first,
                second,
            )

            semantic_pattern = (
                "between_chronological"
            )


    # --------------------------------------------------------
    # PATTERN 3:
    #
    # "X% higher in 2023 than in 2022"
    #
    # This was the Cell-3 bug.
    # --------------------------------------------------------

    if y0 is None:

        m = re.search(
            r"higher\s+in\s+(20\d{2})"
            r"\s+than\s+in\s+(20\d{2})",
            c,
        )


        if m:

            higher_year = int(
                m.group(1)
            )

            comparison_year = int(
                m.group(2)
            )


            y0 = (
                comparison_year
            )

            y1 = (
                higher_year
            )

            semantic_pattern = (
                "higher_in_new_than_old"
            )


    # --------------------------------------------------------
    # PATTERN 4:
    #
    # "Comparing ... totals for 2022 and 2023, ... growth"
    # --------------------------------------------------------

    if y0 is None:

        m = re.search(
            r"totals\s+for\s+(20\d{2})"
            r"\s+and\s+(20\d{2})",
            c,
        )


        if m:

            first = int(
                m.group(1)
            )

            second = int(
                m.group(2)
            )


            y0 = min(
                first,
                second,
            )

            y1 = max(
                first,
                second,
            )

            semantic_pattern = (
                "compare_totals_chronological"
            )


    # --------------------------------------------------------
    # PATTERN 5:
    #
    # "later revenue total was ... above its earlier
    #  total from 2022"
    # --------------------------------------------------------

    if y0 is None:

        m = re.search(
            r"earlier total from\s+(20\d{2})",
            c,
        )


        if m:

            earlier_year = int(
                m.group(1)
            )


            other_years = [
                year

                for year in known_years

                if year != earlier_year
            ]


            if len(other_years) != 1:

                raise ValueError(
                    "Could not resolve later "
                    "revenue year."
                )


            y0 = earlier_year

            y1 = other_years[0]

            semantic_pattern = (
                "later_vs_earlier"
            )


    # --------------------------------------------------------
    # PATTERN 6 fallback:
    #
    # If the claim names the two known evidence years and
    # describes an increase/growth, use chronological direction.
    #
    # This fallback does NOT override "higher in X than Y",
    # because that was already parsed above.
    # --------------------------------------------------------

    if y0 is None:

        claim_years = [
            int(x)

            for x in re.findall(
                r"\b(20\d{2})\b",
                c,
            )
        ]


        matching_years = sorted(
            {
                year

                for year in claim_years

                if year in year_to_value
            }
        )


        if (
            len(matching_years) == 2
            and (
                "increase" in c
                or "growth" in c
                or "rose" in c
                or "higher" in c
            )
        ):

            y0 = matching_years[0]

            y1 = matching_years[1]

            semantic_pattern = (
                "chronological_fallback"
            )


    # --------------------------------------------------------
    # Missing requested year => NEI.
    # --------------------------------------------------------

    if (
        y0 is None
        or y1 is None
    ):

        claim_years = {
            int(x)

            for x in re.findall(
                r"\b(20\d{2})\b",
                c,
            )
        }


        if any(
            year not in year_to_value

            for year in claim_years
        ):

            return (
                "NOT_ENOUGH_INFO",
                (
                    "Requested revenue "
                    "comparison includes "
                    "an unavailable year."
                ),
                "missing_year",
            )


        raise ValueError(
            f"Could not determine revenue "
            f"comparison direction:\n{claim}"
        )


    # --------------------------------------------------------
    # If either required year lacks an evidence value => NEI.
    # --------------------------------------------------------

    if (
        y0 not in year_to_value
        or y1 not in year_to_value
    ):

        return (
            "NOT_ENOUGH_INFO",
            (
                f"Revenue value unavailable "
                f"for {y0}->{y1}"
            ),
            semantic_pattern,
        )


    old = year_to_value[
        y0
    ]

    new = year_to_value[
        y1
    ]


    actual_pct = (
        Fraction(
            new - old,
            old,
        )
        * 100
    )


    truth = (
        actual_pct
        == claimed_pct
    )


    derived = (
        result_from_truth(
            truth
        )
    )


    reasoning = (
        f"pattern={semantic_pattern}; "
        f"old={y0}:{old}; "
        f"new={y1}:{new}; "
        f"({new}-{old})/{old}*100="
        f"{float(actual_pct):g}%; "
        f"claim={claimed_pct}%"
    )


    return (
        derived,
        reasoning,
        semantic_pattern,
    )


# ============================================================
# 5. BUILD RAW ROW LOOKUP
# ============================================================

all_rows = (
    train_rows
    + holdout_rows
)


row_by_id = {
    str(row["id"]): row

    for row in all_rows
}


assert len(
    row_by_id
) == 225


# ============================================================
# 6. RE-AUDIT ALL 60 REVENUE EXAMPLES
# ============================================================

revenue_ids = [
    example_id

    for example_id in row_by_id

    if "_revenue_percentage_"
    in example_id
]


assert len(
    revenue_ids
) == 60


revenue_results = {}


print("\n" + "=" * 78)
print("RE-AUDITING ALL 60 REVENUE EXAMPLES")
print("=" * 78)


for index, example_id in enumerate(
    sorted(
        revenue_ids
    ),
    1,
):

    row = row_by_id[
        example_id
    ]


    derived_label, reasoning, pattern = (
        audit_revenue_corrected(
            row[
                "claim"
            ],

            row[
                "evidence"
            ],
        )
    )


    agreement = (
        derived_label
        == row[
            "label"
        ]
    )


    revenue_results[
        example_id
    ] = {
        "derived_label": (
            derived_label
        ),

        "reasoning": (
            reasoning
        ),

        "pattern": (
            pattern
        ),

        "agreement": (
            agreement
        ),
    }


    if (
        index % 15 == 0
        or index == 60
    ):

        print(
            f"  checked "
            f"{index}/60"
        )


revenue_agreements = sum(
    result[
        "agreement"
    ]

    for result in revenue_results.values()
)


print(
    "\nRevenue semantic agreement:",
    f"{revenue_agreements}/60",
)


assert revenue_agreements == 60, (
    "Corrected revenue auditor still "
    "found a disagreement. "
    "DO NOT TRAIN."
)


# ============================================================
# 7. SHOW THE FIVE PREVIOUS FAILURES
# ============================================================

previous_failures = (
    old_audit_df[
        ~old_audit_df[
            "agreement"
        ]
    ]
)


assert len(
    previous_failures
) == 5


print("\n" + "=" * 78)
print("THE FIVE PREVIOUS FAILURES — CORRECTED")
print("=" * 78)


for _, old_row in (
    previous_failures
    .iterrows()
):

    example_id = str(
        old_row[
            "id"
        ]
    )

    original = row_by_id[
        example_id
    ]

    result = revenue_results[
        example_id
    ]


    print("\n" + "-" * 78)

    print(
        "ID:",
        example_id,
    )

    print(
        "Assigned:",
        original[
            "label"
        ],
    )

    print(
        "Old buggy derived:",
        old_row[
            "derived_label"
        ],
    )

    print(
        "Corrected derived:",
        result[
            "derived_label"
        ],
    )

    print(
        "Pattern:",
        result[
            "pattern"
        ],
    )

    print(
        "Check:",
        result[
            "reasoning"
        ],
    )

    print(
        "Claim:",
        original[
            "claim"
        ],
    )


# ============================================================
# 8. PATCH AUDIT DATAFRAME
#
# Non-revenue examples were already independently audited:
#   165/165 agreement
#
# Revenue is now independently re-audited:
#    60/60 agreement
# ============================================================

final_audit_df = (
    old_audit_df.copy()
)


for idx, audit_row in (
    final_audit_df.iterrows()
):

    example_id = str(
        audit_row[
            "id"
        ]
    )


    if example_id not in revenue_results:
        continue


    result = revenue_results[
        example_id
    ]


    final_audit_df.at[
        idx,
        "derived_label"
    ] = result[
        "derived_label"
    ]


    final_audit_df.at[
        idx,
        "agreement"
    ] = result[
        "agreement"
    ]


    final_audit_df.at[
        idx,
        "parser_failed"
    ] = False


    final_audit_df.at[
        idx,
        "reasoning_check"
    ] = result[
        "reasoning"
    ]


    final_audit_df.at[
        idx,
        "parser_error"
    ] = None


# ============================================================
# 9. HARD SEMANTIC GATE
# ============================================================

agreement_count = int(
    final_audit_df[
        "agreement"
    ].sum()
)


parser_failures = int(
    final_audit_df[
        "parser_failed"
    ].fillna(
        False
    ).sum()
)


disagreements = (
    final_audit_df[
        ~final_audit_df[
            "agreement"
        ]
    ]
)


print("\n" + "=" * 78)
print("FINAL SEMANTIC AUDIT")
print("=" * 78)


print(
    "Total:",
    len(
        final_audit_df
    ),
)

print(
    "Semantic agreements:",
    f"{agreement_count}/225",
)

print(
    "Parser failures:",
    parser_failures,
)

print(
    "Label disagreements:",
    len(
        disagreements
    ),
)


assert len(
    final_audit_df
) == 225

assert agreement_count == 225
assert parser_failures == 0
assert len(disagreements) == 0


# ============================================================
# 10. AUDIT BY SPLIT
# ============================================================

print("\n" + "=" * 78)
print("FINAL AUDIT BY SPLIT")
print("=" * 78)


for split_name in [
    "train",
    "frozen_holdout",
]:

    subset = final_audit_df[
        final_audit_df[
            "split"
        ]
        == split_name
    ]


    print(
        f"{split_name:16} | "
        f"{int(subset['agreement'].sum())}"
        f"/{len(subset)}"
    )


assert int(
    final_audit_df.loc[
        final_audit_df[
            "split"
        ]
        == "train",
        "agreement",
    ].sum()
) == 150


assert int(
    final_audit_df.loc[
        final_audit_df[
            "split"
        ]
        == "frozen_holdout",
        "agreement",
    ].sum()
) == 75


# ============================================================
# 11. AUDIT BY FAMILY
# ============================================================

print("\n" + "=" * 78)
print("FINAL AUDIT BY FAMILY")
print("=" * 78)


family_summary = {}


for family in sorted(
    final_audit_df[
        "family"
    ].unique()
):

    subset = final_audit_df[
        final_audit_df[
            "family"
        ]
        == family
    ]


    n = len(
        subset
    )

    agree = int(
        subset[
            "agreement"
        ].sum()
    )


    family_summary[
        family
    ] = {
        "n": int(
            n
        ),

        "agreement": int(
            agree
        ),
    }


    print(
        f"{family:30} | "
        f"{agree:3d}/{n:3d}"
    )


# ============================================================
# 12. CONTRASTIVE TRIO RECHECK
# ============================================================

def scenario_id_from_example(
    example_id,
):

    return re.sub(
        r"_(S|R|N)$",
        "",
        str(
            example_id
        ),
    )


scenario_groups = defaultdict(
    list
)


for row in all_rows:

    scenario_groups[
        scenario_id_from_example(
            row[
                "id"
            ]
        )
    ].append(
        row
    )


assert len(
    scenario_groups
) == 75


trio_failures = []


for scenario_id, rows in (
    scenario_groups.items()
):

    labels = {
        row["label"]
        for row in rows
    }


    evidence_versions = {
        json.dumps(
            row["evidence"],
            ensure_ascii=False,
            sort_keys=True,
        )

        for row in rows
    }


    if (
        len(rows) != 3
        or labels != LABELS
        or len(
            evidence_versions
        ) != 1
    ):

        trio_failures.append(
            scenario_id
        )


assert len(
    trio_failures
) == 0


print("\n" + "=" * 78)
print("CONTRASTIVE TRIOS")
print("=" * 78)


print(
    "Scenarios:",
    len(
        scenario_groups
    ),
)

print(
    "Valid trios:",
    75,
)

print(
    "Failures:",
    0,
)


# ============================================================
# 13. TRAIN / HOLDOUT SEPARATION RECHECK
# ============================================================

train_ids = {
    row["id"]
    for row in train_rows
}

holdout_ids = {
    row["id"]
    for row in holdout_rows
}


train_claims = {
    norm(
        row["claim"]
    )

    for row in train_rows
}

holdout_claims = {
    norm(
        row["claim"]
    )

    for row in holdout_rows
}


train_meta = pd.read_csv(
    TRAIN_META_PATH
)

holdout_meta = pd.read_csv(
    HOLDOUT_META_PATH
)


train_entities = set(
    train_meta[
        "entity"
    ].astype(str)
)

holdout_entities = set(
    holdout_meta[
        "entity"
    ].astype(str)
)


train_templates = set(
    train_meta[
        "template_id"
    ].astype(str)
)

holdout_templates = set(
    holdout_meta[
        "template_id"
    ].astype(str)
)


assert train_ids.isdisjoint(
    holdout_ids
)

assert train_claims.isdisjoint(
    holdout_claims
)

assert train_entities.isdisjoint(
    holdout_entities
)

assert train_templates.isdisjoint(
    holdout_templates
)


print("\n" + "=" * 78)
print("TRAIN / HOLDOUT SEPARATION")
print("=" * 78)


print(
    "ID overlap:",
    len(
        train_ids
        & holdout_ids
    ),
)

print(
    "Claim overlap:",
    len(
        train_claims
        & holdout_claims
    ),
)

print(
    "Entity overlap:",
    len(
        train_entities
        & holdout_entities
    ),
)

print(
    "Template-ID overlap:",
    len(
        train_templates
        & holdout_templates
    ),
)


# ============================================================
# 14. SAVE CORRECTED FINAL AUDIT
# ============================================================

final_audit_df.to_csv(
    FINAL_AUDIT_CSV,
    index=False,
)


# Empty failure file, with same columns.

final_audit_df.iloc[
    0:0
].to_csv(
    FINAL_FAILURES_CSV,
    index=False,
)


# ============================================================
# 15. AUDIT SUMMARY
# ============================================================

audit_summary = {
    "audit_name": (
        "D4_SYNTHETIC_225_INDEPENDENT_SEMANTIC_AUDIT"
    ),

    "status": (
        "PASSED"
    ),

    "audit_revision": (
        "3B"
    ),

    "auditor_correction": {
        "issue": (
            "Original revenue auditor reversed "
            "the semantic direction of claims "
            "phrased as 'higher in NEW_YEAR than "
            "in OLD_YEAR'."
        ),

        "dataset_rows_changed": 0,

        "labels_changed": 0,

        "auditor_only_changed": True,

        "previous_false_disagreements": 5,

        "corrected_revenue_examples_checked": 60,

        "corrected_revenue_agreement": 60,
    },

    "train": {
        "filename": (
            TRAIN_PATH.name
        ),

        "sha256": (
            train_sha
        ),

        "rows": 150,

        "semantic_agreement": 150,
    },

    "frozen_holdout": {
        "filename": (
            HOLDOUT_PATH.name
        ),

        "sha256": (
            holdout_sha
        ),

        "rows": 75,

        "semantic_agreement": 75,

        "use_for_training": False,
    },

    "total_examples": 225,

    "semantic_agreements": 225,

    "parser_failures": 0,

    "label_disagreements": 0,

    "contrastive_scenarios": 75,

    "contrastive_trio_failures": 0,

    "family_summary": (
        family_summary
    ),
}


with open(
    AUDIT_SUMMARY,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 16. FINAL FROZEN MANIFEST
# ============================================================

frozen_manifest = {
    "artifact": (
        "D4_CONTRASTIVE_CURRICULUM_V1_AUDITED"
    ),

    "status": (
        "SEMANTICALLY_AUDITED_AND_FROZEN"
    ),

    "train": {
        "filename": (
            TRAIN_PATH.name
        ),

        "sha256": (
            train_sha
        ),

        "rows": 150,

        "use_for_training": True,
    },

    "frozen_holdout": {
        "filename": (
            HOLDOUT_PATH.name
        ),

        "sha256": (
            holdout_sha
        ),

        "rows": 75,

        "use_for_training": False,

        "status": (
            "FROZEN_BEFORE_FIRST_D4_TRAINING_RUN"
        ),
    },

    "semantic_audit": {
        "examples_checked": 225,

        "agreements": 225,

        "parser_failures": 0,

        "label_disagreements": 0,

        "contrastive_trio_failures": 0,

        "auditor_revision": (
            "3B"
        ),
    },

    "experimental_rule": (
        "The frozen holdout must never be "
        "included in optimization. Do not "
        "generate additional training examples "
        "from its individual errors before the "
        "planned D4 experiment is evaluated."
    ),
}


with open(
    FINAL_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        frozen_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 17. CREATE AUDITED BACKUP ZIP
# ============================================================

if BACKUP_DIR.exists():

    shutil.rmtree(
        BACKUP_DIR
    )


BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


files_to_copy = [
    TRAIN_PATH,
    HOLDOUT_PATH,
    TRAIN_META_PATH,
    HOLDOUT_META_PATH,
    ORIGINAL_MANIFEST,
    FINAL_MANIFEST,
    FINAL_AUDIT_CSV,
    AUDIT_SUMMARY,
    BUGGY_AUDIT_BACKUP,
]


for source in files_to_copy:

    assert source.exists(), source


    shutil.copy2(
        source,
        BACKUP_DIR
        / source.name,
    )


README_PATH = (
    BACKUP_DIR
    / "README_DO_NOT_CONTAMINATE_HOLDOUT.txt"
)


README_PATH.write_text(
    f"""D4 CONTRASTIVE CURRICULUM V1 — AUDITED

TRAIN
-----
File:
{TRAIN_PATH.name}

Rows:
150

SHA256:
{train_sha}

Use for D4 training:
YES


FROZEN HOLDOUT
--------------
File:
{HOLDOUT_PATH.name}

Rows:
75

SHA256:
{holdout_sha}

Use for training:
NO


SEMANTIC AUDIT
--------------
Final agreement:
225 / 225

Train:
150 / 150

Frozen holdout:
75 / 75

Parser failures:
0

Label disagreements:
0

Contrastive scenario failures:
0


AUDITOR REVISION NOTE
---------------------
The first Cell-3 run reported five apparent revenue
disagreements.

Those were caused by an auditor direction bug for wording
such as:

    "20% higher in 2023 than in 2022"

The first auditor reversed old/new year semantics.

No generated examples were changed.
No labels were changed.

A corrected phrase-aware revenue auditor then independently
checked all 60 revenue examples and obtained 60/60 agreement.


EXPERIMENTAL RULE
-----------------
Never add the frozen holdout to training.

Do not use individual frozen-holdout failures to create
additional training examples before evaluating the planned
D4 experiment.
""",
    encoding="utf-8",
)


existing_zip = Path(
    str(
        ZIP_BASE
    )
    + ".zip"
)


if existing_zip.exists():

    existing_zip.unlink()


zip_path = shutil.make_archive(
    base_name=str(
        ZIP_BASE
    ),

    format="zip",

    root_dir=str(
        BACKUP_DIR.parent
    ),

    base_dir=(
        BACKUP_DIR.name
    ),
)


zip_path = Path(
    zip_path
)


zip_sha = sha256_file(
    zip_path
)

zip_mb = (
    zip_path.stat().st_size
    / 1024**2
)


# ============================================================
# 18. OPTIONAL READ-ONLY HOLDOUT REMINDER
# ============================================================

try:

    HOLDOUT_PATH.chmod(
        0o444
    )

except Exception:

    pass


# ============================================================
# 19. FINAL REPORT
# ============================================================

print("\n" + "=" * 78)
print("D4 DATASET — FINAL AUDIT PASSED")
print("=" * 78)


print(
    "Semantic agreement:",
    "225/225",
)

print(
    "Train:",
    "150/150",
)

print(
    "Frozen holdout:",
    "75/75",
)

print(
    "Parser failures:",
    0,
)

print(
    "Label disagreements:",
    0,
)

print(
    "Contrastive trio failures:",
    0,
)


print(
    "\nDATASET CONTENT CHANGES:",
    0,
)

print(
    "LABEL CHANGES:",
    0,
)

print(
    "AUDITOR FIX ONLY:",
    True,
)


print(
    "\nTRAIN SHA256:"
)

print(
    train_sha
)


print(
    "\nFROZEN HOLDOUT SHA256:"
)

print(
    holdout_sha
)


print(
    "\nFinal audit CSV:",
    FINAL_AUDIT_CSV,
)

print(
    "Final manifest:",
    FINAL_MANIFEST,
)


print(
    "\nAudited backup ZIP:",
    zip_path,
)

print(
    "ZIP size:",
    f"{zip_mb:.3f} MB",
)

print(
    "ZIP SHA256:",
    zip_sha,
)


print("\n" + "=" * 78)
print("CELL 3B COMPLETE")
print("=" * 78)


print(
    "\nNO MODEL was loaded."
)

print(
    "NO GPU was used."
)

print(
    "NO inference occurred."
)

print(
    "NO training occurred."
)


print(
    "\nNEXT:"
)

print(
    "Send me the CELL 3B output."
)

print(
    "If it reports 225/225, "
    "we proceed to the first D4 training run."
)

In [ ]:
# ============================================================
# CELL 4A — VERIFY RECOVERED V3 CONTENT AFTER RUNTIME RESET
#
# This checks semantic/content identity rather than requiring
# the reconstructed JSONL to have the same byte-level SHA as
# the old pre-reset file.
#
# NO GPU
# NO MODEL
# NO TRAINING
# ============================================================

import json
import hashlib
from pathlib import Path
from collections import Counter


WORK = Path("/kaggle/working")

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

CLEAN_PATH = (
    RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

FIT_PATH = (
    RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

DEV_PATH = (
    RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)


EXPECTED_FIT_COUNTS = {
    "SUPPORTS": 296,
    "REFUTES": 264,
    "NOT_ENOUGH_INFO": 235,
}

EXPECTED_DEV_COUNTS = {
    "SUPPORTS": 52,
    "REFUTES": 47,
    "NOT_ENOUGH_INFO": 41,
}

EXPECTED_CLEAN_COUNTS = {
    "SUPPORTS": 348,
    "REFUTES": 311,
    "NOT_ENOUGH_INFO": 276,
}


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def canonical_row(row):

    return {
        "id": str(row["id"]),
        "claim": str(row["claim"]),
        "evidence": list(row["evidence"]),
        "label": str(row["label"]),
    }


def semantic_sha(rows):

    # Stable content hash independent of JSONL formatting
    # and dictionary key serialization.

    canonical = [
        canonical_row(row)
        for row in rows
    ]

    canonical = sorted(
        canonical,
        key=lambda x: x["id"],
    )

    material = "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        for row in canonical
    )

    return hashlib.sha256(
        material.encode("utf-8")
    ).hexdigest()


for path in [
    CLEAN_PATH,
    FIT_PATH,
    DEV_PATH,
]:

    assert path.exists(), path


clean = read_jsonl(CLEAN_PATH)
fit = read_jsonl(FIT_PATH)
dev = read_jsonl(DEV_PATH)


print("=" * 78)
print("CELL 4A — RECOVERED V3 VERIFICATION")
print("=" * 78)

print("\nBYTE-LEVEL FILE SHAs")
print("Clean:", sha256_file(CLEAN_PATH))
print("Fit:  ", sha256_file(FIT_PATH))
print("Dev:  ", sha256_file(DEV_PATH))


print("\nROWS")
print("Clean:", len(clean))
print("Fit:  ", len(fit))
print("Dev:  ", len(dev))


assert len(clean) == 935
assert len(fit) == 795
assert len(dev) == 140


clean_counts = Counter(
    row["label"]
    for row in clean
)

fit_counts = Counter(
    row["label"]
    for row in fit
)

dev_counts = Counter(
    row["label"]
    for row in dev
)


print("\nLABEL COUNTS")
print("Clean:", clean_counts)
print("Fit:  ", fit_counts)
print("Dev:  ", dev_counts)


assert dict(clean_counts) == EXPECTED_CLEAN_COUNTS
assert dict(fit_counts) == EXPECTED_FIT_COUNTS
assert dict(dev_counts) == EXPECTED_DEV_COUNTS


clean_ids = {
    str(row["id"])
    for row in clean
}

fit_ids = {
    str(row["id"])
    for row in fit
}

dev_ids = {
    str(row["id"])
    for row in dev
}


assert len(clean_ids) == 935
assert len(fit_ids) == 795
assert len(dev_ids) == 140

assert fit_ids.isdisjoint(dev_ids)

assert (
    fit_ids | dev_ids
) == clean_ids


# ============================================================
# Verify that each fit/dev row is byte-content equivalent
# at the Python-object level to its corresponding clean row.
# ============================================================

clean_map = {
    str(row["id"]): canonical_row(row)
    for row in clean
}


fit_mismatches = []

for row in fit:

    rid = str(row["id"])

    if canonical_row(row) != clean_map[rid]:

        fit_mismatches.append(rid)


dev_mismatches = []

for row in dev:

    rid = str(row["id"])

    if canonical_row(row) != clean_map[rid]:

        dev_mismatches.append(rid)


print("\nCONTENT CONSISTENCY")
print(
    "Fit rows differing from clean source:",
    len(fit_mismatches),
)

print(
    "Dev rows differing from clean source:",
    len(dev_mismatches),
)


assert len(fit_mismatches) == 0
assert len(dev_mismatches) == 0


print("\nSEMANTIC / CANONICAL SHAs")
print(
    "Clean semantic SHA:",
    semantic_sha(clean),
)

print(
    "Fit semantic SHA:  ",
    semantic_sha(fit),
)

print(
    "Dev semantic SHA:  ",
    semantic_sha(dev),
)


print("\n" + "=" * 78)
print("RECOVERED V3 CONTENT — VERIFIED")
print("=" * 78)

print(
    "935 = 795 fit + 140 dev"
)

print(
    "Fit/dev overlap = 0"
)

print(
    "Fit + dev exactly reconstruct clean935"
)

print(
    "All label distributions match D1."
)

print(
    "All fit/dev row contents match their clean935 source."
)

print(
    "\nThe old file SHA mismatch is therefore only a "
    "serialization/byte-format issue."
)

print(
    "\nSend me this output. I will give you the corrected "
    "Cell 4 with the recovered SHAs."
)

In [ ]:
# ============================================================
# CELL 4B — PROCESSOR/TOKENIZER COMPATIBILITY CHECK
#
# NO MODEL LOAD
# NO TRAINING
# ============================================================

import os
import json
from pathlib import Path

import torch
import transformers

from transformers import (
    AutoProcessor,
    AutoTokenizer,
)


MODEL_ID = "google/gemma-4-12B-it"


# ------------------------------------------------------------
# HF TOKEN
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:

    try:

        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


assert hf_token, "HF_TOKEN unavailable."


print("=" * 78)
print("CELL 4B — PROCESSOR/TOKENIZER CHECK")
print("=" * 78)

print(
    "transformers:",
    transformers.__version__,
)

print(
    "torch:",
    torch.__version__,
)


# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=hf_token,
)


print(
    "\nAutoProcessor returned:",
    processor.__class__.__name__,
)


# New/fallback behavior:
#
# Case A:
#   Gemma4UnifiedProcessor
#       -> processor.tokenizer exists
#
# Case B:
#   TokenizersBackend / tokenizer itself
#       -> processor IS the tokenizer

if hasattr(processor, "tokenizer"):

    tokenizer = processor.tokenizer

    processor_mode = (
        "WRAPPER_WITH_TOKENIZER"
    )

else:

    tokenizer = processor

    processor_mode = (
        "PROCESSOR_IS_TOKENIZER"
    )


print(
    "Mode:",
    processor_mode,
)

print(
    "Tokenizer class:",
    tokenizer.__class__.__name__,
)


assert hasattr(
    processor,
    "apply_chat_template",
), (
    "Returned object lacks apply_chat_template."
)


assert hasattr(
    tokenizer,
    "decode",
), (
    "Tokenizer lacks decode()."
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


# ------------------------------------------------------------
# TEST THE EXACT D1/D4 CHAT FORMAT
# ------------------------------------------------------------

test_messages = [
    {
        "role": "user",

        "content": [
            {
                "type": "text",

                "text": (
                    "Classify the claim using only "
                    "the supplied evidence.\n\n"
                    "Claim:\n"
                    "Example claim.\n\n"
                    "Evidence:\n"
                    "[1] Example evidence."
                ),
            }
        ],
    }
]


encoded = processor.apply_chat_template(
    test_messages,

    tokenize=True,

    return_dict=True,

    return_tensors="pt",

    add_generation_prompt=True,

    enable_thinking=False,
)


print(
    "\nReturned keys:",
    list(encoded.keys()),
)


assert "input_ids" in encoded


ids = encoded["input_ids"][0]


print(
    "Token count:",
    len(ids),
)


tail_ids = ids[
    -30:
]


tail_text = tokenizer.decode(
    tail_ids,
    skip_special_tokens=False,
)


print("\nDECODED PREFIX TAIL")
print("-" * 78)

print(
    repr(tail_text)
)

print("-" * 78)


# ------------------------------------------------------------
# VERIFY ANSWER TOKENIZATION
# ------------------------------------------------------------

for label in [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]:

    answer = (
        "FINAL: "
        + label
    )

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]


    decoded = tokenizer.decode(
        answer_ids,
        skip_special_tokens=False,
    )


    print(
        f"{label:16} | "
        f"tokens={len(answer_ids):2d} | "
        f"{repr(decoded)}"
    )


print("\n" + "=" * 78)
print("CELL 4B PASSED")
print("=" * 78)

print(
    "processor.apply_chat_template works."
)

print(
    "tokenizer.decode works."
)

print(
    "This runtime can proceed with the "
    "text-only Gemma-4 D4 pipeline."
)

In [ ]:
# ============================================================
# CELL 4C — RESTORE KNOWN-GOOD D1 SOFTWARE ENVIRONMENT
#
# Current broken environment:
#   transformers 5.0.0
#
# Known-good environment used successfully for D1/D2/D3:
#   transformers 5.10.1
#   peft         0.19.1
#   bitsandbytes 0.50.1
#
# After this cell:
#   RESTART THE NOTEBOOK KERNEL / SESSION ONCE.
#
# DO NOT rerun Cells 1–3B.
# ============================================================

import sys
import subprocess


packages = [
    "transformers==5.10.1",
    "peft==0.19.1",
    "bitsandbytes==0.50.1",
]


print("=" * 78)
print("CELL 4C — RESTORE D1 PACKAGE VERSIONS")
print("=" * 78)

print("Installing:")
for package in packages:
    print(" ", package)


subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        *packages,
    ]
)


# ------------------------------------------------------------
# IMPORTANT:
# Check versions in a FRESH subprocess because this notebook
# process may still have transformers 5.0.0 imported in memory.
# ------------------------------------------------------------

check_code = r'''
import transformers
import peft
import bitsandbytes

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

from transformers import (
    Gemma4UnifiedConfig,
    Gemma4UnifiedForConditionalGeneration,
)

print("Gemma4UnifiedConfig import: PASS")
print("Gemma4UnifiedForConditionalGeneration import: PASS")
'''


print("\n" + "=" * 78)
print("FRESH-PROCESS VERIFICATION")
print("=" * 78)


subprocess.check_call(
    [
        sys.executable,
        "-c",
        check_code,
    ]
)


print("\n" + "=" * 78)
print("CELL 4C COMPLETE")
print("=" * 78)

print(
    "\nNow RESTART the notebook kernel/session once."
)

print(
    "Do NOT factory-reset/delete the Kaggle session."
)

print(
    "After restart, do NOT rerun Cells 1, 2, 3 or 3B."
)

print(
    "Send me this Cell-4C output first."
)

In [1]:
# ============================================================
# CELL 4D — POST-RESTART FINAL PREFLIGHT
#
# PURPOSE:
# Verify that the restarted Python process is actually using
# the known-good Gemma-4 environment and that all D4 files
# survived the kernel restart.
#
# NO 12B MODEL LOAD
# NO TRAINING
# NO INFERENCE
# ============================================================

import os
import hashlib
from pathlib import Path

import torch
import transformers
import peft
import bitsandbytes

from transformers import (
    AutoConfig,
    AutoProcessor,
    Gemma4UnifiedConfig,
    Gemma4UnifiedForConditionalGeneration,
)


# ============================================================
# 0. EXPECTED ENVIRONMENT
# ============================================================

assert transformers.__version__ == "5.10.1", (
    f"Wrong transformers version: "
    f"{transformers.__version__}"
)

assert peft.__version__ == "0.19.1", (
    f"Wrong PEFT version: {peft.__version__}"
)

assert bitsandbytes.__version__ == "0.50.1", (
    f"Wrong bitsandbytes version: "
    f"{bitsandbytes.__version__}"
)


print("=" * 78)
print("CELL 4D — POST-RESTART FINAL PREFLIGHT")
print("=" * 78)

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("torch:", torch.__version__)


# ============================================================
# 1. PATHS
# ============================================================

WORK = Path("/kaggle/working")

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

D4_DIR = (
    WORK
    / "d4_contrastive_curriculum_v1"
)


FIT_PATH = (
    RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

DEV_PATH = (
    RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)

SYNTH_PATH = (
    D4_DIR
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT_PATH = (
    D4_DIR
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)

MANIFEST_PATH = (
    D4_DIR
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


EXPECTED = {
    FIT_PATH:
        "f69970553bdcb8529933d25fa5c83e9185fdbebb533177b625802dd158867677",

    DEV_PATH:
        "5f9695233d389d56ac48fe8cbee8622fbd9c43bd61f4ac8df8e18788128e395b",

    SYNTH_PATH:
        "17258cbc0e40f4ebd1cd4d583e3a331e59217d62cf4ff36741e8b1e3a7a98f41",

    HOLDOUT_PATH:
        "61f41a537f738ecd153474565c1e22bb678c4f8a9dd096ed9eabfe5bddc2e1b2",
}


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


print("\n" + "=" * 78)
print("ARTIFACT CHECK")
print("=" * 78)


for path, expected_sha in EXPECTED.items():

    assert path.exists(), (
        f"Missing after restart: {path}"
    )

    actual_sha = sha256_file(path)

    assert actual_sha == expected_sha, (
        f"SHA mismatch:\n"
        f"{path}\n"
        f"expected={expected_sha}\n"
        f"actual={actual_sha}"
    )

    print(
        f"PASS | {path.name}"
    )


assert MANIFEST_PATH.exists()

print(
    "PASS |",
    MANIFEST_PATH.name,
)


# ============================================================
# 2. D1 CHAMPION ADAPTER
# ============================================================

D1_ADAPTER_FILE = Path(
    "/kaggle/input/datasets/"
    "omerfarooq223/d1-dataset/"
    "D1_GEMMA4_12B_CHAMPION/"
    "epoch_2_adapter/"
    "adapter_model.safetensors"
)

EXPECTED_D1_SHA = (
    "2c64a87e227e0638be9dc73e021f877f"
    "2536876ccf94cc116e04d60075164f8f"
)


assert D1_ADAPTER_FILE.exists()

assert (
    sha256_file(D1_ADAPTER_FILE)
    == EXPECTED_D1_SHA
)


print(
    "PASS | D1 champion adapter"
)


# ============================================================
# 3. GPU CLEANLINESS
# ============================================================

print("\n" + "=" * 78)
print("GPU CHECK")
print("=" * 78)


assert torch.cuda.is_available(), (
    "GPU is not enabled."
)


print(
    "GPU count:",
    torch.cuda.device_count(),
)

print(
    "GPU0:",
    torch.cuda.get_device_name(0),
)


allocated = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print(
    "GPU0 allocated:",
    f"{allocated:.3f} GB",
)


assert allocated < 0.5, (
    "GPU0 is unexpectedly occupied."
)


# ============================================================
# 4. HF TOKEN
# ============================================================

hf_token = os.environ.get("HF_TOKEN")


if not hf_token:

    try:

        from kaggle_secrets import UserSecretsClient

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:

        hf_token = None


assert hf_token, (
    "HF_TOKEN unavailable."
)


# ============================================================
# 5. GEMMA-4 CONFIG RECOGNITION
#
# This is the exact point that failed under transformers 5.0.0.
# ============================================================

MODEL_ID = "google/gemma-4-12B-it"


print("\n" + "=" * 78)
print("GEMMA-4 ARCHITECTURE CHECK")
print("=" * 78)


config = AutoConfig.from_pretrained(
    MODEL_ID,
    token=hf_token,
)


print(
    "Config class:",
    config.__class__.__name__,
)

print(
    "model_type:",
    config.model_type,
)


assert isinstance(
    config,
    Gemma4UnifiedConfig,
)

assert (
    config.model_type
    == "gemma4_unified"
)


print(
    "Gemma4UnifiedConfig recognition: PASS"
)


# ============================================================
# 6. PROCESSOR CHECK
# ============================================================

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=hf_token,
)


if hasattr(
    processor,
    "tokenizer",
):

    tokenizer = processor.tokenizer

else:

    tokenizer = processor


print(
    "Processor class:",
    processor.__class__.__name__,
)

print(
    "Tokenizer class:",
    tokenizer.__class__.__name__,
)


assert hasattr(
    processor,
    "apply_chat_template",
)

assert hasattr(
    tokenizer,
    "decode",
)


# Exact generation-prefix sanity check.

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Test."
            }
        ],
    }
]


encoded = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
)


tail = tokenizer.decode(
    encoded["input_ids"][0][-20:],
    skip_special_tokens=False,
)


print(
    "\nPrefix tail:",
    repr(tail),
)


assert "<|turn>model" in tail
assert "<|channel>thought" in tail
assert "<channel|>" in tail


print(
    "Gemma-4 generation prefix: PASS"
)


# ============================================================
# 7. FINAL
# ============================================================

print("\n" + "=" * 78)
print("CELL 4D — ALL CHECKS PASSED")
print("=" * 78)

print(
    "Environment: READY"
)

print(
    "Recovered D1 data: READY"
)

print(
    "D4 synthetic train150: READY"
)

print(
    "Frozen holdout75: READY"
)

print(
    "D1 champion adapter: READY"
)

print(
    "Gemma4 Unified architecture: RECOGNIZED"
)

print(
    "GPU0: CLEAN"
)

print(
    "\nNO 12B weights were loaded."
)

print(
    "NO training occurred."
)

print(
    "\nSend me this output."
)

CELL 4D — POST-RESTART FINAL PREFLIGHT
transformers: 5.10.1
peft: 0.19.1
bitsandbytes: 0.50.1
torch: 2.10.0+cu128

ARTIFACT CHECK
PASS | train_fit_v3_semantic_recovered.jsonl
PASS | internal_dev_v3_semantic_recovered.jsonl
PASS | d4_contrastive_train_v1.jsonl
PASS | D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl
PASS | D4_AUDITED_FROZEN_MANIFEST_V1.json
PASS | D1 champion adapter

GPU CHECK
GPU count: 2
GPU0: Tesla T4
GPU0 allocated: 0.000 GB

GEMMA-4 ARCHITECTURE CHECK
Config class: Gemma4UnifiedConfig
model_type: gemma4_unified
Gemma4UnifiedConfig recognition: PASS
Processor class: Gemma4UnifiedProcessor
Tokenizer class: GemmaTokenizer

Prefix tail: '<bos><|turn>user\nTest.<turn|>\n<|turn>model\n<|channel>thought\n<channel|>'
Gemma-4 generation prefix: PASS

CELL 4D — ALL CHECKS PASSED
Environment: READY
Recovered D1 data: READY
D4 synthetic train150: READY
Frozen holdout75: READY
D1 champion adapter: READY
Gemma4 Unified architecture: RECOGNIZED
GPU0: CLEAN

NO 12B weights were loaded.
NO 

In [2]:
# ============================================================
# CELL 4 — D4 CONTROLLED TRAINING EXPERIMENT
#
# QUESTION:
# Does adding the audited 150-example contrastive curriculum
# improve D1's reasoning/generalization?
#
# ------------------------------------------------------------
# D1:
#   real V3 fit = 795
#
# D4:
#   same real V3 fit = 795
#   + audited synthetic train = 150
#   --------------------------------
#   total = 945
#
# ONLY DATA CHANGES.
#
# SAME WINNING D1 RECIPE:
#   google/gemma-4-12B-it
#   4-bit NF4
#   FP16 compute
#   LoRA r=8
#   alpha=16
#   dropout=.05
#   q/k/v/o/gate/up/down
#   BASE prompt
#   completion-only loss
#   batch=1
#   grad accumulation=16
#   LR=2e-4
#   AdamW
#   weight decay=.01
#   cosine scheduler
#   warmup ratio=5%
#   2 epochs
#   seed=42
#   thinking=False
#   greedy inference
#
# EVALUATION POLICY:
#
#   1. D1 frozen baseline:
#        evaluate NEW frozen holdout75 ONCE.
#
#   2. D4:
#        epoch1 -> V3 dev only
#        epoch2 -> V3 dev
#               -> frozen holdout ONCE
#
# We intentionally DO NOT inspect individual frozen-holdout
# failures here.
#
# NO organizer validation.
# NO external30.
# NO old synthetic180.
# ============================================================

import os
import re
import gc
import json
import math
import time
import random
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
)

from transformers import (
    AutoProcessor,
    Gemma4UnifiedForConditionalGeneration,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
)


# ============================================================
# 0. CONFIG
# ============================================================

MODEL_ID = "google/gemma-4-12B-it"

WORK = Path(
    "/kaggle/working"
)

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

D4_DATA_DIR = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

RUN_DIR = (
    WORK
    / "d4_gemma4_12b_d1recipe_real795_plus_audited150"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

REAL_FIT_PATH = (
    RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

DEV_PATH = (
    RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)

SYNTH_TRAIN_PATH = (
    D4_DATA_DIR
    / "d4_contrastive_train_v1.jsonl"
)

FROZEN_HOLDOUT_PATH = (
    D4_DATA_DIR
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)

HOLDOUT_META_PATH = (
    D4_DATA_DIR
    / "D4_FROZEN_HOLDOUT_V1_metadata.csv"
)

AUDITED_MANIFEST = (
    D4_DATA_DIR
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


# ------------------------------------------------------------
# EXPECTED HASHES
# ------------------------------------------------------------

EXPECTED_REAL_FIT_SHA = (
    "f69970553bdcb8529933d25fa5c83e9185fdbebb533177b625802dd158867677"
)

EXPECTED_DEV_SHA = (
    "5f9695233d389d56ac48fe8cbee8622fbd9c43bd61f4ac8df8e18788128e395b"
)

EXPECTED_SYNTH_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)

EXPECTED_D1_ADAPTER_SHA = (
    "2c64a87e227e0638be9dc73e021f877f"
    "2536876ccf94cc116e04d60075164f8f"
)


# ------------------------------------------------------------
# TRAINING RECIPE
# ------------------------------------------------------------

SEED = 42

MAX_LENGTH = 256

LORA_R = 8

LORA_ALPHA = 16

LORA_DROPOUT = 0.05

LR = 2e-4

WEIGHT_DECAY = 0.01

GRAD_ACCUM = 16

EPOCHS = 2

WARMUP_RATIO = 0.05


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


BASE_PROMPT = """Classify the claim using only the supplied evidence.

SUPPORTS: the evidence establishes the claim.
REFUTES: the evidence contradicts the claim.
NOT_ENOUGH_INFO: the evidence neither establishes nor contradicts the specific claim.

End your response exactly as:
FINAL: SUPPORTS
or
FINAL: REFUTES
or
FINAL: NOT_ENOUGH_INFO"""


TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\."
        r"(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\."
        r"(?:gate_proj|up_proj|down_proj)"
    r")"
)


# ============================================================
# 1. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(
                        line
                    )
                )

    return rows


def evidence_list(row):

    evidence = row[
        "evidence"
    ]

    if isinstance(
        evidence,
        str,
    ):

        return [
            evidence
        ]

    return list(
        evidence
    )


def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"

        for i, passage
        in enumerate(
            evidence_list(
                row
            ),
            1,
        )
    )


    return (
        BASE_PROMPT
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",

                    "text": (
                        build_user_text(
                            row
                        )
                    ),
                }
            ],
        }
    ]


FINAL_RE = re.compile(
    r"FINAL\s*:\s*"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)",
    flags=re.I,
)


LABEL_RE = re.compile(
    r"\b"
    r"(NOT_ENOUGH_INFO|SUPPORTS|REFUTES)"
    r"\b",
    flags=re.I,
)


def parse_prediction(text):

    matches = list(
        FINAL_RE.finditer(
            text
        )
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    matches = list(
        LABEL_RE.finditer(
            text
        )
    )


    if matches:

        return (
            matches[-1]
            .group(1)
            .upper()
        )


    return None


def seed_everything():

    random.seed(
        SEED
    )

    np.random.seed(
        SEED
    )

    torch.manual_seed(
        SEED
    )

    torch.cuda.manual_seed_all(
        SEED
    )


    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


seed_everything()


# ============================================================
# 2. VERIFY FILES + HASHES
# ============================================================

for path in [
    REAL_FIT_PATH,
    DEV_PATH,
    SYNTH_TRAIN_PATH,
    FROZEN_HOLDOUT_PATH,
    HOLDOUT_META_PATH,
    AUDITED_MANIFEST,
]:

    assert path.exists(), path


assert (
    sha256_file(
        REAL_FIT_PATH
    )
    == EXPECTED_REAL_FIT_SHA
)

assert (
    sha256_file(
        DEV_PATH
    )
    == EXPECTED_DEV_SHA
)

assert (
    sha256_file(
        SYNTH_TRAIN_PATH
    )
    == EXPECTED_SYNTH_SHA
)

assert (
    sha256_file(
        FROZEN_HOLDOUT_PATH
    )
    == EXPECTED_HOLDOUT_SHA
)


real_rows = read_jsonl(
    REAL_FIT_PATH
)

synth_rows = read_jsonl(
    SYNTH_TRAIN_PATH
)

dev_rows = read_jsonl(
    DEV_PATH
)

holdout_rows = read_jsonl(
    FROZEN_HOLDOUT_PATH
)


assert len(real_rows) == 795
assert len(synth_rows) == 150
assert len(dev_rows) == 140
assert len(holdout_rows) == 75


# ------------------------------------------------------------
# CRITICAL CONTAMINATION CHECK
# ------------------------------------------------------------

real_ids = {
    str(row["id"])
    for row in real_rows
}

synth_ids = {
    str(row["id"])
    for row in synth_rows
}

dev_ids = {
    str(row["id"])
    for row in dev_rows
}

holdout_ids = {
    str(row["id"])
    for row in holdout_rows
}


assert real_ids.isdisjoint(
    synth_ids
)

assert real_ids.isdisjoint(
    dev_ids
)

assert synth_ids.isdisjoint(
    dev_ids
)

assert holdout_ids.isdisjoint(
    real_ids
    | synth_ids
    | dev_ids
)


train_rows = (
    real_rows
    + synth_rows
)


assert len(train_rows) == 945


print("=" * 78)
print("CELL 4 — D4 CONTROLLED TRAINING")
print("=" * 78)


print(
    "Real train:",
    len(real_rows),
)

print(
    "Synthetic train:",
    len(synth_rows),
)

print(
    "TOTAL D4 train:",
    len(train_rows),
)

print(
    "V3 dev:",
    len(dev_rows),
)

print(
    "Frozen holdout:",
    len(holdout_rows),
)


print(
    "\nD4 train labels:",
    Counter(
        row["label"]
        for row in train_rows
    ),
)


print(
    "\nFrozen holdout SHA:"
)

print(
    EXPECTED_HOLDOUT_SHA
)


# ============================================================
# 3. GPU CHECK
# ============================================================

gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


assert torch.cuda.is_available(), (
    "Enable Kaggle GPU first."
)


torch.cuda.set_device(
    0
)


allocated = (
    torch.cuda.memory_allocated(
        0
    )
    / 1024**3
)


print("\n" + "=" * 78)
print("GPU")
print("=" * 78)


print(
    "GPU0:",
    torch.cuda.get_device_name(
        0
    ),
)


print(
    "GPU count:",
    torch.cuda.device_count(),
)


print(
    "GPU0 allocated:",
    f"{allocated:.3f} GB",
)


if allocated > 0.5:

    raise RuntimeError(
        "GPU0 is not clean. "
        "Restart runtime before Cell 4."
    )


# ============================================================
# 4. HF TOKEN
# ============================================================

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )


        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )


    except Exception:

        hf_token = None


if not hf_token:

    raise RuntimeError(
        "HF_TOKEN unavailable."
    )


# ============================================================
# 5. PROCESSOR
# ============================================================

print("\n" + "=" * 78)
print("PROCESSOR")
print("=" * 78)

processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)

# Transformers-version compatible:
# - older behavior: Gemma4UnifiedProcessor with .tokenizer
# - current behavior: TokenizersBackend is already the tokenizer

if hasattr(
    processor,
    "tokenizer",
):
    tokenizer = processor.tokenizer
else:
    tokenizer = processor


if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


print(
    "Processor:",
    processor.__class__.__name__,
)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__,
)

print(
    "Processor is tokenizer:",
    processor is tokenizer,
)

# ============================================================
# 6. QUANTIZATION CONFIG
# ============================================================

quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,

        bnb_4bit_quant_type="nf4",

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


# ============================================================
# 7. GENERIC EVALUATION FUNCTION
# ============================================================

def evaluate_rows(
    model,
    rows,
    name,
    save_path,
    family_map=None,
):

    model.eval()


    predictions = []
    raw_outputs = []


    for index, row in enumerate(
        rows,
        1,
    ):

        inputs = (
            processor.apply_chat_template(
                user_messages(
                    row
                ),

                tokenize=True,

                return_dict=True,

                return_tensors="pt",

                add_generation_prompt=True,

                enable_thinking=False,
            )
        )


        inputs = {
            key: (
                value.to(
                    "cuda:0"
                )

                if torch.is_tensor(
                    value
                )

                else value
            )

            for key, value
            in inputs.items()
        }


        input_length = (
            inputs[
                "input_ids"
            ].shape[-1]
        )


        with torch.inference_mode():

            generated = model.generate(
                **inputs,

                do_sample=False,

                num_beams=1,

                max_new_tokens=24,

                pad_token_id=(
                    tokenizer.pad_token_id
                ),
            )


        new_tokens = (
            generated[
                0,
                input_length:
            ]
        )


        raw = tokenizer.decode(
            new_tokens,
            skip_special_tokens=False,
        )


        pred = parse_prediction(
            raw
        )


        predictions.append(
            pred
        )

        raw_outputs.append(
            raw
        )


        if (
            index % 25 == 0
            or index == len(rows)
        ):

            print(
                f"    {name}: "
                f"{index}/{len(rows)}"
            )


        del inputs
        del generated
        del new_tokens


    gold = [
        row["label"]
        for row in rows
    ]


    metric_preds = [
        p
        if p in LABELS
        else "__INVALID__"

        for p in predictions
    ]


    invalid = sum(
        p not in LABELS
        for p in predictions
    )


    correct = sum(
        g == p

        for g, p in zip(
            gold,
            metric_preds,
        )
    )


    accuracy = accuracy_score(
        gold,
        metric_preds,
    )


    macro_f1 = f1_score(
        gold,
        metric_preds,

        labels=LABELS,

        average="macro",

        zero_division=0,
    )


    cm = confusion_matrix(
        gold,
        metric_preds,

        labels=LABELS,
    )


    records = []


    for row, g, p, raw in zip(
        rows,
        gold,
        metric_preds,
        raw_outputs,
    ):

        record = {
            "id": str(
                row["id"]
            ),

            "gold": (
                g
            ),

            "prediction": (
                p
            ),

            "correct": (
                g == p
            ),

            "raw_output": (
                raw
            ),
        }


        if family_map is not None:

            record[
                "family"
            ] = family_map[
                str(
                    row["id"]
                )
            ]


        records.append(
            record
        )


    result_df = pd.DataFrame(
        records
    )


    result_df.to_csv(
        save_path,
        index=False,
    )


    result = {
        "correct": int(
            correct
        ),

        "total": int(
            len(rows)
        ),

        "accuracy": float(
            accuracy
        ),

        "macro_f1": float(
            macro_f1
        ),

        "invalid": int(
            invalid
        ),

        "confusion_matrix": (
            cm.tolist()
        ),
    }


    # --------------------------------------------------------
    # Aggregate family results ONLY.
    # No individual frozen-holdout errors printed.
    # --------------------------------------------------------

    if family_map is not None:

        family_results = {}


        for family in sorted(
            result_df[
                "family"
            ].unique()
        ):

            subset = result_df[
                result_df[
                    "family"
                ]
                == family
            ]


            f_gold = (
                subset[
                    "gold"
                ].tolist()
            )

            f_pred = (
                subset[
                    "prediction"
                ].tolist()
            )


            family_results[
                family
            ] = {
                "n": int(
                    len(
                        subset
                    )
                ),

                "correct": int(
                    subset[
                        "correct"
                    ].sum()
                ),

                "accuracy": float(
                    accuracy_score(
                        f_gold,
                        f_pred,
                    )
                ),

                "macro_f1": float(
                    f1_score(
                        f_gold,
                        f_pred,

                        labels=LABELS,

                        average="macro",

                        zero_division=0,
                    )
                ),
            }


        result[
            "family_results"
        ] = family_results


    return (
        result,
        result_df,
    )


def print_result(
    title,
    result,
):

    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


    print(
        "Correct:",
        f"{result['correct']}/{result['total']}",
    )

    print(
        "Accuracy:",
        f"{result['accuracy']:.4f}",
    )

    print(
        "Macro-F1:",
        f"{result['macro_f1']:.4f}",
    )

    print(
        "Invalid:",
        result[
            "invalid"
        ],
    )


    print(
        "\nConfusion matrix:"
    )


    for label, row in zip(
        LABELS,
        result[
            "confusion_matrix"
        ],
    ):

        print(
            f"{label:16} "
            f"{row}"
        )


# ============================================================
# 8. HOLDOUT FAMILY MAP
# ============================================================

holdout_meta = pd.read_csv(
    HOLDOUT_META_PATH
)


holdout_family_map = {
    str(row["id"]): str(
        row["family"]
    )

    for row in holdout_meta.to_dict(
        orient="records"
    )
}


assert all(
    str(row["id"])
    in holdout_family_map

    for row in holdout_rows
)


# ============================================================
# 9. LOCATE EXACT D1 CHAMPION ADAPTER
# ============================================================

print("\n" + "=" * 78)
print("LOCATE D1 CHAMPION")
print("=" * 78)


preferred_adapter = Path(
    "/kaggle/input/datasets/"
    "omerfarooq223/d1-dataset/"
    "D1_GEMMA4_12B_CHAMPION/"
    "epoch_2_adapter"
)


D1_ADAPTER_DIR = None


if (
    preferred_adapter.exists()
    and (
        preferred_adapter
        / "adapter_model.safetensors"
    ).exists()
):

    candidate_file = (
        preferred_adapter
        / "adapter_model.safetensors"
    )


    if (
        sha256_file(
            candidate_file
        )
        == EXPECTED_D1_ADAPTER_SHA
    ):

        D1_ADAPTER_DIR = (
            preferred_adapter
        )


if D1_ADAPTER_DIR is None:

    for candidate_file in Path(
        "/kaggle/input"
    ).rglob(
        "adapter_model.safetensors"
    ):

        try:

            if (
                sha256_file(
                    candidate_file
                )
                == EXPECTED_D1_ADAPTER_SHA
            ):

                D1_ADAPTER_DIR = (
                    candidate_file.parent
                )

                break

        except Exception:

            pass


assert D1_ADAPTER_DIR is not None, (
    "Exact D1 adapter not found."
)


print(
    "D1 adapter:",
    D1_ADAPTER_DIR,
)

print(
    "D1 SHA:",
    EXPECTED_D1_ADAPTER_SHA,
)


# ============================================================
# 10. D1 BASELINE ON THE NEW FROZEN HOLDOUT
#
# THIS IS THE FIRST AND ONLY PRE-D4 D1 LOOK AT HOLDOUT75.
# ============================================================

print("\n" + "=" * 78)
print("D1 — LOAD FOR FROZEN HOLDOUT BASELINE")
print("=" * 78)


base_model = (
    Gemma4UnifiedForConditionalGeneration
    .from_pretrained(
        MODEL_ID,

        token=hf_token,

        quantization_config=(
            quant_config
        ),

        device_map={
            "": 0
        },

        dtype=torch.float16,

        low_cpu_mem_usage=True,
    )
)


d1_model = (
    PeftModel
    .from_pretrained(
        base_model,

        D1_ADAPTER_DIR,

        is_trainable=False,
    )
)


d1_model.eval()


print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated(0)/1024**3:.3f} GB",
)


D1_HOLDOUT_PRED_PATH = (
    RUN_DIR
    / "D1_frozen_holdout75_predictions.csv"
)


d1_holdout_result, _ = (
    evaluate_rows(
        d1_model,

        holdout_rows,

        "D1 frozen holdout",

        D1_HOLDOUT_PRED_PATH,

        family_map=(
            holdout_family_map
        ),
    )
)


print_result(
    "D1 — FROZEN HOLDOUT75 BASELINE",
    d1_holdout_result,
)


print(
    "\nAggregate D1 family results:"
)


for family, values in (
    d1_holdout_result[
        "family_results"
    ].items()
):

    print(
        f"{family:30} | "
        f"{values['correct']:2d}/{values['n']:2d} | "
        f"F1={values['macro_f1']:.4f}"
    )


# ============================================================
# 11. COMPLETELY UNLOAD D1
# ============================================================

print("\n" + "=" * 78)
print("UNLOAD D1")
print("=" * 78)


del d1_model
del base_model


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


time.sleep(
    2
)


after_unload = (
    torch.cuda.memory_allocated(
        0
    )
    / 1024**3
)


print(
    "GPU allocated after unload:",
    f"{after_unload:.3f} GB",
)


if after_unload > 0.5:

    raise RuntimeError(
        "D1 did not fully unload. "
        "Restart runtime before D4 training."
    )


# ============================================================
# 12. BUILD EXACT D4 TRAINING TOKENS
# ============================================================

TURN_CLOSE_TEXT = (
    "<turn|>\n"
)


turn_close_ids = (
    tokenizer(
        TURN_CLOSE_TEXT,

        add_special_tokens=False,
    )[
        "input_ids"
    ]
)


def build_training_sample(
    row,
    source,
):

    prompt_encoded = (
        processor.apply_chat_template(
            user_messages(
                row
            ),

            tokenize=True,

            return_dict=True,

            return_tensors="pt",

            add_generation_prompt=True,

            enable_thinking=False,
        )
    )


    prompt_ids = (
        prompt_encoded[
            "input_ids"
        ][0]
        .tolist()
    )


    answer_ids = (
        tokenizer(
            (
                "FINAL: "
                + row[
                    "label"
                ]
            ),

            add_special_tokens=False,
        )[
            "input_ids"
        ]
    )


    full_ids = (
        prompt_ids
        + answer_ids
        + turn_close_ids
    )


    labels = (
        [-100]
        * len(
            prompt_ids
        )
        + answer_ids
        + turn_close_ids
    )


    assert len(
        full_ids
    ) == len(
        labels
    )


    return {
        "id": str(
            row["id"]
        ),

        "source": (
            source
        ),

        "input_ids": (
            torch.tensor(
                full_ids,
                dtype=torch.long,
            )
        ),

        "attention_mask": (
            torch.ones(
                len(
                    full_ids
                ),
                dtype=torch.long,
            )
        ),

        "labels": (
            torch.tensor(
                labels,
                dtype=torch.long,
            )
        ),

        "prompt_length": (
            len(
                prompt_ids
            )
        ),

        "full_length": (
            len(
                full_ids
            )
        ),

        "target_length": (
            len(
                answer_ids
            )
            + len(
                turn_close_ids
            )
        ),
    }


print("\n" + "=" * 78)
print("TOKENIZE D4 TRAIN945")
print("=" * 78)


train_samples = []

full_lengths = []

real_lengths = []

synth_lengths = []


for index, row in enumerate(
    real_rows,
    1,
):

    sample = (
        build_training_sample(
            row,
            "real",
        )
    )


    train_samples.append(
        sample
    )

    full_lengths.append(
        sample[
            "full_length"
        ]
    )

    real_lengths.append(
        sample[
            "full_length"
        ]
    )


for index, row in enumerate(
    synth_rows,
    1,
):

    sample = (
        build_training_sample(
            row,
            "synthetic",
        )
    )


    train_samples.append(
        sample
    )

    full_lengths.append(
        sample[
            "full_length"
        ]
    )

    synth_lengths.append(
        sample[
            "full_length"
        ]
    )


assert len(
    train_samples
) == 945


print(
    "Real lengths min/median/max:",
    min(
        real_lengths
    ),
    int(
        np.median(
            real_lengths
        )
    ),
    max(
        real_lengths
    ),
)


print(
    "Synthetic lengths min/median/max:",
    min(
        synth_lengths
    ),
    int(
        np.median(
            synth_lengths
        )
    ),
    max(
        synth_lengths
    ),
)


print(
    "Combined max:",
    max(
        full_lengths
    ),
)


if max(
    full_lengths
) > MAX_LENGTH:

    raise RuntimeError(
        f"D4 has sequence length "
        f"{max(full_lengths)} > "
        f"D1 MAX_LENGTH={MAX_LENGTH}. "
        f"STOP and send me this output."
    )


print(
    "945/945 fit within D1 max_length=256."
)


# ============================================================
# 13. FRESH BASE MODEL FOR D4
# ============================================================

print("\n" + "=" * 78)
print("D4 — LOAD FRESH GEMMA-4 12B")
print("=" * 78)


seed_everything()


base_model = (
    Gemma4UnifiedForConditionalGeneration
    .from_pretrained(
        MODEL_ID,

        token=hf_token,

        quantization_config=(
            quant_config
        ),

        device_map={
            "": 0
        },

        dtype=torch.float16,

        low_cpu_mem_usage=True,
    )
)


for parameter in (
    base_model.parameters()
):

    parameter.requires_grad = False


try:

    base_model.config.use_cache = False

except Exception:

    pass


# Reset immediately before LoRA initialization.

seed_everything()


# ============================================================
# 14. ATTACH EXACT D1 LoRA RECIPE
# ============================================================

lora_config = (
    LoraConfig(
        r=LORA_R,

        lora_alpha=(
            LORA_ALPHA
        ),

        lora_dropout=(
            LORA_DROPOUT
        ),

        bias="none",

        task_type="CAUSAL_LM",

        target_modules=(
            TARGET_REGEX
        ),
    )
)


model = get_peft_model(
    base_model,
    lora_config,
)


if hasattr(
    model,
    "enable_input_require_grads",
):

    model.enable_input_require_grads()


try:

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

except TypeError:

    model.gradient_checkpointing_enable()


trainable_params = sum(
    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad
)


assert trainable_params == 32_784_384


print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)

print(
    "Expected D1:",
    "32,784,384",
)


print(
    "GPU allocated:",
    f"{torch.cuda.memory_allocated(0)/1024**3:.3f} GB",
)


# ============================================================
# 15. OPTIMIZER / SCHEDULE
#
# Same D1 policy, but dataset is larger:
#
#   945 / 16 => 60 steps/epoch
#   2 epochs => 120 total
#   5% warmup => 6 steps
# ============================================================

steps_per_epoch = math.ceil(
    len(
        train_samples
    )
    / GRAD_ACCUM
)


total_steps = (
    steps_per_epoch
    * EPOCHS
)


warmup_steps = max(
    1,

    round(
        total_steps
        * WARMUP_RATIO
    ),
)


assert steps_per_epoch == 60
assert total_steps == 120
assert warmup_steps == 6


optimizer = torch.optim.AdamW(
    [
        parameter

        for parameter
        in model.parameters()

        if parameter.requires_grad
    ],

    lr=LR,

    weight_decay=(
        WEIGHT_DECAY
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,

        num_warmup_steps=(
            warmup_steps
        ),

        num_training_steps=(
            total_steps
        ),
    )
)


print("\n" + "=" * 78)
print("D4 TRAINING PLAN")
print("=" * 78)


print(
    "Training rows:",
    945,
)

print(
    "  real:",
    795,
)

print(
    "  synthetic:",
    150,
)


print(
    "Physical batch:",
    1,
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM,
)

print(
    "Effective batch:",
    16,
)

print(
    "Steps/epoch:",
    steps_per_epoch,
)

print(
    "Total optimizer steps:",
    total_steps,
)

print(
    "Warmup steps:",
    warmup_steps,
)

print(
    "LR:",
    LR,
)

print(
    "Epochs:",
    EPOCHS,
)


# ============================================================
# 16. BATCH HELPER
# ============================================================

def make_batch(sample):

    return {
        "input_ids": (
            sample[
                "input_ids"
            ]
            .unsqueeze(0)
            .to(
                "cuda:0"
            )
        ),

        "attention_mask": (
            sample[
                "attention_mask"
            ]
            .unsqueeze(0)
            .to(
                "cuda:0"
            )
        ),

        "labels": (
            sample[
                "labels"
            ]
            .unsqueeze(0)
            .to(
                "cuda:0"
            )
        ),
    }


# ============================================================
# 17. TRAIN TWO EPOCHS
# ============================================================

history = []

global_step = 0


torch.cuda.reset_peak_memory_stats(
    0
)


training_start = time.time()


print("\n" + "=" * 78)
print("D4 TRAINING START")
print("=" * 78)


for epoch in range(
    1,
    EPOCHS + 1,
):

    epoch_start = time.time()


    model.train()


    epoch_rng = random.Random(
        SEED + epoch
    )


    order = list(
        range(
            len(
                train_samples
            )
        )
    )


    epoch_rng.shuffle(
        order
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    epoch_loss = 0.0

    real_loss_sum = 0.0
    real_count = 0

    synth_loss_sum = 0.0
    synth_count = 0


    recent_loss = 0.0
    recent_count = 0

    optimizer_steps = 0


    for position, sample_index in enumerate(
        order,
        1,
    ):

        sample = (
            train_samples[
                sample_index
            ]
        )


        batch = make_batch(
            sample
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):

            outputs = model(
                **batch
            )


            raw_loss = (
                outputs.loss
            )


        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                f"Non-finite loss "
                f"epoch={epoch}, "
                f"id={sample['id']}"
            )


        (
            raw_loss
            / GRAD_ACCUM
        ).backward()


        loss_value = float(
            raw_loss
            .detach()
            .cpu()
        )


        epoch_loss += (
            loss_value
        )


        if (
            sample[
                "source"
            ]
            == "real"
        ):

            real_loss_sum += (
                loss_value
            )

            real_count += 1


        else:

            synth_loss_sum += (
                loss_value
            )

            synth_count += 1


        recent_loss += (
            loss_value
        )

        recent_count += 1


        should_step = (
            position
            % GRAD_ACCUM
            == 0

            or position
            == len(order)
        )


        if should_step:

            optimizer.step()

            scheduler.step()


            optimizer.zero_grad(
                set_to_none=True
            )


            optimizer_steps += 1
            global_step += 1


            if (
                optimizer_steps % 5
                == 0

                or optimizer_steps
                == steps_per_epoch
            ):

                print(
                    f"Epoch {epoch} | "
                    f"step "
                    f"{optimizer_steps:02d}/"
                    f"{steps_per_epoch} | "
                    f"global "
                    f"{global_step:03d}/"
                    f"{total_steps} | "
                    f"loss="
                    f"{recent_loss/recent_count:.4f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.8f} | "
                    f"gpu="
                    f"{torch.cuda.memory_allocated(0)/1024**3:.2f}GB | "
                    f"peak="
                    f"{torch.cuda.max_memory_allocated(0)/1024**3:.2f}GB"
                )


                recent_loss = 0.0
                recent_count = 0


        del outputs
        del raw_loss
        del batch


    assert optimizer_steps == 60


    avg_loss = (
        epoch_loss
        / len(
            train_samples
        )
    )


    avg_real_loss = (
        real_loss_sum
        / real_count
    )


    avg_synth_loss = (
        synth_loss_sum
        / synth_count
    )


    # ========================================================
    # SAVE EPOCH
    # ========================================================

    adapter_dir = (
        RUN_DIR
        / f"epoch_{epoch}_adapter"
    )


    model.save_pretrained(
        adapter_dir,

        safe_serialization=True,
    )


    processor.save_pretrained(
        adapter_dir
    )


    adapter_file = (
        adapter_dir
        / "adapter_model.safetensors"
    )


    adapter_sha = (
        sha256_file(
            adapter_file
        )
    )


    # ========================================================
    # V3 DEV — ALLOWED AFTER EACH EPOCH
    # ========================================================

    print("\n" + "-" * 78)

    print(
        f"D4 EPOCH {epoch} — V3 DEV140"
    )

    print("-" * 78)


    dev_pred_path = (
        RUN_DIR
        / f"v3_dev_epoch_{epoch}.csv"
    )


    dev_result, _ = (
        evaluate_rows(
            model,

            dev_rows,

            f"D4 epoch{epoch} dev",

            dev_pred_path,
        )
    )


    print_result(
        f"D4 EPOCH {epoch} — V3 DEV",
        dev_result,
    )


    result = {
        "epoch": (
            epoch
        ),

        "train_loss": (
            avg_loss
        ),

        "real_train_loss": (
            avg_real_loss
        ),

        "synthetic_train_loss": (
            avg_synth_loss
        ),

        "dev": (
            dev_result
        ),

        "adapter_sha256": (
            adapter_sha
        ),

        "adapter_dir": str(
            adapter_dir
        ),

        "epoch_seconds": (
            time.time()
            - epoch_start
        ),
    }


    history.append(
        result
    )


    print(
        "\nTrain loss overall:",
        f"{avg_loss:.4f}",
    )

    print(
        "Train loss real:",
        f"{avg_real_loss:.4f}",
    )

    print(
        "Train loss synthetic:",
        f"{avg_synth_loss:.4f}",
    )


    print(
        "Adapter SHA:",
        adapter_sha,
    )


# ============================================================
# 18. FROZEN HOLDOUT — D4 EPOCH2 ONLY
#
# This is intentionally NOT evaluated after epoch1.
# ============================================================

print("\n" + "=" * 78)
print("D4 EPOCH2 — FIRST AND ONLY FROZEN HOLDOUT EVALUATION")
print("=" * 78)


D4_HOLDOUT_PRED_PATH = (
    RUN_DIR
    / "D4_epoch2_frozen_holdout75_predictions.csv"
)


d4_holdout_result, _ = (
    evaluate_rows(
        model,

        holdout_rows,

        "D4 frozen holdout",

        D4_HOLDOUT_PRED_PATH,

        family_map=(
            holdout_family_map
        ),
    )
)


print_result(
    "D4 EPOCH2 — FROZEN HOLDOUT75",
    d4_holdout_result,
)


# ============================================================
# 19. D1 vs D4 HOLDOUT COMPARISON
# ============================================================

print("\n" + "=" * 78)
print("D1 vs D4 — FROZEN HOLDOUT75")
print("=" * 78)


print(
    "D1:"
)

print(
    f"  Correct: "
    f"{d1_holdout_result['correct']}/75"
)

print(
    f"  Accuracy: "
    f"{d1_holdout_result['accuracy']:.4f}"
)

print(
    f"  Macro-F1: "
    f"{d1_holdout_result['macro_f1']:.4f}"
)


print(
    "\nD4:"
)

print(
    f"  Correct: "
    f"{d4_holdout_result['correct']}/75"
)

print(
    f"  Accuracy: "
    f"{d4_holdout_result['accuracy']:.4f}"
)

print(
    f"  Macro-F1: "
    f"{d4_holdout_result['macro_f1']:.4f}"
)


print(
    "\nDelta D4 - D1:"
)

print(
    f"  Correct: "
    f"{d4_holdout_result['correct'] - d1_holdout_result['correct']:+d}"
)

print(
    f"  Accuracy: "
    f"{d4_holdout_result['accuracy'] - d1_holdout_result['accuracy']:+.4f}"
)

print(
    f"  Macro-F1: "
    f"{d4_holdout_result['macro_f1'] - d1_holdout_result['macro_f1']:+.4f}"
)


# ============================================================
# 20. FAMILY-LEVEL DELTAS
#
# Aggregate only.
# Do not inspect individual holdout errors yet.
# ============================================================

print("\n" + "=" * 78)
print("FROZEN HOLDOUT — FAMILY DELTAS")
print("=" * 78)


print(
    f"{'Family':30} | "
    f"{'D1':>5} | "
    f"{'D4':>5} | "
    f"{'Δ':>4}"
)

print(
    "-" * 55
)


family_delta = {}


for family in sorted(
    d1_holdout_result[
        "family_results"
    ]
):

    d1_values = (
        d1_holdout_result[
            "family_results"
        ][
            family
        ]
    )

    d4_values = (
        d4_holdout_result[
            "family_results"
        ][
            family
        ]
    )


    delta_correct = (
        d4_values[
            "correct"
        ]
        - d1_values[
            "correct"
        ]
    )


    family_delta[
        family
    ] = {
        "D1": (
            d1_values
        ),

        "D4": (
            d4_values
        ),

        "delta_correct": (
            delta_correct
        ),
    }


    print(
        f"{family:30} | "
        f"{d1_values['correct']:2d}/"
        f"{d1_values['n']:2d} | "
        f"{d4_values['correct']:2d}/"
        f"{d4_values['n']:2d} | "
        f"{delta_correct:+3d}"
    )


# ============================================================
# 21. V3 DEV COMPARISON AGAINST D1 CHAMPION
# ============================================================

D1_DEV_CORRECT = 134
D1_DEV_ACC = 0.9571
D1_DEV_F1 = 0.9596


print("\n" + "=" * 78)
print("D1 vs D4 — V3 DEV")
print("=" * 78)


print(
    "D1 champion:"
)

print(
    "  134/140"
)

print(
    "  Acc=0.9571"
)

print(
    "  F1=0.9596"
)


for result in history:

    dev = result[
        "dev"
    ]


    print(
        f"\nD4 epoch {result['epoch']}:"
    )

    print(
        f"  {dev['correct']}/140"
    )

    print(
        f"  Acc={dev['accuracy']:.4f}"
    )

    print(
        f"  F1={dev['macro_f1']:.4f}"
    )

    print(
        f"  Delta F1 vs D1="
        f"{dev['macro_f1'] - D1_DEV_F1:+.4f}"
    )


# ============================================================
# 22. SAVE SUMMARY
# ============================================================

SUMMARY_PATH = (
    RUN_DIR
    / "D4_TRAINING_AND_FROZEN_HOLDOUT_SUMMARY.json"
)


summary = {
    "experiment": (
        "D4_Gemma4_12B_D1_recipe_"
        "real795_plus_audited150"
    ),

    "controlled_change": (
        "Training data only: "
        "added 150 audited contrastive examples "
        "to D1's same 795 real fit examples."
    ),

    "model": (
        MODEL_ID
    ),

    "data": {
        "real_train_rows": 795,

        "real_train_sha256": (
            EXPECTED_REAL_FIT_SHA
        ),

        "synthetic_train_rows": 150,

        "synthetic_train_sha256": (
            EXPECTED_SYNTH_SHA
        ),

        "combined_train_rows": 945,

        "dev_rows": 140,

        "dev_sha256": (
            EXPECTED_DEV_SHA
        ),

        "frozen_holdout_rows": 75,

        "frozen_holdout_sha256": (
            EXPECTED_HOLDOUT_SHA
        ),
    },

    "recipe": {
        "lora_r": 8,

        "lora_alpha": 16,

        "lora_dropout": 0.05,

        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],

        "trainable_parameters": (
            trainable_params
        ),

        "lr": (
            LR
        ),

        "weight_decay": (
            WEIGHT_DECAY
        ),

        "batch": 1,

        "gradient_accumulation": (
            GRAD_ACCUM
        ),

        "effective_batch": 16,

        "epochs": 2,

        "steps_per_epoch": (
            steps_per_epoch
        ),

        "total_steps": (
            total_steps
        ),

        "warmup_ratio": (
            WARMUP_RATIO
        ),

        "warmup_steps": (
            warmup_steps
        ),

        "scheduler": (
            "cosine"
        ),

        "seed": (
            SEED
        ),

        "prompt": (
            "BASE"
        ),

        "thinking": False,

        "inference": (
            "greedy"
        ),
    },

    "D1_reference": {
        "adapter_sha256": (
            EXPECTED_D1_ADAPTER_SHA
        ),

        "v3_dev": {
            "correct": 134,
            "total": 140,
            "accuracy": (
                D1_DEV_ACC
            ),
            "macro_f1": (
                D1_DEV_F1
            ),
        },

        "new_frozen_holdout": (
            d1_holdout_result
        ),
    },

    "D4_history": (
        history
    ),

    "D4_epoch2_frozen_holdout": (
        d4_holdout_result
    ),

    "frozen_holdout_delta": {
        "correct": (
            d4_holdout_result[
                "correct"
            ]
            - d1_holdout_result[
                "correct"
            ]
        ),

        "accuracy": (
            d4_holdout_result[
                "accuracy"
            ]
            - d1_holdout_result[
                "accuracy"
            ]
        ),

        "macro_f1": (
            d4_holdout_result[
                "macro_f1"
            ]
            - d1_holdout_result[
                "macro_f1"
            ]
        ),
    },

    "family_delta": (
        family_delta
    ),

    "experimental_note": (
        "Frozen holdout was created and audited "
        "before D4 training. D4 epoch1 was not "
        "evaluated on it. D4 epoch2 received one "
        "frozen-holdout evaluation."
    ),

    "runtime_seconds": float(
        time.time()
        - training_start
    ),
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 23. FINAL
# ============================================================

print("\n" + "=" * 78)
print("CELL 4 COMPLETE")
print("=" * 78)


print(
    "Run directory:",
    RUN_DIR,
)

print(
    "Summary:",
    SUMMARY_PATH,
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT run organizer300, external30, "
    "or old synthetic180 yet."
)

print(
    "Do NOT inspect individual frozen-holdout "
    "errors yet."
)

print(
    "\nSend me:"
)

print(
    "1. D1 — FROZEN HOLDOUT75 BASELINE"
)

print(
    "2. D4 EPOCH 1 — V3 DEV"
)

print(
    "3. D4 EPOCH 2 — V3 DEV"
)

print(
    "4. D1 vs D4 — FROZEN HOLDOUT75"
)

print(
    "5. FROZEN HOLDOUT — FAMILY DELTAS"
)

CELL 4 — D4 CONTROLLED TRAINING
Real train: 795
Synthetic train: 150
TOTAL D4 train: 945
V3 dev: 140
Frozen holdout: 75

D4 train labels: Counter({'SUPPORTS': 346, 'REFUTES': 314, 'NOT_ENOUGH_INFO': 285})

Frozen holdout SHA:
61f41a537f738ecd153474565c1e22bb678c4f8a9dd096ed9eabfe5bddc2e1b2

GPU
GPU0: Tesla T4
GPU count: 2
GPU0 allocated: 0.000 GB

PROCESSOR
Processor: Gemma4UnifiedProcessor
Tokenizer: GemmaTokenizer
Processor is tokenizer: False

LOCATE D1 CHAMPION


D1 adapter: /kaggle/input/datasets/omerfarooq223/d1-dataset/D1_GEMMA4_12B_CHAMPION/epoch_2_adapter
D1 SHA: 2c64a87e227e0638be9dc73e021f877f2536876ccf94cc116e04d60075164f8f

D1 — LOAD FOR FROZEN HOLDOUT BASELINE


model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

GPU allocated: 7.274 GB
    D1 frozen holdout: 25/75
    D1 frozen holdout: 50/75
    D1 frozen holdout: 75/75

D1 — FROZEN HOLDOUT75 BASELINE
Correct: 58/75
Accuracy: 0.7733
Macro-F1: 0.7750
Invalid: 0

Confusion matrix:
SUPPORTS         [16, 9, 0]
REFUTES          [7, 18, 0]
NOT_ENOUGH_INFO  [1, 0, 24]

Aggregate D1 family results:
complete_manifest              |  6/ 6 | F1=1.0000
depot_comparison               |  3/ 3 | F1=1.0000
derived_comparison_transfer    |  3/ 6 | F1=0.4667
duration_threshold             |  2/ 3 | F1=0.5556
percentage_threshold           |  4/ 6 | F1=0.5556
project_location_mapping       |  9/ 9 | F1=1.0000
range_or_bound                 |  9/ 9 | F1=1.0000
revenue_percentage             | 11/15 | F1=0.6825
temporal_transition            |  4/ 6 | F1=0.6667
two_component_sum              |  7/12 | F1=0.4908

UNLOAD D1
GPU allocated after unload: 0.008 GB

TOKENIZE D4 TRAIN945
Real lengths min/median/max: 173 191 233
Synthetic lengths min/median/max: 192 207 2

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Trainable parameters: 32,784,384
Expected D1: 32,784,384
GPU allocated: 7.282 GB

D4 TRAINING PLAN
Training rows: 945
  real: 795
  synthetic: 150
Physical batch: 1
Gradient accumulation: 16
Effective batch: 16
Steps/epoch: 60
Total optimizer steps: 120
Warmup steps: 6
LR: 0.0002
Epochs: 2

D4 TRAINING START
Epoch 1 | step 05/60 | global 005/120 | loss=2.4760 | lr=0.00016667 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 10/60 | global 010/120 | loss=0.2510 | lr=0.00019939 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 15/60 | global 015/120 | loss=0.0985 | lr=0.00019694 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 20/60 | global 020/120 | loss=0.1062 | lr=0.00019265 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 25/60 | global 025/120 | loss=0.0647 | lr=0.00018660 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 30/60 | global 030/120 | loss=0.0584 | lr=0.00017891 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 35/60 | global 035/120 | loss=0.0700 | lr=0.00016973 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 40/60 | gl

In [3]:
# ============================================================
# CELL 5 — D4 BROAD GENERALIZATION EVALUATION
#
# Evaluate exact D4 epoch2 on:
#
#   1. Organizer validation300
#   2. External stress30
#   3. Old synthetic180
#
# Compare directly with saved D1 predictions.
#
# Then calculate a FIVE-SET pooled diagnostic:
#
#   V3 dev140
#   Frozen new holdout75
#   External30
#   Organizer300
#   Old synthetic180
#   ------------------------
#   TOTAL = 725 examples
#
# IMPORTANT:
#   This pooled number is a diagnostic, NOT hidden-test accuracy.
#
# NO TRAINING.
# NO MODEL MODIFICATION.
# NO INDIVIDUAL ERROR INSPECTION.
#
# Run immediately after Cell 4 in SAME runtime.
# ============================================================

import ast
import json
import hashlib
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
)


# ============================================================
# 0. CONFIG
# ============================================================

WORK = Path("/kaggle/working")

RUN_DIR = (
    WORK
    / "d4_gemma4_12b_d1recipe_real795_plus_audited150"
)

D1_ROOT = Path(
    "/kaggle/input/datasets/"
    "omerfarooq223/d1-dataset/"
    "D1_GEMMA4_12B_CHAMPION"
)


LABELS = [
    "SUPPORTS",
    "REFUTES",
    "NOT_ENOUGH_INFO",
]


EXPECTED_D4_EPOCH2_SHA = (
    "03b134b92e46f44b3802bf56366843625"
    "0078a2ffef175e35c0c3c5b5ccf42d7"
)

EXPECTED_ORGANIZER_SHA = (
    "722a5f111693996369a02d26eda08f00"
    "cdfb51f5b8b842addb752814f03716c9"
)


# ============================================================
# 1. REQUIRE CURRENT D4 MODEL
# ============================================================

print("=" * 78)
print("CELL 5 — D4 BROAD GENERALIZATION")
print("=" * 78)


assert "model" in globals(), (
    "D4 model is no longer in memory.\n"
    "Do NOT continue. Send me this message."
)

assert "evaluate_rows" in globals(), (
    "Cell-4 evaluate_rows() function is missing.\n"
    "Do NOT continue."
)

assert "processor" in globals()
assert "tokenizer" in globals()


# ============================================================
# 2. VERIFY EXACT D4 EPOCH2 ADAPTER
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


D4_ADAPTER_FILE = (
    RUN_DIR
    / "epoch_2_adapter"
    / "adapter_model.safetensors"
)


assert D4_ADAPTER_FILE.exists(), (
    D4_ADAPTER_FILE
)


actual_d4_sha = sha256_file(
    D4_ADAPTER_FILE
)


assert (
    actual_d4_sha
    == EXPECTED_D4_EPOCH2_SHA
), (
    "Wrong D4 adapter checkpoint."
)


print(
    "D4 epoch2 adapter SHA:",
    actual_d4_sha,
)

print(
    "Exact D4 checkpoint: VERIFIED"
)


# ============================================================
# 3. D1 PREDICTION ARTIFACTS
# ============================================================

D1_DEV_PATH = (
    D1_ROOT
    / "internal_dev_epoch_2.csv"
)

D1_EXTERNAL_PATH = (
    D1_ROOT
    / "evaluation_artifacts"
    / "d1_12b_external_30_predictions.csv"
)

D1_ORGANIZER_PATH = (
    D1_ROOT
    / "evaluation_artifacts"
    / "d1_12b_organizer_300_predictions.csv"
)

D1_SYNTH_PATH = (
    D1_ROOT
    / "evaluation_artifacts"
    / "d1_12b_synthetic_180_predictions.csv"
)


D1_HOLDOUT_PATH = (
    RUN_DIR
    / "D1_frozen_holdout75_predictions.csv"
)


D4_DEV_PATH = (
    RUN_DIR
    / "v3_dev_epoch_2.csv"
)

D4_HOLDOUT_PATH = (
    RUN_DIR
    / "D4_epoch2_frozen_holdout75_predictions.csv"
)


for path in [
    D1_DEV_PATH,
    D1_EXTERNAL_PATH,
    D1_ORGANIZER_PATH,
    D1_SYNTH_PATH,
    D1_HOLDOUT_PATH,
    D4_DEV_PATH,
    D4_HOLDOUT_PATH,
]:

    assert path.exists(), path


# ============================================================
# 4. DATA HELPERS
# ============================================================

def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def normalize_label(value):

    value = str(
        value
    ).strip().upper()

    assert value in LABELS, (
        f"Unexpected label: {value}"
    )

    return value


def parse_evidence_cell(value):

    if isinstance(value, list):

        return [
            str(x)
            for x in value
        ]


    if pd.isna(value):

        return []


    text = str(
        value
    ).strip()


    # JSON first.

    try:

        parsed = json.loads(
            text
        )

        if isinstance(
            parsed,
            list,
        ):

            return [
                str(x)
                for x in parsed
            ]

        return [
            str(parsed)
        ]

    except Exception:

        pass


    # Python-list representation fallback.

    try:

        parsed = ast.literal_eval(
            text
        )

        if isinstance(
            parsed,
            list,
        ):

            return [
                str(x)
                for x in parsed
            ]

    except Exception:

        pass


    return [
        text
    ]


def rows_from_prediction_csv(path):

    df = pd.read_csv(
        path
    )


    required = {
        "id",
        "gold",
        "claim",
        "evidence",
    }


    missing = (
        required
        - set(
            df.columns
        )
    )


    assert not missing, (
        f"{path.name} missing columns: "
        f"{missing}\n"
        f"Available: {df.columns.tolist()}"
    )


    rows = []


    for record in df.to_dict(
        orient="records"
    ):

        rows.append({
            "id": str(
                record["id"]
            ),

            "claim": str(
                record["claim"]
            ),

            "evidence": (
                parse_evidence_cell(
                    record["evidence"]
                )
            ),

            "label": (
                normalize_label(
                    record["gold"]
                )
            ),
        })


    return rows


# ============================================================
# 5. LOCATE EXACT ORGANIZER VALIDATION300
# ============================================================

print("\n" + "=" * 78)
print("LOCATE ORGANIZER VALIDATION300")
print("=" * 78)


organizer_path = None


for candidate in Path(
    "/kaggle/input"
).rglob(
    "*.jsonl"
):

    try:

        if (
            candidate.stat().st_size
            < 50 * 1024 * 1024
        ):

            if (
                sha256_file(
                    candidate
                )
                == EXPECTED_ORGANIZER_SHA
            ):

                organizer_path = (
                    candidate
                )

                break

    except Exception:

        pass


assert organizer_path is not None, (
    "Organizer validation300 not found."
)


organizer_rows = read_jsonl(
    organizer_path
)


assert len(
    organizer_rows
) == 300


print(
    "Organizer path:",
    organizer_path,
)

print(
    "Organizer rows:",
    len(
        organizer_rows
    ),
)


# ============================================================
# 6. RECOVER EXTERNAL30 + SYNTH180 INPUTS
#
# The saved D1 prediction artifacts contain the original
# claim/evidence/gold data used in those evaluations.
# ============================================================

external_rows = (
    rows_from_prediction_csv(
        D1_EXTERNAL_PATH
    )
)

synth_rows = (
    rows_from_prediction_csv(
        D1_SYNTH_PATH
    )
)


assert len(
    external_rows
) == 30

assert len(
    synth_rows
) == 180


print(
    "External rows:",
    len(
        external_rows
    ),
)

print(
    "Old synthetic rows:",
    len(
        synth_rows
    ),
)


# ============================================================
# 7. VERIFY GOLD LABELS AGAINST D1 FILES
# ============================================================

def gold_from_csv(path):

    df = pd.read_csv(
        path
    )

    return {
        str(row["id"]):
            normalize_label(
                row["gold"]
            )

        for row in df.to_dict(
            orient="records"
        )
    }


organizer_gold = gold_from_csv(
    D1_ORGANIZER_PATH
)


assert len(
    organizer_gold
) == 300


for row in organizer_rows:

    rid = str(
        row["id"]
    )

    assert rid in organizer_gold

    assert (
        normalize_label(
            row["label"]
        )
        == organizer_gold[
            rid
        ]
    )


print(
    "Organizer gold alignment: 300/300 PASS"
)


# ============================================================
# 8. EVALUATE D4 — ORGANIZER300
# ============================================================

print("\n" + "=" * 78)
print("D4 — ORGANIZER VALIDATION300")
print("=" * 78)


D4_ORGANIZER_PATH = (
    RUN_DIR
    / "D4_epoch2_organizer300_predictions.csv"
)


d4_organizer_result, d4_organizer_df = (
    evaluate_rows(
        model,

        organizer_rows,

        "D4 organizer300",

        D4_ORGANIZER_PATH,
    )
)


print_result(
    "D4 — ORGANIZER300",
    d4_organizer_result,
)


# ============================================================
# 9. EVALUATE D4 — EXTERNAL30
# ============================================================

print("\n" + "=" * 78)
print("D4 — EXTERNAL STRESS30")
print("=" * 78)


D4_EXTERNAL_PATH = (
    RUN_DIR
    / "D4_epoch2_external30_predictions.csv"
)


d4_external_result, d4_external_df = (
    evaluate_rows(
        model,

        external_rows,

        "D4 external30",

        D4_EXTERNAL_PATH,
    )
)


print_result(
    "D4 — EXTERNAL30",
    d4_external_result,
)


# ============================================================
# 10. EVALUATE D4 — OLD SYNTHETIC180
# ============================================================

print("\n" + "=" * 78)
print("D4 — OLD SYNTHETIC180")
print("=" * 78)


D4_SYNTH_PATH = (
    RUN_DIR
    / "D4_epoch2_old_synthetic180_predictions.csv"
)


d4_synth_result, d4_synth_df = (
    evaluate_rows(
        model,

        synth_rows,

        "D4 old synthetic180",

        D4_SYNTH_PATH,
    )
)


print_result(
    "D4 — OLD SYNTHETIC180",
    d4_synth_result,
)


# ============================================================
# 11. GENERIC METRICS FROM SAVED CSV
# ============================================================

def metrics_from_prediction_csv(
    path,
):

    df = pd.read_csv(
        path
    )


    assert {
        "gold",
        "prediction",
    }.issubset(
        df.columns
    )


    gold = [
        normalize_label(x)
        for x in df[
            "gold"
        ].tolist()
    ]


    predictions = []


    for value in df[
        "prediction"
    ].tolist():

        value = str(
            value
        ).strip().upper()


        if value in LABELS:

            predictions.append(
                value
            )

        else:

            predictions.append(
                "__INVALID__"
            )


    return {
        "correct": int(
            sum(
                g == p
                for g, p
                in zip(
                    gold,
                    predictions,
                )
            )
        ),

        "total": int(
            len(
                gold
            )
        ),

        "accuracy": float(
            accuracy_score(
                gold,
                predictions,
            )
        ),

        "macro_f1": float(
            f1_score(
                gold,
                predictions,

                labels=LABELS,

                average="macro",

                zero_division=0,
            )
        ),
    }


# ============================================================
# 12. D1 SECONDARY BASELINES
# ============================================================

d1_secondary = {
    "organizer300": (
        metrics_from_prediction_csv(
            D1_ORGANIZER_PATH
        )
    ),

    "external30": (
        metrics_from_prediction_csv(
            D1_EXTERNAL_PATH
        )
    ),

    "synthetic180": (
        metrics_from_prediction_csv(
            D1_SYNTH_PATH
        )
    ),
}


d4_secondary = {
    "organizer300": (
        d4_organizer_result
    ),

    "external30": (
        d4_external_result
    ),

    "synthetic180": (
        d4_synth_result
    ),
}


# ============================================================
# 13. SECONDARY GENERALIZATION COMPARISON
# ============================================================

print("\n" + "=" * 78)
print("D1 vs D4 — SECONDARY GENERALIZATION")
print("=" * 78)


print(
    f"{'Dataset':17} | "
    f"{'D1 correct':>10} | "
    f"{'D4 correct':>10} | "
    f"{'Δ':>4} | "
    f"{'D1 F1':>7} | "
    f"{'D4 F1':>7} | "
    f"{'ΔF1':>7}"
)

print(
    "-" * 86
)


for dataset_name in [
    "organizer300",
    "external30",
    "synthetic180",
]:

    d1 = d1_secondary[
        dataset_name
    ]

    d4 = d4_secondary[
        dataset_name
    ]


    print(
        f"{dataset_name:17} | "
        f"{d1['correct']:3d}/{d1['total']:<3d} | "
        f"{d4['correct']:3d}/{d4['total']:<3d} | "
        f"{d4['correct']-d1['correct']:+4d} | "
        f"{d1['macro_f1']:.4f} | "
        f"{d4['macro_f1']:.4f} | "
        f"{d4['macro_f1']-d1['macro_f1']:+.4f}"
    )


# ============================================================
# 14. MOVEMENT ANALYSIS
#
# Aggregate only — no individual errors printed.
# ============================================================

def movement_summary(
    d1_path,
    d4_path,
):

    d1 = pd.read_csv(
        d1_path
    )

    d4 = pd.read_csv(
        d4_path
    )


    needed = {
        "id",
        "gold",
        "prediction",
    }


    assert needed.issubset(
        d1.columns
    )

    assert needed.issubset(
        d4.columns
    )


    d1 = d1[
        [
            "id",
            "gold",
            "prediction",
        ]
    ].copy()


    d4 = d4[
        [
            "id",
            "gold",
            "prediction",
        ]
    ].copy()


    d1["id"] = (
        d1["id"]
        .astype(str)
    )

    d4["id"] = (
        d4["id"]
        .astype(str)
    )


    merged = d1.merge(
        d4,

        on="id",

        suffixes=(
            "_d1",
            "_d4",
        ),

        validate="one_to_one",
    )


    assert len(
        merged
    ) == len(
        d1
    )


    assert all(
        merged[
            "gold_d1"
        ].astype(str)
        ==
        merged[
            "gold_d4"
        ].astype(str)
    )


    d1_correct = (
        merged[
            "prediction_d1"
        ]
        ==
        merged[
            "gold_d1"
        ]
    )

    d4_correct = (
        merged[
            "prediction_d4"
        ]
        ==
        merged[
            "gold_d4"
        ]
    )


    return {
        "both_correct": int(
            (
                d1_correct
                & d4_correct
            ).sum()
        ),

        "d4_fixes_d1": int(
            (
                ~d1_correct
                & d4_correct
            ).sum()
        ),

        "d4_breaks_d1": int(
            (
                d1_correct
                & ~d4_correct
            ).sum()
        ),

        "both_wrong": int(
            (
                ~d1_correct
                & ~d4_correct
            ).sum()
        ),

        "prediction_changes": int(
            (
                merged[
                    "prediction_d1"
                ]
                != merged[
                    "prediction_d4"
                ]
            ).sum()
        ),
    }


movement = {
    "organizer300": (
        movement_summary(
            D1_ORGANIZER_PATH,
            D4_ORGANIZER_PATH,
        )
    ),

    "external30": (
        movement_summary(
            D1_EXTERNAL_PATH,
            D4_EXTERNAL_PATH,
        )
    ),

    "synthetic180": (
        movement_summary(
            D1_SYNTH_PATH,
            D4_SYNTH_PATH,
        )
    ),
}


print("\n" + "=" * 78)
print("D1 → D4 MOVEMENT")
print("=" * 78)


print(
    f"{'Dataset':17} | "
    f"{'Both ✓':>6} | "
    f"{'D4 fixes':>8} | "
    f"{'D4 breaks':>9} | "
    f"{'Both ✗':>6} | "
    f"{'Pred Δ':>6}"
)

print(
    "-" * 76
)


for dataset_name in [
    "organizer300",
    "external30",
    "synthetic180",
]:

    m = movement[
        dataset_name
    ]


    print(
        f"{dataset_name:17} | "
        f"{m['both_correct']:6d} | "
        f"{m['d4_fixes_d1']:8d} | "
        f"{m['d4_breaks_d1']:9d} | "
        f"{m['both_wrong']:6d} | "
        f"{m['prediction_changes']:6d}"
    )


# ============================================================
# 15. FIVE-SET POOLED DIAGNOSTIC — 725 EXAMPLES
#
# IMPORTANT:
# This is NOT "final accuracy".
#
# It mixes:
#   real dev,
#   frozen targeted holdout,
#   external stress,
#   organizer validation,
#   old synthetic.
#
# Still useful as one aggregate diagnostic.
# ============================================================

d1_five_paths = {
    "v3_dev140": (
        D1_DEV_PATH
    ),

    "frozen75": (
        D1_HOLDOUT_PATH
    ),

    "external30": (
        D1_EXTERNAL_PATH
    ),

    "organizer300": (
        D1_ORGANIZER_PATH
    ),

    "synthetic180": (
        D1_SYNTH_PATH
    ),
}


d4_five_paths = {
    "v3_dev140": (
        D4_DEV_PATH
    ),

    "frozen75": (
        D4_HOLDOUT_PATH
    ),

    "external30": (
        D4_EXTERNAL_PATH
    ),

    "organizer300": (
        D4_ORGANIZER_PATH
    ),

    "synthetic180": (
        D4_SYNTH_PATH
    ),
}


def pooled_metrics(path_dict):

    pooled_gold = []
    pooled_pred = []

    set_rows = {}


    for dataset_name, path in (
        path_dict.items()
    ):

        df = pd.read_csv(
            path
        )


        gold = [
            normalize_label(x)

            for x in df[
                "gold"
            ].tolist()
        ]


        pred = []


        for value in df[
            "prediction"
        ].tolist():

            value = str(
                value
            ).strip().upper()


            pred.append(
                value
                if value in LABELS
                else "__INVALID__"
            )


        pooled_gold.extend(
            gold
        )

        pooled_pred.extend(
            pred
        )


        set_rows[
            dataset_name
        ] = {
            "correct": int(
                sum(
                    g == p

                    for g, p
                    in zip(
                        gold,
                        pred,
                    )
                )
            ),

            "total": int(
                len(
                    gold
                )
            ),
        }


    assert len(
        pooled_gold
    ) == 725


    return {
        "correct": int(
            sum(
                g == p

                for g, p
                in zip(
                    pooled_gold,
                    pooled_pred,
                )
            )
        ),

        "total": 725,

        "accuracy": float(
            accuracy_score(
                pooled_gold,
                pooled_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                pooled_gold,
                pooled_pred,

                labels=LABELS,

                average="macro",

                zero_division=0,
            )
        ),

        "sets": (
            set_rows
        ),
    }


d1_pooled = pooled_metrics(
    d1_five_paths
)

d4_pooled = pooled_metrics(
    d4_five_paths
)


print("\n" + "=" * 78)
print("FIVE-SET POOLED DIAGNOSTIC — 725 EXAMPLES")
print("=" * 78)


print(
    "D1:"
)

print(
    f"  Correct: "
    f"{d1_pooled['correct']}/725"
)

print(
    f"  Accuracy: "
    f"{d1_pooled['accuracy']:.4f}"
)

print(
    f"  Macro-F1: "
    f"{d1_pooled['macro_f1']:.4f}"
)


print(
    "\nD4:"
)

print(
    f"  Correct: "
    f"{d4_pooled['correct']}/725"
)

print(
    f"  Accuracy: "
    f"{d4_pooled['accuracy']:.4f}"
)

print(
    f"  Macro-F1: "
    f"{d4_pooled['macro_f1']:.4f}"
)


print(
    "\nD4 - D1:"
)

print(
    f"  Correct: "
    f"{d4_pooled['correct'] - d1_pooled['correct']:+d}"
)

print(
    f"  Accuracy: "
    f"{d4_pooled['accuracy'] - d1_pooled['accuracy']:+.4f}"
)

print(
    f"  Macro-F1: "
    f"{d4_pooled['macro_f1'] - d1_pooled['macro_f1']:+.4f}"
)


print(
    "\nReminder:"
)

print(
    "This 725-example number is a POOLED DIAGNOSTIC,"
)

print(
    "NOT an estimate of hidden-test accuracy."
)


# ============================================================
# 16. SAVE SUMMARY
# ============================================================

SUMMARY_PATH = (
    RUN_DIR
    / "D4_BROAD_GENERALIZATION_COMPARISON.json"
)


summary = {
    "D4_adapter_sha256": (
        actual_d4_sha
    ),

    "secondary_results": {
        dataset_name: {
            "D1": (
                d1_secondary[
                    dataset_name
                ]
            ),

            "D4": (
                d4_secondary[
                    dataset_name
                ]
            ),

            "movement": (
                movement[
                    dataset_name
                ]
            ),
        }

        for dataset_name in [
            "organizer300",
            "external30",
            "synthetic180",
        ]
    },

    "five_set_pooled_diagnostic": {
        "total_examples": 725,

        "D1": (
            d1_pooled
        ),

        "D4": (
            d4_pooled
        ),

        "delta_correct": int(
            d4_pooled[
                "correct"
            ]
            - d1_pooled[
                "correct"
            ]
        ),

        "delta_accuracy": float(
            d4_pooled[
                "accuracy"
            ]
            - d1_pooled[
                "accuracy"
            ]
        ),

        "delta_macro_f1": float(
            d4_pooled[
                "macro_f1"
            ]
            - d1_pooled[
                "macro_f1"
            ]
        ),

        "warning": (
            "Pooled diagnostic mixes multiple "
            "distributions and is not hidden-test "
            "accuracy."
        ),
    },
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 17. FINAL
# ============================================================

print("\n" + "=" * 78)
print("CELL 5 COMPLETE")
print("=" * 78)


print(
    "Summary:",
    SUMMARY_PATH,
)


print(
    "\nNO TRAINING occurred."
)

print(
    "NO model parameters were changed."
)

print(
    "Individual errors were NOT printed."
)


print(
    "\nSend me these sections:"
)

print(
    "1. D4 — ORGANIZER300"
)

print(
    "2. D4 — EXTERNAL30"
)

print(
    "3. D4 — OLD SYNTHETIC180"
)

print(
    "4. D1 vs D4 — SECONDARY GENERALIZATION"
)

print(
    "5. D1 → D4 MOVEMENT"
)

print(
    "6. FIVE-SET POOLED DIAGNOSTIC — 725 EXAMPLES"
)

CELL 5 — D4 BROAD GENERALIZATION
D4 epoch2 adapter SHA: 03b134b92e46f44b3802bf563668436250078a2ffef175e35c0c3c5b5ccf42d7
Exact D4 checkpoint: VERIFIED

LOCATE ORGANIZER VALIDATION300
Organizer path: /kaggle/input/datasets/omerfarooq223/fine-tunning-d/Gemma Fine Tuning Paticipants /data/validation.jsonl
Organizer rows: 300
External rows: 30
Old synthetic rows: 180
Organizer gold alignment: 300/300 PASS

D4 — ORGANIZER VALIDATION300
    D4 organizer300: 25/300
    D4 organizer300: 50/300
    D4 organizer300: 75/300
    D4 organizer300: 100/300
    D4 organizer300: 125/300
    D4 organizer300: 150/300
    D4 organizer300: 175/300
    D4 organizer300: 200/300
    D4 organizer300: 225/300
    D4 organizer300: 250/300
    D4 organizer300: 275/300
    D4 organizer300: 300/300

D4 — ORGANIZER300
Correct: 275/300
Accuracy: 0.9167
Macro-F1: 0.9161
Invalid: 0

Confusion matrix:
SUPPORTS         [84, 13, 3]
REFUTES          [8, 92, 0]
NOT_ENOUGH_INFO  [0, 1, 99]

D4 — EXTERNAL STRESS30
    D4 exte

In [4]:
# ============================================================
# CELL 6 — PACKAGE PROVEN D4 CHAMPION
#
# Proven D4:
#   Gemma-4 12B
#   795 real + audited 150 synthetic = 945
#   epoch 2
#
# Adapter SHA:
#   03b134b92e46f44b3802bf563668436250078a2ffef175e35c0c3c5b5ccf42d7
#
# This package contains:
#   - exact D4 epoch2 adapter
#   - processor/tokenizer files saved with adapter
#   - D4 training summary
#   - broad evaluation summary
#   - prediction CSVs
#   - exact fit795 / dev140
#   - audited synthetic150
#   - frozen holdout75
#   - semantic audit / manifests
#   - reproducibility manifest
#
# It DOES NOT contain the 23.9 GB Gemma-4 base weights.
#
# NO TRAINING
# NO MODEL MODIFICATION
# ============================================================

import json
import shutil
import hashlib
from pathlib import Path


WORK = Path("/kaggle/working")

D4_RUN = (
    WORK
    / "d4_gemma4_12b_d1recipe_real795_plus_audited150"
)

DATA_RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

D4_DATA = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

SOURCE_ADAPTER = (
    D4_RUN
    / "epoch_2_adapter"
)

EXPECTED_ADAPTER_SHA = (
    "03b134b92e46f44b3802bf56366843625"
    "0078a2ffef175e35c0c3c5b5ccf42d7"
)

EXPECTED_SYNTH_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)

PACKAGE_DIR = (
    WORK
    / "D4_GEMMA4_12B_CHAMPION"
)

ZIP_BASE = (
    WORK
    / "D4_GEMMA4_12B_CHAMPION"
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def copy_if_exists(source, destination_dir):

    if source.exists():

        destination_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            source,
            destination_dir / source.name,
        )

        return True

    return False


# ============================================================
# VERIFY EXACT CHAMPION
# ============================================================

print("=" * 78)
print("CELL 6 — PACKAGE D4 CHAMPION")
print("=" * 78)


adapter_file = (
    SOURCE_ADAPTER
    / "adapter_model.safetensors"
)

assert adapter_file.exists(), adapter_file


actual_adapter_sha = sha256_file(
    adapter_file
)


assert (
    actual_adapter_sha
    == EXPECTED_ADAPTER_SHA
)


print(
    "D4 adapter SHA:",
    actual_adapter_sha,
)

print(
    "Exact D4 champion checkpoint: VERIFIED"
)


# ============================================================
# VERIFY DATA
# ============================================================

fit_path = (
    DATA_RECOVERY
    / "train_fit_v3_semantic_recovered.jsonl"
)

dev_path = (
    DATA_RECOVERY
    / "internal_dev_v3_semantic_recovered.jsonl"
)

clean935_path = (
    DATA_RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

synth_path = (
    D4_DATA
    / "d4_contrastive_train_v1.jsonl"
)

holdout_path = (
    D4_DATA
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)


for path in [
    fit_path,
    dev_path,
    clean935_path,
    synth_path,
    holdout_path,
]:

    assert path.exists(), path


assert (
    sha256_file(synth_path)
    == EXPECTED_SYNTH_SHA
)

assert (
    sha256_file(holdout_path)
    == EXPECTED_HOLDOUT_SHA
)


# ============================================================
# BUILD CLEAN PACKAGE
# ============================================================

if PACKAGE_DIR.exists():

    shutil.rmtree(
        PACKAGE_DIR
    )


PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Exact adapter directory
# ------------------------------------------------------------

adapter_destination = (
    PACKAGE_DIR
    / "epoch_2_adapter"
)


shutil.copytree(
    SOURCE_ADAPTER,
    adapter_destination,
)


# ------------------------------------------------------------
# Training data
# ------------------------------------------------------------

data_dir = (
    PACKAGE_DIR
    / "training_data"
)

data_dir.mkdir(
    parents=True,
    exist_ok=True,
)


for source in [
    fit_path,
    dev_path,
    clean935_path,
    synth_path,
    holdout_path,
]:

    shutil.copy2(
        source,
        data_dir / source.name,
    )


# ------------------------------------------------------------
# D4 data audit files
# ------------------------------------------------------------

audit_dir = (
    PACKAGE_DIR
    / "data_audit"
)

audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)


audit_candidates = [
    D4_DATA
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json",

    D4_DATA
    / "d4_curriculum_v1_manifest.json",

    D4_DATA
    / "d4_contrastive_train_v1_metadata.csv",

    D4_DATA
    / "D4_FROZEN_HOLDOUT_V1_metadata.csv",

    D4_DATA
    / "semantic_audit"
    / "D4_SYNTHETIC_225_SEMANTIC_AUDIT.csv",

    D4_DATA
    / "semantic_audit"
    / "D4_SYNTHETIC_225_AUDIT_SUMMARY.json",
]


for source in audit_candidates:

    copy_if_exists(
        source,
        audit_dir,
    )


# ------------------------------------------------------------
# Evaluation artifacts
# ------------------------------------------------------------

eval_dir = (
    PACKAGE_DIR
    / "evaluation_artifacts"
)

eval_dir.mkdir(
    parents=True,
    exist_ok=True,
)


evaluation_candidates = [
    D4_RUN
    / "D4_TRAINING_AND_FROZEN_HOLDOUT_SUMMARY.json",

    D4_RUN
    / "D4_BROAD_GENERALIZATION_COMPARISON.json",

    D4_RUN
    / "v3_dev_epoch_1.csv",

    D4_RUN
    / "v3_dev_epoch_2.csv",

    D4_RUN
    / "D1_frozen_holdout75_predictions.csv",

    D4_RUN
    / "D4_epoch2_frozen_holdout75_predictions.csv",

    D4_RUN
    / "D4_epoch2_organizer300_predictions.csv",

    D4_RUN
    / "D4_epoch2_external30_predictions.csv",

    D4_RUN
    / "D4_epoch2_old_synthetic180_predictions.csv",
]


copied_eval = []


for source in evaluation_candidates:

    if copy_if_exists(
        source,
        eval_dir,
    ):

        copied_eval.append(
            source.name
        )


# ============================================================
# REPRODUCIBILITY MANIFEST
# ============================================================

manifest = {
    "artifact": (
        "D4_GEMMA4_12B_CHAMPION"
    ),

    "status": (
        "PROVEN_CHAMPION_BEFORE_FINAL_ALL935_TRAINING"
    ),

    "base_model": (
        "google/gemma-4-12B-it"
    ),

    "base_weights_included": False,

    "adapter": {
        "epoch": 2,

        "sha256": (
            actual_adapter_sha
        ),

        "path_in_package": (
            "epoch_2_adapter"
        ),
    },

    "training_data": {
        "real_fit_rows": 795,

        "synthetic_rows": 150,

        "total_rows": 945,

        "synthetic_sha256": (
            EXPECTED_SYNTH_SHA
        ),

        "frozen_holdout_sha256": (
            EXPECTED_HOLDOUT_SHA
        ),

        "frozen_holdout_used_for_training": False,
    },

    "recipe": {
        "lora_r": 8,

        "lora_alpha": 16,

        "lora_dropout": 0.05,

        "targets": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],

        "trainable_parameters": (
            32_784_384
        ),

        "quantization": (
            "4-bit NF4 double quant"
        ),

        "compute_dtype": (
            "float16"
        ),

        "physical_batch": 1,

        "gradient_accumulation": 16,

        "effective_batch": 16,

        "learning_rate": (
            2e-4
        ),

        "weight_decay": (
            0.01
        ),

        "scheduler": (
            "cosine"
        ),

        "warmup_ratio": (
            0.05
        ),

        "epochs": 2,

        "seed": 42,

        "max_length": 256,

        "completion_only_loss": True,

        "thinking": False,

        "inference": (
            "greedy"
        ),
    },

    "evaluation": {
        "v3_dev140": {
            "correct": 129,
            "total": 140,
            "accuracy": 0.9214,
            "macro_f1": 0.9259,
        },

        "frozen_holdout75": {
            "correct": 65,
            "total": 75,
            "accuracy": 0.8667,
            "macro_f1": 0.8658,
        },

        "organizer300": {
            "correct": 275,
            "total": 300,
            "accuracy": 0.9167,
            "macro_f1": 0.9161,
        },

        "external30": {
            "correct": 27,
            "total": 30,
            "accuracy": 0.9000,
            "macro_f1": 0.9048,
        },

        "old_synthetic180": {
            "correct": 143,
            "total": 180,
            "accuracy": 0.7944,
            "macro_f1": 0.7866,
        },

        "five_set_pooled_diagnostic": {
            "correct": 639,
            "total": 725,
            "accuracy": 0.8814,
            "macro_f1": 0.8820,
            "warning": (
                "Diagnostic only; not hidden-test accuracy."
            ),
        },
    },

    "evaluation_artifacts_copied": (
        copied_eval
    ),

    "important_note": (
        "This is the exact empirically validated D4 adapter. "
        "Preserve it even if the later all-935 production "
        "training experiment performs differently."
    ),
}


manifest_path = (
    PACKAGE_DIR
    / "CHAMPION_MANIFEST.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# README
# ============================================================

readme = (
    PACKAGE_DIR
    / "README.txt"
)


readme.write_text(
    f"""D4 GEMMA-4 12B CHAMPION
==========================

BASE MODEL
----------
google/gemma-4-12B-it

The 23.9 GB base-model weights are NOT included.

EXACT CHAMPION ADAPTER
----------------------
epoch_2_adapter/

SHA256:
{actual_adapter_sha}

TRAINING
--------
795 semantically-clean real fit examples
+ 150 audited contrastive examples
= 945 examples

LoRA:
r=8
alpha=16
dropout=0.05

Targets:
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj

2 epochs
LR 2e-4
weight decay 0.01
cosine schedule
5% warmup
batch 1
gradient accumulation 16
seed 42
completion-only loss
greedy inference
thinking disabled

RESULTS
-------
V3 dev140:
129/140 = 92.14%

Frozen holdout75:
65/75 = 86.67%

Organizer validation300:
275/300 = 91.67%

External30:
27/30 = 90.00%

Old synthetic180:
143/180 = 79.44%

Five-set pooled diagnostic:
639/725 = 88.14%

The pooled diagnostic is NOT hidden-test accuracy.

IMPORTANT
---------
This package is the proven D4 champion before the
all-935 final-production training experiment.
Do not overwrite it.
""",
    encoding="utf-8",
)


# ============================================================
# ZIP
# ============================================================

existing_zip = Path(
    str(ZIP_BASE) + ".zip"
)


if existing_zip.exists():

    existing_zip.unlink()


zip_path = Path(
    shutil.make_archive(
        base_name=str(
            ZIP_BASE
        ),

        format="zip",

        root_dir=str(
            PACKAGE_DIR.parent
        ),

        base_dir=(
            PACKAGE_DIR.name
        ),
    )
)


zip_sha = sha256_file(
    zip_path
)

zip_mb = (
    zip_path.stat().st_size
    / 1024**2
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

packaged_adapter = (
    PACKAGE_DIR
    / "epoch_2_adapter"
    / "adapter_model.safetensors"
)


assert (
    sha256_file(
        packaged_adapter
    )
    == EXPECTED_ADAPTER_SHA
)


print("\n" + "=" * 78)
print("D4 CHAMPION — SAVED")
print("=" * 78)


print(
    "Package:",
    PACKAGE_DIR,
)

print(
    "ZIP:",
    zip_path,
)

print(
    "ZIP size:",
    f"{zip_mb:.2f} MB",
)

print(
    "ZIP SHA256:",
    zip_sha,
)

print(
    "\nPackaged adapter SHA:",
    sha256_file(
        packaged_adapter
    ),
)

print(
    "Exact D4 champion preserved: YES"
)

print(
    "\nYou can now proceed to Cell 7."
)

CELL 6 — PACKAGE D4 CHAMPION
D4 adapter SHA: 03b134b92e46f44b3802bf563668436250078a2ffef175e35c0c3c5b5ccf42d7
Exact D4 champion checkpoint: VERIFIED

D4 CHAMPION — SAVED
Package: /kaggle/working/D4_GEMMA4_12B_CHAMPION
ZIP: /kaggle/working/D4_GEMMA4_12B_CHAMPION.zip
ZIP size: 120.71 MB
ZIP SHA256: a6d0d10c266e96ee88122dd14afb13d132e487481a889905844ac2d9d7fc4fb2

Packaged adapter SHA: 03b134b92e46f44b3802bf563668436250078a2ffef175e35c0c3c5b5ccf42d7
Exact D4 champion preserved: YES

You can now proceed to Cell 7.


In [5]:
# ============================================================
# CELL 7 — FINAL ALL-935 PRODUCTION TRAINING
#
# FROZEN D4 RECIPE
#
# Training:
#   935 semantically-clean REAL examples
# + 150 SAME audited contrastive examples
# = 1085 total
#
# IMPORTANT:
#
# - Fresh Gemma-4 12B base.
# - Fresh LoRA initialization.
# - NOT continued from D4.
# - Exact D4 recipe.
# - Fixed 2 epochs.
# - NO dev-set model selection.
# - Frozen holdout75 is NOT used.
# - Organizer300 is NOT used.
# - External30 is NOT used.
# - Old synthetic180 is NOT used.
#
# This creates a FINAL PRODUCTION CANDIDATE.
#
# The proven D4 945-trained champion remains preserved
# separately by Cell 6.
# ============================================================

import os
import re
import gc
import json
import math
import time
import random
import shutil
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np

import torch
import transformers
import peft
import bitsandbytes

from transformers import (
    AutoProcessor,
    Gemma4UnifiedForConditionalGeneration,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
)


# ============================================================
# 0. ENVIRONMENT
# ============================================================

assert transformers.__version__ == "5.10.1"
assert peft.__version__ == "0.19.1"
assert bitsandbytes.__version__ == "0.50.1"


MODEL_ID = (
    "google/gemma-4-12B-it"
)

SEED = 42

MAX_LENGTH = 256

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

LR = 2e-4
WEIGHT_DECAY = 0.01

GRAD_ACCUM = 16
EPOCHS = 2
WARMUP_RATIO = 0.05


WORK = Path(
    "/kaggle/working"
)

RECOVERY = (
    WORK
    / "d1_recovery_and_consolidated_audit"
)

D4_DATA = (
    WORK
    / "d4_contrastive_curriculum_v1"
)

RUN_DIR = (
    WORK
    / "FINAL_gemma4_12b_d4recipe_real935_plus_audited150"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REAL935_PATH = (
    RECOVERY
    / "train_clean_v3_semantic_recovered.jsonl"
)

SYNTH150_PATH = (
    D4_DATA
    / "d4_contrastive_train_v1.jsonl"
)

HOLDOUT75_PATH = (
    D4_DATA
    / "D4_FROZEN_HOLDOUT_V1_DO_NOT_TRAIN.jsonl"
)


EXPECTED_REAL935_SHA = (
    "b706dbf0c0b4aab4cbcd07bb89c5d018"
    "f7c41e47a55b757016ef3d07a9713337"
)

EXPECTED_SYNTH150_SHA = (
    "17258cbc0e40f4ebd1cd4d583e3a331e"
    "59217d62cf4ff36741e8b1e3a7a98f41"
)

EXPECTED_HOLDOUT75_SHA = (
    "61f41a537f738ecd153474565c1e22bb6"
    "78c4f8a9dd096ed9eabfe5bddc2e1b2"
)


BASE_PROMPT = """Classify the claim using only the supplied evidence.

SUPPORTS: the evidence establishes the claim.
REFUTES: the evidence contradicts the claim.
NOT_ENOUGH_INFO: the evidence neither establishes nor contradicts the specific claim.

End your response exactly as:
FINAL: SUPPORTS
or
FINAL: REFUTES
or
FINAL: NOT_ENOUGH_INFO"""


TARGET_REGEX = (
    r"model\.language_model\.layers\.\d+\."
    r"(?:"
        r"self_attn\."
        r"(?:q_proj|k_proj|v_proj|o_proj)"
        r"|"
        r"mlp\."
        r"(?:gate_proj|up_proj|down_proj)"
    r")"
)


# ============================================================
# 1. HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
    ) as f:

        for line in f:

            if line.strip():

                rows.append(
                    json.loads(line)
                )

    return rows


def seed_everything():

    random.seed(SEED)

    np.random.seed(SEED)

    torch.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def evidence_list(row):

    evidence = row["evidence"]

    if isinstance(
        evidence,
        str,
    ):
        return [evidence]

    return list(evidence)


def build_user_text(row):

    evidence_text = "\n".join(
        f"[{i}] {passage}"

        for i, passage
        in enumerate(
            evidence_list(row),
            1,
        )
    )

    return (
        BASE_PROMPT
        + "\n\nClaim:\n"
        + row["claim"]
        + "\n\nEvidence:\n"
        + evidence_text
    )


def user_messages(row):

    return [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",

                    "text": build_user_text(
                        row
                    ),
                }
            ],
        }
    ]


# ============================================================
# 2. VERIFY EXACT DATA
# ============================================================

print("=" * 78)
print("CELL 7 — FINAL ALL-935 PRODUCTION TRAINING")
print("=" * 78)


for path in [
    REAL935_PATH,
    SYNTH150_PATH,
    HOLDOUT75_PATH,
]:

    assert path.exists(), path


assert (
    sha256_file(
        REAL935_PATH
    )
    == EXPECTED_REAL935_SHA
)

assert (
    sha256_file(
        SYNTH150_PATH
    )
    == EXPECTED_SYNTH150_SHA
)

assert (
    sha256_file(
        HOLDOUT75_PATH
    )
    == EXPECTED_HOLDOUT75_SHA
)


real_rows = read_jsonl(
    REAL935_PATH
)

synth_rows = read_jsonl(
    SYNTH150_PATH
)

holdout_rows = read_jsonl(
    HOLDOUT75_PATH
)


assert len(real_rows) == 935
assert len(synth_rows) == 150
assert len(holdout_rows) == 75


real_ids = {
    str(row["id"])
    for row in real_rows
}

synth_ids = {
    str(row["id"])
    for row in synth_rows
}

holdout_ids = {
    str(row["id"])
    for row in holdout_rows
}


assert len(real_ids) == 935
assert len(synth_ids) == 150
assert len(holdout_ids) == 75

assert real_ids.isdisjoint(
    synth_ids
)

assert holdout_ids.isdisjoint(
    real_ids | synth_ids
)


train_rows = (
    real_rows
    + synth_rows
)


assert len(
    train_rows
) == 1085


label_counts = Counter(
    row["label"]
    for row in train_rows
)


assert label_counts == Counter({
    "SUPPORTS": 398,
    "REFUTES": 361,
    "NOT_ENOUGH_INFO": 326,
})


print(
    "Real semantic-clean train:",
    len(real_rows),
)

print(
    "Audited synthetic train:",
    len(synth_rows),
)

print(
    "TOTAL FINAL TRAIN:",
    len(train_rows),
)

print(
    "Labels:",
    label_counts,
)

print(
    "\nFrozen holdout rows:",
    len(holdout_rows),
)

print(
    "Frozen holdout used in training: NO"
)


# ============================================================
# 3. CLEAN GPU
#
# Cell 5 may still have D4 loaded.
# ============================================================

print("\n" + "=" * 78)
print("GPU CLEANUP")
print("=" * 78)


for variable_name in [
    "model",
    "base_model",
    "d1_model",
]:

    if variable_name in globals():

        try:
            del globals()[
                variable_name
            ]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


if hasattr(
    torch.cuda,
    "ipc_collect",
):

    torch.cuda.ipc_collect()


time.sleep(2)


assert torch.cuda.is_available()


torch.cuda.set_device(0)


allocated = (
    torch.cuda.memory_allocated(0)
    / 1024**3
)


print(
    "GPU0:",
    torch.cuda.get_device_name(0),
)

print(
    "GPU0 allocated:",
    f"{allocated:.3f} GB",
)


assert allocated < 0.5, (
    "GPU did not cleanly unload. "
    "Restart kernel and rerun only Cell 7."
)


# ============================================================
# 4. HF TOKEN
# ============================================================

hf_token = os.environ.get(
    "HF_TOKEN"
)


if not hf_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )


        hf_token = (
            UserSecretsClient()
            .get_secret(
                "HF_TOKEN"
            )
        )

    except Exception:

        hf_token = None


if not hf_token:

    print(
        "WARNING: HF_TOKEN unavailable; "
        "cached/public download may still work."
    )


# ============================================================
# 5. PROCESSOR
# ============================================================

print("\n" + "=" * 78)
print("PROCESSOR")
print("=" * 78)


processor = (
    AutoProcessor
    .from_pretrained(
        MODEL_ID,
        token=hf_token,
    )
)


if hasattr(
    processor,
    "tokenizer",
):

    tokenizer = (
        processor.tokenizer
    )

else:

    tokenizer = (
        processor
    )


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


print(
    "Processor:",
    processor.__class__.__name__,
)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__,
)


# ============================================================
# 6. EXACT TRAINING TOKEN CONSTRUCTION
# ============================================================

TURN_CLOSE_TEXT = (
    "<turn|>\n"
)


turn_close_ids = (
    tokenizer(
        TURN_CLOSE_TEXT,
        add_special_tokens=False,
    )[
        "input_ids"
    ]
)


def build_training_sample(
    row,
    source,
):

    prompt_encoded = (
        processor.apply_chat_template(
            user_messages(row),

            tokenize=True,

            return_dict=True,

            return_tensors="pt",

            add_generation_prompt=True,

            enable_thinking=False,
        )
    )


    prompt_ids = (
        prompt_encoded[
            "input_ids"
        ][0]
        .tolist()
    )


    answer_ids = (
        tokenizer(
            "FINAL: "
            + row["label"],

            add_special_tokens=False,
        )[
            "input_ids"
        ]
    )


    full_ids = (
        prompt_ids
        + answer_ids
        + turn_close_ids
    )


    labels = (
        [-100] * len(prompt_ids)
        + answer_ids
        + turn_close_ids
    )


    assert len(full_ids) == len(labels)


    return {
        "id": str(
            row["id"]
        ),

        "source": source,

        "input_ids": torch.tensor(
            full_ids,
            dtype=torch.long,
        ),

        "attention_mask": torch.ones(
            len(full_ids),
            dtype=torch.long,
        ),

        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),

        "prompt_length": len(
            prompt_ids
        ),

        "full_length": len(
            full_ids
        ),
    }


print("\n" + "=" * 78)
print("TOKENIZE FINAL TRAIN1085")
print("=" * 78)


train_samples = []

real_lengths = []
synth_lengths = []


for index, row in enumerate(
    real_rows,
    1,
):

    sample = build_training_sample(
        row,
        "real",
    )

    train_samples.append(
        sample
    )

    real_lengths.append(
        sample[
            "full_length"
        ]
    )


    if index % 250 == 0:

        print(
            f"  real tokenized "
            f"{index}/935"
        )


for index, row in enumerate(
    synth_rows,
    1,
):

    sample = build_training_sample(
        row,
        "synthetic",
    )

    train_samples.append(
        sample
    )

    synth_lengths.append(
        sample[
            "full_length"
        ]
    )


assert len(
    train_samples
) == 1085


print(
    "\nReal min/median/max:",
    min(real_lengths),
    int(
        np.median(
            real_lengths
        )
    ),
    max(real_lengths),
)

print(
    "Synthetic min/median/max:",
    min(synth_lengths),
    int(
        np.median(
            synth_lengths
        )
    ),
    max(synth_lengths),
)


max_length_seen = max(
    max(real_lengths),
    max(synth_lengths),
)


print(
    "Combined max:",
    max_length_seen,
)


assert max_length_seen <= MAX_LENGTH


print(
    "1085/1085 fit within "
    "max_length=256."
)


# ============================================================
# 7. FRESH GEMMA-4 12B
# ============================================================

quant_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,

        bnb_4bit_quant_type=(
            "nf4"
        ),

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)


print("\n" + "=" * 78)
print("LOAD FRESH GEMMA-4 12B")
print("=" * 78)


seed_everything()


base_model = (
    Gemma4UnifiedForConditionalGeneration
    .from_pretrained(
        MODEL_ID,

        token=hf_token,

        quantization_config=(
            quant_config
        ),

        device_map={
            "": 0
        },

        dtype=torch.float16,

        low_cpu_mem_usage=True,
    )
)


for parameter in (
    base_model.parameters()
):

    parameter.requires_grad = False


try:

    base_model.config.use_cache = False

except Exception:

    pass


print(
    "Base GPU allocated:",
    f"{torch.cuda.memory_allocated(0)/1024**3:.3f} GB",
)


# ============================================================
# 8. FRESH EXACT D4 LoRA
# ============================================================

seed_everything()


lora_config = (
    LoraConfig(
        r=LORA_R,

        lora_alpha=(
            LORA_ALPHA
        ),

        lora_dropout=(
            LORA_DROPOUT
        ),

        bias="none",

        task_type="CAUSAL_LM",

        target_modules=(
            TARGET_REGEX
        ),
    )
)


model = get_peft_model(
    base_model,
    lora_config,
)


if hasattr(
    model,
    "enable_input_require_grads",
):

    model.enable_input_require_grads()


try:

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

except TypeError:

    model.gradient_checkpointing_enable()


trainable_params = sum(
    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad
)


assert (
    trainable_params
    == 32_784_384
)


print(
    "Trainable parameters:",
    f"{trainable_params:,}",
)

print(
    "Exact D4 LoRA capacity: PASS"
)


# ============================================================
# 9. FIXED TRAINING GEOMETRY
#
# ceil(1085 / 16) = 68 steps/epoch
# 68 * 2 = 136
# 5% warmup = round(6.8) = 7
# ============================================================

steps_per_epoch = math.ceil(
    len(train_samples)
    / GRAD_ACCUM
)

total_steps = (
    steps_per_epoch
    * EPOCHS
)

warmup_steps = max(
    1,

    round(
        total_steps
        * WARMUP_RATIO
    ),
)


assert steps_per_epoch == 68
assert total_steps == 136
assert warmup_steps == 7


optimizer = torch.optim.AdamW(
    [
        parameter

        for parameter
        in model.parameters()

        if parameter.requires_grad
    ],

    lr=LR,

    weight_decay=(
        WEIGHT_DECAY
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,

        num_warmup_steps=(
            warmup_steps
        ),

        num_training_steps=(
            total_steps
        ),
    )
)


print("\n" + "=" * 78)
print("FINAL TRAINING PLAN")
print("=" * 78)


print(
    "Rows:",
    1085,
)

print(
    "Real:",
    935,
)

print(
    "Synthetic:",
    150,
)

print(
    "Physical batch:",
    1,
)

print(
    "Grad accumulation:",
    16,
)

print(
    "Effective batch:",
    16,
)

print(
    "Steps/epoch:",
    steps_per_epoch,
)

print(
    "Total steps:",
    total_steps,
)

print(
    "Warmup:",
    warmup_steps,
)

print(
    "Epochs:",
    2,
)

print(
    "LR:",
    LR,
)

print(
    "NO validation-based checkpoint selection."
)


# ============================================================
# 10. BATCH HELPER
# ============================================================

def make_batch(sample):

    return {
        "input_ids": (
            sample[
                "input_ids"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),

        "attention_mask": (
            sample[
                "attention_mask"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),

        "labels": (
            sample[
                "labels"
            ]
            .unsqueeze(0)
            .to("cuda:0")
        ),
    }


# ============================================================
# 11. FIXED 2-EPOCH TRAINING
# ============================================================

history = []

global_step = 0


torch.cuda.reset_peak_memory_stats(
    0
)


training_start = time.time()


print("\n" + "=" * 78)
print("FINAL TRAINING START")
print("=" * 78)


for epoch in range(
    1,
    EPOCHS + 1,
):

    epoch_start = time.time()

    model.train()


    epoch_rng = random.Random(
        SEED + epoch
    )


    order = list(
        range(
            len(train_samples)
        )
    )


    epoch_rng.shuffle(
        order
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    epoch_loss = 0.0

    real_loss_sum = 0.0
    real_count = 0

    synth_loss_sum = 0.0
    synth_count = 0

    recent_loss = 0.0
    recent_count = 0

    optimizer_steps = 0


    for position, sample_index in enumerate(
        order,
        1,
    ):

        sample = (
            train_samples[
                sample_index
            ]
        )


        batch = make_batch(
            sample
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):

            outputs = model(
                **batch
            )

            raw_loss = outputs.loss


        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                f"Non-finite loss "
                f"epoch={epoch} "
                f"id={sample['id']}"
            )


        (
            raw_loss
            / GRAD_ACCUM
        ).backward()


        loss_value = float(
            raw_loss
            .detach()
            .cpu()
        )


        epoch_loss += (
            loss_value
        )


        if sample["source"] == "real":

            real_loss_sum += (
                loss_value
            )

            real_count += 1

        else:

            synth_loss_sum += (
                loss_value
            )

            synth_count += 1


        recent_loss += (
            loss_value
        )

        recent_count += 1


        should_step = (
            position
            % GRAD_ACCUM
            == 0

            or position
            == len(order)
        )


        if should_step:

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )


            optimizer_steps += 1
            global_step += 1


            if (
                optimizer_steps % 5 == 0
                or optimizer_steps
                == steps_per_epoch
            ):

                print(
                    f"Epoch {epoch} | "
                    f"step "
                    f"{optimizer_steps:02d}/"
                    f"{steps_per_epoch} | "
                    f"global "
                    f"{global_step:03d}/"
                    f"{total_steps} | "
                    f"loss="
                    f"{recent_loss/recent_count:.4f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.8f} | "
                    f"gpu="
                    f"{torch.cuda.memory_allocated(0)/1024**3:.2f}GB | "
                    f"peak="
                    f"{torch.cuda.max_memory_allocated(0)/1024**3:.2f}GB"
                )


                recent_loss = 0.0
                recent_count = 0


        del outputs
        del raw_loss
        del batch


    assert (
        optimizer_steps
        == steps_per_epoch
    )


    avg_loss = (
        epoch_loss
        / len(train_samples)
    )

    avg_real_loss = (
        real_loss_sum
        / real_count
    )

    avg_synth_loss = (
        synth_loss_sum
        / synth_count
    )


    # --------------------------------------------------------
    # SAVE CHECKPOINT
    # --------------------------------------------------------

    adapter_dir = (
        RUN_DIR
        / f"epoch_{epoch}_adapter"
    )


    model.save_pretrained(
        adapter_dir,
        safe_serialization=True,
    )


    processor.save_pretrained(
        adapter_dir
    )


    adapter_file = (
        adapter_dir
        / "adapter_model.safetensors"
    )


    adapter_sha = sha256_file(
        adapter_file
    )


    history.append({
        "epoch": epoch,

        "train_loss": (
            avg_loss
        ),

        "real_train_loss": (
            avg_real_loss
        ),

        "synthetic_train_loss": (
            avg_synth_loss
        ),

        "adapter_sha256": (
            adapter_sha
        ),

        "adapter_dir": str(
            adapter_dir
        ),

        "epoch_seconds": (
            time.time()
            - epoch_start
        ),
    })


    print("\n" + "-" * 78)

    print(
        f"EPOCH {epoch} COMPLETE"
    )

    print("-" * 78)

    print(
        "Overall loss:",
        f"{avg_loss:.4f}",
    )

    print(
        "Real loss:",
        f"{avg_real_loss:.4f}",
    )

    print(
        "Synthetic loss:",
        f"{avg_synth_loss:.4f}",
    )

    print(
        "Adapter SHA:",
        adapter_sha,
    )


# ============================================================
# 12. FINAL EPOCH-2 MANIFEST
# ============================================================

FINAL_ADAPTER_DIR = (
    RUN_DIR
    / "epoch_2_adapter"
)

FINAL_ADAPTER_FILE = (
    FINAL_ADAPTER_DIR
    / "adapter_model.safetensors"
)

FINAL_ADAPTER_SHA = (
    sha256_file(
        FINAL_ADAPTER_FILE
    )
)


SUMMARY_PATH = (
    RUN_DIR
    / "FINAL_ALL935_TRAINING_SUMMARY.json"
)


summary = {
    "experiment": (
        "FINAL_Gemma4_12B_D4_recipe_"
        "real935_plus_audited150"
    ),

    "status": (
        "FINAL_PRODUCTION_CANDIDATE"
    ),

    "base_model": (
        MODEL_ID
    ),

    "training": {
        "real_semantic_clean_rows": 935,

        "real_sha256": (
            EXPECTED_REAL935_SHA
        ),

        "audited_synthetic_rows": 150,

        "synthetic_sha256": (
            EXPECTED_SYNTH150_SHA
        ),

        "total_rows": 1085,

        "labels": dict(
            label_counts
        ),
    },

    "frozen_holdout": {
        "rows": 75,

        "sha256": (
            EXPECTED_HOLDOUT75_SHA
        ),

        "used_for_training": False,

        "used_for_checkpoint_selection": False,
    },

    "recipe": {
        "lora_r": 8,

        "lora_alpha": 16,

        "lora_dropout": 0.05,

        "trainable_parameters": (
            trainable_params
        ),

        "lr": (
            LR
        ),

        "weight_decay": (
            WEIGHT_DECAY
        ),

        "physical_batch": 1,

        "gradient_accumulation": 16,

        "effective_batch": 16,

        "epochs": 2,

        "steps_per_epoch": (
            steps_per_epoch
        ),

        "total_steps": (
            total_steps
        ),

        "warmup_steps": (
            warmup_steps
        ),

        "scheduler": (
            "cosine"
        ),

        "seed": (
            SEED
        ),

        "max_length": (
            MAX_LENGTH
        ),

        "thinking": False,

        "completion_only": True,
    },

    "history": history,

    "final_epoch": 2,

    "final_adapter_sha256": (
        FINAL_ADAPTER_SHA
    ),

    "important_note": (
        "No dev/validation/holdout metric was "
        "used for checkpoint selection. Epoch 2 "
        "was fixed before training."
    ),
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. PACKAGE FINAL PRODUCTION CANDIDATE
# ============================================================

PACKAGE_DIR = (
    WORK
    / "FINAL_GEMMA4_12B_ALL935_PLUS150"
)


if PACKAGE_DIR.exists():

    shutil.rmtree(
        PACKAGE_DIR
    )


PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copytree(
    FINAL_ADAPTER_DIR,
    PACKAGE_DIR
    / "epoch_2_adapter",
)


shutil.copy2(
    SUMMARY_PATH,
    PACKAGE_DIR
    / SUMMARY_PATH.name,
)


manifest_copy = (
    D4_DATA
    / "D4_AUDITED_FROZEN_MANIFEST_V1.json"
)


if manifest_copy.exists():

    shutil.copy2(
        manifest_copy,
        PACKAGE_DIR
        / manifest_copy.name,
    )


readme = (
    PACKAGE_DIR
    / "README.txt"
)


readme.write_text(
    f"""FINAL GEMMA-4 12B PRODUCTION CANDIDATE
=========================================

Training:
935 semantically-clean real examples
+ 150 audited contrastive examples
= 1085 examples

Recipe:
Frozen from proven D4 champion.

Fresh base model.
Fresh LoRA initialization.
Two fixed epochs.
No validation-based checkpoint selection.

Final epoch-2 adapter SHA256:
{FINAL_ADAPTER_SHA}

IMPORTANT:
The separately saved D4_GEMMA4_12B_CHAMPION.zip remains
the empirically validated champion.

This all-935 model is the final-production candidate trained
after model/data strategy was frozen.
""",
    encoding="utf-8",
)


ZIP_BASE = (
    WORK
    / "FINAL_GEMMA4_12B_ALL935_PLUS150"
)


existing_zip = Path(
    str(ZIP_BASE)
    + ".zip"
)


if existing_zip.exists():

    existing_zip.unlink()


zip_path = Path(
    shutil.make_archive(
        base_name=str(
            ZIP_BASE
        ),

        format="zip",

        root_dir=str(
            PACKAGE_DIR.parent
        ),

        base_dir=(
            PACKAGE_DIR.name
        ),
    )
)


zip_sha = sha256_file(
    zip_path
)

zip_mb = (
    zip_path.stat().st_size
    / 1024**2
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print("\n" + "=" * 78)
print("CELL 7 COMPLETE — FINAL PRODUCTION CANDIDATE")
print("=" * 78)


print(
    "Final training rows:",
    1085,
)

print(
    "Final epoch:",
    2,
)

print(
    "Final adapter:",
    FINAL_ADAPTER_DIR,
)

print(
    "Final adapter SHA256:",
    FINAL_ADAPTER_SHA,
)


print(
    "\nSummary:",
    SUMMARY_PATH,
)

print(
    "Package ZIP:",
    zip_path,
)

print(
    "ZIP size:",
    f"{zip_mb:.2f} MB",
)

print(
    "ZIP SHA256:",
    zip_sha,
)


print(
    "\nNO validation set was used."
)

print(
    "NO frozen holdout was evaluated."
)

print(
    "NO organizer validation was evaluated."
)

print(
    "NO external30 was evaluated."
)

print(
    "NO old synthetic180 was evaluated."
)


print(
    "\nIMPORTANT:"
)

print(
    "Keep BOTH packages:"
)

print(
    "1. D4_GEMMA4_12B_CHAMPION.zip"
)

print(
    "2. FINAL_GEMMA4_12B_ALL935_PLUS150.zip"
)


print(
    "\nSend me the complete Cell-7 output."
)

CELL 7 — FINAL ALL-935 PRODUCTION TRAINING
Real semantic-clean train: 935
Audited synthetic train: 150
TOTAL FINAL TRAIN: 1085
Labels: Counter({'SUPPORTS': 398, 'REFUTES': 361, 'NOT_ENOUGH_INFO': 326})

Frozen holdout rows: 75
Frozen holdout used in training: NO

GPU CLEANUP
GPU0: Tesla T4
GPU0 allocated: 0.383 GB

PROCESSOR
Processor: Gemma4UnifiedProcessor
Tokenizer: GemmaTokenizer

TOKENIZE FINAL TRAIN1085
  real tokenized 250/935
  real tokenized 500/935
  real tokenized 750/935

Real min/median/max: 173 191 233
Synthetic min/median/max: 192 207 232
Combined max: 233
1085/1085 fit within max_length=256.

LOAD FRESH GEMMA-4 12B


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Base GPU allocated: 7.536 GB
Trainable parameters: 32,784,384
Exact D4 LoRA capacity: PASS

FINAL TRAINING PLAN
Rows: 1085
Real: 935
Synthetic: 150
Physical batch: 1
Grad accumulation: 16
Effective batch: 16
Steps/epoch: 68
Total steps: 136
Warmup: 7
Epochs: 2
LR: 0.0002
NO validation-based checkpoint selection.

FINAL TRAINING START
Epoch 1 | step 05/68 | global 005/136 | loss=2.7273 | lr=0.00014286 | gpu=7.63GB | peak=8.58GB
Epoch 1 | step 10/68 | global 010/136 | loss=0.3210 | lr=0.00019973 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 15/68 | global 015/136 | loss=0.1149 | lr=0.00019811 | gpu=7.64GB | peak=8.64GB
Epoch 1 | step 20/68 | global 020/136 | loss=0.1078 | lr=0.00019503 | gpu=7.65GB | peak=8.64GB
Epoch 1 | step 25/68 | global 025/136 | loss=0.0799 | lr=0.00019054 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 30/68 | global 030/136 | loss=0.0683 | lr=0.00018472 | gpu=7.63GB | peak=8.64GB
Epoch 1 | step 35/68 | global 035/136 | loss=0.0524 | lr=0.00017764 | gpu=7.63GB | peak=8.65GB

In [6]:
# ============================================================
# RECOVER OLD SYNTHETIC180 FROM D1 PREDICTION ARTIFACT
#
# No model loading.
# No inference.
# ============================================================

from pathlib import Path
import pandas as pd
import json
import ast
import zipfile


INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

OUT = (
    WORK
    / "synthetic_a2_v1_train_RECOVERED.jsonl"
)


# ============================================================
# 1. SEARCH NORMAL FILES FIRST
# ============================================================

matches = list(
    INPUT.rglob(
        "d1_12b_synthetic_180_predictions.csv"
    )
)


csv_path = None


if matches:

    csv_path = matches[0]

    print(
        "Found directly:",
        csv_path,
    )


# ============================================================
# 2. OTHERWISE SEARCH INSIDE UPLOADED ZIP FILES
# ============================================================

if csv_path is None:

    for zip_path in INPUT.rglob("*.zip"):

        try:

            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zf:

                candidates = [
                    name
                    for name in zf.namelist()
                    if name.endswith(
                        "d1_12b_synthetic_180_predictions.csv"
                    )
                ]


                if candidates:

                    member = candidates[0]

                    extracted = (
                        WORK
                        / "recovered_d1_synthetic_artifact"
                    )

                    extracted.mkdir(
                        parents=True,
                        exist_ok=True,
                    )


                    zf.extract(
                        member,
                        extracted,
                    )


                    csv_path = (
                        extracted
                        / member
                    )


                    print(
                        "Recovered from ZIP:",
                        zip_path,
                    )

                    print(
                        "Member:",
                        member,
                    )

                    break

        except Exception:

            pass


assert csv_path is not None, (
    "Could not find "
    "d1_12b_synthetic_180_predictions.csv "
    "inside /kaggle/input."
)


# ============================================================
# 3. LOAD
# ============================================================

df = pd.read_csv(
    csv_path
)


print(
    "\nColumns:",
    df.columns.tolist(),
)

print(
    "Rows:",
    len(df),
)


assert len(df) == 180


required = {
    "id",
    "claim",
    "evidence",
    "gold",
}


assert required.issubset(
    df.columns
), (
    f"Missing columns: "
    f"{required - set(df.columns)}"
)


# ============================================================
# 4. EVIDENCE PARSER
# ============================================================

def parse_evidence(value):

    if isinstance(
        value,
        list,
    ):

        return [
            str(x)
            for x in value
        ]


    text = str(
        value
    ).strip()


    # JSON-style list
    try:

        obj = json.loads(
            text
        )

        if isinstance(
            obj,
            list,
        ):

            return [
                str(x)
                for x in obj
            ]

    except Exception:

        pass


    # Python-style list
    try:

        obj = ast.literal_eval(
            text
        )

        if isinstance(
            obj,
            list,
        ):

            return [
                str(x)
                for x in obj
            ]

    except Exception:

        pass


    return [
        text
    ]


# ============================================================
# 5. RECONSTRUCT JSONL
# ============================================================

rows = []


for record in df.to_dict(
    orient="records"
):

    label = str(
        record["gold"]
    ).strip().upper()


    assert label in {
        "SUPPORTS",
        "REFUTES",
        "NOT_ENOUGH_INFO",
    }


    rows.append({
        "id": str(
            record["id"]
        ),

        "claim": str(
            record["claim"]
        ),

        "evidence": (
            parse_evidence(
                record["evidence"]
            )
        ),

        "label": label,
    })


with open(
    OUT,
    "w",
    encoding="utf-8",
) as f:

    for row in rows:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )


# ============================================================
# 6. CHECK
# ============================================================

from collections import Counter


labels = Counter(
    row["label"]
    for row in rows
)


print("\n" + "=" * 78)
print("OLD SYNTHETIC180 RECOVERED")
print("=" * 78)

print(
    "Rows:",
    len(rows),
)

print(
    "Labels:",
    labels,
)

print(
    "Output:",
    OUT,
)

print(
    "\nNOTE:"
)

print(
    "This preserves the original 180 examples "
    "semantically."
)

print(
    "Its byte-level SHA may differ from the old "
    "JSONL because JSON serialization can differ."
)

Found directly: /kaggle/input/datasets/omerfarooq223/d1-dataset/D1_GEMMA4_12B_CHAMPION/evaluation_artifacts/d1_12b_synthetic_180_predictions.csv

Columns: ['id', 'gold', 'prediction', 'correct', 'claim', 'evidence', 'raw_output']
Rows: 180

OLD SYNTHETIC180 RECOVERED
Rows: 180
Labels: Counter({'SUPPORTS': 60, 'REFUTES': 60, 'NOT_ENOUGH_INFO': 60})
Output: /kaggle/working/synthetic_a2_v1_train_RECOVERED.jsonl

NOTE:
This preserves the original 180 examples semantically.
Its byte-level SHA may differ from the old JSONL because JSON serialization can differ.
